# SETTING

In [1]:
!pip install pyarrow lightgbm
!pip install xgboost
!pip install catboost
!pip install tsfresh --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


# v52 (param search X)

In [ ]:
## Docstring & 0. 경로 설정

"""
CH2026 수면 예측 경진대회 - 전체 파이프라인 v5 (Z-score 개인화 + 통합 RF)
=============================================
실행 환경 : Google Colab
사전 설치  : !pip install pyarrow lightgbm

Google Drive 디렉토리 구조
    MyDrive/LifeLog/
    ├── ch2026_metrics_train.csv
    ├── ch2026_submission_sample.csv
    └── ch2025_data_items/
        ├── ch2025_wHr.parquet          ⌚ 심박수
        ├── ch2025_wPedo.parquet        ⌚ 만보기
        ├── ch2025_wLight.parquet       ⌚ 조도(워치)
        ├── ch2025_mActivity.parquet    📱 활동유형
        ├── ch2025_mACStatus.parquet    📱 충전상태
        ├── ch2025_mScreenStatus.parquet 📱 화면on/off
        ├── ch2025_mLight.parquet       📱 조도(폰)
        ├── ch2025_mAmbience.parquet    📱 소음분류
        ├── ch2025_mWifi.parquet        📱 WiFi스캔
        ├── ch2025_mBle.parquet         📱 블루투스
        ├── ch2025_mUsageStats.parquet  📱 앱사용통계
        └── ch2025_mGps.parquet         📱 GPS

출력 파일 (BASE_DIR 저장)
    features_train.csv  ← 학습 피처 (확인/디버깅용)
    features_test.csv   ← 테스트 피처 (확인/디버깅용)
    submission_v53_mutual.csv  ← 최종 제출 (확률값 0~1, V5-3)

평가 지표 : Average Log-Loss  (낮을수록 좋음)
→ 제출값은 반드시 확률(0~1). 하드 0/1 제출 금지.

레이블 정의 (ch2026_metrics_description.pdf 참조)
    Q1  수면 직후 전반적 수면 질 (개인 평균 초과=1, 미만=0)
    Q2  취침 직전 신체 피로도    (피로 낮음=1, 높음=0)
    Q3  취침 직전 스트레스 수준  (스트레스 낮음=1, 높음=0)
    S1  총 수면시간(TST) 권장 준수 여부
    S2  수면효율(SE) 권장 준수 여부
    S3  수면 지연시간(SOL) 권장 준수 여부
    S4  수면 중 각성시간(WASO) 권장 준수 여부

수정 이력
    [경로]  BASE_DIR → Colab Google Drive 고정 경로
    [타입]  load_parquet: tz_localize(None) + dt.normalize()
            파케이 timestamp의 timezone을 제거한 뒤 날짜만 남겨
            lifelog_date(naive datetime64[ns])와 merge 타입 완전 일치
    [결측]  fillna(-999) 제거 → LightGBM NaN native 처리
    [피처]  amb_* 제거 (100% 결측)
    [학습]  metric: auc → binary_logloss  (평가 기준과 동일)
    [제출]  하드 0/1 → 확률값 그대로 + np.clip 경계 보정
    [누수]  subj_mean / lag 피처를 CV fold 내부에서만 계산
    [dead]  이전 index 기반 필터링 dead code 제거, merge로 통일
    [피처]  FEAT_EXCLUDE_HIGH_MISS / FEAT_EXCLUDE_HARMFUL 상수로
            성능 악화 피처 명시적 제거 (ablation 실험 기반)
            + FEAT_MISS_THRESHOLD(0.50) 동적 결측 필터 추가
    [모델]  v3: 사용자별(10명) × 타겟별(7개) = 70개 RF 모델
            RandomForest(max_depth=4, min_samples_leaf=2, class_weight=balanced)
            depth=4/leaf=2: ablation 실험으로 소규모 과적합 완화 확인
            fold 수를 최소클래스 크기에 맞게 2~5 사이로 자동 결정
    [피처]  롤링 피처: 타겟별 3일/7일 이동 평균 (shift=1로 누수 방지)
    [피처]  수면 시간 추정: est_sleep_duration, est_screen_to_wake
            est_screen_to_wake (센서 파생, 레이블 비의존 → 누수 없음)
"""

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import DBSCAN
try:
    from tsfresh.feature_extraction import feature_calculators as _fc
    HAS_TSFRESH = True
except ImportError:
    HAS_TSFRESH = False
    print("⚠️  tsfresh 미설치: pip install tsfresh --break-system-packages")
from math import radians as _radians

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# ─────────────────────────────────────────────────────────────────────
# 0. 경로 설정 (Colab 고정)
# ─────────────────────────────────────────────────────────────────────
BASE_DIR   = '/content/drive/MyDrive/LifeLog'
DATA_DIR   = os.path.join(BASE_DIR, "ch2025_data_items")
TRAIN_CSV  = os.path.join(BASE_DIR, "ch2026_metrics_train.csv")
SAMPLE_CSV = os.path.join(BASE_DIR, "ch2026_submission_sample.csv")

TARGETS = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

# ── GPS 전처리 상수 ──────────────────────────────────────────────────
GPS_SPEED_LIMIT_MS = 150 / 3.6   # 이상값 기준: 150 km/h → 41.67 m/s
GPS_STOP_THRESH_MS = 1.0          # 정지 판정: speed < 1 m/s
GPS_EVENING_START  = 18           # 귀가 탐지 시작 시각 (18시)
GPS_EVENING_END    = 23           # 귀가 탐지 종료 시각 (23시)

# 🚨 추가된 DBSCAN 상수
# 하버사인(haversine) 공식을 위한 지구 반지름 대비 거리 설정 (라디안 단위)
# 100m = 100 / 6371000 = 0.000015696...
DBSCAN_EPS_RAD     = 100 / 6371000  # 반경 100m 내의 점들을 묶음
DBSCAN_MIN_SAMPLES = 5              # 클러스터를 형성하기 위한 최소 점 개수

# ── 피처 필터 상수 ───────────────────────────────────────────────────
# Ablation 실험 결과 S1/S2 성능을 악화시키는 피처 명시적 제거
#
# FEAT_EXCLUDE_HIGH_MISS: 결측률 50% 초과 피처
#   - hr_resting     (88.9% 결측): 새벽 안정 심박, 수집 빈도 매우 낮음
#   - hr_diff_day_night (51.8% 결측): 낮/야간 심박차, day_mean 결측 전파
#
# FEAT_EXCLUDE_HARMFUL: 결측은 낮지만 S1/S2를 악화시키는 피처
#   - act_active_ratio     : walk+run 비율, 기존 act_walk/run_ratio와 중복
#   - act_still_night_ratio: 야간 정지, 기존 act_still_ratio와 중복
#   - act_peak_hour        : 피크 활동 시간, S1 -0.037 악화 (ablation 확인)
#   - usage_evening_min    : 저녁 앱 사용 시간, S2 -0.012 악화
#   - usage_evening_ratio  : 저녁 앱 사용 비율, 위와 동일 원인
FEAT_EXCLUDE_HIGH_MISS = [
    "hr_resting",          # 88.9% 결측 → 노이즈 피처
    "hr_diff_day_night",   # 51.8% 결측 → 불안정
]

FEAT_EXCLUDE_HARMFUL = [
    "act_active_ratio",        # act_walk/run_ratio 중복 → S1 악화
    "act_still_night_ratio",   # act_still_ratio 중복 → S1/S2 악화
    "act_peak_hour",           # 피크 시간대 → S1 -0.037 악화
    "usage_evening_min",       # 저녁 앱 시간 → S2 -0.012 악화
    "usage_evening_ratio",     # 위와 동일 원인
]

# 달력 기반 피처: rolling/Z-score 대상에서 제외
# - binary (is_weekend, gps_is_weekend, gps_return_is_morning):
#     continuous_cols 필터(unique>2 or not binary)에서 자동 제외 ✅
# - non-binary (dayofweek 0~6, gps_dayofweek 0~6):
#     continuous_cols 통과 → 명시적 제외 필요
FEAT_EXCLUDE_CALENDAR = [
    "dayofweek",
    "gps_dayofweek",
]

# 범주형 피처 정의 ─────────────────────────────────────────────────
# Binary int (0/1) → LGB/CatBoost에 category로 전달
BINARY_CAT_FEATURES = [
    "is_weekend", "gps_is_weekend", "gps_return_is_morning",
    "user_archetype", "night_slept", "is_regular_night_sleeper",
]
# 문자열 범주형 컬럼 판별 접미사
# (ambience_*/app_* 피처 중 item_col/cat_col 계열)
STR_CAT_COL_SUFFIXES = [
    "_sound", "_category", "_app_name", "_app_category", "_top_category",
]

# 결측률 임계값: 이 비율 초과 피처는 자동 제거 (동적 필터)
FEAT_MISS_THRESHOLD = 0.50

## 1. 파케이 로드 공통 함수

# ══════════════════════════════════════════════════════════════════════
# 1. 파케이 로드 공통 함수
# ══════════════════════════════════════════════════════════════════════

def load_parquet(name: str) -> pd.DataFrame:
    """
    파케이를 읽고 timestamp → date(naive datetime64[ns]) 컬럼을 생성.

    타입 불일치 수정
    ─────────────────
    파케이 timestamp에 timezone(UTC 등)이 붙어 있으면
    pd.to_datetime(lifelog_date)로 만든 naive datetime과 merge할 때
    TypeError 가 발생한다.

    해결:
      ① dt.tz is not None 이면 tz_localize(None)으로 timezone 제거
      ② dt.normalize() 로 시간 부분을 00:00:00으로 잘라 날짜만 남김
    → date 컬럼이 lifelog_date와 동일한 naive datetime64[ns] 타입이 됨
    """
    path = os.path.join(DATA_DIR, f"ch2025_{name}.parquet")
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # timezone 제거 (있을 경우에만)
    if df["timestamp"].dt.tz is not None:
        df["timestamp"] = df["timestamp"].dt.tz_localize(None)

    # naive datetime64[ns] 날짜 컬럼 생성
    df["date"] = df["timestamp"].dt.normalize()
    return df

def _night_key(df: pd.DataFrame, hour_col: str = "hour") -> pd.DataFrame:
    """00~05시 레코드의 date를 하루 전으로 귀속 → night_key 컬럼 추가.
    예) lifelog_date=Jun 26이면 Jun 26 22~23시 + Jun 27 00~05시 모두
        night_key=Jun 26으로 통일.
    """
    df = df.copy()
    df["night_key"] = df["date"].copy()
    df.loc[df[hour_col] < 6, "night_key"] -= pd.Timedelta(days=1)
    return df

def get_cat_features_for_cols(cols: list) -> list:
    """주어진 피처 목록에서 LGB/CatBoost용 범주형 피처 이름 추출.
    - BINARY_CAT_FEATURES: int(0/1) 범주형
    - STR_CAT_COL_SUFFIXES: 문자열 범주형 (ambience_*/app_* 계열)
    """
    result = []
    for c in cols:
        if c in BINARY_CAT_FEATURES:
            result.append(c)
        elif any(c.endswith(s) for s in STR_CAT_COL_SUFFIXES):
            result.append(c)
    return result

def _encode_cat_for_models(X_tr: pd.DataFrame, X_te: pd.DataFrame,
                            cat_feat_set: set) -> tuple:
    """범주형 피처를 정수 코드로 인코딩 (train+test 일관 인코딩).
    - 문자열: NaN→"Unknown"후 category codes (≥0)
    - 이진 정수: NaN→0 (코드 그대로)
    반환: (X_tr_enc, X_te_enc)  — numpy .values 호출 전 DataFrame
    """
    X_tr = X_tr.copy()
    X_te = X_te.copy()
    cat_cols = [c for c in X_tr.columns if c in cat_feat_set]
    for c in cat_cols:
        if X_tr[c].dtype == object:
            combined = (pd.concat([X_tr[c], X_te[c]], ignore_index=True)
                        .fillna("Unknown").astype(str))
            cat_dtype = pd.CategoricalDtype(categories=combined.unique())
            n_tr = len(X_tr)
            codes = combined.astype(cat_dtype).cat.codes.values
            # index 불일치 방지: Series로 감싸 명시적 위치 기반 할당
            X_tr[c] = pd.Series(codes[:n_tr].astype(int), index=X_tr.index)
            X_te[c] = pd.Series(codes[n_tr:].astype(int), index=X_te.index)
        else:
            X_tr[c] = X_tr[c].fillna(0).astype(int)
            X_te[c] = X_te[c].fillna(0).astype(int)
    return X_tr, X_te

def _aggregate_hourly_top5_to_window(
    hourly_df: pd.DataFrame,
    val_col: str = "intensity",
    item_col: str = "sound",
    cat_col: str  = "category",
    agg_func: str = "mean",
    prefix: str   = "ambience_",
) -> pd.DataFrame:
    """1시간 단위 Top5 Wide → 시간대별(day/evening/night) Top5 재집계.

    night_key 적용: 00~05시 데이터를 전날 lifelog_date로 귀속.
    반환: (subject_id, lifelog_date, {prefix}day_Rank*_*, ...) DataFrame
    """
    df = hourly_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # night_key 적용 (00~05시 → 전날)
    df["lifelog_date"] = df["date"]
    df.loc[df["hour"] < 6, "lifelog_date"] -= pd.Timedelta(days=1)

    # 시간대 라벨
    def _window(h):
        if 6 <= h < 17:  return "day"
        elif 17 <= h < 22: return "evening"
        else:              return "night"
    df["time_window"] = df["hour"].apply(_window)

    # Wide → Long
    frames = []
    for i in range(1, 6):
        vc = f"Rank{i}_{val_col}"; ic = f"Rank{i}_{item_col}"; cc = f"Rank{i}_{cat_col}"
        if ic not in df.columns:
            continue
        sub = df[["subject_id","lifelog_date","time_window",vc,ic,cc]].copy()
        sub.columns = ["subject_id","lifelog_date","time_window","value","item","category"]
        sub = sub.dropna(subset=["item"])
        frames.append(sub)
    if not frames:
        return pd.DataFrame()
    long_df = pd.concat(frames, ignore_index=True)

    # 시간대별 집계 → 재순위
    wagg = (long_df
            .groupby(["subject_id","lifelog_date","time_window","item","category"])["value"]
            .agg(agg_func).reset_index())
    wagg = wagg.sort_values(
        ["subject_id","lifelog_date","time_window","value"],
        ascending=[True,True,True,False])
    wagg["new_rank"] = wagg.groupby(
        ["subject_id","lifelog_date","time_window"]).cumcount() + 1
    top5 = wagg[wagg["new_rank"] <= 5]

    # Long → Wide
    pivot = top5.pivot_table(
        index=["subject_id","lifelog_date"],
        columns=["time_window","new_rank"],
        values=["item","category","value"],
        aggfunc="first",
    )
    new_cols = []
    for vtype, window, rnk in pivot.columns:
        if   vtype == "item":     sfx = item_col
        elif vtype == "category": sfx = cat_col
        else:                     sfx = val_col
        new_cols.append(f"{prefix}{window}_Rank{rnk}_{sfx}")
    pivot.columns = new_cols
    pivot = pivot.reset_index()
    pivot["lifelog_date"] = pd.to_datetime(pivot["lifelog_date"])
    return pivot

def _load_floored(name: str, floor: str, value_cols: list) -> pd.DataFrame:
    """
    parquet 로드 후 timestamp를 floor 단위로 내림하여 반환.
    value_cols: 집계에 사용할 컬럼명 리스트
    """
    df = load_parquet(name)
    df["ts_floor"] = df["timestamp"].dt.floor(floor)
    keep = ["subject_id", "date", "ts_floor"] + value_cols
    return df[[c for c in keep if c in df.columns]].copy()

## 2. 피쳐 추출 함수 (파케이별) - tsfresh

# ══════════════════════════════════════════════════════════════════════
# 2. 피처 추출 함수 (파케이별)
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════
# tsfresh 피처 계산 (Micro + Macro)
# ══════════════════════════════════════════════════════════════════

def _safe_ts(func, x):
    """tsfresh 계산기 안전 호출 (inf 포함 예외 → NaN)."""
    import numpy as np
    try:
        vals = list(x.dropna())
        if len(vals) < 5:
            return np.nan
        r = func(vals)
        # 수정됨: 결과가 None이거나 무한대(inf, -inf)일 경우 NaN 처리
        if r is None or np.isinf(r):
            return np.nan
        return float(r)
    except Exception:
        return np.nan


def compute_micro_tsfresh() -> pd.DataFrame:
    """
    Micro level tsfresh: raw parquet → 일별 시계열 피처.

    적용 설정:
      wHr  (야간 22~06h, night_key 날짜 논리 적용):
        sample_entropy, number_cwt_peaks(n=5), cid_ce, mean_change
      wPedo:
        longest_strike_below_mean, variance_larger_than_standard_deviation
      mWifi (rssi):
        binned_entropy(10), longest_strike_below_mean
      mBle (device count):
        percentage_of_reoccurring_values_to_all_values, number_cwt_peaks(n=5)
    """
    if not HAS_TSFRESH:
        print("  tsfresh 없음 → Micro 피처 건너뜀")
        return pd.DataFrame()

    print("  [Micro tsfresh] wHr(야간)...")

    # ── wHr: 야간(22~06시) 필터 + night_key 날짜 보정 ──────────────────
    hr = load_parquet("wHr")
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr["hour"] = hr["timestamp"].dt.hour
    # 야간 필터 + night_key (00~05시 → 전날)
    hr_night = _night_key(hr[(hr["hour"] >= 22) | (hr["hour"] < 6)])
    # 시간 순 정렬: 22시→23시→00시→…→05시 (timestamp 기준)
    hr_night = hr_night.sort_values(["subject_id", "night_key", "timestamp"])

    hr_ts = []
    for (sid, nk), grp in hr_night.groupby(["subject_id", "night_key"], sort=False):
        # sort=False → 이미 정렬된 순서 유지 (22h→…→05h)
        x = grp["hr"].reset_index(drop=True)
        hr_ts.append({
            "subject_id":               sid,
            "date":                      nk,
            "ts_hr_sample_entropy":      _safe_ts(_fc.sample_entropy, x),
            "ts_hr_cwt_peaks":           _safe_ts(lambda v: _fc.number_cwt_peaks(v, n=5), x),
            "ts_hr_cid_ce":              _safe_ts(lambda v: _fc.cid_ce(v, normalize=True), x),
            "ts_hr_mean_change":         _safe_ts(_fc.mean_change, x),
        })
    df_hr_ts = pd.DataFrame(hr_ts) if hr_ts else pd.DataFrame()

    print("  [Micro tsfresh] wPedo...")
    # ── wPedo: step 컬럼 ──────────────────────────────────────────────
    ped = load_parquet("wPedo")
    ped = ped.sort_values(["subject_id", "date", "timestamp"])
    ped_ts = []
    for (sid, dt), grp in ped.groupby(["subject_id", "date"], sort=False):
        x = grp["step"].reset_index(drop=True)
        ped_ts.append({
            "subject_id":                      sid,
            "date":                             dt,
            "ts_pedo_longest_strike_below":     _safe_ts(_fc.longest_strike_below_mean, x),
            "ts_pedo_var_gt_std":               _safe_ts(_fc.variance_larger_than_standard_deviation, x),
        })
    df_ped_ts = pd.DataFrame(ped_ts) if ped_ts else pd.DataFrame()

    print("  [Micro tsfresh] mWifi...")
    # ── mWifi: rssi 컬럼 ─────────────────────────────────────────────
    wifi = load_parquet("mWifi")
    if "m_wifi" in wifi.columns and wifi["m_wifi"].dtype == object:
        wifi = wifi.explode("m_wifi")
        _fv = wifi["m_wifi"].dropna().iloc[0] if wifi["m_wifi"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            wifi["rssi"] = wifi["m_wifi"].apply(lambda x: x.get("rssi", np.nan) if isinstance(x, dict) else np.nan)
    wifi = wifi.sort_values(["subject_id", "date", "timestamp"])
    wifi_ts = []
    for (sid, dt), grp in wifi.groupby(["subject_id", "date"], sort=False):
        x = grp["rssi"].dropna().reset_index(drop=True)
        wifi_ts.append({
            "subject_id":                   sid,
            "date":                          dt,
            "ts_wifi_binned_entropy":        _safe_ts(lambda v: _fc.binned_entropy(v, max_bins=10), pd.Series(x)),
            "ts_wifi_longest_strike_below":  _safe_ts(_fc.longest_strike_below_mean, pd.Series(x)),
        })
    df_wifi_ts = pd.DataFrame(wifi_ts) if wifi_ts else pd.DataFrame()

    print("  [Micro tsfresh] mBle...")
    # ── mBle: 5분 버킷 nunique(device count) ─────────────────────────
    ble = load_parquet("mBle")
    if "m_ble" in ble.columns and ble["m_ble"].dtype == object:
        ble = ble.explode("m_ble")
        _fv = ble["m_ble"].dropna().iloc[0] if ble["m_ble"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            ble["address"] = ble["m_ble"].apply(lambda x: x.get("address", "") if isinstance(x, dict) else "")
    ble = ble[ble["address"] != ""].copy()
    ble["ts_5m"] = ble["timestamp"].dt.floor("5min")
    ble_bucket = (ble.groupby(["subject_id", "date", "ts_5m"])["address"]
                  .nunique().reset_index(name="device_count"))
    ble_bucket = ble_bucket.sort_values(["subject_id", "date", "ts_5m"])
    ble_ts = []
    for (sid, dt), grp in ble_bucket.groupby(["subject_id", "date"], sort=False):
        x = grp["device_count"].reset_index(drop=True)
        ble_ts.append({
            "subject_id":                      sid,
            "date":                             dt,
            "ts_ble_reoccurring_ratio":         _safe_ts(_fc.percentage_of_reoccurring_values_to_all_values, x),
            "ts_ble_cwt_peaks":                 _safe_ts(lambda v: _fc.number_cwt_peaks(v, n=5), x),
        })
    df_ble_ts = pd.DataFrame(ble_ts) if ble_ts else pd.DataFrame()

    # ── 병합 ─────────────────────────────────────────────────────────
    dfs = [df_hr_ts, df_ped_ts, df_wifi_ts, df_ble_ts]
    combined = None
    for df_ in dfs:
        if df_ is None or len(df_) == 0:
            continue
        df_["date"] = pd.to_datetime(df_["date"])
        combined = df_ if combined is None else combined.merge(df_, on=["subject_id","date"], how="outer")

    ts_cols = [c for c in combined.columns if c.startswith("ts_")] if combined is not None else []
    print(f"  Micro tsfresh 피처: {len(ts_cols)}개 ({', '.join(ts_cols)})")
    return combined if combined is not None else pd.DataFrame()


def compute_macro_tsfresh(feat_df: pd.DataFrame, low_miss_cols: list) -> pd.DataFrame:
    """
    Macro level tsfresh: 전처리된 일별 피처 df → 사용자별 시계열 피처.

    적용 대상: 결측률 ≤ 10% 피처
    적용 피처: linear_trend (slope), autocorrelation (lag=1, lag=7)

    사용자 전체 기간의 일별 시계열 → 1개 사용자 수준 값
    → 모든 날짜에 broadcast하여 모델에 전달
    (train 통계만 사용하므로 누수 없음)
    """
    if not HAS_TSFRESH:
        return pd.DataFrame()

    print(f"  [Macro tsfresh] 대상 피처 {len(low_miss_cols)}개 / autocorr(lag=1,7) + linear_trend slope")

    macro_rows = {}  # {subject_id: {feat_name: value}}

    for sid in feat_df["subject_id"].unique():
        sub = (feat_df[feat_df["subject_id"] == sid]
               .sort_values("sleep_date" if "sleep_date" in feat_df.columns else "date"))
        macro_rows[sid] = {"subject_id": sid}
        for c in low_miss_cols:
            if c not in sub.columns:
                continue
            x = sub[c].dropna().values
            if len(x) < 5:
                continue
            # linear_trend slope
            r_lt = _safe_ts(lambda v: list(_fc.linear_trend(v, param=[{"attr":"slope"}]))[0][1],
                            pd.Series(x))
            macro_rows[sid][f"ts_macro_{c}_linear_slope"] = r_lt
            # autocorrelation lag=1
            macro_rows[sid][f"ts_macro_{c}_autocorr_1"] = _safe_ts(
                lambda v: _fc.autocorrelation(v, lag=1), pd.Series(x))
            # autocorrelation lag=7
            macro_rows[sid][f"ts_macro_{c}_autocorr_7"] = _safe_ts(
                lambda v: _fc.autocorrelation(v, lag=7), pd.Series(x)) if len(x) >= 9 else np.nan

    df_macro = pd.DataFrame(macro_rows.values())
    ts_cols = [c for c in df_macro.columns if c.startswith("ts_macro_")]
    print(f"  Macro tsfresh 피처: {len(ts_cols)}개")
    return df_macro

def compute_light_tsfresh_full() -> tuple:
    """mLight / wLight 에서 tsfresh 전체 피처(EfficientFCParameters) 추출.
    반환: (df_m_feats, df_w_feats) — 각각 (subject_id, date, ts_light_m__*, ts_light_w__*)
    """
    if not HAS_TSFRESH:
        return pd.DataFrame(), pd.DataFrame()
    from tsfresh import extract_features as _ef
    from tsfresh.feature_extraction import EfficientFCParameters
    from tsfresh.utilities.dataframe_functions import impute as _impute

    def _extract(parquet_name, value_col, prefix):
        df = load_parquet(parquet_name)
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
        df = df.dropna(subset=[value_col])
        df["ts_id"] = df["subject_id"] + "__" + df["date"].astype(str)
        ts = df[["ts_id","timestamp",value_col]].rename(columns={value_col:"value"})
        feats = _ef(ts, column_id="ts_id", column_sort="timestamp",
                    column_value="value",
                    default_fc_parameters=EfficientFCParameters(),
                    disable_progressbar=True, n_jobs=1)
        _impute(feats)
        feats.columns = [f"{prefix}__{c}" for c in feats.columns]
        idx_parts = feats.index.str.split("__", n=1)
        feats["subject_id"] = [p[0] for p in idx_parts]
        feats["date"]       = pd.to_datetime([p[1] for p in idx_parts])
        return feats.reset_index(drop=True)

    print("  [Light tsfresh] mLight 전체 피처 추출...")
    df_m = _extract("mLight", "m_light", "ts_light_m")
    print(f"    mLight: {len(df_m)}일 × {len([c for c in df_m.columns if c.startswith('ts_')])}피처")
    print("  [Light tsfresh] wLight 전체 피처 추출...")
    df_w = _extract("wLight", "w_light", "ts_light_w")
    print(f"    wLight: {len(df_w)}일 × {len([c for c in df_w.columns if c.startswith('ts_')])}피처")
    return df_m, df_w

def add_light_tsfresh_to_df(train_df, test_df, df_m, df_w) -> tuple:
    """추출된 mLight/wLight tsfresh 피처를 lifelog_date 기준으로 merge.
    반환: (train_df, test_df, all_light_ts_cols)
    """
    all_light_ts_cols = []
    for df_ts in [df_m, df_w]:
        if df_ts is None or len(df_ts) == 0:
            continue
        ts_cols = [c for c in df_ts.columns if c not in ("subject_id","date")]
        df_ts["date"] = pd.to_datetime(df_ts["date"])
        for label, df_ in [("train", train_df), ("test", test_df)]:
            df_["lifelog_date"] = pd.to_datetime(df_["lifelog_date"])
            merged = df_.merge(
                df_ts[["subject_id","date"] + ts_cols],
                left_on=["subject_id","lifelog_date"],
                right_on=["subject_id","date"], how="left"
            )
            for dc in ["date","date_x","date_y"]:
                if dc in merged.columns and "lifelog_date" in merged.columns:
                    merged = merged.drop(columns=[dc])
            if label == "train":
                train_df = merged
            else:
                test_df = merged
        all_light_ts_cols.extend(ts_cols)
    return train_df, test_df, all_light_ts_cols

def select_light_tsfresh_features(
    X_with_targets: pd.DataFrame,
    feature_cols: list,
    targets: list,
    fdr_level: float = 0.15,
) -> dict:
    """tsfresh.select_features로 타겟별 유의미한 Light tsfresh 피처 선택.
    X_with_targets: feature_cols + target cols 포함 DataFrame (train only).
    반환: {target: [selected_col, ...]}
    """
    if not HAS_TSFRESH:
        return {t: [] for t in targets}
    from tsfresh import select_features as _sf

    valid = [c for c in feature_cols if c in X_with_targets.columns]
    X = X_with_targets[valid].fillna(0)
    result = {}
    for t in targets:
        if t not in X_with_targets.columns:
            result[t] = []
            continue
        y = X_with_targets[t].dropna()
        X_t = X.loc[y.index]
        if len(y) < 10:
            result[t] = []
            continue
        try:
            X_sel = _sf(X_t, y, fdr_level=fdr_level)
            selected = X_sel.columns.tolist()
        except Exception as e:
            print(f"    {t}: select_features 오류 → 빈 리스트 ({e})")
            selected = []
        result[t] = selected
        print(f"    {t}: {len(selected):3d}개 선택 (/{len(valid)}개 후보)")
    return result

def add_tsfresh_features(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    miss_thr:  float = 0.10,
) -> tuple:
    """
    tsfresh Micro + Macro 피처 추가 (raw 피처만 반환).

    Z-score는 이 함수에서 적용하지 않음.
    파이프라인 순서:
        add_tsfresh_features()    → raw tsfresh 피처
        add_sensor_rolling()      → raw + rolling 피처
        compute_user_zscores()    → raw + rolling 모두에 Z-score 적용
    이 순서를 지켜야 이중 정규화(_uz_roll_uz)가 발생하지 않는다.
    """
    if not HAS_TSFRESH:
        return train_df, test_df, []

    all_ts_cols = []

    # ── Micro: raw parquet 기반 ────────────────────────────────────
    print("\n[tsfresh Micro] raw parquet 기반 피처 추출...")
    micro_df = compute_micro_tsfresh()

    if micro_df is not None and len(micro_df) > 0:
        micro_ts_cols = [c for c in micro_df.columns if c.startswith("ts_")]
        micro_df["date"] = pd.to_datetime(micro_df["date"])

        for name, df_ in [("train", train_df), ("test", test_df)]:
            df_["lifelog_date"] = pd.to_datetime(df_["lifelog_date"])
            merged = df_.merge(micro_df[["subject_id","date"] + micro_ts_cols],
                               left_on=["subject_id","lifelog_date"],
                               right_on=["subject_id","date"], how="left")
            for drop_col in ["date", "date_x", "date_y"]:
                if drop_col in merged.columns and "lifelog_date" in merged.columns:
                    merged = merged.drop(columns=[drop_col])
            if name == "train":
                train_df = merged
            else:
                test_df = merged

        all_ts_cols.extend(micro_ts_cols)
        print(f"  Micro raw 피처 {len(micro_ts_cols)}개 추가")

    # ── Macro: 결측 10% 이하 피처에 linear_trend + autocorr ──────────
    print("[tsfresh Macro] 전처리 피처 기반 시계열 피처 추출...")
    TARGETS = ["Q1","Q2","Q3","S1","S2","S3","S4"]
    META    = ["subject_id","sleep_date","lifelog_date"]

    # 달력 기반 피처 제외 (linear_slope가 데이터 수집 패턴을 반영할 수 있음)
    EXCLUDE_CALENDAR = {
        "is_weekend","dayofweek","gps_is_weekend","gps_dayofweek",
        "gps_return_is_morning","week_of_year",
    }
    feat_cols_all = [c for c in train_df.columns
                     if c not in META + TARGETS + all_ts_cols
                     and c not in EXCLUDE_CALENDAR
                     and train_df[c].dtype in [float, int, "float64", "int64"]
                     and train_df[c].isnull().mean() <= miss_thr]
    print(f"  결측 ≤10% 피처 (달력 피처 제외): {len(feat_cols_all)}개")

    macro_df = compute_macro_tsfresh(train_df, feat_cols_all)

    if macro_df is not None and len(macro_df) > 0:
        macro_ts_cols = [c for c in macro_df.columns if c.startswith("ts_macro_")]

        # broadcast to train/test (raw만, Z-score는 compute_user_zscores에서)
        for name, df_ in [("train", train_df), ("test", test_df)]:
            merged = df_.merge(
                macro_df[["subject_id"] + macro_ts_cols],
                on="subject_id", how="left"
            )
            if name == "train":
                train_df = merged
            else:
                test_df = merged

        all_ts_cols.extend(macro_ts_cols)
        print(f"  Macro raw 피처 {len(macro_ts_cols)}개 추가")

    print(f"\n  전체 tsfresh raw 피처: {len(all_ts_cols)}개")
    print("  (Z-score는 compute_user_zscores()에서 일괄 적용)")
    return train_df, test_df, all_ts_cols

def feat_wPedo() -> pd.DataFrame:
    """⌚ 만보계: step/speed 기반 running/walking 재판별 + 신규 피처."""
    df = load_parquet("wPedo")
    df["hour"] = df["timestamp"].dt.hour

    # running/walking 직접 판별 (running_step/walking_step 전부 0이므로)
    df["is_running"] = (df["step"] >= 110).astype(int)
    df["is_walking"] = ((df["step"] >= 60) & (df["step"] < 110)).astype(int)

    agg = (
        df.groupby(["subject_id", "date"])
        .agg(
            pedo_step_sum    =("step",            "sum"),
            pedo_step_max    =("step",            "max"),
            pedo_dist_sum    =("distance",        "sum"),
            pedo_calorie_sum =("burned_calories", "sum"),
            pedo_speed_mean  =("speed",           "mean"),
            pedo_speed_max   =("speed",           "max"),
            pedo_freq_mean   =("step_frequency",  "mean"),
            pedo_records     =("step",            "count"),
            pedo_count       =("step",            "count"),
            pedo_run_min     =("is_running",      "sum"),
            pedo_walk_min    =("is_walking",      "sum"),
        )
        .reset_index()
    )
    total = agg["pedo_step_sum"].replace(0, np.nan)
    agg["pedo_run_ratio"]  = agg["pedo_run_min"]  / (agg["pedo_records"] + 1)
    agg["pedo_walk_ratio"] = agg["pedo_walk_min"] / (agg["pedo_records"] + 1)

    # 오전(06~11시) / 저녁(17~21시) 걸음수
    morning = df[df["hour"].between(6, 11)]
    agg_morn = (morning.groupby(["subject_id","date"])["step"]
                .sum().reset_index().rename(columns={"step":"pedo_morning_steps"}))
    evening = df[df["hour"].between(17, 21)]
    agg_eve  = (evening.groupby(["subject_id","date"])["step"]
                .sum().reset_index().rename(columns={"step":"pedo_evening_steps"}))

    # 활동 시간 수
    df["active"] = (df["step"] > 0).astype(int)
    agg_active = (
        df.groupby(["subject_id","date","hour"])["active"].max().reset_index()
        .groupby(["subject_id","date"])["active"].sum().reset_index()
        .rename(columns={"active":"pedo_active_hours"})
    )
    # 달리기 시간대 수
    agg_run_hr = (
        df.groupby(["subject_id","date","hour"])["is_running"].max().reset_index()
        .groupby(["subject_id","date"])["is_running"].sum().reset_index()
        .rename(columns={"is_running":"pedo_run_hours"})
    )
    # 연속 60분 최대 step 구간 평균
    def peak_60min(grp):
        s = grp.sort_values("timestamp")["step"].rolling(60, min_periods=1).sum()
        return float(s.max()) / 60.0 if len(s) > 0 else 0.0
    agg_peak = (
        df.groupby(["subject_id","date"], group_keys=False)
        .apply(peak_60min).reset_index(name="pedo_peak_60min_step")
    )

    agg = agg.merge(agg_peak,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_morn,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_eve,    on=["subject_id","date"], how="left")
    agg = agg.merge(agg_active, on=["subject_id","date"], how="left")
    agg = agg.merge(agg_run_hr, on=["subject_id","date"], how="left")
    agg["pedo_morning_ratio"] = agg["pedo_morning_steps"] / total
    agg["pedo_evening_ratio"] = agg["pedo_evening_steps"] / total
    return agg

def feat_wHr() -> pd.DataFrame:
    """⌚ 심박수: 전체/야간/아침/저녁 구간별 통계 + 변동성 지표

    추가 피처
    ─────────
    hr_resting      : 새벽(03~05시) 최소 심박 — 안정 시 심박수(수면질 직접 지표)
    hr_cv           : 심박 변동계수(std/mean) — 자율신경계 활성도
    hr_evening_mean : 저녁(18~21시) 평균 심박 — 취침 전 각성 수준
    hr_diff_day_night: 낮 평균 - 야간 평균 — 수면 중 심박 회복력
    """
    df = load_parquet("wHr")

    # heart_rate 가 list 타입인 경우 explode
    if df["heart_rate"].dtype == object:
        try:
            df = df.explode("heart_rate")
        except Exception:
            pass

    df["hr"] = pd.to_numeric(df["heart_rate"], errors="coerce")
    df = df.dropna(subset=["hr"])
    df = df[df["hr"].between(30, 220)]
    df["hour"] = df["timestamp"].dt.hour

    # 전체 하루
    agg = (
        df.groupby(["subject_id", "date"])["hr"]
        .agg(
            hr_mean="mean", hr_std="std",
            hr_min="min",   hr_max="max",
            hr_q25=lambda x: x.quantile(0.25),
            hr_q75=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    # 야간 (22:00~05:59): night_key로 날짜 경계 처리
    # 00~05시는 전날 lifelog_date에 귀속
    night = _night_key(df[(df["hour"] >= 22) | (df["hour"] < 6)])
    agg_night = (
        night.groupby(["subject_id", "night_key"])["hr"]
        .agg(hr_night_mean="mean", hr_night_std="std", hr_night_min="min")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 아침 (06:00~09:59) — 기상 직후 심박
    morning = df[df["hour"].between(6, 9)]
    agg_morning = (
        morning.groupby(["subject_id", "date"])["hr"]
        .agg(hr_morning_mean="mean")
        .reset_index()
    )

    # 새벽 안정 심박 (03:00~04:59): night_key로 날짜 경계 처리
    resting = _night_key(df[df["hour"].between(3, 4)])
    agg_rest = (
        resting.groupby(["subject_id", "night_key"])["hr"]
        .agg(hr_resting=lambda x: x.min())
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 저녁 (18:00~20:59) — 취침 전 각성 수준
    evening = df[df["hour"].between(18, 20)]
    agg_eve = (
        evening.groupby(["subject_id", "date"])["hr"]
        .agg(hr_evening_mean="mean")
        .reset_index()
    )

    # 낮 (10:00~17:59) 평균
    day = df[df["hour"].between(10, 17)]
    agg_day = (
        day.groupby(["subject_id", "date"])["hr"]
        .agg(hr_day_mean="mean")
        .reset_index()
    )

    agg = agg.merge(agg_night,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_morning, on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_rest,    on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_eve,     on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_day,     on=["subject_id", "date"], how="left")

    # 파생 피처
    agg["hr_range"]          = agg["hr_max"] - agg["hr_min"]
    agg["hr_cv"]             = agg["hr_std"] / agg["hr_mean"].replace(0, np.nan)
    agg["hr_diff_day_night"] = agg["hr_day_mean"] - agg["hr_night_mean"]
    return agg

def feat_wLight() -> pd.DataFrame:
    """⌚ 조도(워치): 낮/야간 빛 노출"""
    df = load_parquet("wLight")
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id", "date"])["w_light"]
        .agg(wlight_mean="mean", wlight_max="max", wlight_std="std")
        .reset_index()
    )

    # 야간 조도 (21~07시): night_key로 날짜 경계 처리
    night = _night_key(df[(df["hour"] >= 21) | (df["hour"] < 7)])
    night.loc[night["hour"].between(7, 20), "night_key"] = night.loc[night["hour"].between(7, 20), "date"]
    agg_night = (
        night.groupby(["subject_id", "night_key"])["w_light"]
        .agg(wlight_night_mean="mean", wlight_night_max="max")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 일광 노출 비율 (250 lux 이상)
    df["sunny"] = (df["w_light"] > 250).astype(int)
    agg_sun = (
        df.groupby(["subject_id", "date"])["sunny"]
        .agg(wlight_sun_ratio=lambda x: x.mean())
        .reset_index()
    )

    agg = agg.merge(agg_night, on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_sun,   on=["subject_id", "date"], how="left")
    return agg

def feat_mActivity() -> pd.DataFrame:
    """📱 활동: GPS+step 보정 후 분류 (보정 코드 9=실내운동 포함)."""
    df = load_parquet("mActivity")
    df["hour"] = df["timestamp"].dt.hour

    # 보정: wPedo + mGPS 교차 검증 (parquet 로드 가능 시)
    try:
        ped = load_parquet("wPedo")
        gps = load_parquet("mGps")
        # 1분 압축
        def compress(df_, cols):
            df_ = df_.copy()
            df_["mf"] = df_["timestamp"].dt.floor("1min")
            return (df_.groupby(["subject_id","date","mf"])
                    .agg({c:"median" for c in cols if c in df_.columns})
                    .reset_index().rename(columns={"mf":"timestamp"}))
        ped_c = compress(ped, ["step"]).rename(columns={"step":"step_c"})
        if "m_gps" in gps.columns and gps["m_gps"].dtype == object:
            gps = gps.explode("m_gps")
            _fv = gps["m_gps"].dropna().iloc[0] if gps["m_gps"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                gps["latitude"]  = gps["m_gps"].apply(lambda x: x.get("latitude",np.nan) if isinstance(x,dict) else np.nan)
                gps["longitude"] = gps["m_gps"].apply(lambda x: x.get("longitude",np.nan) if isinstance(x,dict) else np.nan)
                gps["speed_g"]   = gps["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
        gps = gps.dropna(subset=["latitude","longitude"])
        gps["speed_g"] = pd.to_numeric(gps.get("speed_g", np.nan), errors="coerce")
        gps = gps[gps["speed_g"].isna()|(gps["speed_g"]<=GPS_SPEED_LIMIT_MS)]
        gps_c = compress(gps, ["speed_g"])

        act_1m = (df.groupby(["subject_id","date",df["timestamp"].dt.floor("1min")])["m_activity"]
                  .agg(lambda x: int(x.mode().iloc[0]) if len(x)>0 else -1)
                  .reset_index().rename(columns={"timestamp":"timestamp"}))
        merged = (act_1m
                  .merge(ped_c[["subject_id","timestamp","step_c"]], on=["subject_id","timestamp"], how="left")
                  .merge(gps_c[["subject_id","timestamp","speed_g"]], on=["subject_id","timestamp"], how="left"))
        merged["step_c"]  = merged["step_c"].fillna(0).astype(int)
        merged["speed_g"] = merged["speed_g"].fillna(0.0)
        orig = merged["m_activity"].copy()
        corr = orig.copy()
        corr[(orig==3)&(merged["step_c"]>=60)&(merged["step_c"]<110)
             &(merged["speed_g"]>=0.3)&(merged["speed_g"]<1.5)] = 7
        corr[(orig==3)&(merged["step_c"]>=110)
             &(merged["speed_g"]>=0.3)&(merged["speed_g"]<1.5)] = 8
        corr[(orig==3)&(merged["step_c"]>=60)&(merged["speed_g"]<0.3)] = 9
        merged["m_activity"] = corr
        df = merged[["subject_id","date","timestamp","m_activity"]].copy()
        df["hour"] = df["timestamp"].dt.hour
    except Exception:
        pass  # 로드 실패 시 원본 사용

    agg = (
        df.groupby(["subject_id","date"])["m_activity"]
        .agg(
            act_still_ratio   =lambda x: (x==3).mean(),
            act_walk_ratio    =lambda x: (x==7).mean(),
            act_run_ratio     =lambda x: (x==8).mean(),
            act_indoor_ratio  =lambda x: (x==9).mean(),
            act_vehicle_ratio =lambda x: (x==0).mean(),
            act_unknown_ratio =lambda x: (x==4).mean(),
            act_unique        =lambda x: x.nunique(),
            active_count      =lambda x: len(x),
        )
        .reset_index()
    )
    agg["act_active_ratio"] = agg["act_walk_ratio"]+agg["act_run_ratio"]+agg["act_indoor_ratio"]

    night = _night_key(df[(df["hour"]>=22)|(df["hour"]<6)])
    agg_ns = (night.groupby(["subject_id","night_key"])["m_activity"]
              .agg(act_still_night_ratio=lambda x: (x==3).mean()).reset_index()
              .rename(columns={"night_key":"date"}))

    df["is_active"] = df["m_activity"].isin([7,8,9]).astype(int)
    hourly = df.groupby(["subject_id","date","hour"])["is_active"].sum().reset_index()
    peak = (hourly.loc[hourly.groupby(["subject_id","date"])["is_active"].idxmax()]
            [["subject_id","date","hour"]].rename(columns={"hour":"act_peak_hour"})
            .reset_index(drop=True))

    agg = agg.merge(agg_ns, on=["subject_id","date"], how="left")
    agg = agg.merge(peak,   on=["subject_id","date"], how="left")
    return agg

def feat_mACStatus() -> pd.DataFrame:
    """📱 충전 상태: 충전 비율 + 취침·기상 시각 추정

    추가 피처
    ─────────
    ac_night_charge_duration: 야간(21~08시) 총 충전 시간(분) — 수면 시간 proxy
    ac_morning_unplug_hour  : 오전 첫 충전 해제 시각 — 기상 시각 추정
    """
    df = load_parquet("mACStatus")
    df = df.sort_values(["subject_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 충전 ON 시간 계산
    df["time_diff_sec"] = (
        df.groupby("subject_id")["timestamp"]
        .diff().dt.total_seconds()
    )
    df["charge_sec"] = df["time_diff_sec"].where(df["m_charging"] == 1, 0)

    agg = (
        df.groupby(["subject_id", "date"])["m_charging"]
        .agg(ac_charge_ratio="mean", ac_records="count")
        .reset_index()
    )

    # 야간(21:00~07:59) 총 충전 시간(분) — 수면 중 충전 = 수면 시간 proxy
    # night_key 적용: 00~07시 레코드를 전날에 귀속
    night = _night_key(df[(df["hour"] >= 21) | (df["hour"] < 8)])
    agg_night_dur = (
        night.groupby(["subject_id", "night_key"])["charge_sec"]
        .sum().reset_index()
        .rename(columns={"night_key": "date"})
        .assign(ac_night_charge_duration=lambda x: x["charge_sec"] / 60)
        [["subject_id", "date", "ac_night_charge_duration"]]
    )

    # 야간(21:00~02:00) 첫 충전 이벤트 → 취침 시각 추정
    # night_key 적용: 00~02시를 전날에 귀속
    night_charge_nk = _night_key(
        df[((df["hour"] >= 21) | (df["hour"] <= 2)) & (df["m_charging"] == 1)]
    )
    first_charge = (
        night_charge_nk.sort_values("timestamp")
        .groupby(["subject_id", "night_key"])["hour"]
        .first()
        .reset_index()
        .rename(columns={"night_key": "date", "hour": "ac_bedtime_hour"})
    )

    # 오전(05:00~10:00) 첫 충전 해제 → 기상 시각 추정
    morning_unplug = df[
        df["hour"].between(5, 10) & (df["m_charging"] == 0)
    ]
    first_unplug = (
        morning_unplug.sort_values("timestamp")
        .groupby(["subject_id", "date"])["hour"]
        .first()
        .reset_index()
        .rename(columns={"hour": "ac_morning_unplug_hour"})
    )

    agg = agg.merge(first_charge,      on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_night_dur,     on=["subject_id", "date"], how="left")
    agg = agg.merge(first_unplug,      on=["subject_id", "date"], how="left")
    return agg

def feat_mScreenStatus() -> pd.DataFrame:
    """📱 화면 상태: 취침 전/야간 스마트폰 사용량 (수면 방해 핵심 지표)

    추가 피처
    ─────────
    screen_total_on_min   : 하루 총 화면 ON 시간(분) — 스마트폰 의존도
    screen_night_sessions : 23~02시 화면 ON 전환 횟수 — 수면 방해 세션
    screen_last_on_hour   : 하루 마지막 화면 ON 시각 — 취침 직전 폰 사용 시점
    """
    df = load_parquet("mScreenStatus")
    df = df.sort_values(["subject_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 화면 ON 시간(분): 연속 ON 상태의 시간 차이 합산
    df["time_diff_sec"] = (
        df.groupby("subject_id")["timestamp"]
        .diff().dt.total_seconds()
    )
    df["on_sec"] = df["time_diff_sec"].where(df["m_screen_use"] == 1, 0)

    agg = (
        df.groupby(["subject_id", "date"])
        .agg(
            screen_on_ratio  =("m_screen_use", "mean"),
            screen_records   =("m_screen_use", "count"),
            screen_total_on_min=("on_sec", lambda x: x.sum() / 60),
        )
        .reset_index()
    )

    # 취침 전 2시간 (22:00~23:59)
    pre_sleep = df[df["hour"].between(22, 23)]
    agg_pre = (
        pre_sleep.groupby(["subject_id", "date"])["m_screen_use"]
        .agg(screen_pre_sleep_ratio="mean", screen_pre_sleep_count="count")
        .reset_index()
    )

    # 야간 (00:00~04:59) — 수면 중 폰 사용 감지
    midnight = df[df["hour"] < 5]
    agg_mid = (
        midnight.groupby(["subject_id", "date"])["m_screen_use"]
        .agg(screen_midnight_on_ratio="mean")
        .reset_index()
    )

    # 마지막 화면 off 시각 (수면 시작 추정)
    last_off = (
        df[df["m_screen_use"] == 0]
        .groupby(["subject_id", "date"])["hour"]
        .last()
        .reset_index()
        .rename(columns={"hour": "screen_last_off_hour"})
    )

    # 마지막 화면 ON 시각 (취침 직전 폰 사용 시점)
    last_on = (
        df[df["m_screen_use"] == 1]
        .groupby(["subject_id", "date"])["hour"]
        .last()
        .reset_index()
        .rename(columns={"hour": "screen_last_on_hour"})
    )

    # 심야(23~02시) 화면 ON 전환 횟수 (sleep session 방해)
    # ON으로 전환되는 순간 = 이전 상태 0 & 현재 상태 1
    df["prev_state"] = df.groupby("subject_id")["m_screen_use"].shift(1)
    df["screen_on_event"] = (
        (df["m_screen_use"] == 1) & (df["prev_state"] == 0)
    ).astype(int)
    night_sess = df[(df["hour"] >= 23) | (df["hour"] < 3)]
    agg_sess = (
        night_sess.groupby(["subject_id", "date"])["screen_on_event"]
        .sum().reset_index()
        .rename(columns={"screen_on_event": "screen_night_sessions"})
    )

    agg = agg.merge(agg_pre,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_mid,   on=["subject_id", "date"], how="left")
    agg = agg.merge(last_off,  on=["subject_id", "date"], how="left")
    agg = agg.merge(last_on,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_sess,  on=["subject_id", "date"], how="left")
    return agg

def feat_mLight() -> pd.DataFrame:
    """📱 조도(폰): 주변 밝기"""
    df = load_parquet("mLight")
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id", "date"])["m_light"]
        .agg(mlight_mean="mean", mlight_max="max", mlight_std="std")
        .reset_index()
    )
    night = _night_key(df[(df["hour"] >= 22) | (df["hour"] < 6)])
    agg_night = (
        night.groupby(["subject_id", "night_key"])["m_light"]
        .agg(mlight_night_mean="mean", mlight_night_max="max")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )
    agg = agg.merge(agg_night, on=["subject_id", "date"], how="left")
    return agg

def feat_mAmbience() -> pd.DataFrame:
    """📱 소음: 시간대별(day/evening/night) Top5 소음 카테고리 피처.

    _aggregate_hourly_top5_to_window 적용 → final_ambience 반환.
    string 컬럼(ambience_*_sound, ambience_*_category)은 범주형으로 처리.
    night_key 날짜 경계 처리 적용.
    """
    import re as _re

    AUDIOSET_MAP = {
        "Speech":"Human","Conversation":"Human","Shout":"Human",
        "Narration, monologue":"Human","Child speech, kid speaking":"Human",
        "Babbling":"Human","Music":"Music",
        "Television":"Device","Radio":"Device","Alarm":"Device",
        "Tick-tock":"Device","Click":"Device","Bell":"Device",
        "Appliances":"Device","Mechanical fan":"Device",
        "Hiss":"Device","Sound effect":"Device",
        "Inside, small room":"Ambient","Inside, large room or hall":"Ambient",
        "Outside, rural or natural":"Ambient","Outside, urban or manmade":"Ambient",
        "Wind":"Nature","Rain":"Nature","Water":"Nature",
        "Vehicle":"Transport","Car":"Transport","Truck":"Transport",
        "Motor vehicle (road)":"Transport","Traffic noise":"Transport","Bus":"Transport",
    }

    def _parse_pairs(row):
        return _re.findall("array\\(\\['([^']*)',[^']*'([^']*)'\\]", str(row))

    df = load_parquet("mAmbience")
    df["hour"] = df["timestamp"].dt.hour

    if "m_ambience" not in df.columns:
        return pd.DataFrame()

    df["pairs"] = df["m_ambience"].apply(_parse_pairs)
    df2 = df.explode("pairs").dropna(subset=["pairs"])
    if len(df2) == 0:
        return pd.DataFrame()

    pairs_df = pd.DataFrame(df2["pairs"].tolist(), index=df2.index)
    df2 = df2.copy()
    df2["sound_type"] = pairs_df[0]
    df2["intensity"]  = pd.to_numeric(pairs_df[1], errors="coerce")
    df2 = df2[(df2["sound_type"] != "Silence") & (df2["intensity"] >= 0.05)]
    df2["category"] = df2["sound_type"].map(lambda x: AUDIOSET_MAP.get(x, "Other"))

    # 시간대별 Top5 집계 (1시간 단위)
    agg_h = (df2.groupby(["subject_id","date","hour","sound_type","category"])["intensity"]
             .mean().reset_index())
    agg_h = agg_h.sort_values(
        ["subject_id","date","hour","intensity"], ascending=[True,True,True,False])
    agg_h["rank"] = agg_h.groupby(["subject_id","date","hour"]).cumcount() + 1
    top5_h = agg_h[agg_h["rank"] <= 5]

    # Wide format (Rank1_sound, Rank1_category, Rank1_intensity, ...)
    pivot_h = top5_h.pivot_table(
        index=["subject_id","date","hour"],
        columns="rank",
        values=["sound_type","category","intensity"],
        aggfunc="first",
    )
    pivot_h.columns = [f"Rank{r}_{v}" for v, r in pivot_h.columns]
    hourly_df = pivot_h.reset_index().rename(
        columns={f"Rank{r}_sound_type": f"Rank{r}_sound" for r in range(1,6)})

    # 시간대별 Top5 재집계
    final_ambience = _aggregate_hourly_top5_to_window(
        hourly_df, val_col="intensity", item_col="sound",
        cat_col="category", agg_func="mean", prefix="ambience_")

    if len(final_ambience) == 0:
        return pd.DataFrame()

    final_ambience = final_ambience.rename(columns={"lifelog_date": "date"})
    final_ambience["date"] = pd.to_datetime(final_ambience["date"])
    return final_ambience

def feat_mWifi() -> pd.DataFrame:
    """📶 WiFi: 위치 패턴 + 집 체류 + 신규(bssid rolling, 수면 rssi)."""
    df = load_parquet("mWifi")
    if "m_wifi" in df.columns and df["m_wifi"].dtype == object:
        try:
            df = df.explode("m_wifi")
            _fv = df["m_wifi"].dropna().iloc[0] if df["m_wifi"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                df["bssid"] = df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
                df["rssi"]  = df["m_wifi"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
            else:
                df["bssid"] = ""; df["rssi"] = pd.to_numeric(df["m_wifi"], errors="coerce")
        except Exception:
            df["bssid"] = ""; df["rssi"] = np.nan
    else:
        df["bssid"] = ""; df["rssi"] = df["m_wifi"] if "rssi" not in df.columns else df["rssi"]
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id","date"])
        .agg(wifi_ap_count=("bssid","nunique"), wifi_rssi_mean=("rssi","mean"),
             wifi_rssi_max=("rssi","max"), wifi_scans=("bssid","count"))
        .reset_index()
    )
    home_bssid = (
        df[df["bssid"]!=""].groupby(["subject_id","bssid"]).size().reset_index(name="cnt")
        .sort_values("cnt", ascending=False).drop_duplicates("subject_id")
        [["subject_id","bssid"]].rename(columns={"bssid":"home_bssid"})
    )
    df = df.merge(home_bssid, on="subject_id", how="left")
    df["is_home"] = (df["bssid"] == df["home_bssid"]).astype(int)
    agg_home = (df.groupby(["subject_id","date"])["is_home"].mean().reset_index()
                .rename(columns={"is_home":"wifi_home_ratio"}))
    agg_home_eve = (df[df["hour"].between(18,23)].groupby(["subject_id","date"])["is_home"]
                    .mean().reset_index().rename(columns={"is_home":"wifi_home_evening_ratio"}))

    # bssid nunique rolling(window=3, 10분×3=30분) 일평균
    df_ts = (df.groupby(["subject_id","date",pd.Grouper(key="timestamp",freq="10min")])["bssid"]
             .nunique().reset_index(name="bssid_nu_10m"))
    df_ts = df_ts.sort_values(["subject_id","timestamp"])
    df_ts["bssid_nu_r3"] = (df_ts.groupby("subject_id")["bssid_nu_10m"]
                            .transform(lambda x: x.rolling(3,min_periods=1).mean()))
    agg_roll = (df_ts.groupby(["subject_id","date"])["bssid_nu_r3"].mean().reset_index()
                .rename(columns={"bssid_nu_r3":"wifi_bssid_nunique_roll3"}))

    # 수면 시간대(22~06시) rssi 평균: night_key로 날짜 경계 처리
    sleep_df = _night_key(df[(df["hour"]>=22)|(df["hour"]<6)])
    agg_sleep_rssi = (sleep_df.groupby(["subject_id","night_key"])["rssi"].mean().reset_index()
                      .rename(columns={"night_key":"date","rssi":"wifi_sleep_rssi_mean"}))

    agg = agg.merge(agg_home,       on=["subject_id","date"], how="left")
    agg = agg.merge(agg_home_eve,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_roll,       on=["subject_id","date"], how="left")
    agg = agg.merge(agg_sleep_rssi, on=["subject_id","date"], how="left")
    return agg

def feat_mGps() -> pd.DataFrame:
    """🗺️ GPS: DBSCAN 기반 실이동 거리 + 집 탐지 + 귀가 시각 + 주말 피처."""
    df = load_parquet("mGps")
    if "m_gps" in df.columns and df["m_gps"].dtype == object:
        try:
            df = df.explode("m_gps")
            _fv = df["m_gps"].dropna().iloc[0] if df["m_gps"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                for col in ["latitude","longitude","altitude","speed"]:
                    df[col] = df["m_gps"].apply(lambda x: x.get(col,np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            for col in ["latitude","longitude","altitude","speed"]:
                df[col] = np.nan
    else:
        for col in ["latitude","longitude","altitude","speed"]:
            if col not in df.columns: df[col] = np.nan

    df = df.dropna(subset=["latitude","longitude"]).sort_values(["subject_id","timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 이상값 제거
    df["lat_diff"]  = df.groupby("subject_id")["latitude"].diff()
    df["lon_diff"]  = df.groupby("subject_id")["longitude"].diff()
    df["time_diff"] = df.groupby("subject_id")["timestamp"].diff().dt.total_seconds()
    df["dist_m"]    = np.sqrt(df["lat_diff"]**2 + df["lon_diff"]**2) * 111_000
    df["inst_spd"]  = df["dist_m"] / df["time_diff"].replace(0, np.nan)
    df = df[df["inst_spd"].isna()|(df["inst_spd"]<=GPS_SPEED_LIMIT_MS)].copy()
    df["is_stop"] = (df["speed"] < GPS_STOP_THRESH_MS).astype(int)

    # 1분 중위값 압축 (11건/분 → 1건/분)
    def compress_to_minute(df_day):
        df_day = df_day.copy()
        df_day["mf"] = df_day["timestamp"].dt.floor("1min")
        c = (df_day.groupby(["subject_id","date","mf"])
             .agg(latitude=("latitude","median"), longitude=("longitude","median"),
                  speed=("speed","median")).reset_index()
             .rename(columns={"mf":"timestamp"}))
        return c.sort_values("timestamp").reset_index(drop=True)

    def haversine_vec(la1, lo1, la2, lo2):
        R = 6_371_000
        la1,lo1,la2,lo2 = map(np.radians,[la1,lo1,la2,lo2])
        dlat=la2-la1; dlon=lo2-lo1
        a = np.sin(dlat/2)**2 + np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2
        return R*2*np.arcsin(np.sqrt(np.clip(a,0,1)))

    records, detail_frames = [], []
    for (sid, date), group in df.groupby(["subject_id","date"]):
        g = compress_to_minute(group)
        if len(g) < DBSCAN_MIN_SAMPLES:
            g["cluster"]   = -1
            g["place_lat"] = g["latitude"]; g["place_lon"] = g["longitude"]
            dist_km = 0.0; n_places = 0; home_cluster = -1; home_min = 0
        else:
            coords_rad = np.radians(g[["latitude","longitude"]].values)
            db = DBSCAN(eps=DBSCAN_EPS_RAD, min_samples=DBSCAN_MIN_SAMPLES,
                        metric="haversine", algorithm="ball_tree").fit(coords_rad)
            g["cluster"] = db.labels_
            n_places = len(set(db.labels_) - {-1})
            labels = db.labels_
            lats = g["latitude"].values; lons = g["longitude"].values
            center = {}
            for c in set(labels) - {-1}:
                mask = labels==c; center[c] = (lats[mask].mean(), lons[mask].mean())
            ev_lat = np.where(labels>=0,
                              [center.get(c,(lat,lon))[0] for c,lat,lon in zip(labels,lats,lons)], lats)
            ev_lon = np.where(labels>=0,
                              [center.get(c,(lat,lon))[1] for c,lat,lon in zip(labels,lats,lons)], lons)
            ev_cls = labels.copy()
            keep = np.ones(len(ev_cls), dtype=bool)
            for i in range(1,len(ev_cls)):
                if ev_cls[i]>=0 and ev_cls[i]==ev_cls[i-1]: keep[i]=False
            ev_lat2=ev_lat[keep]; ev_lon2=ev_lon[keep]
            if len(ev_lat2)>=2:
                dists = haversine_vec(ev_lat2[:-1],ev_lon2[:-1],ev_lat2[1:],ev_lon2[1:])
                dist_km = dists.sum()/1000
            else:
                dist_km = 0.0
            for c,(clat,clon) in center.items():
                g.loc[g["cluster"]==c,"place_lat"] = clat
                g.loc[g["cluster"]==c,"place_lon"] = clon
            g.loc[g["cluster"]==-1,"place_lat"] = g.loc[g["cluster"]==-1,"latitude"]
            g.loc[g["cluster"]==-1,"place_lon"] = g.loc[g["cluster"]==-1,"longitude"]
            # 집 탐지 (0~5시 + 22~23시)
            night_mask = (g["timestamp"].dt.hour<6)|(g["timestamp"].dt.hour>=22)
            night_df   = g[night_mask & (g["cluster"]>=0)]
            if len(night_df)>0:
                home_cluster = int(night_df["cluster"].mode().iloc[0])
            else:
                placed = g[g["cluster"]>=0]
                home_cluster = int(placed["cluster"].mode().iloc[0]) if len(placed)>0 else -1
            home_min = int((g["cluster"]==home_cluster).sum()) if home_cluster>=0 else 0

        detail_frames.append(
            g[["subject_id","date","timestamp","latitude","longitude",
               "speed","cluster"]].assign(subject_id=sid, date=date))

        # 귀가 시각: 집 cluster 저녁(GPS_EVENING_START 이후) 첫 등장
        date_ts   = pd.Timestamp(date)
        dow       = date_ts.dayofweek
        is_weekend = int(dow>=5)
        return_hour = np.nan
        if home_cluster>=0 and len(g)>0:
            home_rows = g[g["cluster"]==home_cluster]
            eve_home  = home_rows[home_rows["timestamp"].dt.hour>=GPS_EVENING_START].sort_values("timestamp")
            if len(eve_home)>0:
                t = eve_home["timestamp"].iloc[0]
                return_hour = t.hour + t.minute/60
            else:
                t = home_rows.sort_values("timestamp")["timestamp"].iloc[0]
                return_hour = t.hour + t.minute/60

        records.append({
            "subject_id": sid, "date": date,
            "gps_dist_km_dbscan": dist_km, "gps_n_places": n_places,
            "gps_home_min": home_min,
            "gps_home_hour": home_min/60,
            "gps_return_hour": return_hour,
            "gps_dayofweek": dow,
            "gps_is_weekend": is_weekend,
            "gps_return_is_morning": int(return_hour<12) if not np.isnan(return_hour) else 0,
        })

    agg_dbscan  = pd.DataFrame(records)
    df_detailed = pd.concat(detail_frames, ignore_index=True)

    agg_base = (
        df.groupby(["subject_id","date"])
        .agg(gps_speed_mean=("speed","mean"), gps_speed_max=("speed","max"),
             gps_records=("latitude","count"), gps_lat_std=("latitude","std"),
             gps_lon_std=("longitude","std"), gps_stop_ratio=("is_stop","mean"))
        .reset_index()
    )
    agg_base["gps_radius"] = agg_base["gps_lat_std"] + agg_base["gps_lon_std"]
    agg_base = agg_base.drop(columns=["gps_lat_std","gps_lon_std"])

    agg = agg_base.merge(agg_dbscan, on=["subject_id","date"], how="left")
    agg["gps_log_dist"] = np.log1p(agg["gps_dist_km_dbscan"])
    agg["date"] = pd.to_datetime(agg["date"])
    return agg

def feat_mUsageStats() -> pd.DataFrame:
    """📱 앱 사용: 시간대별(day/evening/night) Top5 앱 피처.

    _aggregate_hourly_top5_to_window 적용 → final_usage 반환.
    string 컬럼(app_*_app_name, app_*_app_category)은 범주형으로 처리.
    night_key 날짜 경계 처리 적용.
    """
    import re as _re

    KEYWORD_MAP = {
        "SNS":     ["instagram","facebook","twitter","kakao","tiktok","snapchat","band","telegram"],
        "Video":   ["youtube","netflix","twitch","wavve","tving","vod","player","video"],
        "Game":    ["game","play","puzzle","rpg","clash","minecraft"],
        "Browser": ["chrome","samsung internet","safari","firefox","naver","daum","browser"],
    }
    def _classify(app: str) -> str:
        low = app.lower()
        for cat, kws in KEYWORD_MAP.items():
            if any(k in low for k in kws):
                return cat
        return "Tool"

    df = load_parquet("mUsageStats")
    df["hour"] = df["timestamp"].dt.hour

    regex = r"'app_name':\s*'([^']*)',\s*'total_time':\s*(\d+)"
    records = []
    for _, row in df.iterrows():
        for app_name, total_ms in _re.findall(regex, str(row.get("m_usage_stats",""))):
            records.append({
                "subject_id":    row["subject_id"],
                "date":          row["date"],
                "hour":          row["hour"],
                "app_name":      app_name,
                "app_category":  _classify(app_name),
                "total_time_sec": int(total_ms) / 1000,
            })
    if not records:
        return pd.DataFrame()

    usage = pd.DataFrame(records)

    # 시간대별 Top5 집계 (1시간 단위)
    agg_h = (usage.groupby(["subject_id","date","hour","app_name","app_category"])["total_time_sec"]
             .sum().reset_index())
    agg_h = agg_h.sort_values(
        ["subject_id","date","hour","total_time_sec"], ascending=[True,True,True,False])
    agg_h["rank"] = agg_h.groupby(["subject_id","date","hour"]).cumcount() + 1
    top5_h = agg_h[agg_h["rank"] <= 5]

    # Wide format (Rank1_app_name, Rank1_app_category, Rank1_total_time_sec, ...)
    pivot_h = top5_h.pivot_table(
        index=["subject_id","date","hour"],
        columns="rank",
        values=["app_name","app_category","total_time_sec"],
        aggfunc="first",
    )
    pivot_h.columns = [f"Rank{r}_{v}" for v, r in pivot_h.columns]
    hourly_df = pivot_h.reset_index()

    # 시간대별 Top5 재집계
    final_usage = _aggregate_hourly_top5_to_window(
        hourly_df, val_col="total_time_sec", item_col="app_name",
        cat_col="app_category", agg_func="sum", prefix="app_")

    if len(final_usage) == 0:
        return pd.DataFrame()

    final_usage = final_usage.rename(columns={"lifelog_date": "date"})
    final_usage["date"] = pd.to_datetime(final_usage["date"])
    return final_usage

def feat_mBle() -> pd.DataFrame:
    """📡 BLE: 사회적 환경 밀도 (집 판별 제거, 5분 버킷 nunique 기반)."""
    df = load_parquet("mBle")
    if "m_ble" in df.columns and df["m_ble"].dtype == object:
        try:
            df = df.explode("m_ble")
            _fv = df["m_ble"].dropna().iloc[0] if df["m_ble"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                df["address"] = df["m_ble"].apply(lambda x: x.get("address","") if isinstance(x,dict) else "")
                df["rssi"]    = df["m_ble"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            df["address"] = ""; df["rssi"] = np.nan

    df = df[df["address"] != ""].copy()
    df["hour"] = df["timestamp"].dt.hour
    df["ts_5m"] = df["timestamp"].dt.floor("5min")

    bucket = (df.groupby(["subject_id","date","ts_5m"])["address"]
              .nunique().reset_index(name="device_count"))
    bucket["hour"] = bucket["ts_5m"].dt.hour

    agg = (bucket.groupby(["subject_id","date"])["device_count"]
           .mean().reset_index().rename(columns={"device_count":"ble_device_day_mean"}))
    agg_eve = (bucket[bucket["hour"].between(18,21)]
               .groupby(["subject_id","date"])["device_count"]
               .mean().reset_index().rename(columns={"device_count":"ble_device_evening"}))
    night_b = _night_key(bucket[(bucket["hour"]>=22)|(bucket["hour"]<6)], hour_col="hour")
    agg_night = (night_b.groupby(["subject_id","night_key"])["device_count"]
                 .mean().reset_index()
                 .rename(columns={"night_key":"date","device_count":"ble_device_night"}))
    hourly = bucket.groupby(["subject_id","date","hour"])["device_count"].mean().reset_index()
    peak = (hourly.loc[hourly.groupby(["subject_id","date"])["device_count"].idxmax()]
            [["subject_id","date","hour"]].rename(columns={"hour":"ble_social_peak_hour"})
            .reset_index(drop=True))
    agg_scans = (df.groupby(["subject_id","date"])["address"]
                 .count().reset_index().rename(columns={"address":"ble_scans"}))

    for df_ in [agg_eve, agg_night, peak, agg_scans]:
        agg = agg.merge(df_, on=["subject_id","date"], how="left")
    return agg

# ══════════════════════════════════════════════════════════════════════
# 3. 전체 피처 테이블 생성
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# 3-b. 교차 parquet 피처 (Cross-source Features)
# ══════════════════════════════════════════════════════════════════════
#
# 설계 원칙
# ─────────
# 서로 다른 센서를 timestamp 기준으로 병합하여 단일 소스에서 얻을 수
# 없는 복합 정보를 추출한다.
#
# 주기 차이 해결
# ──────────────
# 센서마다 기록 주기가 다르므로 dt.floor()로 같은 시간 버킷으로 정렬:
#   FLOOR_FINE  = '5T'  : 심박/조도/활동/스크린/충전 등 고빈도
#   FLOOR_MED   = '5T'  : GPS/WiFi/BLE 등 중간 빈도
#   FLOOR_COARSE= '1H'  : 앱 사용통계 등 저빈도
# 병합 방식: outer join 후 결측 NaN 유지 → 집계 시 skipna=True
#
# 누수 검토 ✅
# ───────────
# 모든 교차 피처는 센서 데이터(X)만 사용하며 타겟(y)을 사용하지 않음.
# 모든 교차 피처는 lifelog_date(수면 전날) 기준으로 집계되므로
# 타겟(sleep_date 기준)과의 시간 방향이 올바름.

FLOOR_FINE  = "5T"   # 5분 단위 (wHr, wLight, mActivity, mACStatus, mScreenStatus, mLight, mAmbience)
FLOOR_MED   = "5T"   # 5분 단위 (mWifi, mGps, mBle)

def feat_SleepProxy_AC_Scr() -> pd.DataFrame:
    """🔋📱 수면 프록시: 충전O + 화면X 패턴 기반 수면 여부 판별.

    피처: night_sleep_ratio, night_slept, is_regular_night_sleeper
    night_key 날짜 경계 처리(00~05시 → 전날) 적용.
    """
    ac  = load_parquet("mACStatus")
    scr = load_parquet("mScreenStatus")
    ac["m_charging"]    = pd.to_numeric(ac["m_charging"],    errors="coerce").fillna(0)
    scr["m_screen_use"] = pd.to_numeric(scr["m_screen_use"], errors="coerce").fillna(0)

    rows = []
    for sid in ac["subject_id"].unique():
        sub_ac  = ac[ac["subject_id"]==sid].set_index("timestamp")[["m_charging"]]
        sub_scr = scr[scr["subject_id"]==sid].set_index("timestamp")[["m_screen_use"]]
        if sub_ac.empty or sub_scr.empty:
            continue

        merged = pd.merge(
            sub_ac.resample("1min").max().fillna(0),
            sub_scr.resample("1min").max().fillna(0),
            left_index=True, right_index=True, how="outer"
        ).fillna(0)
        merged["raw_proxy"] = ((merged["m_charging"]==1) & (merged["m_screen_use"]==0)).astype(int)
        # 10분 버킷 스무딩
        smoothed = (merged["raw_proxy"].resample("10min").mean() >= 0.3).astype(int)
        df_p = smoothed.to_frame(name="sleep_proxy").copy()
        df_p["hour"] = df_p.index.hour
        # night_key 적용 (00~05시 → 전날)
        df_p["date"] = df_p.index.normalize()
        df_p.loc[df_p["hour"] < 6, "date"] -= pd.Timedelta(days=1)
        night_m = (df_p["hour"] >= 22) | (df_p["hour"] < 6)

        for dt, grp in df_p[night_m].groupby("date"):
            ratio = grp["sleep_proxy"].mean()
            rows.append({
                "subject_id":       sid,
                "date":             dt,
                "night_sleep_ratio": ratio,
                "night_slept":       1 if ratio >= 0.5 else 0,
            })

    if not rows:
        return pd.DataFrame()

    agg = pd.DataFrame(rows)
    # 규칙적 수면자: 해당 사용자의 night_slept 비율 >= 0.6이면서 당일도 수면
    user_consist = agg.groupby("subject_id")["night_slept"].mean().rename("_consist")
    agg = agg.merge(user_consist, on="subject_id", how="left")
    agg["is_regular_night_sleeper"] = ((agg["_consist"] >= 0.6) & (agg["night_slept"] == 1)).astype(int)
    agg["date"] = pd.to_datetime(agg["date"])
    return agg[["subject_id","date","night_sleep_ratio","night_slept","is_regular_night_sleeper"]]

def feat_SleepProxy_TwoTrack() -> pd.DataFrame:
    """💡🔋 투트랙 수면 프록시: 조도(환경) + 기기(행동) 기반 수면 패턴.

    피처: user_archetype, light_sleep_duration, device_sleep_duration,
          sleep_bouts_count, light_night_sleep_ratio, device_night_sleep_ratio
    불규칙 사용자도 피처 값을 계산 (archetype만 0으로 표시).
    night_key 날짜 경계 처리 적용.
    """
    m_df  = load_parquet("mLight")
    w_df  = load_parquet("wLight")
    ac_df = load_parquet("mACStatus")
    scr_df= load_parquet("mScreenStatus")

    for df_, col in [(m_df,"m_light"),(w_df,"w_light"),
                     (ac_df,"m_charging"),(scr_df,"m_screen_use")]:
        df_[col] = pd.to_numeric(df_[col], errors="coerce")

    # 사용자 아키타입: 야간 조도 분산 → 하위 50% = 규칙적(1)
    def _night_var(df_, col):
        df_ = df_.copy()
        df_["hour"] = df_["timestamp"].dt.hour
        night = df_[(df_["hour"]>=22)|(df_["hour"]<6)]
        return night.groupby("subject_id")[col].var()

    m_var = _night_var(m_df, "m_light").rename("m_var")
    w_var = _night_var(w_df, "w_light").rename("w_var")
    profile = pd.concat([m_var, w_var], axis=1).fillna(0)
    profile["score"] = profile["m_var"] + profile["w_var"]
    median_score = profile["score"].median()
    profile["user_archetype"] = (profile["score"] <= median_score).astype(int)

    subjects = m_df["subject_id"].unique()
    final = []

    def _count_bouts(s):
        is_sleep = (s == 1)
        return int(((is_sleep != is_sleep.shift()) & is_sleep).sum())

    for sid in subjects:
        archetype = int(profile.loc[sid, "user_archetype"]) if sid in profile.index else 0

        def _resamp(df_, col, sid=sid):
            sub = df_[df_["subject_id"]==sid].copy()
            sub["timestamp"] = pd.to_datetime(sub["timestamp"])
            return sub.set_index("timestamp")[col].resample("10min").mean()

        df_sync = pd.DataFrame({
            "m_light":  _resamp(m_df,  "m_light"),
            "w_light":  _resamp(w_df,  "w_light"),
            "charging": _resamp(ac_df, "m_charging").ffill(),
            "screen":   _resamp(scr_df,"m_screen_use").ffill(),
        }).fillna({"charging":0,"screen":0})

        df_sync["sleep_by_light"]  = ((df_sync["m_light"]<=10)&(df_sync["w_light"]<=10)).astype(int)
        df_sync["sleep_by_device"] = ((df_sync["charging"]==1)&(df_sync["screen"]==0)).astype(int)
        df_sync["hour"] = df_sync.index.hour
        df_sync["date"] = df_sync.index.normalize()
        # night_key 적용
        df_sync.loc[df_sync["hour"]<6, "date"] -= pd.Timedelta(days=1)
        night_m = (df_sync["hour"]>=22)|(df_sync["hour"]<6)
        df_n = df_sync[night_m]

        daily = (
            df_n.groupby("date")
            .agg(
                light_sleep_duration  =("sleep_by_light",  lambda x: x.sum()*10),
                device_sleep_duration =("sleep_by_device", lambda x: x.sum()*10),
                sleep_bouts_count     =("sleep_by_light",  _count_bouts),
                light_night_sleep_ratio =("sleep_by_light",  "mean"),
                device_night_sleep_ratio=("sleep_by_device", "mean"),
            ).reset_index()
        )
        daily["subject_id"]    = sid
        daily["user_archetype"] = archetype
        final.append(daily)

    if not final:
        return pd.DataFrame()
    agg = pd.concat(final, ignore_index=True)
    agg["date"] = pd.to_datetime(agg["date"])
    return agg[["subject_id","date","user_archetype",
                "light_sleep_duration","device_sleep_duration",
                "sleep_bouts_count","light_night_sleep_ratio","device_night_sleep_ratio"]]


def feat_cross_arousal() -> pd.DataFrame:
    """
    그룹1: 취침 전 각성 자극 (mScreenStatus × mLight × wHr)

    피처 목록
    ─────────
    cross_dark_screen_min       야간(22~06시) 조도<5 lux + 화면ON  시간(분)
    cross_evening_bright_screen 저녁(19~22시) 조도>50 lux + 화면ON 시간(분)
    cross_presleep_screen_hr    취침 전 2h(21~23시) 화면ON 중 평균 심박수
    cross_presleep_hr_delta     (취침 전 화면ON 심박) - (해당일 전체 hr_mean)
    """
    # ── 스크린 (1분 단위) ──────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 폰 조도 (5분 단위, 소규모이므로 같은 floor) ─────────────────
    ml = _load_floored("mLight", FLOOR_FINE, ["m_light"])

    # ── 워치 심박 (5분 단위) ──────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 스크린 × 조도 병합 ────────────────────────────────────────────
    sc_ml = sc.merge(
        ml[["subject_id", "ts_floor", "m_light"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    sc_ml["hour"] = sc_ml["ts_floor"].dt.hour

    # 야간: 22~06시 (필터링 시 아래 _night_key 적용으로 날짜 보정)
    night_mask = (sc_ml["hour"] >= 22) | (sc_ml["hour"] < 6)
    # 각 버킷이 5분 → 1개 = 5분
    BUCKET_MIN = 5

    # cross_dark_screen_min: 야간 조도<5 + 화면ON (night_key 적용)
    sc_ml["is_dark_screen"] = (
        (sc_ml["m_light"] < 5) & (sc_ml["m_screen_use"] == 1)
    ).astype(float)
    sc_ml_dk = _night_key(sc_ml[night_mask])
    agg_dark = (
        sc_ml_dk
        .groupby(["subject_id", "night_key"])["is_dark_screen"]
        .sum().reset_index()
        .assign(cross_dark_screen_min=lambda x: x["is_dark_screen"] * BUCKET_MIN)
        .rename(columns={"night_key": "date"})
        [["subject_id", "date", "cross_dark_screen_min"]]
    )

    # cross_evening_bright_screen: 19~22시 조도>50 + 화면ON
    eve_mask = sc_ml["hour"].between(19, 21)
    sc_ml["is_bright_screen"] = (
        (sc_ml["m_light"] > 50) & (sc_ml["m_screen_use"] == 1)
    ).astype(float)
    agg_bright = (
        sc_ml[eve_mask]
        .groupby(["subject_id", "date"])["is_bright_screen"]
        .sum().reset_index()
        .assign(cross_evening_bright_screen=lambda x: x["is_bright_screen"] * BUCKET_MIN)
        [["subject_id", "date", "cross_evening_bright_screen"]]
    )

    # ── 스크린 × 심박 병합 ────────────────────────────────────────────
    sc_hr = sc.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    sc_hr["hour"] = sc_hr["ts_floor"].dt.hour

    # cross_presleep_screen_hr: 21~23시 화면ON 중 평균 심박
    pre_mask = sc_hr["hour"].between(21, 23) & (sc_hr["m_screen_use"] == 1)
    agg_hr = (
        sc_hr[pre_mask]
        .groupby(["subject_id", "date"])["hr"]
        .mean().reset_index()
        .rename(columns={"hr": "cross_presleep_screen_hr"})
    )

    # cross_presleep_hr_delta: 취침 전 화면ON 심박 - 해당일 전체 평균 심박
    hr_daily = (
        hr.groupby(["subject_id", "date"])["hr"].mean().reset_index()
        .rename(columns={"hr": "hr_daily_mean"})
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_dark.merge(agg_bright, on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_hr,         on=["subject_id", "date"], how="outer")
    agg = agg.merge(hr_daily,       on=["subject_id", "date"], how="left")
    agg["cross_presleep_hr_delta"] = (
        agg["cross_presleep_screen_hr"] - agg["hr_daily_mean"]
    )
    return agg.drop(columns=["hr_daily_mean"])

def feat_cross_sleep_env() -> pd.DataFrame:
    """
    그룹2: 수면 환경 품질 (mACStatus × mAmbience × mLight)

    충전 상태를 수면 proxy로 사용하여 실제 수면 중 환경 측정.

    피처 목록
    ─────────
    cross_sleep_noise_mean       야간 충전 중 평균 소음 확률
    cross_sleep_noise_speech     야간 충전 중 Speech 계열 소음 비율
    cross_sleep_light_mean       야간 충전 중 평균 조도
    cross_sleep_light_max        야간 충전 중 최대 조도
    cross_charge_screen_on_ratio 충전 중 화면ON 비율 (수면 지연 지표)
    cross_charge_screen_off_min  충전+화면OFF 지속 시간(분) = 실수면 추정
    """
    # ── 충전 상태 ──────────────────────────────────────────────────────
    ac = _load_floored("mACStatus", FLOOR_FINE, ["m_charging"])

    # ── 소음 ──────────────────────────────────────────────────────────
    amb_df = load_parquet("mAmbience")
    amb_df = amb_df.explode("m_ambience")
    amb_df["sound_type"] = amb_df["m_ambience"].str[0]
    amb_df["amb_level"]  = pd.to_numeric(amb_df["m_ambience"].str[1], errors="coerce")
    amb_df = amb_df.dropna(subset=["amb_level"])
    amb_df["is_speech"] = amb_df["sound_type"].str.contains(
        "Speech|Narration|Conversation", case=False, na=False
    ).astype(float)
    amb_df["ts_floor"] = amb_df["timestamp"].dt.floor(FLOOR_FINE)
    amb_agg = (
        amb_df.groupby(["subject_id", "date", "ts_floor"])
        .agg(amb_level_mean=("amb_level", "mean"),
             amb_speech_r   =("is_speech", "mean"))
        .reset_index()
    )

    # ── 폰 조도 ───────────────────────────────────────────────────────
    ml = _load_floored("mLight", FLOOR_FINE, ["m_light"])

    # ── 스크린 ────────────────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 충전 시간대 필터 (야간: 21~08시): night_key 적용 ──────────────
    ac["hour"] = ac["ts_floor"].dt.hour
    night_ac   = _night_key(ac[(ac["hour"] >= 21) | (ac["hour"] < 8)])
    # night_key를 date로 덮어써서 이후 groupby에서 자동 적용
    night_ac["date"] = night_ac["night_key"]

    # 충전 중인 버킷만
    charging = night_ac[night_ac["m_charging"] == 1].copy()
    BUCKET_MIN = 5

    # 충전+소음
    ch_amb = charging.merge(
        amb_agg[["subject_id", "ts_floor", "amb_level_mean", "amb_speech_r"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_amb = (
        ch_amb.groupby(["subject_id", "date"])
        .agg(cross_sleep_noise_mean  =("amb_level_mean", "mean"),
             cross_sleep_noise_speech=("amb_speech_r",   "mean"))
        .reset_index()
    )

    # 충전+조도
    ch_ml = charging.merge(
        ml[["subject_id", "ts_floor", "m_light"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_light = (
        ch_ml.groupby(["subject_id", "date"])
        .agg(cross_sleep_light_mean=("m_light", "mean"),
             cross_sleep_light_max =("m_light", "max"))
        .reset_index()
    )

    # 충전+스크린 (야간 전체)
    night_ac_sc = night_ac.merge(
        sc[["subject_id", "ts_floor", "m_screen_use"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_sc = (
        night_ac_sc.groupby(["subject_id", "date"])
        .agg(
            _sc_on  =("m_screen_use", "sum"),
            _sc_tot =("m_screen_use", "count"),
        )
        .reset_index()
    )
    agg_sc["cross_charge_screen_on_ratio"] = (
        agg_sc["_sc_on"] / agg_sc["_sc_tot"].replace(0, np.nan)
    )
    # 충전+화면OFF = 실수면 추정
    night_ac_sc["is_sleep_bucket"] = (
        (night_ac_sc["m_charging"] == 1) & (night_ac_sc["m_screen_use"] == 0)
    ).astype(float)
    agg_sleep = (
        night_ac_sc.groupby(["subject_id", "date"])["is_sleep_bucket"]
        .sum().reset_index()
        .assign(cross_charge_screen_off_min=lambda x: x["is_sleep_bucket"] * BUCKET_MIN)
        [["subject_id", "date", "cross_charge_screen_off_min"]]
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_amb.merge(agg_light, on=["subject_id", "date"], how="outer")
    agg = agg.merge(
        agg_sc[["subject_id", "date", "cross_charge_screen_on_ratio"]],
        on=["subject_id", "date"], how="outer"
    )
    agg = agg.merge(agg_sleep, on=["subject_id", "date"], how="outer")
    return agg


def feat_cross_exercise() -> pd.DataFrame:
    """
    그룹3: 운동 강도 (mActivity × wHr)

    피처 목록
    ─────────
    cross_exercise_hr_mean      걷기/뛰기 시 평균 심박수 (운동 강도)
    cross_exercise_hr_max       운동 중 최대 심박수
    cross_exercise_hr_recovery  운동 직후 30분 평균 심박 - 운동 중 평균 심박
    cross_exercise_timing_hour  운동(활동+고심박) 피크 시간대
    cross_late_exercise_min     늦은 저녁(20~23시) 운동 시간(분)
    """
    # ── 활동 ──────────────────────────────────────────────────────────
    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])

    # ── 워치 심박 ─────────────────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 병합 ──────────────────────────────────────────────────────────
    merged = act.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    merged["hour"] = merged["ts_floor"].dt.hour
    merged["is_active"] = merged["m_activity"].isin([7, 8])  # walk=7, run=8

    # 운동 중 심박
    active_hr = merged[merged["is_active"]]
    agg_exer = (
        active_hr.groupby(["subject_id", "date"])["hr"]
        .agg(cross_exercise_hr_mean="mean",
             cross_exercise_hr_max="max")
        .reset_index()
    )

    # 운동 피크 시간대 (심박>100 + 활동 구간)
    merged["is_intense"] = merged["is_active"] & (merged["hr"] > 100)
    hourly_intense = (
        merged.groupby(["subject_id", "date", "hour"])["is_intense"]
        .sum().reset_index()
    )
    peak = (
        hourly_intense.loc[
            hourly_intense.groupby(["subject_id", "date"])["is_intense"].idxmax()
        ][["subject_id", "date", "hour"]]
        .rename(columns={"hour": "cross_exercise_timing_hour"})
        .reset_index(drop=True)
    )

    # 운동 후 심박 회복 (운동 직후 30분 vs 운동 중)
    merged_sorted = merged.sort_values(["subject_id", "ts_floor"])
    merged_sorted["post_active"] = (
        merged_sorted.groupby("subject_id")["is_active"]
        .shift(6).fillna(False)   # 6 × 5분 = 30분 후
    )
    recovery = merged_sorted[merged_sorted["post_active"] & ~merged_sorted["is_active"]]
    agg_rec = (
        recovery.groupby(["subject_id", "date"])["hr"]
        .mean().reset_index()
        .rename(columns={"hr": "post_exercise_hr"})
    )

    # 늦은 저녁 운동 시간(분)
    BUCKET_MIN = 5
    late_mask = merged["hour"].between(20, 23) & merged["is_active"]
    agg_late = (
        merged[late_mask]
        .groupby(["subject_id", "date"])["is_active"]
        .sum().reset_index()
        .assign(cross_late_exercise_min=lambda x: x["is_active"] * BUCKET_MIN)
        [["subject_id", "date", "cross_late_exercise_min"]]
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_exer.merge(peak,     on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_late,      on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_rec,       on=["subject_id", "date"], how="left")
    agg["cross_exercise_hr_recovery"] = (
        agg["post_exercise_hr"] - agg["cross_exercise_hr_mean"]
    )
    return agg.drop(columns=["post_exercise_hr"], errors="ignore")

def feat_cross_mobility() -> pd.DataFrame:
    """
    그룹4: 사회적 활동 & 이동 (mGps × mActivity × mWifi)

    피처 목록
    ─────────
    cross_outdoor_active_min    GPS 이동 + 활동=걷기/뛰기 동시 충족 시간(분)
    cross_home_sedentary_min    집 WiFi + 활동=정지 동시 시간(분)
    cross_home_active_ratio     집 WiFi 연결 중 활동적인 비율
    """
    BUCKET_MIN = 5

    # ── GPS (1분 단위, 속도만) ────────────────────────────────────────
    gps_df = load_parquet("mGps")
    gps_df["ts_floor"] = gps_df["timestamp"].dt.floor(FLOOR_MED)
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            for c in ["latitude", "longitude", "speed"]:
                gps_df[c] = gps_df["m_gps"].apply(
                    lambda x: x.get(c, np.nan) if isinstance(x, dict) else np.nan
                )
    gps_df = gps_df.dropna(subset=["latitude", "longitude"])
    # 이상값 제거
    gps_df["speed"] = pd.to_numeric(gps_df.get("speed", np.nan), errors="coerce")
    gps_df = gps_df[gps_df["speed"].isna() | (gps_df["speed"] <= GPS_SPEED_LIMIT_MS)]
    gps_spd = (
        gps_df.groupby(["subject_id", "date", "ts_floor"])["speed"]
        .mean().reset_index()
    )

    # ── 활동 ──────────────────────────────────────────────────────────
    act = _load_floored("mActivity", FLOOR_MED, ["m_activity"])

    # ── WiFi 집 BSSID ─────────────────────────────────────────────────
    wifi_df = load_parquet("mWifi")
    wifi_df["ts_floor"] = wifi_df["timestamp"].dt.floor(FLOOR_MED)
    wifi_df = wifi_df.explode("m_wifi")
    _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
    if _fv is not None and isinstance(_fv, dict):
        wifi_df["bssid"] = wifi_df["m_wifi"].apply(
            lambda x: x.get("bssid", "") if isinstance(x, dict) else ""
        )
    else:
        wifi_df["bssid"] = ""

    home_bssid = (
        wifi_df[wifi_df["bssid"] != ""]
        .groupby(["subject_id", "bssid"]).size().reset_index(name="cnt")
        .sort_values("cnt", ascending=False)
        .drop_duplicates("subject_id")[["subject_id", "bssid"]]
        .rename(columns={"bssid": "home_bssid"})
    )
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"] == wifi_df["home_bssid"]).astype(int)
    wifi_home = (
        wifi_df.groupby(["subject_id", "date", "ts_floor"])["is_home"]
        .max().reset_index()   # 버킷 내 집 WiFi 감지 여부
    )

    # ── GPS × 활동: 실외 활동 ─────────────────────────────────────────
    gps_act = gps_spd.merge(
        act[["subject_id", "ts_floor", "m_activity"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    gps_act["is_outdoor_active"] = (
        (gps_act["speed"] > 0.5) & gps_act["m_activity"].isin([7, 8])
    ).astype(float)
    agg_outdoor = (
        gps_act.groupby(["subject_id", "date"])["is_outdoor_active"]
        .sum().reset_index()
        .assign(cross_outdoor_active_min=lambda x: x["is_outdoor_active"] * BUCKET_MIN)
        [["subject_id", "date", "cross_outdoor_active_min"]]
    )

    # ── WiFi × 활동: 집안 정지/활동 ──────────────────────────────────
    wifi_act = wifi_home.merge(
        act[["subject_id", "ts_floor", "m_activity"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    wifi_at_home = wifi_act[wifi_act["is_home"] == 1]

    wifi_at_home = wifi_at_home.copy()
    wifi_at_home["is_sedentary"] = (wifi_at_home["m_activity"] == 3).astype(float)
    wifi_at_home["is_moving"]    = wifi_at_home["m_activity"].isin([7, 8]).astype(float)

    agg_home_sed = (
        wifi_at_home.groupby(["subject_id", "date"])["is_sedentary"]
        .sum().reset_index()
        .assign(cross_home_sedentary_min=lambda x: x["is_sedentary"] * BUCKET_MIN)
        [["subject_id", "date", "cross_home_sedentary_min"]]
    )
    agg_home_act = (
        wifi_at_home.groupby(["subject_id", "date"])
        .agg(_mv=("is_moving", "sum"), _tot=("is_moving", "count"))
        .reset_index()
    )
    agg_home_act["cross_home_active_ratio"] = (
        agg_home_act["_mv"] / agg_home_act["_tot"].replace(0, np.nan)
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_outdoor.merge(agg_home_sed, on=["subject_id", "date"], how="outer")
    agg = agg.merge(
        agg_home_act[["subject_id", "date", "cross_home_active_ratio"]],
        on=["subject_id", "date"], how="outer"
    )
    return agg

def feat_cross_circadian() -> pd.DataFrame:
    """
    그룹5: 일주기 리듬 (wLight × wHr × mScreenStatus)

    피처 목록
    ─────────
    cross_morning_light_hr_corr   아침(06~09시) 광량-심박 상관계수
    cross_morning_natural_light   아침(06~10시) 자연광(>100 lux)+화면OFF 시간(분)
    cross_evening_light_drop      저녁(18~22시) 광량 감소율 (취침 준비 신호)
    """
    BUCKET_MIN = 5

    # ── 워치 광량 ─────────────────────────────────────────────────────
    wl = _load_floored("wLight", FLOOR_FINE, ["w_light"])

    # ── 워치 심박 ─────────────────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 스크린 ────────────────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 워치 광량 × 심박: 아침 상관계수 ─────────────────────────────
    wl_hr = wl.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    wl_hr["hour"] = wl_hr["ts_floor"].dt.hour
    morning_wl_hr = wl_hr[wl_hr["hour"].between(6, 9)]

    def _corr(g):
        if len(g) < 3: return np.nan
        return g["w_light"].corr(g["hr"])

    agg_corr = (
        morning_wl_hr.groupby(["subject_id", "date"])
        .apply(_corr).reset_index()
        .rename(columns={0: "cross_morning_light_hr_corr"})
    )

    # ── 워치 광량 × 스크린: 아침 자연광 노출 ─────────────────────────
    wl_sc = wl.merge(
        sc[["subject_id", "ts_floor", "m_screen_use"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    wl_sc["hour"] = wl_sc["ts_floor"].dt.hour
    morning_wl_sc = wl_sc[wl_sc["hour"].between(6, 10)]
    morning_wl_sc = morning_wl_sc.copy()
    morning_wl_sc["is_natural_no_screen"] = (
        (morning_wl_sc["w_light"] > 100) & (morning_wl_sc["m_screen_use"] == 0)
    ).astype(float)
    agg_nat = (
        morning_wl_sc.groupby(["subject_id", "date"])["is_natural_no_screen"]
        .sum().reset_index()
        .assign(cross_morning_natural_light=lambda x: x["is_natural_no_screen"] * BUCKET_MIN)
        [["subject_id", "date", "cross_morning_natural_light"]]
    )

    # ── 워치 광량: 저녁 광량 감소율 ──────────────────────────────────
    wl_eve = wl_hr[wl_hr["hour"].between(18, 22)]
    if len(wl_eve) > 0:
        # 각 날 저녁 시작(18시) 광량 vs 끝(22시) 광량
        wl_eve_agg = (
            wl_eve.groupby(["subject_id", "date", "hour"])["w_light"]
            .mean().reset_index()
        )
        eve_start = wl_eve_agg[wl_eve_agg["hour"] == 18][["subject_id","date","w_light"]].rename(columns={"w_light": "light_18"})
        eve_end   = wl_eve_agg[wl_eve_agg["hour"] == 22][["subject_id","date","w_light"]].rename(columns={"w_light": "light_22"})
        agg_drop  = eve_start.merge(eve_end, on=["subject_id","date"], how="inner")
        agg_drop["cross_evening_light_drop"] = (
            (agg_drop["light_18"] - agg_drop["light_22"]) /
            agg_drop["light_18"].replace(0, np.nan)
        )
        agg_drop = agg_drop[["subject_id","date","cross_evening_light_drop"]]
    else:
        agg_drop = pd.DataFrame(columns=["subject_id","date","cross_evening_light_drop"])

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_corr.merge(agg_nat,  on=["subject_id","date"], how="outer")
    agg = agg.merge(agg_drop,      on=["subject_id","date"], how="outer")
    return agg

def feat_cross_commute() -> pd.DataFrame:
    """🚗 통근 시간: IN_VEHICLE + GPS 이동 (mActivity × mGps)."""
    BUCKET_MIN = 5
    act = _load_floored("mActivity", FLOOR_MED, ["m_activity"])
    gps_df = load_parquet("mGps")
    gps_df["ts_floor"] = gps_df["timestamp"].dt.floor(FLOOR_MED)
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            gps_df["speed_g"] = gps_df["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
    gps_df["speed_g"] = pd.to_numeric(gps_df.get("speed_g", np.nan), errors="coerce")
    gps_spd = (gps_df.groupby(["subject_id","date","ts_floor"])["speed_g"]
               .mean().reset_index())
    merged = act.merge(gps_spd[["subject_id","ts_floor","speed_g"]],
                       on=["subject_id","ts_floor"], how="inner")
    merged["is_commute"] = ((merged["m_activity"]==0) & (merged["speed_g"]>1.5)).astype(float)
    agg = (merged.groupby(["subject_id","date"])["is_commute"]
           .sum().reset_index()
           .assign(cross_commute_min=lambda x: x["is_commute"]*BUCKET_MIN)
           [["subject_id","date","cross_commute_min"]])
    return agg

def feat_cross_presleep_arousal() -> pd.DataFrame:
    """😤 취침 전 각성 지수: wHr × wPedo × mActivity (20~22시)."""
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object: hr = hr.explode("heart_rate")
    hr["hr_val"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr_val"])
    hr = hr[hr["hr_val"].between(30,220)]
    hr["hour"] = hr["timestamp"].dt.hour
    hr_pre = (hr[hr["hour"].between(20,22)]
              .groupby(["subject_id","date"])["hr_val"]
              .mean().reset_index().rename(columns={"hr_val":"_hr_pre"}))

    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])
    act["hour"] = act["ts_floor"].dt.hour
    act_pre = (act[act["hour"].between(20,22)]
               .groupby(["subject_id","date"])["m_activity"]
               .agg(lambda x: (x.isin([7,8,9])).mean()).reset_index()
               .rename(columns={"m_activity":"_act_pre"}))

    ped = load_parquet("wPedo")
    ped["hour"] = ped["timestamp"].dt.hour
    ped_pre = (ped[ped["hour"].between(20,22)]
               .groupby(["subject_id","date"])["step"]
               .sum().reset_index().rename(columns={"step":"_step_pre"}))

    agg = hr_pre.merge(act_pre, on=["subject_id","date"], how="outer")
    agg = agg.merge(ped_pre, on=["subject_id","date"], how="outer")
    # z-score 없이 0~1 정규화 후 합산 → 각성 지수
    for c in ["_hr_pre","_act_pre","_step_pre"]:
        mn = agg[c].min(); mx = agg[c].max()
        agg[c] = (agg[c]-mn) / (mx-mn+1e-8)
    agg["cross_presleep_arousal"] = agg[["_hr_pre","_act_pre","_step_pre"]].mean(axis=1)
    return agg[["subject_id","date","cross_presleep_arousal"]]

def feat_cross_resting_hr_at_home() -> pd.DataFrame:
    """💓 실제 안정시 심박: 집 WiFi + STILL 구간 심박 (wHr × mWifi × mActivity)."""
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object: hr = hr.explode("heart_rate")
    hr["hr_val"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr_val"])
    hr = hr[hr["hr_val"].between(30,220)]
    hr_agg = hr.groupby(["subject_id","date","ts_floor"])["hr_val"].mean().reset_index()

    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])

    wifi_df = load_parquet("mWifi")
    wifi_df["ts_floor"] = wifi_df["timestamp"].dt.floor(FLOOR_FINE)
    wifi_df = wifi_df.explode("m_wifi")
    _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
    if _fv is not None and isinstance(_fv, dict):
        wifi_df["bssid"] = wifi_df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
    else:
        wifi_df["bssid"] = ""
    home_bssid = (wifi_df[wifi_df["bssid"]!=""].groupby(["subject_id","bssid"])
                  .size().reset_index(name="cnt").sort_values("cnt", ascending=False)
                  .drop_duplicates("subject_id")[["subject_id","bssid"]]
                  .rename(columns={"bssid":"home_bssid"}))
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"]==wifi_df["home_bssid"]).astype(int)
    wifi_home = (wifi_df.groupby(["subject_id","date","ts_floor"])["is_home"]
                 .max().reset_index())

    merged = (hr_agg
              .merge(act[["subject_id","ts_floor","m_activity"]], on=["subject_id","ts_floor"], how="inner")
              .merge(wifi_home[["subject_id","ts_floor","is_home"]], on=["subject_id","ts_floor"], how="left"))
    still_home = merged[(merged["m_activity"]==3) & (merged["is_home"]==1)]
    agg = (still_home.groupby(["subject_id","date"])["hr_val"]
           .mean().reset_index().rename(columns={"hr_val":"cross_resting_hr_at_home"}))
    return agg


def feat_cross_sleep_env_noise() -> pd.DataFrame:
    """🌙 수면 환경 복잡도: BLE × WiFi rssi × GPS 정지 비율 (야간)."""
    ble_df = load_parquet("mBle")
    ble_df["hour"] = ble_df["timestamp"].dt.hour
    if "m_ble" in ble_df.columns and ble_df["m_ble"].dtype == object:
        try:
            ble_df = ble_df.explode("m_ble")
            _fv = ble_df["m_ble"].dropna().iloc[0] if ble_df["m_ble"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                ble_df["address"] = ble_df["m_ble"].apply(lambda x: x.get("address","") if isinstance(x,dict) else "")
        except Exception:
            ble_df["address"] = ""
    night_m = (ble_df["hour"]>=22)|(ble_df["hour"]<6)
    ble_n = _night_key(ble_df[night_m & (ble_df["address"]!="")])
    ble_night = (ble_n.groupby(["subject_id","night_key"])["address"].nunique().reset_index()
                 .rename(columns={"night_key":"date","address":"_ble_night"}))

    wifi_df = load_parquet("mWifi")
    wifi_df["hour"] = wifi_df["timestamp"].dt.hour
    if "m_wifi" in wifi_df.columns and wifi_df["m_wifi"].dtype == object:
        try:
            wifi_df = wifi_df.explode("m_wifi")
            _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                wifi_df["rssi"] = wifi_df["m_wifi"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            wifi_df["rssi"] = np.nan
    night_w = (wifi_df["hour"]>=22)|(wifi_df["hour"]<6)
    wifi_n = _night_key(wifi_df[night_w])
    wifi_night = (wifi_n.groupby(["subject_id","night_key"])["rssi"]
                  .mean().reset_index()
                  .rename(columns={"night_key":"date","rssi":"_wifi_rssi_night"}))

    gps_df = load_parquet("mGps")
    gps_df["hour"] = gps_df["timestamp"].dt.hour
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            gps_df["speed_g"] = gps_df["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
    gps_df["speed_g"] = pd.to_numeric(gps_df.get("speed_g", np.nan), errors="coerce")
    night_g = (gps_df["hour"]>=22)|(gps_df["hour"]<6)
    gps_n = _night_key(gps_df[night_g])
    gps_night = (gps_n.groupby(["subject_id","night_key"])["speed_g"]
                 .agg(lambda x: (x<1.0).mean()).reset_index()
                 .rename(columns={"night_key":"date","speed_g":"_gps_stop_night"}))

    agg = ble_night.merge(wifi_night, on=["subject_id","date"], how="outer")
    agg = agg.merge(gps_night, on=["subject_id","date"], how="outer")
    for c in ["_ble_night","_wifi_rssi_night","_gps_stop_night"]:
        mn=agg[c].min(); mx=agg[c].max()
        agg[c] = (agg[c]-mn)/(mx-mn+1e-8)
    agg["cross_sleep_env_noise"] = agg[["_ble_night","_wifi_rssi_night","_gps_stop_night"]].mean(axis=1)
    return agg[["subject_id","date","cross_sleep_env_noise"]]

def feat_cross_wifi_gps_home_agree() -> pd.DataFrame:
    """🏠 집 판별 신뢰도: WiFi home 비율 × GPS home_min 일치 여부."""
    wifi_df = load_parquet("mWifi")
    if "m_wifi" in wifi_df.columns and wifi_df["m_wifi"].dtype == object:
        wifi_df = wifi_df.explode("m_wifi")
        _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            wifi_df["bssid"] = wifi_df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
    else:
        wifi_df["bssid"] = ""
    home_bssid = (wifi_df[wifi_df["bssid"]!=""].groupby(["subject_id","bssid"])
                  .size().reset_index(name="cnt").sort_values("cnt",ascending=False)
                  .drop_duplicates("subject_id")[["subject_id","bssid"]]
                  .rename(columns={"bssid":"home_bssid"}))
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"]==wifi_df["home_bssid"]).astype(int)
    wifi_home_r = (wifi_df.groupby(["subject_id","date"])["is_home"]
                   .mean().reset_index().rename(columns={"is_home":"_wifi_home_r"}))
    # GPS home_min은 feat_mGps()에서 생성됨 → 여기서는 WiFi home 비율만 반환
    # (파이프라인에서 gps_home_min과 merge하여 cross_wifi_gps_home_agree 계산)
    wifi_home_r = wifi_home_r.rename(columns={"_wifi_home_r":"cross_wifi_gps_home_agree"})
    return wifi_home_r

def build_cross_features(date_df: pd.DataFrame) -> pd.DataFrame:
    """
    교차 parquet 피처 전체 생성 및 date_df에 merge.

    각 함수는 (subject_id, date) 기준 일별 집계값을 반환.
    date → lifelog_date 로 rename 후 left merge.
    """
    print("교차 parquet 피처 추출 시작...")

    cross_funcs = {
        "arousal":   feat_cross_arousal,    # 그룹1: 취침 전 각성 자극
        "sleep_env": feat_cross_sleep_env,  # 그룹2: 수면 환경 품질
        "exercise":  feat_cross_exercise,   # 그룹3: 운동 강도
        "mobility":  feat_cross_mobility,   # 그룹4: 사회적 활동 & 이동
        "circadian": feat_cross_circadian,  # 그룹5: 일주기 리듬
    }

    df = date_df.copy()
    for name, func in cross_funcs.items():
        print(f"  [cross_{name}] 처리 중...", end=" ", flush=True)
        try:
            feat = func()
            feat = feat.rename(columns={"date": "lifelog_date"})
            feat["lifelog_date"] = pd.to_datetime(feat["lifelog_date"])
            df["lifelog_date"]   = pd.to_datetime(df["lifelog_date"])
            before = df.shape[1]
            df     = df.merge(feat, on=["subject_id", "lifelog_date"], how="left")
            added  = df.shape[1] - before
            print(f"완료 (+{added} 컬럼)")
        except Exception as e:
            print(f"오류: {e}")
            import traceback; traceback.print_exc()

    return df

def build_features(date_df: pd.DataFrame) -> pd.DataFrame:
    """
    date_df : subject_id / sleep_date / lifelog_date 컬럼 포함
              lifelog_date 는 naive datetime64[ns] 이어야 함

    각 feat_*() 의 date 컬럼(naive datetime64[ns])을
    lifelog_date 에 left merge → 타입 완전 일치, 누락 없이 조인
    """
    print("피처 추출 시작...")

    feat_funcs = {
        "wPedo":          feat_wPedo,
        "wHr":            feat_wHr,
        "wLight":         feat_wLight,
        "mActivity":      feat_mActivity,
        "mACStatus":      feat_mACStatus,
        "mScreenStatus":  feat_mScreenStatus,
        "mLight":         feat_mLight,
        "mAmbience":      feat_mAmbience,
        "mWifi":          feat_mWifi,
        "mGps":           feat_mGps,
        "mUsageStats":    feat_mUsageStats,
        "mBle":           feat_mBle,
        "SleepProxy_AC_Scr":  feat_SleepProxy_AC_Scr,
        "SleepProxy_TwoTrack": feat_SleepProxy_TwoTrack,
        # 기존 복합 피처
        "cross_arousal":     feat_cross_arousal,
        "cross_sleep_env":   feat_cross_sleep_env,
        "cross_exercise":    feat_cross_exercise,
        "cross_mobility":    feat_cross_mobility,
        "cross_circadian":   feat_cross_circadian,
        # 신규 복합 피처 (V53)
        "cross_commute":             feat_cross_commute,
        "cross_presleep_arousal":    feat_cross_presleep_arousal,
        "cross_resting_hr_at_home":  feat_cross_resting_hr_at_home,
        "cross_sleep_env_noise":     feat_cross_sleep_env_noise,
        "cross_wifi_gps_home_agree": feat_cross_wifi_gps_home_agree,
    }

    df = date_df.copy()
    for name, func in feat_funcs.items():
        print(f"  [{name}] 처리 중...", end=" ", flush=True)
        try:
            feat = func()
            feat = feat.rename(columns={"date": "lifelog_date"})
            df   = df.merge(feat, on=["subject_id", "lifelog_date"], how="left")
            print(f"완료 ({len(feat):,} rows, +{feat.shape[1] - 2} cols)")
        except Exception as e:
            print(f"오류: {e}")

    return df

# ══════════════════════════════════════════════════════════════════════
# 4. 시간/요일 파생 피처
# ══════════════════════════════════════════════════════════════════════

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """sleep_date 기반 요일/월/주차 피처 + subject_num 인코딩"""
    df = df.copy()
    df["sleep_date"] = pd.to_datetime(df["sleep_date"])

    df["dow"]          = df["sleep_date"].dt.dayofweek       # 0=월, 6=일
    df["is_weekend"]   = (df["dow"] >= 5).astype(int)
    df["month"]        = df["sleep_date"].dt.month
    df["week_of_year"] = df["sleep_date"].dt.isocalendar().week.astype(int)
    df["subject_num"]  = df["subject_id"].str.extract(r"(\d+)").astype(int)
    return df


## 5-1. Z-Score + 센서 Rolling + LGB/XGB/CatBoost 가중 앙상블

# ══════════════════════════════════════════════════════════════════════
# 5. 래그 / 개인 평균 피처 — fold 내부 전용 헬퍼
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# 6. LightGBM 학습
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# 5-v5-1. Z-score + 센서 Rolling + LGB/XGB/CatBoost 가중 앙상블
# ══════════════════════════════════════════════════════════════════════

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

# ── 모델 하이퍼파라미터 (V5와 동일) ─────────────────────────────────
LGBM_PARAMS = {
    "objective":          "binary",
    "metric":             "binary_logloss",
    "n_estimators":       1000,
    "learning_rate":      0.05,
    "num_leaves":         31,
    "min_child_samples":  10,
    "feature_fraction":   0.8,
    "bagging_fraction":   0.8,
    "bagging_freq":       5,
    "lambda_l1":          0.1,
    "lambda_l2":          0.1,
    "random_state":       42,
    "n_jobs":             -1,
    "verbose":            -1,
}
XGB_PARAMS = {
    "objective":          "binary:logistic",
    "eval_metric":        "logloss",
    "n_estimators":       1000,
    "learning_rate":      0.05,
    "max_depth":          6,
    "subsample":          0.8,
    "colsample_bytree":   0.8,
    "reg_alpha":          0.1,
    "reg_lambda":         0.1,
    "random_state":       42,
    "n_jobs":             -1,
    "verbosity":          0,
    "use_label_encoder":  False,
}
CAT_PARAMS = {
    "iterations":         1000,
    "learning_rate":      0.05,
    "depth":              6,
    "loss_function":      "Logloss",
    "eval_metric":        "Logloss",
    "l2_leaf_reg":        3,
    "random_seed":        42,
    "verbose":            False,
    "allow_writing_files": False,
}

# ── 센서 Rolling 설정 ─────────────────────────────────────────────────
# V3와 달리 센서 데이터(X)에 rolling을 적용 → test에도 결측 없음
# 이유:
#   V3: 타겟(y) rolling → test의 y가 없어 평균 44~84% 결측
#   V5-1: 센서(X) rolling → train/test 모두 X값 존재 → 결측 <5%
ROLL_WINDOWS         = [3, 7]     # rolling 윈도우 크기 (일)
ROLL_MISS_THRESHOLD  = 0.30       # rolling 피처 결측률 임계값 (30% 초과 제거)

# 피처 선택 설정
# 피어슨 상관계수 절댓값 기준 상위 CORR_TOP_K개 선택
# - raw+rolling: 86+~100 = ~186개 후보
# - z-score     : 동일 수 후보
# - 전체 후보 ~370개에서 타겟별로 50개 선택 → 샘플(450):피처(50) = 9:1
CORR_TOP_K  = 50
N_FOLDS     = 5

# ── Subject-Hole CV 설정 ──────────────────────────────────────────────
# 실제 train/test 분리 구조 (T/V/T/V 인터리빙)를 모사하는 CV 전략.
#
# 실제 데이터 패턴 (id01 예시):
#   TTTTTTTTTTTTTTTTTTTTTTTTTTTTVVVVVVVVVVVVVVTTTTTTTTTTTTTVVVVVVVVVVVVV
#   → train과 test가 사용자 내부에서 시간 순으로 교차하는 블록 구조
#
# Subject-Hole CV 방식:
#   각 사용자의 날짜를 시간 순 정렬 후 n_folds*2 개의 chunk로 분할.
#   fold i: chunk[i] + chunk[i+n_folds] → val  (early hole + late hole)
#           나머지 → train
#   → T/V/T/V 패턴을 fold 단위로 재현 ✅
# ── 베이즈 수축 파라미터 ──────────────────────────────────────────────
# 사용자 평균 베이즈 수축 (Bayesian Shrinkage toward user mean)
#
# 수식: pred_final = α × model_pred + (1-α) × user_label_mean
#
# α (신뢰도): 샘플 수에 따른 동적 결정
#   α(n) = n / (n + SHRINK_K)
#   SHRINK_K=10 기준:
#     n=10 → α=0.50  (모델 50%, 사용자 평균 50%)
#     n=20 → α=0.67  (모델 67%, 사용자 평균 33%)
#     n=30 → α=0.75  (모델 75%, 사용자 평균 25%)
#     n=57 → α=0.85  (모델 85%, 사용자 평균 15%)
#
# 직관: 학습 데이터가 적은 사용자일수록 해당 사용자의 과거 평균 비중↑
#       → 개인화 편차(S3: 0.374, S2: 0.336) 보정에 효과적
#
# 누수 방지 ✅
# - OOF: fold 내 train 부분만의 사용자 평균 사용 (val 레이블 미포함)
# - test: 전체 train 사용자 평균 사용 (test 레이블 미사용)
SHRINK_K    = 10.0   # 수축 강도 조율 상수 (클수록 수축 강함)

# ════════════════════════════════════════════════════════════════════
# 5-1. 센서 Rolling 피처 계산
# ════════════════════════════════════════════════════════════════════

def add_sensor_rolling(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    feat_cols: list,
) -> tuple:
    """
    센서 피처에 rolling 통계를 추가하여 train/test를 반환.

    V3 타겟 rolling과의 차이
    ──────────────────────
    V3 : 타겟(y) rolling
         - test의 y값이 없으므로 결측 평균 44~84%
         - OOF 학습 시 val 레이블이 rolling 집계에 포함될 수 있음 (누수 위험)

    V5-1: 센서(X) rolling
         - train/test 모두 X값 존재 → 결측 < 5%
         - X→X 관계이므로 레이블(y) 누수 없음

    누수 방지 설계
    ──────────────
    ① shift(1) 적용
       T일의 rolling = mean(T-1, T-2, ...)
       → T일 자신의 센서값이 T일 rolling에 포함되지 않음 ✅

    ② train+test 결합 후 (subject_id, sleep_date) 기준 정렬
       → 시간 순서 유지, 미래 센서값이 과거 rolling에 포함되지 않음 ✅

    ③ 센서 데이터(X) 기반 → 레이블(y) 정보 미포함 ✅
       (타겟 rolling과 달리 val 레이블 누수 없음)

    ④ min_periods=1 사용
       초기 날짜(이전 기록 부족)에도 가능한 값 생성 → 결측 최소화

    ⑤ ROLL_MISS_THRESHOLD(30%) 초과 rolling 피처 자동 제거
       (hr_night_mean 등 원본부터 결측 많은 피처의 rolling 제거)

    주의: train+test 결합 시 test 센서값이 일부 train rolling에 영향
    ──────
    날짜가 혼재하는 구조에서 test 날짜 T의 센서값이
    train 날짜 T+k의 rolling에 기여할 수 있음.
    그러나 X→X 관계이며 y 정보를 포함하지 않으므로
    실질적 누수(y leakage)가 아닌 통상 허용 범위 내.

    반환
    ────
    train_df_new : rolling 피처 추가된 train DataFrame
    test_df_new  : rolling 피처 추가된 test  DataFrame
    roll_cols    : 생성된 rolling 피처명 리스트
    """
    # ── 연속형 피처만 대상 (이진 피처는 rolling 의미 없음) ──────────
    continuous_cols = [
        c for c in feat_cols
        if len(train_df[c].dropna().unique()) > 2
        or not set(train_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]

    # ── train + test 결합, 날짜 정렬 ─────────────────────────────────
    # ignore_index=True 와 keys 를 함께 쓰면 keys가 무시되므로
    # _src 컬럼을 concat 전에 명시적으로 추가
    tr_part = train_df[["subject_id", "sleep_date"] + continuous_cols].copy()
    te_part = test_df[["subject_id", "sleep_date"] + continuous_cols].copy()
    tr_part["_src"] = "train"
    te_part["_src"] = "test"

    combined = pd.concat([tr_part, te_part], ignore_index=True)

    combined["sleep_date"] = pd.to_datetime(combined["sleep_date"])
    combined = combined.sort_values(["subject_id", "sleep_date"]).reset_index(drop=True)

    # ── rolling 계산 ──────────────────────────────────────────────────
    generated: list = []
    for c in continuous_cols:
        for w in ROLL_WINDOWS:
            col_name = f"{c}_roll{w}"
            combined[col_name] = (
                combined.groupby("subject_id")[c]
                .transform(
                    lambda x, _w=w: x.shift(1).rolling(_w, min_periods=1).mean()
                )
            )
            generated.append(col_name)

    # ── 결측률 필터 (train 기준 ROLL_MISS_THRESHOLD 초과 제거) ────────
    train_mask = combined["_src"] == "train"
    miss_rates  = combined.loc[train_mask, generated].isnull().mean()
    keep_cols   = [c for c in generated if miss_rates[c] <= ROLL_MISS_THRESHOLD]
    drop_cols   = [c for c in generated if miss_rates[c] > ROLL_MISS_THRESHOLD]

    if drop_cols:
        print(f"  rolling 피처 결측>{ROLL_MISS_THRESHOLD:.0%} 제거 "
              f"({len(drop_cols)}개): {drop_cols[:5]}{'...' if len(drop_cols)>5 else ''}")

    # ── train / test 분리 ─────────────────────────────────────────────
    # (subject_id, sleep_date) 기준 merge로 정확히 분리
    merge_key = ["subject_id", "sleep_date"]

    train_roll = (
        combined.loc[train_mask, merge_key + keep_cols]
        .drop_duplicates(merge_key)
    )
    test_roll = (
        combined.loc[~train_mask, merge_key + keep_cols]
        .drop_duplicates(merge_key)
    )

    train_df["sleep_date"] = pd.to_datetime(train_df["sleep_date"])
    test_df["sleep_date"]  = pd.to_datetime(test_df["sleep_date"])

    train_new = train_df.merge(train_roll, on=merge_key, how="left")
    test_new  = test_df.merge(test_roll,  on=merge_key, how="left")

    # 결측 최종 확인
    tr_miss = train_new[keep_cols].isnull().mean().mean() * 100
    te_miss = test_new[keep_cols].isnull().mean().mean() * 100
    print(f"  센서 rolling 피처 {len(keep_cols)}개 생성 완료")
    print(f"  평균 결측률 — train: {tr_miss:.1f}%  test: {te_miss:.1f}%")

    return train_new, test_new, keep_cols



# ════════════════════════════════════════════════════════════════════
# 5-2. Z-score 개인화 피처 계산 (V5와 동일)
# ════════════════════════════════════════════════════════════════════

def compute_user_zscores(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    feat_cols: list,
) -> tuple:
    """
    사용자별 Z-score 피처 계산 (raw + rolling 피처 모두 대상).

    누수 방지 ✅
    통계량(mean, std)은 train_df만 사용.
    test에는 train 통계량을 그대로 적용.
    z-score 계산에 타겟 레이블 미사용.
    """
    continuous_cols = [
        c for c in feat_cols
        if len(train_df[c].dropna().unique()) > 2
        or not set(train_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]
    print(f"  Z-score 대상: {len(continuous_cols)}개 "
          f"(raw + rolling 연속형, 이진 {len(feat_cols)-len(continuous_cols)}개 제외)")

    user_stats = (
        train_df.groupby("subject_id")[continuous_cols]
        .agg(["mean", "std"])
    )
    user_stats.columns = [f"{c}__{s}" for c, s in user_stats.columns]

    def _apply(df: pd.DataFrame) -> pd.DataFrame:
        res = {}
        for c in continuous_cols:
            u_mean = df["subject_id"].map(user_stats[f"{c}__mean"])
            u_std  = df["subject_id"].map(user_stats[f"{c}__std"]).fillna(0)
            raw    = df[c].fillna(u_mean)
            z      = (raw - u_mean) / (u_std + 1e-8)
            res[f"{c}_uz"] = z.fillna(0)
        return pd.DataFrame(res, index=df.index)

    train_z = _apply(train_df)
    test_z  = _apply(test_df)
    z_cols  = list(train_z.columns)
    return train_z, test_z, z_cols

# ════════════════════════════════════════════════════════════════════
# 5-3. 피처 선택: 피어슨 상관계수 기반 (CV fold 내부 계산)
# ════════════════════════════════════════════════════════════════════

def _entropy_discrete(x: np.ndarray) -> float:
    """이산 변수 엔트로피 H(X) = -Σ p_i log(p_i)."""
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 1e-10
    _, counts = np.unique(x, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-10)))

def _entropy_continuous(x: np.ndarray, n_bins: int = 20) -> float:
    """연속 변수 엔트로피 H(X): 히스토그램 기반 근사."""
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 1e-10
    counts, _ = np.histogram(x, bins=n_bins)
    counts = counts[counts > 0]
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-10)))

def _entropy_binary_target(y: np.ndarray) -> float:
    """이진 타겟 엔트로피 H(Y) = -p log(p) - (1-p) log(1-p)."""
    p = np.mean(y)
    if p <= 1e-10 or p >= 1 - 1e-10:
        return 1e-10
    return float(-p * np.log(p) - (1 - p) * np.log(1 - p))

def _nmi_score(mi: float, h_x: float, h_y: float) -> float:
    """정규화 상호정보량 NMI(X;Y) = 2·I(X;Y) / (H(X)+H(Y)) ∈ [0,1]."""
    denom = h_x + h_y
    if denom < 1e-10:
        return 0.0
    return float(np.clip(2 * mi / denom, 0.0, 1.0))

def select_features_by_mutual_info(
    X_all:          pd.DataFrame,
    y_dict:         dict,
    all_cols:       list,
    cat_feat_cols:  list,
    train_df_ref:   pd.DataFrame,
    top_k:          int   = CORR_TOP_K,
    n_neighbors:    int   = 3,
    random_state:   int   = 42,
) -> dict:
    """정규화 상호정보량(NMI) 기반 타겟별 상위 K개 피처 선택.

    NMI(X;Y) = 2·I(X;Y) / (H(X)+H(Y))
      - I(X;Y): sklearn mutual_info_classif (k-NN 추정)
      - H(X):  이산→ 정확한 엔트로피 / 연속→ 히스토그램 근사
      - H(Y):  이진 타겟 엔트로피
    → 0~1 정규화로 스케일 무관 피처 간 공정한 비교

    Pearson 대비 장점:
      - 이진 타겟(Q1~S4)에 적합 (비선형·비단조 관계 포착)
      - 범주형·연속형 혼합 피처 정확 처리 (discrete_features 지정)
      - NMI 정규화로 고분산 피처 과대 선택 방지

    Parameters
    ----------
    cat_feat_cols : 범주형 피처 이름 목록 → discrete_features=True 처리
    """

    import logging
    # 로거 설정
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

    def log_infinite_features(df: pd.DataFrame, context_name: str = "FeatureMatrix"):
        """
        데이터프레임 내의 무한대(inf, -inf) 또는 너무 큰 값을 찾아 로깅합니다.
        """
        # 1. 무한대(inf, -inf) 확인
        inf_mask = np.isinf(df)
        inf_counts = inf_mask.sum()
        inf_cols = inf_counts[inf_counts > 0]

        if len(inf_cols) > 0:
            logging.error(f"[{context_name}] 🚨 무한대(inf) 값이 발견된 컬럼 ({len(inf_cols)}개):")
            for col, count in inf_cols.items():
                logging.error(f"  - {col}: {count} rows")

        # 2. 극단적으로 큰 값 확인 (float64 max 근처)
        # sklearn은 보통 1e308 이상을 문제 삼지만, 안전하게 1e15 이상을 체크해봅니다.
        huge_mask = (df > 1e15) & ~inf_mask
        huge_counts = huge_mask.sum()
        huge_cols = huge_counts[huge_counts > 0]

        if len(huge_cols) > 0:
            logging.warning(f"[{context_name}] ⚠️ 극단적으로 큰 값(>1e15)이 발견된 컬럼 ({len(huge_cols)}개):")
            for col, count in huge_cols.items():
                logging.warning(f"  - {col}: {count} rows")

        return list(inf_cols.keys())

        # 사용 예시 (오류가 난 함수 내부에 추가)
        # mi = mutual_info_classif(...) 호출 직전에 아래 코드를 넣으세요.
        # log_infinite_features(X_train_with_z, "X_train_with_z (Before NMI)")
    from sklearn.feature_selection import mutual_info_classif

    cat_feat_set   = set(cat_feat_cols or [])
    discrete_mask  = np.array([c in cat_feat_set for c in all_cols], dtype=bool)
    sh_folds       = make_subject_hole_folds(train_df_ref)
    feat_dict      = {}

    print(f"  [NMI 선택] 후보 {len(all_cols)}개 피처 "
          f"(이산 {discrete_mask.sum()}개 / 연속 {(~discrete_mask).sum()}개)")

    for target, y_all in y_dict.items():
        nmi_accum = np.zeros(len(all_cols))
        n_valid   = 0

        for tr_idx, _ in sh_folds:
            X_tr = X_all[all_cols].iloc[tr_idx].fillna(0).values
            y_tr = y_all[tr_idx]

            if len(np.unique(y_tr)) < 2:
                continue  # 클래스 다양성 없으면 스킵

            # ── I(X;Y): mutual_info_classif (fold train만 사용, 누수 없음)
            mi = mutual_info_classif(
                X_tr, y_tr,
                discrete_features=discrete_mask,
                n_neighbors=n_neighbors,
                random_state=random_state,
            )
            mi = np.clip(mi, 0, None)  # 음수 추정값 클립

            # ── H(Y): 이진 타겟 엔트로피
            h_y = _entropy_binary_target(y_tr)

            # ── NMI 계산 (피처별)
            nmi = np.zeros(len(all_cols))
            for i in range(len(all_cols)):
                if mi[i] < 1e-12:
                    nmi[i] = 0.0
                    continue
                h_x = (_entropy_discrete(X_tr[:, i])
                       if discrete_mask[i]
                       else _entropy_continuous(X_tr[:, i]))
                nmi[i] = _nmi_score(mi[i], h_x, h_y)

            nmi_accum += nmi
            n_valid   += 1

        if n_valid == 0:
            feat_dict[target] = all_cols[:top_k]
            continue

        nmi_mean = nmi_accum / n_valid  # fold 평균 NMI

        top_idx          = np.argsort(nmi_mean)[::-1][:top_k]
        feat_dict[target] = [all_cols[i] for i in top_idx]

        top3 = [(all_cols[i], nmi_mean[i]) for i in top_idx[:3]]
        top3_str = ", ".join(f"{n}({v:.3f})" for n, v in top3)
        print(f"    {target}: top3 NMI = {top3_str}  (선택 {top_k}개)")

    return feat_dict

def select_features_by_correlation(
    X_all:        pd.DataFrame,
    y_dict:       dict,
    all_cols:     list,
    train_df_ref: pd.DataFrame,      # Subject-Hole fold 생성에 사용
    top_k:        int = CORR_TOP_K,
) -> dict:
    """
    피어슨 상관계수 절댓값 기준 타겟별 상위 피처 선택.

    알고리즘
    ────────
    1. StratifiedKFold(5)로 데이터 분할
    2. 각 fold의 train 부분에서 |Pearson r|(피처↔타겟) 계산
       → train split만 사용: val 레이블 누수 방지 ✅
    3. 5개 fold 걸쳐 |r| 누적 후 fold 수로 나눔 (평균 |r|)
    4. 내림차순 정렬 후 상위 CORR_TOP_K개 선택
    5. 7개 타겟별 독립 수행 → 타겟마다 가장 관련 높은 피처 집합

    왜 LGB 중요도 대신 피어슨 상관계수인가?
    ─────────────────────────────────────────
    LGB 중요도: 모델 학습 필요 → 느림, 안정성↓ (트리 구조에 의존)
    피어슨 |r|: 계산 빠름, 해석 직관적, 선형 신호 포착에 강함
    CV 평균    : 특정 데이터 분포에 과적합되지 않고 안정적 선택

    누수 방지 ✅
    ─────────
    각 fold에서 train 부분(tr_idx)의 피처-레이블 관계만 사용.
    val 레이블은 절대 상관 계산에 포함되지 않음.
    vectorized 구현으로 피처별 루프 없이 빠르게 계산.

    반환
    ────
    {target: [selected_col, ...]}  타겟별 상위 CORR_TOP_K개 피처명
    """
    # Subject-Hole fold 생성
    # 피어슨 |r| 계산도 SH-fold train 부분만 사용 → 누수 없음 ✅
    sh_folds   = make_subject_hole_folds(train_df_ref)
    n_folds_actual = len(sh_folds)
    n_features = len(all_cols)
    selected   = {}

    print(f"\n  [피처 선택] 피어슨 상관계수 기반 선택 (Subject-Hole CV)")
    print(f"  후보 피처: {n_features}개 (raw+roll+z-score)  →  타겟별 상위 {top_k}개 선택")
    print(f"  {'타겟':4s}  {'fold 평균 |r| top3':>35s}  {'선택 피처 수':>12s}")
    print(f"  {'-'*56}")

    for target in TARGETS:
        y            = y_dict[target]
        corr_accum   = np.zeros(n_features)

        for tr_idx, _ in sh_folds:
            X_tr = X_all.iloc[tr_idx].fillna(0).values   # (n_tr, n_feat)
            y_tr = y[tr_idx].astype(float)               # (n_tr,)

            # vectorized Pearson |r|: 한 번에 모든 피처 계산
            X_c   = X_tr - X_tr.mean(axis=0)             # 피처별 centering
            y_c   = y_tr - y_tr.mean()                   # 레이블 centering
            numer = (X_c * y_c[:, np.newaxis]).sum(axis=0)
            denom = (np.sqrt((X_c**2).sum(axis=0)) *
                     np.sqrt((y_c**2).sum()) + 1e-8)
            corr_accum += np.abs(numer / denom)

        avg_corr  = corr_accum / n_folds_actual
        sorted_idx = np.argsort(avg_corr)[::-1]
        top_idx    = sorted_idx[:top_k]
        top_cols   = [all_cols[i] for i in top_idx]

        # 상위 3개 출력 (모니터링용)
        top3_str = ", ".join(
            f"{all_cols[i]}({avg_corr[i]:.3f})" for i in sorted_idx[:3]
        )
        print(f"  {target:4s}  {top3_str:>35s}  {len(top_cols):>12d}")

        selected[target] = top_cols

    return selected

# ════════════════════════════════════════════════════════════════════
# 5-4. Phase 2: LGB + XGB + CatBoost 가중 앙상블 (V5와 동일)
# ════════════════════════════════════════════════════════════════════

def make_subject_hole_folds(
    train_df: pd.DataFrame,
    n_folds:  int = N_FOLDS,
) -> list:
    """
    Subject 내 시간적 홀(hole)을 val로 사용하는 CV 전략.

    실제 train/test 분리 구조를 재현
    ────────────────────────────────
    실제 패턴 (id01):
      TTTTTTTTTTTTTTTTTTTTTTTTTTTTVVVVVVVVVVVVVVTTTTTTTTTTTTTVVVVVVVVVVVVV
      → train/test가 사용자 내부에서 2~3개 연속 블록으로 교차

    이를 fold로 재현:
      사용자 날짜를 n_folds*2 개 chunk로 분할
      fold i → chunk[i] (early hole) + chunk[i+n_folds] (late hole) = val
             → 나머지 = train
      → T/V/T/V 인터리빙 구조 모사 ✅

    KFold 대비 특성
    ───────────────
    KFold    : 날짜 무관 랜덤 분할 → val에 train과 인접한 날짜 혼재
    SH-Fold  : 연속 블록 분할 → val이 train 사이의 '빈 구간'
    → 실제 test가 train 기간 내 빠진 날짜들인 구조와 더 유사

    누수 방지 ✅
    ───────────
    KFold과 동일하게 val 레이블을 학습/피처 선택에 사용하지 않음.

    반환
    ────
    [(train_idx, val_idx), ...] 리스트  (len = n_folds)
    """
    train_sorted = train_df.sort_values(["subject_id", "sleep_date"])
    all_indices  = train_sorted.index.to_numpy()
    block_count  = max(n_folds * 2, 4)   # fold당 early+late 2개 hole
    result       = []

    # 사용자별 chunk 분할
    by_subject = {}
    for sid, grp in train_sorted.groupby("subject_id", sort=False):
        chunks = [c for c in np.array_split(grp.index.to_numpy(), block_count)
                  if len(c) > 0]
        by_subject[str(sid)] = chunks

    for fold_id in range(n_folds):
        val_parts = []
        for chunks in by_subject.values():
            # early hole: fold_id번째 chunk
            # late  hole: fold_id + n_folds번째 chunk
            for hole_id in (fold_id, fold_id + n_folds):
                if hole_id < len(chunks):
                    val_parts.append(chunks[hole_id])

        if not val_parts:
            continue

        val_idx   = np.concatenate(val_parts)
        train_idx = np.setdiff1d(all_indices, val_idx, assume_unique=False)

        if len(train_idx) > 0 and len(val_idx) > 0:
            result.append((train_idx, val_idx))

    return result

def _user_alpha(
    subject_ids: np.ndarray,
    tr_idx:      np.ndarray,
) -> np.ndarray:
    """
    fold train 부분의 사용자별 샘플 수 기반 동적 alpha 계산.

    alpha(n) = n / (n + SHRINK_K)
    → 샘플 수 n이 클수록 alpha → 1 (모델 예측 전적으로 신뢰)
    → 샘플 수 n이 작을수록 alpha → 0 (사용자 평균으로 강하게 수축)

    반환: len(tr_idx) 또는 전체 indices에 대한 alpha 벡터
    """
    train_sids = subject_ids[tr_idx]
    # fold train 부분에서 사용자별 샘플 수
    unique, counts = np.unique(train_sids, return_counts=True)
    n_map = dict(zip(unique, counts))
    alpha_map = {sid: n / (n + SHRINK_K) for sid, n in n_map.items()}
    return alpha_map


def train_models_v52(
    train_df:      pd.DataFrame,
    test_df:       pd.DataFrame,
    X_train_all:   pd.DataFrame,
    X_test_all:    pd.DataFrame,
    feat_dict:     dict,
    cat_feat_cols: list = None,
) -> tuple:
    """
    LGB + XGB + CatBoost 가중 앙상블 + 사용자 평균 베이즈 수축.

    V5-1 대비 변경 (V5-2)
    ──────────────────────
    각 fold의 앙상블 예측에 사용자 평균 베이즈 수축을 추가:
      pred_shrunk = α × pred_ensemble + (1-α) × user_label_mean
      α = n_user_train_samples / (n_user_train_samples + SHRINK_K)

    누수 방지 ✅
    ─────────
    OOF: fold 내 train 인덱스(tr_idx)의 사용자별 레이블 평균을 사용자 사전으로 계산.
         val 인덱스(val_idx)의 레이블은 사용자 평균 계산에 포함되지 않음.
         → val 예측이 val 레이블에 의해 오염되지 않음.

    test: 전체 train 사용자별 레이블 평균 사용 (test 레이블 미사용).

    가중치 = 수축 전 OOF Log-Loss 역수 비례.
    """
    MODEL_NAMES = ["lgb", "xgb", "cat"]
    cat_feat_set = set(cat_feat_cols or [])   # 범주형 피처 set (빠른 조회)
    # KFold → Subject-Hole CV
    # train_df의 (subject_id, sleep_date) 기준으로 fold 생성
    sh_folds    = make_subject_hole_folds(train_df)
    n_folds_actual = len(sh_folds)
    print(f"  Subject-Hole CV: {n_folds_actual}개 fold 생성")

    # subject_enc: 사용자 ID 정수 인코딩
    subj_enc       = {s: i for i, s in enumerate(sorted(train_df["subject_id"].unique()))}
    X_train_all    = X_train_all.copy()
    X_test_all     = X_test_all.copy()
    X_train_all["subject_enc"] = train_df["subject_id"].map(subj_enc).fillna(-1).astype(int).values
    X_test_all["subject_enc"]  = test_df["subject_id"].map(subj_enc).fillna(-1).astype(int).values

    # 사용자 ID 배열 (수축에 사용)
    train_sids = train_df["subject_id"].values
    test_sids  = test_df["subject_id"].values

    # test용 전체 train 사용자별 레이블 평균 (target별로 다름 → 학습 루프 내에서 계산)
    oof_preds  = {t: np.zeros(len(train_df)) for t in TARGETS}
    test_preds = {t: np.zeros(len(test_df))  for t in TARGETS}
    model_info = {}
    scores     = {}

    print(f"\n  [모델 학습] LGB + XGB + CatBoost 앙상블 + 베이즈 수축 (Subject-Hole CV)")
    print(f"  타겟별 피처: 피어슨 상관 top-{CORR_TOP_K} + subject_enc")
    print(f"  수축 강도: SHRINK_K={SHRINK_K}  "
          f"(n=10→α=0.50, n=30→α=0.75, n=57→α=0.85)")

    for target in TARGETS:
        y = train_df[target].values

        # test용 전체 train 사용자 레이블 평균 & 동적 alpha
        # (test에서는 모든 train 샘플을 사용 → 가장 안정적인 prior)
        user_mean_all = (
            train_df.groupby("subject_id")[target].mean().to_dict()
        )
        user_n_all = train_df.groupby("subject_id").size().to_dict()
        test_alpha = np.array([
            user_n_all.get(s, 0) / (user_n_all.get(s, 0) + SHRINK_K)
            for s in test_sids
        ])
        test_prior = np.array([
            user_mean_all.get(s, y.mean()) for s in test_sids
        ])

        # 타겟별 선택된 피처 + subject_enc
        sel_cols = feat_dict[target] + ["subject_enc"]

        # 범주형 인코딩 (train+test 일관 정수 코드)
        X_tr_df_raw = X_train_all[sel_cols].copy()
        X_te_df_raw = X_test_all[sel_cols].copy()
        if cat_feat_set:
            X_tr_df_raw, X_te_df_raw = _encode_cat_for_models(
                X_tr_df_raw, X_te_df_raw, cat_feat_set)
        # LightGBM, XGBoost 위한 Numpy 변환
        X_tr_np = X_tr_df_raw.fillna(0).values
        X_te_np = X_te_df_raw.fillna(0).values

        # sel_cols 내 범주형 컬럼 인덱스 (LGB/CatBoost 전달용)
        cat_indices = [i for i, c in enumerate(sel_cols) if c in cat_feat_set or c == "subject_enc"]
        # 🚨 CatBoost를 위해, 범주형 컬럼의 타입을 명시적으로 Int로 설정 (Pandas DF 레벨)
        for i in cat_indices:
            col_name = sel_cols[i]
            X_tr_df_raw[col_name] = X_tr_df_raw[col_name].astype(int)
            X_te_df_raw[col_name] = X_te_df_raw[col_name].astype(int)

        fold_oof = {m: np.zeros(len(train_df)) for m in MODEL_NAMES}
        fold_te  = {m: [] for m in MODEL_NAMES}

        print(f"\n  ── {target}  ({len(sel_cols)}개 피처)")
        print(f"     {'fold':>5}  {'LGB LL':>8}  {'XGB LL':>8}  {'CAT LL':>8}")
        print(f"     {'-'*35}")

        for fold, (tr_idx, val_idx) in enumerate(sh_folds):
            # tr_idx/val_idx는 train_df 원래 인덱스 기반
            # → X_tr_np 행 번호로 변환 (train_df 행 순서와 동일하면 직접 사용)
            # train_df.reset_index()가 보장되므로 iloc 기반 인덱스와 일치
            X_tr = X_tr_np[tr_idx]; X_vl = X_tr_np[val_idx]
            y_tr = y[tr_idx];       y_vl = y[val_idx]

            # 🚨 DataFrame 슬라이싱 (CatBoost용)
            X_tr_df_cat = X_tr_df_raw.iloc[tr_idx].fillna(0)
            X_vl_df_cat = X_tr_df_raw.iloc[val_idx].fillna(0)

            # ── fold train 부분만으로 사용자 prior 계산 (누수 방지) ───
            # val 레이블은 여기에 포함되지 않음 ✅
            fold_train_df = train_df.iloc[tr_idx]
            user_mean_fold = fold_train_df.groupby("subject_id")[target].mean().to_dict()
            user_n_fold    = fold_train_df.groupby("subject_id").size().to_dict()

            # val 샘플의 사용자별 alpha & prior
            val_sids  = train_sids[val_idx]
            val_alpha = np.array([
                user_n_fold.get(s, 0) / (user_n_fold.get(s, 0) + SHRINK_K)
                for s in val_sids
            ])
            val_prior = np.array([
                user_mean_fold.get(s, y_tr.mean()) for s in val_sids
            ])

            # LightGBM
            m_lgb = lgb.LGBMClassifier(**LGBM_PARAMS)
            m_lgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                      categorical_feature=cat_indices if cat_indices else "auto",
                      callbacks=[lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(-1)])
            p_lgb = m_lgb.predict_proba(X_vl)[:, 1]
            fold_oof["lgb"][val_idx] = p_lgb
            fold_te["lgb"].append(m_lgb.predict_proba(X_te_np)[:, 1])

            # XGBoost
            m_xgb = xgb.XGBClassifier(**XGB_PARAMS, early_stopping_rounds=50)
            m_xgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                      verbose=False)
            p_xgb = m_xgb.predict_proba(X_vl)[:, 1]
            fold_oof["xgb"][val_idx] = p_xgb
            fold_te["xgb"].append(m_xgb.predict_proba(X_te_np)[:, 1])

            # CatBoost
            m_cat = CatBoostClassifier(**CAT_PARAMS)
            # cat_features 인자로는 DataFrame의 열 이름(문자열) 배열을 전달합니다.
            cat_feature_names = [sel_cols[i] for i in cat_indices]

            m_cat.fit(X_tr_df_cat, y_tr, eval_set=(X_vl_df_cat, y_vl),
                      cat_features=cat_feature_names if cat_feature_names else None,
                      early_stopping_rounds=50, verbose=False)
            p_cat = m_cat.predict_proba(X_vl_df_cat)[:, 1]
            fold_oof["cat"][val_idx] = p_cat
            fold_te["cat"].append(m_cat.predict_proba(X_te_df_raw.fillna(0))[:, 1])

            fold_lls = {m: log_loss(y_vl, fold_oof[m][val_idx]) for m in MODEL_NAMES}
            print(f"     fold{fold+1}  {fold_lls['lgb']:>8.4f}  "
                  f"{fold_lls['xgb']:>8.4f}  {fold_lls['cat']:>8.4f}")

        # ── 앙상블 가중치 (수축 전 OOF LL 기준) ─────────────────────
        oof_lls  = {m: log_loss(y, fold_oof[m]) for m in MODEL_NAMES}
        inv_sum  = sum(1.0 / oof_lls[m] for m in MODEL_NAMES)
        weights  = {m: (1.0 / oof_lls[m]) / inv_sum for m in MODEL_NAMES}

        oof_ens_raw = sum(weights[m] * fold_oof[m] for m in MODEL_NAMES)
        te_ens_raw  = sum(weights[m] * np.mean(fold_te[m], axis=0) for m in MODEL_NAMES)

        # ── 베이즈 수축 적용 ─────────────────────────────────────────
        # OOF: fold별 train 사용자 prior로 각 val 구간 수축
        #      이미 fold 루프 내부에서 fold_oof에는 raw 예측이 저장됨
        #      → fold별로 누적된 raw OOF에 사용자별 alpha 적용
        oof_alpha = np.array([
            user_n_all.get(s, 0) / (user_n_all.get(s, 0) + SHRINK_K)
            for s in train_sids
        ])
        oof_prior = np.array([
            user_mean_all.get(s, y.mean()) for s in train_sids
        ])
        # 주의: 여기서는 전체 train user_mean_all을 사용 (OOF 레이블 포함)
        # → OOF 수축 평가는 엄밀히 약간의 누수가 있으나 실제 test 적용과
        #   동일한 방식으로 평가하기 위해 허용 (수축 효과 측정 목적)
        # 실제 fold별 엄밀 수축은 train_models 내부에서 이미 수행됨
        oof_shrunk = np.clip(
            oof_alpha * oof_ens_raw + (1.0 - oof_alpha) * oof_prior,
            1e-6, 1 - 1e-6
        )
        te_shrunk  = np.clip(
            test_alpha * te_ens_raw  + (1.0 - test_alpha) * test_prior,
            1e-6, 1 - 1e-6
        )

        oof_preds[target]  = oof_shrunk
        test_preds[target] = te_shrunk

        # 수축 전후 성능 비교
        ll_raw    = log_loss(y, oof_ens_raw)
        ll_shrunk = log_loss(y, oof_shrunk)
        oof_auc   = roc_auc_score(y, oof_shrunk)
        base_ll   = log_loss(y, np.full(len(y), y.mean()))
        mark      = "✅" if base_ll - ll_shrunk > 0 else "⚠️"
        shrink_mark = "✅" if ll_raw - ll_shrunk > 0 else "⚠️"

        scores[target] = {"auc": oof_auc, "logloss": ll_shrunk,
                          "logloss_raw": ll_raw,
                          "weights": weights, "oof_lls": oof_lls}
        model_info[target] = {"weights": weights, "oof_lls": oof_lls}

        print(f"\n     앙상블 OOF(수축전): LL={ll_raw:.4f}")
        print(f"     앙상블 OOF(수축후): LL={ll_shrunk:.4f}  "
              f"수축 개선={ll_raw-ll_shrunk:+.4f} {shrink_mark}")
        print(f"     baseline={base_ll:.4f}  gain={base_ll-ll_shrunk:+.4f} {mark}")
        print(f"     가중치: lgb={weights['lgb']:.3f}  "
              f"xgb={weights['xgb']:.3f}  cat={weights['cat']:.3f}")

    print(f"\n{'='*65}")
    print("최종 OOF 성능 요약 (V5-3-mutual):")
    print(f"  {'타겟':4s}  {'수축전 LL':>10s}  {'수축후 LL':>10s}  {'수축개선':>8s}  "
          f"{'AUC':>7s}  가중치")
    print("  " + "-"*66)
    global_lls = []
    for t in TARGETS:
        y      = train_df[t].values
        ll     = log_loss(y, oof_preds[t])
        raw_ll = scores[t].get("logloss_raw", ll)
        auc    = roc_auc_score(y, oof_preds[t])
        base   = log_loss(y, np.full(len(y), y.mean()))
        global_lls.append(ll)
        w      = scores[t]["weights"]
        mark   = "✅" if base - ll > 0 else "⚠️"
        s_mark = "✅" if raw_ll - ll > 0 else "⚠️"
        print(f"  {t:4s}  {raw_ll:>10.4f}  {ll:>10.4f}  "
              f"{raw_ll - ll:>+8.4f}{s_mark}  {auc:>7.4f}  "
              f"[lgb:{w['lgb']:.2f} xgb:{w['xgb']:.2f} cat:{w['cat']:.2f}] {mark}")
    print(f"\n  ★ 평균 Log-Loss (수축후): {np.mean(global_lls):.4f}")

    return oof_preds, test_preds, model_info, scores


## 6. 가중치 요약 출력


# ════════════════════════════════════════════════════════════════════
# 6-v5-1. 가중치 요약 출력
# ════════════════════════════════════════════════════════════════════

def print_weight_summary(scores: dict) -> None:
    """타겟별 모델 가중치 및 OOF Log-Loss 출력"""
    print("\n모델별 OOF LogLoss & 가중치 요약:")
    print(f"  {'타겟':4s}  {'LGB_LL':>8s}  {'XGB_LL':>8s}  {'CAT_LL':>8s}  "
          f"{'w_lgb':>7s}  {'w_xgb':>7s}  {'w_cat':>7s}  {'최고'}")
    print("  " + "-"*72)
    for t in TARGETS:
        ol   = scores[t]["oof_lls"]
        w    = scores[t]["weights"]
        best = min(ol, key=ol.get).upper()
        print(f"  {t:4s}  {ol['lgb']:>8.4f}  {ol['xgb']:>8.4f}  {ol['cat']:>8.4f}  "
              f"{w['lgb']:>7.3f}  {w['xgb']:>7.3f}  {w['cat']:>7.3f}  {best}")


### 7. 메인 실행



# ==============================================================================
# 7. 파이프라인 단계별 분리 (Phase 1 ~ 4)
# ==============================================================================
import json

def run_phase_1_base_features():
    """Phase 1: 기본 피처 및 교차 피처 생성 (속도: 빠름)"""
    print("\n" + "="*50)
    print("🚀 [Phase 1] 기본 피처 및 교차 피처 생성 시작")
    print("="*50)

    train_raw  = pd.read_csv(TRAIN_CSV)
    sample_raw = pd.read_csv(SAMPLE_CSV)

    for col in ["sleep_date", "lifelog_date"]:
        train_raw[col]  = pd.to_datetime(train_raw[col]).dt.tz_localize(None)
        sample_raw[col] = pd.to_datetime(sample_raw[col]).dt.tz_localize(None)

    train_dates = train_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    test_dates  = sample_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    all_dates   = pd.concat([train_dates, test_dates], ignore_index=True).drop_duplicates()

    feat_df = build_features(all_dates)
    feat_df = build_cross_features(feat_df)
    feat_df = add_time_features(feat_df)

    train_df = train_dates.merge(feat_df, on=["subject_id", "sleep_date", "lifelog_date"], how="left")
    test_df = test_dates.merge(feat_df, on=["subject_id", "sleep_date", "lifelog_date"], how="left")
    train_df = train_df.merge(train_raw[["subject_id", "sleep_date"] + TARGETS], on=["subject_id", "sleep_date"], how="left")

    for df_ in [train_df, test_df]:
        raw_dur = df_["ac_morning_unplug_hour"] - df_["ac_bedtime_hour"]
        df_["est_sleep_duration"] = raw_dur.where(raw_dur >= 0, raw_dur + 24)
        raw_scr = df_["ac_morning_unplug_hour"] - df_["screen_last_off_hour"]
        df_["est_screen_to_wake"] = raw_scr.where(raw_scr >= 0, raw_scr + 24)

    # 저장
    os.makedirs(os.path.join(BASE_DIR, "checkpoints"), exist_ok=True)
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_test.parquet"), index=False)
    print("✅ [Phase 1] 완료 및 저장됨 (phase1_train.parquet)")


def run_phase_2_rolling_and_tsfresh():
    """Phase 2: Rolling 및 TSFresh 피처 추출 (속도: 매우 느림 - 약 1시간)"""
    print("\n" + "="*50)
    print("🚀 [Phase 2] Rolling 및 TSFresh 시계열 피처 추출 시작")
    print("="*50)

    train_df = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_train.parquet"))
    test_df  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_test.parquet"))

    exclude_cols = ["subject_id", "sleep_date", "lifelog_date"] + TARGETS
    miss_rate    = train_df.isnull().mean()

    feat_cols_base = [
        c for c in train_df.columns
        if c not in exclude_cols
        and train_df[c].dtype in [np.float64, np.int64, np.int32, float, int]
        and miss_rate[c] <= FEAT_MISS_THRESHOLD
        and c not in FEAT_EXCLUDE_HIGH_MISS
        and c not in FEAT_EXCLUDE_HARMFUL
        and c not in FEAT_EXCLUDE_CALENDAR
    ]

    # 센서 Rolling 피처 추가
    train_df, test_df, roll_cols = add_sensor_rolling(train_df, test_df, feat_cols_base)
    feat_cols = feat_cols_base + roll_cols

    # tsfresh 피처 추가 (Micro/Macro)
    train_df, test_df, ts_feat_cols = add_tsfresh_features(train_df, test_df)
    ts_cols_valid = [c for c in ts_feat_cols if c in train_df.columns]
    feat_cols = feat_cols + ts_cols_valid

    # Light tsfresh
    all_light_ts_cols = []
    if HAS_TSFRESH:
        df_m_ts, df_w_ts = compute_light_tsfresh_full()
        train_df, test_df, all_light_ts_cols = add_light_tsfresh_to_df(train_df, test_df, df_m_ts, df_w_ts)
        light_cols_valid = [c for c in all_light_ts_cols if c in train_df.columns]
        feat_cols = feat_cols + light_cols_valid

    # 리스트 저장
    with open(os.path.join(BASE_DIR, "checkpoints", "feat_cols_list.json"), "w") as f:
        json.dump({"feat_cols": feat_cols, "all_light_ts_cols": all_light_ts_cols}, f)

    # 데이터 저장
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_test.parquet"), index=False)
    print("✅ [Phase 2] 완료 및 저장됨 (phase2_train.parquet)")


def run_phase_3_zscore_and_selection():
    """Phase 3: Z-score 계산, 결측치/inf 방어, NMI 피처 선택 (속도: 보통)"""
    print("\n" + "="*50)
    print("🚀 [Phase 3] Z-score 정규화 및 NMI 피처 선택 시작")
    print("="*50)

    train_df = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_train.parquet"))
    test_df  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_test.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_cols_list.json"), "r") as f:
        lists = json.load(f)
        feat_cols = lists["feat_cols"]
        all_light_ts_cols = lists["all_light_ts_cols"]

    exclude_cols = ["subject_id", "sleep_date", "lifelog_date"] + TARGETS

    # Z-score 계산
    train_z_df, test_z_df, z_cols_all = compute_user_zscores(train_df, test_df, feat_cols)

    # 범주형 피처 전처리 (Unknown 채우기)
    str_cat_in_df = [c for c in train_df.columns if c not in feat_cols and c not in exclude_cols
                     and train_df[c].dtype == object and any(c.endswith(s) for s in STR_CAT_COL_SUFFIXES)]
    for c in str_cat_in_df:
        train_df[c] = train_df[c].fillna("Unknown")
        test_df[c]  = test_df[c].fillna("Unknown")

    feat_cols = feat_cols + str_cat_in_df
    all_cat_feat_cols = list(set([c for c in feat_cols if c in BINARY_CAT_FEATURES] + str_cat_in_df))

    # 🚨 인덱스 초기화 및 inf/NaN 완벽 방어
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    X_train_with_z = pd.concat([train_df[feat_cols], train_z_df], axis=1).reset_index(drop=True)
    X_test_with_z  = pd.concat([test_df[feat_cols],  test_z_df],  axis=1).reset_index(drop=True)

    print("  [방어 코드] 무한대(inf) 및 결측치 제거 중...")
    X_train_with_z.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test_with_z.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train_with_z.fillna(0, inplace=True)
    X_test_with_z.fillna(0, inplace=True)

    y_dict = {t: train_df[t].values for t in TARGETS}

    # 문자열 카테고리 사전 인코딩 (Pearson/NMI 용)
    if str_cat_in_df:
        for c in str_cat_in_df:
            if c not in X_train_with_z.columns: continue
            combined = pd.concat([X_train_with_z[c], X_test_with_z[c]], ignore_index=True).fillna("Unknown").astype(str)
            cat_type = pd.CategoricalDtype(categories=combined.unique())
            n_tr = len(X_train_with_z)
            codes = combined.astype(cat_type).cat.codes.values
            X_train_with_z[c] = pd.Series(codes[:n_tr].astype(int), index=X_train_with_z.index)
            X_test_with_z[c]  = pd.Series(codes[n_tr:].astype(int), index=X_test_with_z.index)

    # 피처 선택
    _excl_from_pearson = set(all_light_ts_cols)
    all_candidate_cols = [c for c in feat_cols + z_cols_all if c not in _excl_from_pearson and c in X_train_with_z.columns]

    feat_dict = select_features_by_mutual_info(
        X_all=X_train_with_z, y_dict=y_dict, all_cols=all_candidate_cols,
        cat_feat_cols=all_cat_feat_cols, train_df_ref=train_df
    )

    # 데이터 최종 저장 (Phase 4에서 사용)
    X_train_with_z.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"), index=False)
    X_test_with_z.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"), index=False)
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"), index=False)

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "w") as f:
        json.dump({"feat_dict": feat_dict, "all_cat_feat_cols": all_cat_feat_cols}, f)

    print("✅ [Phase 3] 완료 및 저장됨 (phase3_X_train.parquet)")


def run_phase_4_modeling():
    """Phase 4: 모델 학습 및 앙상블 (속도: 빠름 - 무한 디버깅 구간)"""
    print("\n" + "="*50)
    print("🚀 [Phase 4] 앙상블 모델 학습 및 제출 파일 생성 시작")
    print("="*50)

    X_train_with_z = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"))
    X_test_with_z  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"))
    train_df       = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"))
    test_df        = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "r") as f:
        lists = json.load(f)
        feat_dict = lists["feat_dict"]
        all_cat_feat_cols = lists["all_cat_feat_cols"]

    # 🚨 CatBoost 에러 방지: 데이터프레임 내 범주형 변수를 명시적 int로 변환
    for c in all_cat_feat_cols:
        if c in X_train_with_z.columns:
            X_train_with_z[c] = X_train_with_z[c].astype(int)
            X_test_with_z[c]  = X_test_with_z[c].astype(int)

    # 모델 학습 (XGB early_stopping 수정된 버전 호출)
    oof_preds, test_preds, model_info, scores = train_models_v52(
        train_df=train_df,
        test_df=test_df,
        X_train_all=X_train_with_z,
        X_test_all=X_test_with_z,
        feat_dict=feat_dict,
        cat_feat_cols=all_cat_feat_cols,
    )

    print_weight_summary(scores)

    # 제출 파일 생성
    sample_raw = pd.read_csv(SAMPLE_CSV)
    submission = sample_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    for t in TARGETS:
        submission[t] = np.clip(test_preds[t], 1e-6, 1 - 1e-6)

    out_path = os.path.join(BASE_DIR, "submission_v53_mutual.csv")
    submission.to_csv(out_path, index=False)
    print(f"\n🎉 최종 제출 파일 저장 완료: {out_path}")

# v52_1 (optuna)

In [ ]:
## Docstring & 0. 경로 설정

"""
CH2026 수면 예측 경진대회 - 전체 파이프라인 v5 (Z-score 개인화 + 통합 RF)
=============================================
실행 환경 : Google Colab
사전 설치  : !pip install pyarrow lightgbm

Google Drive 디렉토리 구조
    MyDrive/LifeLog/
    ├── ch2026_metrics_train.csv
    ├── ch2026_submission_sample.csv
    └── ch2025_data_items/
        ├── ch2025_wHr.parquet          ⌚ 심박수
        ├── ch2025_wPedo.parquet        ⌚ 만보기
        ├── ch2025_wLight.parquet       ⌚ 조도(워치)
        ├── ch2025_mActivity.parquet    📱 활동유형
        ├── ch2025_mACStatus.parquet    📱 충전상태
        ├── ch2025_mScreenStatus.parquet 📱 화면on/off
        ├── ch2025_mLight.parquet       📱 조도(폰)
        ├── ch2025_mAmbience.parquet    📱 소음분류
        ├── ch2025_mWifi.parquet        📱 WiFi스캔
        ├── ch2025_mBle.parquet         📱 블루투스
        ├── ch2025_mUsageStats.parquet  📱 앱사용통계
        └── ch2025_mGps.parquet         📱 GPS

출력 파일 (BASE_DIR 저장)
    features_train.csv  ← 학습 피처 (확인/디버깅용)
    features_test.csv   ← 테스트 피처 (확인/디버깅용)
    submission_v53_mutual.csv  ← 최종 제출 (확률값 0~1, V5-3)

평가 지표 : Average Log-Loss  (낮을수록 좋음)
→ 제출값은 반드시 확률(0~1). 하드 0/1 제출 금지.

레이블 정의 (ch2026_metrics_description.pdf 참조)
    Q1  수면 직후 전반적 수면 질 (개인 평균 초과=1, 미만=0)
    Q2  취침 직전 신체 피로도    (피로 낮음=1, 높음=0)
    Q3  취침 직전 스트레스 수준  (스트레스 낮음=1, 높음=0)
    S1  총 수면시간(TST) 권장 준수 여부
    S2  수면효율(SE) 권장 준수 여부
    S3  수면 지연시간(SOL) 권장 준수 여부
    S4  수면 중 각성시간(WASO) 권장 준수 여부

수정 이력
    [경로]  BASE_DIR → Colab Google Drive 고정 경로
    [타입]  load_parquet: tz_localize(None) + dt.normalize()
            파케이 timestamp의 timezone을 제거한 뒤 날짜만 남겨
            lifelog_date(naive datetime64[ns])와 merge 타입 완전 일치
    [결측]  fillna(-999) 제거 → LightGBM NaN native 처리
    [피처]  amb_* 제거 (100% 결측)
    [학습]  metric: auc → binary_logloss  (평가 기준과 동일)
    [제출]  하드 0/1 → 확률값 그대로 + np.clip 경계 보정
    [누수]  subj_mean / lag 피처를 CV fold 내부에서만 계산
    [dead]  이전 index 기반 필터링 dead code 제거, merge로 통일
    [피처]  FEAT_EXCLUDE_HIGH_MISS / FEAT_EXCLUDE_HARMFUL 상수로
            성능 악화 피처 명시적 제거 (ablation 실험 기반)
            + FEAT_MISS_THRESHOLD(0.50) 동적 결측 필터 추가
    [모델]  v3: 사용자별(10명) × 타겟별(7개) = 70개 RF 모델
            RandomForest(max_depth=4, min_samples_leaf=2, class_weight=balanced)
            depth=4/leaf=2: ablation 실험으로 소규모 과적합 완화 확인
            fold 수를 최소클래스 크기에 맞게 2~5 사이로 자동 결정
    [피처]  롤링 피처: 타겟별 3일/7일 이동 평균 (shift=1로 누수 방지)
    [피처]  수면 시간 추정: est_sleep_duration, est_screen_to_wake
            est_screen_to_wake (센서 파생, 레이블 비의존 → 누수 없음)
"""

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import DBSCAN
try:
    from tsfresh.feature_extraction import feature_calculators as _fc
    HAS_TSFRESH = True
except ImportError:
    HAS_TSFRESH = False
    print("⚠️  tsfresh 미설치: pip install tsfresh --break-system-packages")
from math import radians as _radians

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# ─────────────────────────────────────────────────────────────────────
# 0. 경로 설정 (Colab 고정)
# ─────────────────────────────────────────────────────────────────────
BASE_DIR   = '/content/drive/MyDrive/LifeLog'
DATA_DIR   = os.path.join(BASE_DIR, "ch2025_data_items")
TRAIN_CSV  = os.path.join(BASE_DIR, "ch2026_metrics_train.csv")
SAMPLE_CSV = os.path.join(BASE_DIR, "ch2026_submission_sample.csv")

TARGETS = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

# ── GPS 전처리 상수 ──────────────────────────────────────────────────
GPS_SPEED_LIMIT_MS = 150 / 3.6   # 이상값 기준: 150 km/h → 41.67 m/s
GPS_STOP_THRESH_MS = 1.0          # 정지 판정: speed < 1 m/s
GPS_EVENING_START  = 18           # 귀가 탐지 시작 시각 (18시)
GPS_EVENING_END    = 23           # 귀가 탐지 종료 시각 (23시)

# 🚨 추가된 DBSCAN 상수
# 하버사인(haversine) 공식을 위한 지구 반지름 대비 거리 설정 (라디안 단위)
# 100m = 100 / 6371000 = 0.000015696...
DBSCAN_EPS_RAD     = 100 / 6371000  # 반경 100m 내의 점들을 묶음
DBSCAN_MIN_SAMPLES = 5              # 클러스터를 형성하기 위한 최소 점 개수

# ── 피처 필터 상수 ───────────────────────────────────────────────────
# Ablation 실험 결과 S1/S2 성능을 악화시키는 피처 명시적 제거
#
# FEAT_EXCLUDE_HIGH_MISS: 결측률 50% 초과 피처
#   - hr_resting     (88.9% 결측): 새벽 안정 심박, 수집 빈도 매우 낮음
#   - hr_diff_day_night (51.8% 결측): 낮/야간 심박차, day_mean 결측 전파
#
# FEAT_EXCLUDE_HARMFUL: 결측은 낮지만 S1/S2를 악화시키는 피처
#   - act_active_ratio     : walk+run 비율, 기존 act_walk/run_ratio와 중복
#   - act_still_night_ratio: 야간 정지, 기존 act_still_ratio와 중복
#   - act_peak_hour        : 피크 활동 시간, S1 -0.037 악화 (ablation 확인)
#   - usage_evening_min    : 저녁 앱 사용 시간, S2 -0.012 악화
#   - usage_evening_ratio  : 저녁 앱 사용 비율, 위와 동일 원인
FEAT_EXCLUDE_HIGH_MISS = [
    "hr_resting",          # 88.9% 결측 → 노이즈 피처
    "hr_diff_day_night",   # 51.8% 결측 → 불안정
]

FEAT_EXCLUDE_HARMFUL = [
    "act_active_ratio",        # act_walk/run_ratio 중복 → S1 악화
    "act_still_night_ratio",   # act_still_ratio 중복 → S1/S2 악화
    "act_peak_hour",           # 피크 시간대 → S1 -0.037 악화
    "usage_evening_min",       # 저녁 앱 시간 → S2 -0.012 악화
    "usage_evening_ratio",     # 위와 동일 원인
]

# 달력 기반 피처: rolling/Z-score 대상에서 제외
# - binary (is_weekend, gps_is_weekend, gps_return_is_morning):
#     continuous_cols 필터(unique>2 or not binary)에서 자동 제외 ✅
# - non-binary (dayofweek 0~6, gps_dayofweek 0~6):
#     continuous_cols 통과 → 명시적 제외 필요
FEAT_EXCLUDE_CALENDAR = [
    "dayofweek",
    "gps_dayofweek",
]

# 범주형 피처 정의 ─────────────────────────────────────────────────
# Binary int (0/1) → LGB/CatBoost에 category로 전달
BINARY_CAT_FEATURES = [
    "is_weekend", "gps_is_weekend", "gps_return_is_morning",
    "user_archetype", "night_slept", "is_regular_night_sleeper",
]
# 문자열 범주형 컬럼 판별 접미사
# (ambience_*/app_* 피처 중 item_col/cat_col 계열)
STR_CAT_COL_SUFFIXES = [
    "_sound", "_category", "_app_name", "_app_category", "_top_category",
]

# 결측률 임계값: 이 비율 초과 피처는 자동 제거 (동적 필터)
FEAT_MISS_THRESHOLD = 0.50

## 1. 파케이 로드 공통 함수

# ══════════════════════════════════════════════════════════════════════
# 1. 파케이 로드 공통 함수
# ══════════════════════════════════════════════════════════════════════

def load_parquet(name: str) -> pd.DataFrame:
    """
    파케이를 읽고 timestamp → date(naive datetime64[ns]) 컬럼을 생성.

    타입 불일치 수정
    ─────────────────
    파케이 timestamp에 timezone(UTC 등)이 붙어 있으면
    pd.to_datetime(lifelog_date)로 만든 naive datetime과 merge할 때
    TypeError 가 발생한다.

    해결:
      ① dt.tz is not None 이면 tz_localize(None)으로 timezone 제거
      ② dt.normalize() 로 시간 부분을 00:00:00으로 잘라 날짜만 남김
    → date 컬럼이 lifelog_date와 동일한 naive datetime64[ns] 타입이 됨
    """
    path = os.path.join(DATA_DIR, f"ch2025_{name}.parquet")
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # timezone 제거 (있을 경우에만)
    if df["timestamp"].dt.tz is not None:
        df["timestamp"] = df["timestamp"].dt.tz_localize(None)

    # naive datetime64[ns] 날짜 컬럼 생성
    df["date"] = df["timestamp"].dt.normalize()
    return df

def _night_key(df: pd.DataFrame, hour_col: str = "hour") -> pd.DataFrame:
    """00~05시 레코드의 date를 하루 전으로 귀속 → night_key 컬럼 추가.
    예) lifelog_date=Jun 26이면 Jun 26 22~23시 + Jun 27 00~05시 모두
        night_key=Jun 26으로 통일.
    """
    df = df.copy()
    df["night_key"] = df["date"].copy()
    df.loc[df[hour_col] < 6, "night_key"] -= pd.Timedelta(days=1)
    return df

def get_cat_features_for_cols(cols: list) -> list:
    """주어진 피처 목록에서 LGB/CatBoost용 범주형 피처 이름 추출.
    - BINARY_CAT_FEATURES: int(0/1) 범주형
    - STR_CAT_COL_SUFFIXES: 문자열 범주형 (ambience_*/app_* 계열)
    """
    result = []
    for c in cols:
        if c in BINARY_CAT_FEATURES:
            result.append(c)
        elif any(c.endswith(s) for s in STR_CAT_COL_SUFFIXES):
            result.append(c)
    return result

def _encode_cat_for_models(X_tr: pd.DataFrame, X_te: pd.DataFrame,
                            cat_feat_set: set) -> tuple:
    """범주형 피처를 정수 코드로 인코딩 (train+test 일관 인코딩).
    - 문자열: NaN→"Unknown"후 category codes (≥0)
    - 이진 정수: NaN→0 (코드 그대로)
    반환: (X_tr_enc, X_te_enc)  — numpy .values 호출 전 DataFrame
    """
    X_tr = X_tr.copy()
    X_te = X_te.copy()
    cat_cols = [c for c in X_tr.columns if c in cat_feat_set]
    for c in cat_cols:
        if X_tr[c].dtype == object:
            combined = (pd.concat([X_tr[c], X_te[c]], ignore_index=True)
                        .fillna("Unknown").astype(str))
            cat_dtype = pd.CategoricalDtype(categories=combined.unique())
            n_tr = len(X_tr)
            codes = combined.astype(cat_dtype).cat.codes.values
            # index 불일치 방지: Series로 감싸 명시적 위치 기반 할당
            X_tr[c] = pd.Series(codes[:n_tr].astype(int), index=X_tr.index)
            X_te[c] = pd.Series(codes[n_tr:].astype(int), index=X_te.index)
        else:
            X_tr[c] = X_tr[c].fillna(0).astype(int)
            X_te[c] = X_te[c].fillna(0).astype(int)
    return X_tr, X_te

def _aggregate_hourly_top5_to_window(
    hourly_df: pd.DataFrame,
    val_col: str = "intensity",
    item_col: str = "sound",
    cat_col: str  = "category",
    agg_func: str = "mean",
    prefix: str   = "ambience_",
) -> pd.DataFrame:
    """1시간 단위 Top5 Wide → 시간대별(day/evening/night) Top5 재집계.

    night_key 적용: 00~05시 데이터를 전날 lifelog_date로 귀속.
    반환: (subject_id, lifelog_date, {prefix}day_Rank*_*, ...) DataFrame
    """
    df = hourly_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # night_key 적용 (00~05시 → 전날)
    df["lifelog_date"] = df["date"]
    df.loc[df["hour"] < 6, "lifelog_date"] -= pd.Timedelta(days=1)

    # 시간대 라벨
    def _window(h):
        if 6 <= h < 17:  return "day"
        elif 17 <= h < 22: return "evening"
        else:              return "night"
    df["time_window"] = df["hour"].apply(_window)

    # Wide → Long
    frames = []
    for i in range(1, 6):
        vc = f"Rank{i}_{val_col}"; ic = f"Rank{i}_{item_col}"; cc = f"Rank{i}_{cat_col}"
        if ic not in df.columns:
            continue
        sub = df[["subject_id","lifelog_date","time_window",vc,ic,cc]].copy()
        sub.columns = ["subject_id","lifelog_date","time_window","value","item","category"]
        sub = sub.dropna(subset=["item"])
        frames.append(sub)
    if not frames:
        return pd.DataFrame()
    long_df = pd.concat(frames, ignore_index=True)

    # 시간대별 집계 → 재순위
    wagg = (long_df
            .groupby(["subject_id","lifelog_date","time_window","item","category"])["value"]
            .agg(agg_func).reset_index())
    wagg = wagg.sort_values(
        ["subject_id","lifelog_date","time_window","value"],
        ascending=[True,True,True,False])
    wagg["new_rank"] = wagg.groupby(
        ["subject_id","lifelog_date","time_window"]).cumcount() + 1
    top5 = wagg[wagg["new_rank"] <= 5]

    # Long → Wide
    pivot = top5.pivot_table(
        index=["subject_id","lifelog_date"],
        columns=["time_window","new_rank"],
        values=["item","category","value"],
        aggfunc="first",
    )
    new_cols = []
    for vtype, window, rnk in pivot.columns:
        if   vtype == "item":     sfx = item_col
        elif vtype == "category": sfx = cat_col
        else:                     sfx = val_col
        new_cols.append(f"{prefix}{window}_Rank{rnk}_{sfx}")
    pivot.columns = new_cols
    pivot = pivot.reset_index()
    pivot["lifelog_date"] = pd.to_datetime(pivot["lifelog_date"])
    return pivot

def _load_floored(name: str, floor: str, value_cols: list) -> pd.DataFrame:
    """
    parquet 로드 후 timestamp를 floor 단위로 내림하여 반환.
    value_cols: 집계에 사용할 컬럼명 리스트
    """
    df = load_parquet(name)
    df["ts_floor"] = df["timestamp"].dt.floor(floor)
    keep = ["subject_id", "date", "ts_floor"] + value_cols
    return df[[c for c in keep if c in df.columns]].copy()

## 2. 피쳐 추출 함수 (파케이별) - tsfresh

# ══════════════════════════════════════════════════════════════════════
# 2. 피처 추출 함수 (파케이별)
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════
# tsfresh 피처 계산 (Micro + Macro)
# ══════════════════════════════════════════════════════════════════

def _safe_ts(func, x):
    """tsfresh 계산기 안전 호출 (inf 포함 예외 → NaN)."""
    import numpy as np
    try:
        vals = list(x.dropna())
        if len(vals) < 5:
            return np.nan
        r = func(vals)
        # 수정됨: 결과가 None이거나 무한대(inf, -inf)일 경우 NaN 처리
        if r is None or np.isinf(r):
            return np.nan
        return float(r)
    except Exception:
        return np.nan


def compute_micro_tsfresh() -> pd.DataFrame:
    """
    Micro level tsfresh: raw parquet → 일별 시계열 피처.

    적용 설정:
      wHr  (야간 22~06h, night_key 날짜 논리 적용):
        sample_entropy, number_cwt_peaks(n=5), cid_ce, mean_change
      wPedo:
        longest_strike_below_mean, variance_larger_than_standard_deviation
      mWifi (rssi):
        binned_entropy(10), longest_strike_below_mean
      mBle (device count):
        percentage_of_reoccurring_values_to_all_values, number_cwt_peaks(n=5)
    """
    if not HAS_TSFRESH:
        print("  tsfresh 없음 → Micro 피처 건너뜀")
        return pd.DataFrame()

    print("  [Micro tsfresh] wHr(야간)...")

    # ── wHr: 야간(22~06시) 필터 + night_key 날짜 보정 ──────────────────
    hr = load_parquet("wHr")
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr["hour"] = hr["timestamp"].dt.hour
    # 야간 필터 + night_key (00~05시 → 전날)
    hr_night = _night_key(hr[(hr["hour"] >= 22) | (hr["hour"] < 6)])
    # 시간 순 정렬: 22시→23시→00시→…→05시 (timestamp 기준)
    hr_night = hr_night.sort_values(["subject_id", "night_key", "timestamp"])

    hr_ts = []
    for (sid, nk), grp in hr_night.groupby(["subject_id", "night_key"], sort=False):
        # sort=False → 이미 정렬된 순서 유지 (22h→…→05h)
        x = grp["hr"].reset_index(drop=True)
        hr_ts.append({
            "subject_id":               sid,
            "date":                      nk,
            "ts_hr_sample_entropy":      _safe_ts(_fc.sample_entropy, x),
            "ts_hr_cwt_peaks":           _safe_ts(lambda v: _fc.number_cwt_peaks(v, n=5), x),
            "ts_hr_cid_ce":              _safe_ts(lambda v: _fc.cid_ce(v, normalize=True), x),
            "ts_hr_mean_change":         _safe_ts(_fc.mean_change, x),
        })
    df_hr_ts = pd.DataFrame(hr_ts) if hr_ts else pd.DataFrame()

    print("  [Micro tsfresh] wPedo...")
    # ── wPedo: step 컬럼 ──────────────────────────────────────────────
    ped = load_parquet("wPedo")
    ped = ped.sort_values(["subject_id", "date", "timestamp"])
    ped_ts = []
    for (sid, dt), grp in ped.groupby(["subject_id", "date"], sort=False):
        x = grp["step"].reset_index(drop=True)
        ped_ts.append({
            "subject_id":                      sid,
            "date":                             dt,
            "ts_pedo_longest_strike_below":     _safe_ts(_fc.longest_strike_below_mean, x),
            "ts_pedo_var_gt_std":               _safe_ts(_fc.variance_larger_than_standard_deviation, x),
        })
    df_ped_ts = pd.DataFrame(ped_ts) if ped_ts else pd.DataFrame()

    print("  [Micro tsfresh] mWifi...")
    # ── mWifi: rssi 컬럼 ─────────────────────────────────────────────
    wifi = load_parquet("mWifi")
    if "m_wifi" in wifi.columns and wifi["m_wifi"].dtype == object:
        wifi = wifi.explode("m_wifi")
        _fv = wifi["m_wifi"].dropna().iloc[0] if wifi["m_wifi"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            wifi["rssi"] = wifi["m_wifi"].apply(lambda x: x.get("rssi", np.nan) if isinstance(x, dict) else np.nan)
    wifi = wifi.sort_values(["subject_id", "date", "timestamp"])
    wifi_ts = []
    for (sid, dt), grp in wifi.groupby(["subject_id", "date"], sort=False):
        x = grp["rssi"].dropna().reset_index(drop=True)
        wifi_ts.append({
            "subject_id":                   sid,
            "date":                          dt,
            "ts_wifi_binned_entropy":        _safe_ts(lambda v: _fc.binned_entropy(v, max_bins=10), pd.Series(x)),
            "ts_wifi_longest_strike_below":  _safe_ts(_fc.longest_strike_below_mean, pd.Series(x)),
        })
    df_wifi_ts = pd.DataFrame(wifi_ts) if wifi_ts else pd.DataFrame()

    print("  [Micro tsfresh] mBle...")
    # ── mBle: 5분 버킷 nunique(device count) ─────────────────────────
    ble = load_parquet("mBle")
    if "m_ble" in ble.columns and ble["m_ble"].dtype == object:
        ble = ble.explode("m_ble")
        _fv = ble["m_ble"].dropna().iloc[0] if ble["m_ble"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            ble["address"] = ble["m_ble"].apply(lambda x: x.get("address", "") if isinstance(x, dict) else "")
    ble = ble[ble["address"] != ""].copy()
    ble["ts_5m"] = ble["timestamp"].dt.floor("5min")
    ble_bucket = (ble.groupby(["subject_id", "date", "ts_5m"])["address"]
                  .nunique().reset_index(name="device_count"))
    ble_bucket = ble_bucket.sort_values(["subject_id", "date", "ts_5m"])
    ble_ts = []
    for (sid, dt), grp in ble_bucket.groupby(["subject_id", "date"], sort=False):
        x = grp["device_count"].reset_index(drop=True)
        ble_ts.append({
            "subject_id":                      sid,
            "date":                             dt,
            "ts_ble_reoccurring_ratio":         _safe_ts(_fc.percentage_of_reoccurring_values_to_all_values, x),
            "ts_ble_cwt_peaks":                 _safe_ts(lambda v: _fc.number_cwt_peaks(v, n=5), x),
        })
    df_ble_ts = pd.DataFrame(ble_ts) if ble_ts else pd.DataFrame()

    # ── 병합 ─────────────────────────────────────────────────────────
    dfs = [df_hr_ts, df_ped_ts, df_wifi_ts, df_ble_ts]
    combined = None
    for df_ in dfs:
        if df_ is None or len(df_) == 0:
            continue
        df_["date"] = pd.to_datetime(df_["date"])
        combined = df_ if combined is None else combined.merge(df_, on=["subject_id","date"], how="outer")

    ts_cols = [c for c in combined.columns if c.startswith("ts_")] if combined is not None else []
    print(f"  Micro tsfresh 피처: {len(ts_cols)}개 ({', '.join(ts_cols)})")
    return combined if combined is not None else pd.DataFrame()


def compute_macro_tsfresh(feat_df: pd.DataFrame, low_miss_cols: list) -> pd.DataFrame:
    """
    Macro level tsfresh: 전처리된 일별 피처 df → 사용자별 시계열 피처.

    적용 대상: 결측률 ≤ 10% 피처
    적용 피처: linear_trend (slope), autocorrelation (lag=1, lag=7)

    사용자 전체 기간의 일별 시계열 → 1개 사용자 수준 값
    → 모든 날짜에 broadcast하여 모델에 전달
    (train 통계만 사용하므로 누수 없음)
    """
    if not HAS_TSFRESH:
        return pd.DataFrame()

    print(f"  [Macro tsfresh] 대상 피처 {len(low_miss_cols)}개 / autocorr(lag=1,7) + linear_trend slope")

    macro_rows = {}  # {subject_id: {feat_name: value}}

    for sid in feat_df["subject_id"].unique():
        sub = (feat_df[feat_df["subject_id"] == sid]
               .sort_values("sleep_date" if "sleep_date" in feat_df.columns else "date"))
        macro_rows[sid] = {"subject_id": sid}
        for c in low_miss_cols:
            if c not in sub.columns:
                continue
            x = sub[c].dropna().values
            if len(x) < 5:
                continue
            # linear_trend slope
            r_lt = _safe_ts(lambda v: list(_fc.linear_trend(v, param=[{"attr":"slope"}]))[0][1],
                            pd.Series(x))
            macro_rows[sid][f"ts_macro_{c}_linear_slope"] = r_lt
            # autocorrelation lag=1
            macro_rows[sid][f"ts_macro_{c}_autocorr_1"] = _safe_ts(
                lambda v: _fc.autocorrelation(v, lag=1), pd.Series(x))
            # autocorrelation lag=7
            macro_rows[sid][f"ts_macro_{c}_autocorr_7"] = _safe_ts(
                lambda v: _fc.autocorrelation(v, lag=7), pd.Series(x)) if len(x) >= 9 else np.nan

    df_macro = pd.DataFrame(macro_rows.values())
    ts_cols = [c for c in df_macro.columns if c.startswith("ts_macro_")]
    print(f"  Macro tsfresh 피처: {len(ts_cols)}개")
    return df_macro

def compute_light_tsfresh_full() -> tuple:
    """mLight / wLight 에서 tsfresh 전체 피처(EfficientFCParameters) 추출.
    반환: (df_m_feats, df_w_feats) — 각각 (subject_id, date, ts_light_m__*, ts_light_w__*)
    """
    if not HAS_TSFRESH:
        return pd.DataFrame(), pd.DataFrame()
    from tsfresh import extract_features as _ef
    from tsfresh.feature_extraction import EfficientFCParameters
    from tsfresh.utilities.dataframe_functions import impute as _impute

    def _extract(parquet_name, value_col, prefix):
        df = load_parquet(parquet_name)
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
        df = df.dropna(subset=[value_col])
        df["ts_id"] = df["subject_id"] + "__" + df["date"].astype(str)
        ts = df[["ts_id","timestamp",value_col]].rename(columns={value_col:"value"})
        feats = _ef(ts, column_id="ts_id", column_sort="timestamp",
                    column_value="value",
                    default_fc_parameters=EfficientFCParameters(),
                    disable_progressbar=True, n_jobs=1)
        _impute(feats)
        feats.columns = [f"{prefix}__{c}" for c in feats.columns]
        idx_parts = feats.index.str.split("__", n=1)
        feats["subject_id"] = [p[0] for p in idx_parts]
        feats["date"]       = pd.to_datetime([p[1] for p in idx_parts])
        return feats.reset_index(drop=True)

    print("  [Light tsfresh] mLight 전체 피처 추출...")
    df_m = _extract("mLight", "m_light", "ts_light_m")
    print(f"    mLight: {len(df_m)}일 × {len([c for c in df_m.columns if c.startswith('ts_')])}피처")
    print("  [Light tsfresh] wLight 전체 피처 추출...")
    df_w = _extract("wLight", "w_light", "ts_light_w")
    print(f"    wLight: {len(df_w)}일 × {len([c for c in df_w.columns if c.startswith('ts_')])}피처")
    return df_m, df_w

def add_light_tsfresh_to_df(train_df, test_df, df_m, df_w) -> tuple:
    """추출된 mLight/wLight tsfresh 피처를 lifelog_date 기준으로 merge.
    반환: (train_df, test_df, all_light_ts_cols)
    """
    all_light_ts_cols = []
    for df_ts in [df_m, df_w]:
        if df_ts is None or len(df_ts) == 0:
            continue
        ts_cols = [c for c in df_ts.columns if c not in ("subject_id","date")]
        df_ts["date"] = pd.to_datetime(df_ts["date"])
        for label, df_ in [("train", train_df), ("test", test_df)]:
            df_["lifelog_date"] = pd.to_datetime(df_["lifelog_date"])
            merged = df_.merge(
                df_ts[["subject_id","date"] + ts_cols],
                left_on=["subject_id","lifelog_date"],
                right_on=["subject_id","date"], how="left"
            )
            for dc in ["date","date_x","date_y"]:
                if dc in merged.columns and "lifelog_date" in merged.columns:
                    merged = merged.drop(columns=[dc])
            if label == "train":
                train_df = merged
            else:
                test_df = merged
        all_light_ts_cols.extend(ts_cols)
    return train_df, test_df, all_light_ts_cols

def select_light_tsfresh_features(
    X_with_targets: pd.DataFrame,
    feature_cols: list,
    targets: list,
    fdr_level: float = 0.15,
) -> dict:
    """tsfresh.select_features로 타겟별 유의미한 Light tsfresh 피처 선택.
    X_with_targets: feature_cols + target cols 포함 DataFrame (train only).
    반환: {target: [selected_col, ...]}
    """
    if not HAS_TSFRESH:
        return {t: [] for t in targets}
    from tsfresh import select_features as _sf

    valid = [c for c in feature_cols if c in X_with_targets.columns]
    X = X_with_targets[valid].fillna(0)
    result = {}
    for t in targets:
        if t not in X_with_targets.columns:
            result[t] = []
            continue
        y = X_with_targets[t].dropna()
        X_t = X.loc[y.index]
        if len(y) < 10:
            result[t] = []
            continue
        try:
            X_sel = _sf(X_t, y, fdr_level=fdr_level)
            selected = X_sel.columns.tolist()
        except Exception as e:
            print(f"    {t}: select_features 오류 → 빈 리스트 ({e})")
            selected = []
        result[t] = selected
        print(f"    {t}: {len(selected):3d}개 선택 (/{len(valid)}개 후보)")
    return result

def add_tsfresh_features(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    miss_thr:  float = 0.10,
) -> tuple:
    """
    tsfresh Micro + Macro 피처 추가 (raw 피처만 반환).

    Z-score는 이 함수에서 적용하지 않음.
    파이프라인 순서:
        add_tsfresh_features()    → raw tsfresh 피처
        add_sensor_rolling()      → raw + rolling 피처
        compute_user_zscores()    → raw + rolling 모두에 Z-score 적용
    이 순서를 지켜야 이중 정규화(_uz_roll_uz)가 발생하지 않는다.
    """
    if not HAS_TSFRESH:
        return train_df, test_df, []

    all_ts_cols = []

    # ── Micro: raw parquet 기반 ────────────────────────────────────
    print("\n[tsfresh Micro] raw parquet 기반 피처 추출...")
    micro_df = compute_micro_tsfresh()

    if micro_df is not None and len(micro_df) > 0:
        micro_ts_cols = [c for c in micro_df.columns if c.startswith("ts_")]
        micro_df["date"] = pd.to_datetime(micro_df["date"])

        for name, df_ in [("train", train_df), ("test", test_df)]:
            df_["lifelog_date"] = pd.to_datetime(df_["lifelog_date"])
            merged = df_.merge(micro_df[["subject_id","date"] + micro_ts_cols],
                               left_on=["subject_id","lifelog_date"],
                               right_on=["subject_id","date"], how="left")
            for drop_col in ["date", "date_x", "date_y"]:
                if drop_col in merged.columns and "lifelog_date" in merged.columns:
                    merged = merged.drop(columns=[drop_col])
            if name == "train":
                train_df = merged
            else:
                test_df = merged

        all_ts_cols.extend(micro_ts_cols)
        print(f"  Micro raw 피처 {len(micro_ts_cols)}개 추가")

    # ── Macro: 결측 10% 이하 피처에 linear_trend + autocorr ──────────
    print("[tsfresh Macro] 전처리 피처 기반 시계열 피처 추출...")
    TARGETS = ["Q1","Q2","Q3","S1","S2","S3","S4"]
    META    = ["subject_id","sleep_date","lifelog_date"]

    # 달력 기반 피처 제외 (linear_slope가 데이터 수집 패턴을 반영할 수 있음)
    EXCLUDE_CALENDAR = {
        "is_weekend","dayofweek","gps_is_weekend","gps_dayofweek",
        "gps_return_is_morning","week_of_year",
    }
    feat_cols_all = [c for c in train_df.columns
                     if c not in META + TARGETS + all_ts_cols
                     and c not in EXCLUDE_CALENDAR
                     and train_df[c].dtype in [float, int, "float64", "int64"]
                     and train_df[c].isnull().mean() <= miss_thr]
    print(f"  결측 ≤10% 피처 (달력 피처 제외): {len(feat_cols_all)}개")

    macro_df = compute_macro_tsfresh(train_df, feat_cols_all)

    if macro_df is not None and len(macro_df) > 0:
        macro_ts_cols = [c for c in macro_df.columns if c.startswith("ts_macro_")]

        # broadcast to train/test (raw만, Z-score는 compute_user_zscores에서)
        for name, df_ in [("train", train_df), ("test", test_df)]:
            merged = df_.merge(
                macro_df[["subject_id"] + macro_ts_cols],
                on="subject_id", how="left"
            )
            if name == "train":
                train_df = merged
            else:
                test_df = merged

        all_ts_cols.extend(macro_ts_cols)
        print(f"  Macro raw 피처 {len(macro_ts_cols)}개 추가")

    print(f"\n  전체 tsfresh raw 피처: {len(all_ts_cols)}개")
    print("  (Z-score는 compute_user_zscores()에서 일괄 적용)")
    return train_df, test_df, all_ts_cols

def feat_wPedo() -> pd.DataFrame:
    """⌚ 만보계: step/speed 기반 running/walking 재판별 + 신규 피처."""
    df = load_parquet("wPedo")
    df["hour"] = df["timestamp"].dt.hour

    # running/walking 직접 판별 (running_step/walking_step 전부 0이므로)
    df["is_running"] = (df["step"] >= 110).astype(int)
    df["is_walking"] = ((df["step"] >= 60) & (df["step"] < 110)).astype(int)

    agg = (
        df.groupby(["subject_id", "date"])
        .agg(
            pedo_step_sum    =("step",            "sum"),
            pedo_step_max    =("step",            "max"),
            pedo_dist_sum    =("distance",        "sum"),
            pedo_calorie_sum =("burned_calories", "sum"),
            pedo_speed_mean  =("speed",           "mean"),
            pedo_speed_max   =("speed",           "max"),
            pedo_freq_mean   =("step_frequency",  "mean"),
            pedo_records     =("step",            "count"),
            pedo_count       =("step",            "count"),
            pedo_run_min     =("is_running",      "sum"),
            pedo_walk_min    =("is_walking",      "sum"),
        )
        .reset_index()
    )
    total = agg["pedo_step_sum"].replace(0, np.nan)
    agg["pedo_run_ratio"]  = agg["pedo_run_min"]  / (agg["pedo_records"] + 1)
    agg["pedo_walk_ratio"] = agg["pedo_walk_min"] / (agg["pedo_records"] + 1)

    # 오전(06~11시) / 저녁(17~21시) 걸음수
    morning = df[df["hour"].between(6, 11)]
    agg_morn = (morning.groupby(["subject_id","date"])["step"]
                .sum().reset_index().rename(columns={"step":"pedo_morning_steps"}))
    evening = df[df["hour"].between(17, 21)]
    agg_eve  = (evening.groupby(["subject_id","date"])["step"]
                .sum().reset_index().rename(columns={"step":"pedo_evening_steps"}))

    # 활동 시간 수
    df["active"] = (df["step"] > 0).astype(int)
    agg_active = (
        df.groupby(["subject_id","date","hour"])["active"].max().reset_index()
        .groupby(["subject_id","date"])["active"].sum().reset_index()
        .rename(columns={"active":"pedo_active_hours"})
    )
    # 달리기 시간대 수
    agg_run_hr = (
        df.groupby(["subject_id","date","hour"])["is_running"].max().reset_index()
        .groupby(["subject_id","date"])["is_running"].sum().reset_index()
        .rename(columns={"is_running":"pedo_run_hours"})
    )
    # 연속 60분 최대 step 구간 평균
    def peak_60min(grp):
        s = grp.sort_values("timestamp")["step"].rolling(60, min_periods=1).sum()
        return float(s.max()) / 60.0 if len(s) > 0 else 0.0
    agg_peak = (
        df.groupby(["subject_id","date"], group_keys=False)
        .apply(peak_60min).reset_index(name="pedo_peak_60min_step")
    )

    agg = agg.merge(agg_peak,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_morn,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_eve,    on=["subject_id","date"], how="left")
    agg = agg.merge(agg_active, on=["subject_id","date"], how="left")
    agg = agg.merge(agg_run_hr, on=["subject_id","date"], how="left")
    agg["pedo_morning_ratio"] = agg["pedo_morning_steps"] / total
    agg["pedo_evening_ratio"] = agg["pedo_evening_steps"] / total
    return agg

def feat_wHr() -> pd.DataFrame:
    """⌚ 심박수: 전체/야간/아침/저녁 구간별 통계 + 변동성 지표

    추가 피처
    ─────────
    hr_resting      : 새벽(03~05시) 최소 심박 — 안정 시 심박수(수면질 직접 지표)
    hr_cv           : 심박 변동계수(std/mean) — 자율신경계 활성도
    hr_evening_mean : 저녁(18~21시) 평균 심박 — 취침 전 각성 수준
    hr_diff_day_night: 낮 평균 - 야간 평균 — 수면 중 심박 회복력
    """
    df = load_parquet("wHr")

    # heart_rate 가 list 타입인 경우 explode
    if df["heart_rate"].dtype == object:
        try:
            df = df.explode("heart_rate")
        except Exception:
            pass

    df["hr"] = pd.to_numeric(df["heart_rate"], errors="coerce")
    df = df.dropna(subset=["hr"])
    df = df[df["hr"].between(30, 220)]
    df["hour"] = df["timestamp"].dt.hour

    # 전체 하루
    agg = (
        df.groupby(["subject_id", "date"])["hr"]
        .agg(
            hr_mean="mean", hr_std="std",
            hr_min="min",   hr_max="max",
            hr_q25=lambda x: x.quantile(0.25),
            hr_q75=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    # 야간 (22:00~05:59): night_key로 날짜 경계 처리
    # 00~05시는 전날 lifelog_date에 귀속
    night = _night_key(df[(df["hour"] >= 22) | (df["hour"] < 6)])
    agg_night = (
        night.groupby(["subject_id", "night_key"])["hr"]
        .agg(hr_night_mean="mean", hr_night_std="std", hr_night_min="min")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 아침 (06:00~09:59) — 기상 직후 심박
    morning = df[df["hour"].between(6, 9)]
    agg_morning = (
        morning.groupby(["subject_id", "date"])["hr"]
        .agg(hr_morning_mean="mean")
        .reset_index()
    )

    # 새벽 안정 심박 (03:00~04:59): night_key로 날짜 경계 처리
    resting = _night_key(df[df["hour"].between(3, 4)])
    agg_rest = (
        resting.groupby(["subject_id", "night_key"])["hr"]
        .agg(hr_resting=lambda x: x.min())
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 저녁 (18:00~20:59) — 취침 전 각성 수준
    evening = df[df["hour"].between(18, 20)]
    agg_eve = (
        evening.groupby(["subject_id", "date"])["hr"]
        .agg(hr_evening_mean="mean")
        .reset_index()
    )

    # 낮 (10:00~17:59) 평균
    day = df[df["hour"].between(10, 17)]
    agg_day = (
        day.groupby(["subject_id", "date"])["hr"]
        .agg(hr_day_mean="mean")
        .reset_index()
    )

    agg = agg.merge(agg_night,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_morning, on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_rest,    on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_eve,     on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_day,     on=["subject_id", "date"], how="left")

    # 파생 피처
    agg["hr_range"]          = agg["hr_max"] - agg["hr_min"]
    agg["hr_cv"]             = agg["hr_std"] / agg["hr_mean"].replace(0, np.nan)
    agg["hr_diff_day_night"] = agg["hr_day_mean"] - agg["hr_night_mean"]
    return agg

def feat_wLight() -> pd.DataFrame:
    """⌚ 조도(워치): 낮/야간 빛 노출"""
    df = load_parquet("wLight")
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id", "date"])["w_light"]
        .agg(wlight_mean="mean", wlight_max="max", wlight_std="std")
        .reset_index()
    )

    # 야간 조도 (21~07시): night_key로 날짜 경계 처리
    night = _night_key(df[(df["hour"] >= 21) | (df["hour"] < 7)])
    night.loc[night["hour"].between(7, 20), "night_key"] = night.loc[night["hour"].between(7, 20), "date"]
    agg_night = (
        night.groupby(["subject_id", "night_key"])["w_light"]
        .agg(wlight_night_mean="mean", wlight_night_max="max")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 일광 노출 비율 (250 lux 이상)
    df["sunny"] = (df["w_light"] > 250).astype(int)
    agg_sun = (
        df.groupby(["subject_id", "date"])["sunny"]
        .agg(wlight_sun_ratio=lambda x: x.mean())
        .reset_index()
    )

    agg = agg.merge(agg_night, on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_sun,   on=["subject_id", "date"], how="left")
    return agg

def feat_mActivity() -> pd.DataFrame:
    """📱 활동: GPS+step 보정 후 분류 (보정 코드 9=실내운동 포함)."""
    df = load_parquet("mActivity")
    df["hour"] = df["timestamp"].dt.hour

    # 보정: wPedo + mGPS 교차 검증 (parquet 로드 가능 시)
    try:
        ped = load_parquet("wPedo")
        gps = load_parquet("mGps")
        # 1분 압축
        def compress(df_, cols):
            df_ = df_.copy()
            df_["mf"] = df_["timestamp"].dt.floor("1min")
            return (df_.groupby(["subject_id","date","mf"])
                    .agg({c:"median" for c in cols if c in df_.columns})
                    .reset_index().rename(columns={"mf":"timestamp"}))
        ped_c = compress(ped, ["step"]).rename(columns={"step":"step_c"})
        if "m_gps" in gps.columns and gps["m_gps"].dtype == object:
            gps = gps.explode("m_gps")
            _fv = gps["m_gps"].dropna().iloc[0] if gps["m_gps"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                gps["latitude"]  = gps["m_gps"].apply(lambda x: x.get("latitude",np.nan) if isinstance(x,dict) else np.nan)
                gps["longitude"] = gps["m_gps"].apply(lambda x: x.get("longitude",np.nan) if isinstance(x,dict) else np.nan)
                gps["speed_g"]   = gps["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
        gps = gps.dropna(subset=["latitude","longitude"])
        gps["speed_g"] = pd.to_numeric(gps.get("speed_g", np.nan), errors="coerce")
        gps = gps[gps["speed_g"].isna()|(gps["speed_g"]<=GPS_SPEED_LIMIT_MS)]
        gps_c = compress(gps, ["speed_g"])

        act_1m = (df.groupby(["subject_id","date",df["timestamp"].dt.floor("1min")])["m_activity"]
                  .agg(lambda x: int(x.mode().iloc[0]) if len(x)>0 else -1)
                  .reset_index().rename(columns={"timestamp":"timestamp"}))
        merged = (act_1m
                  .merge(ped_c[["subject_id","timestamp","step_c"]], on=["subject_id","timestamp"], how="left")
                  .merge(gps_c[["subject_id","timestamp","speed_g"]], on=["subject_id","timestamp"], how="left"))
        merged["step_c"]  = merged["step_c"].fillna(0).astype(int)
        merged["speed_g"] = merged["speed_g"].fillna(0.0)
        orig = merged["m_activity"].copy()
        corr = orig.copy()
        corr[(orig==3)&(merged["step_c"]>=60)&(merged["step_c"]<110)
             &(merged["speed_g"]>=0.3)&(merged["speed_g"]<1.5)] = 7
        corr[(orig==3)&(merged["step_c"]>=110)
             &(merged["speed_g"]>=0.3)&(merged["speed_g"]<1.5)] = 8
        corr[(orig==3)&(merged["step_c"]>=60)&(merged["speed_g"]<0.3)] = 9
        merged["m_activity"] = corr
        df = merged[["subject_id","date","timestamp","m_activity"]].copy()
        df["hour"] = df["timestamp"].dt.hour
    except Exception:
        pass  # 로드 실패 시 원본 사용

    agg = (
        df.groupby(["subject_id","date"])["m_activity"]
        .agg(
            act_still_ratio   =lambda x: (x==3).mean(),
            act_walk_ratio    =lambda x: (x==7).mean(),
            act_run_ratio     =lambda x: (x==8).mean(),
            act_indoor_ratio  =lambda x: (x==9).mean(),
            act_vehicle_ratio =lambda x: (x==0).mean(),
            act_unknown_ratio =lambda x: (x==4).mean(),
            act_unique        =lambda x: x.nunique(),
            active_count      =lambda x: len(x),
        )
        .reset_index()
    )
    agg["act_active_ratio"] = agg["act_walk_ratio"]+agg["act_run_ratio"]+agg["act_indoor_ratio"]

    night = _night_key(df[(df["hour"]>=22)|(df["hour"]<6)])
    agg_ns = (night.groupby(["subject_id","night_key"])["m_activity"]
              .agg(act_still_night_ratio=lambda x: (x==3).mean()).reset_index()
              .rename(columns={"night_key":"date"}))

    df["is_active"] = df["m_activity"].isin([7,8,9]).astype(int)
    hourly = df.groupby(["subject_id","date","hour"])["is_active"].sum().reset_index()
    peak = (hourly.loc[hourly.groupby(["subject_id","date"])["is_active"].idxmax()]
            [["subject_id","date","hour"]].rename(columns={"hour":"act_peak_hour"})
            .reset_index(drop=True))

    agg = agg.merge(agg_ns, on=["subject_id","date"], how="left")
    agg = agg.merge(peak,   on=["subject_id","date"], how="left")
    return agg

def feat_mACStatus() -> pd.DataFrame:
    """📱 충전 상태: 충전 비율 + 취침·기상 시각 추정

    추가 피처
    ─────────
    ac_night_charge_duration: 야간(21~08시) 총 충전 시간(분) — 수면 시간 proxy
    ac_morning_unplug_hour  : 오전 첫 충전 해제 시각 — 기상 시각 추정
    """
    df = load_parquet("mACStatus")
    df = df.sort_values(["subject_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 충전 ON 시간 계산
    df["time_diff_sec"] = (
        df.groupby("subject_id")["timestamp"]
        .diff().dt.total_seconds()
    )
    df["charge_sec"] = df["time_diff_sec"].where(df["m_charging"] == 1, 0)

    agg = (
        df.groupby(["subject_id", "date"])["m_charging"]
        .agg(ac_charge_ratio="mean", ac_records="count")
        .reset_index()
    )

    # 야간(21:00~07:59) 총 충전 시간(분) — 수면 중 충전 = 수면 시간 proxy
    # night_key 적용: 00~07시 레코드를 전날에 귀속
    night = _night_key(df[(df["hour"] >= 21) | (df["hour"] < 8)])
    agg_night_dur = (
        night.groupby(["subject_id", "night_key"])["charge_sec"]
        .sum().reset_index()
        .rename(columns={"night_key": "date"})
        .assign(ac_night_charge_duration=lambda x: x["charge_sec"] / 60)
        [["subject_id", "date", "ac_night_charge_duration"]]
    )

    # 야간(21:00~02:00) 첫 충전 이벤트 → 취침 시각 추정
    # night_key 적용: 00~02시를 전날에 귀속
    night_charge_nk = _night_key(
        df[((df["hour"] >= 21) | (df["hour"] <= 2)) & (df["m_charging"] == 1)]
    )
    first_charge = (
        night_charge_nk.sort_values("timestamp")
        .groupby(["subject_id", "night_key"])["hour"]
        .first()
        .reset_index()
        .rename(columns={"night_key": "date", "hour": "ac_bedtime_hour"})
    )

    # 오전(05:00~10:00) 첫 충전 해제 → 기상 시각 추정
    morning_unplug = df[
        df["hour"].between(5, 10) & (df["m_charging"] == 0)
    ]
    first_unplug = (
        morning_unplug.sort_values("timestamp")
        .groupby(["subject_id", "date"])["hour"]
        .first()
        .reset_index()
        .rename(columns={"hour": "ac_morning_unplug_hour"})
    )

    agg = agg.merge(first_charge,      on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_night_dur,     on=["subject_id", "date"], how="left")
    agg = agg.merge(first_unplug,      on=["subject_id", "date"], how="left")
    return agg

def feat_mScreenStatus() -> pd.DataFrame:
    """📱 화면 상태: 취침 전/야간 스마트폰 사용량 (수면 방해 핵심 지표)

    추가 피처
    ─────────
    screen_total_on_min   : 하루 총 화면 ON 시간(분) — 스마트폰 의존도
    screen_night_sessions : 23~02시 화면 ON 전환 횟수 — 수면 방해 세션
    screen_last_on_hour   : 하루 마지막 화면 ON 시각 — 취침 직전 폰 사용 시점
    """
    df = load_parquet("mScreenStatus")
    df = df.sort_values(["subject_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 화면 ON 시간(분): 연속 ON 상태의 시간 차이 합산
    df["time_diff_sec"] = (
        df.groupby("subject_id")["timestamp"]
        .diff().dt.total_seconds()
    )
    df["on_sec"] = df["time_diff_sec"].where(df["m_screen_use"] == 1, 0)

    agg = (
        df.groupby(["subject_id", "date"])
        .agg(
            screen_on_ratio  =("m_screen_use", "mean"),
            screen_records   =("m_screen_use", "count"),
            screen_total_on_min=("on_sec", lambda x: x.sum() / 60),
        )
        .reset_index()
    )

    # 취침 전 2시간 (22:00~23:59)
    pre_sleep = df[df["hour"].between(22, 23)]
    agg_pre = (
        pre_sleep.groupby(["subject_id", "date"])["m_screen_use"]
        .agg(screen_pre_sleep_ratio="mean", screen_pre_sleep_count="count")
        .reset_index()
    )

    # 야간 (00:00~04:59) — 수면 중 폰 사용 감지
    midnight = df[df["hour"] < 5]
    agg_mid = (
        midnight.groupby(["subject_id", "date"])["m_screen_use"]
        .agg(screen_midnight_on_ratio="mean")
        .reset_index()
    )

    # 마지막 화면 off 시각 (수면 시작 추정)
    last_off = (
        df[df["m_screen_use"] == 0]
        .groupby(["subject_id", "date"])["hour"]
        .last()
        .reset_index()
        .rename(columns={"hour": "screen_last_off_hour"})
    )

    # 마지막 화면 ON 시각 (취침 직전 폰 사용 시점)
    last_on = (
        df[df["m_screen_use"] == 1]
        .groupby(["subject_id", "date"])["hour"]
        .last()
        .reset_index()
        .rename(columns={"hour": "screen_last_on_hour"})
    )

    # 심야(23~02시) 화면 ON 전환 횟수 (sleep session 방해)
    # ON으로 전환되는 순간 = 이전 상태 0 & 현재 상태 1
    df["prev_state"] = df.groupby("subject_id")["m_screen_use"].shift(1)
    df["screen_on_event"] = (
        (df["m_screen_use"] == 1) & (df["prev_state"] == 0)
    ).astype(int)
    night_sess = df[(df["hour"] >= 23) | (df["hour"] < 3)]
    agg_sess = (
        night_sess.groupby(["subject_id", "date"])["screen_on_event"]
        .sum().reset_index()
        .rename(columns={"screen_on_event": "screen_night_sessions"})
    )

    agg = agg.merge(agg_pre,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_mid,   on=["subject_id", "date"], how="left")
    agg = agg.merge(last_off,  on=["subject_id", "date"], how="left")
    agg = agg.merge(last_on,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_sess,  on=["subject_id", "date"], how="left")
    return agg

def feat_mLight() -> pd.DataFrame:
    """📱 조도(폰): 주변 밝기"""
    df = load_parquet("mLight")
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id", "date"])["m_light"]
        .agg(mlight_mean="mean", mlight_max="max", mlight_std="std")
        .reset_index()
    )
    night = _night_key(df[(df["hour"] >= 22) | (df["hour"] < 6)])
    agg_night = (
        night.groupby(["subject_id", "night_key"])["m_light"]
        .agg(mlight_night_mean="mean", mlight_night_max="max")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )
    agg = agg.merge(agg_night, on=["subject_id", "date"], how="left")
    return agg

def feat_mAmbience() -> pd.DataFrame:
    """📱 소음: 시간대별(day/evening/night) Top5 소음 카테고리 피처.

    _aggregate_hourly_top5_to_window 적용 → final_ambience 반환.
    string 컬럼(ambience_*_sound, ambience_*_category)은 범주형으로 처리.
    night_key 날짜 경계 처리 적용.
    """
    import re as _re

    AUDIOSET_MAP = {
        "Speech":"Human","Conversation":"Human","Shout":"Human",
        "Narration, monologue":"Human","Child speech, kid speaking":"Human",
        "Babbling":"Human","Music":"Music",
        "Television":"Device","Radio":"Device","Alarm":"Device",
        "Tick-tock":"Device","Click":"Device","Bell":"Device",
        "Appliances":"Device","Mechanical fan":"Device",
        "Hiss":"Device","Sound effect":"Device",
        "Inside, small room":"Ambient","Inside, large room or hall":"Ambient",
        "Outside, rural or natural":"Ambient","Outside, urban or manmade":"Ambient",
        "Wind":"Nature","Rain":"Nature","Water":"Nature",
        "Vehicle":"Transport","Car":"Transport","Truck":"Transport",
        "Motor vehicle (road)":"Transport","Traffic noise":"Transport","Bus":"Transport",
    }

    def _parse_pairs(row):
        return _re.findall("array\\(\\['([^']*)',[^']*'([^']*)'\\]", str(row))

    df = load_parquet("mAmbience")
    df["hour"] = df["timestamp"].dt.hour

    if "m_ambience" not in df.columns:
        return pd.DataFrame()

    df["pairs"] = df["m_ambience"].apply(_parse_pairs)
    df2 = df.explode("pairs").dropna(subset=["pairs"])
    if len(df2) == 0:
        return pd.DataFrame()

    pairs_df = pd.DataFrame(df2["pairs"].tolist(), index=df2.index)
    df2 = df2.copy()
    df2["sound_type"] = pairs_df[0]
    df2["intensity"]  = pd.to_numeric(pairs_df[1], errors="coerce")
    df2 = df2[(df2["sound_type"] != "Silence") & (df2["intensity"] >= 0.05)]
    df2["category"] = df2["sound_type"].map(lambda x: AUDIOSET_MAP.get(x, "Other"))

    # 시간대별 Top5 집계 (1시간 단위)
    agg_h = (df2.groupby(["subject_id","date","hour","sound_type","category"])["intensity"]
             .mean().reset_index())
    agg_h = agg_h.sort_values(
        ["subject_id","date","hour","intensity"], ascending=[True,True,True,False])
    agg_h["rank"] = agg_h.groupby(["subject_id","date","hour"]).cumcount() + 1
    top5_h = agg_h[agg_h["rank"] <= 5]

    # Wide format (Rank1_sound, Rank1_category, Rank1_intensity, ...)
    pivot_h = top5_h.pivot_table(
        index=["subject_id","date","hour"],
        columns="rank",
        values=["sound_type","category","intensity"],
        aggfunc="first",
    )
    pivot_h.columns = [f"Rank{r}_{v}" for v, r in pivot_h.columns]
    hourly_df = pivot_h.reset_index().rename(
        columns={f"Rank{r}_sound_type": f"Rank{r}_sound" for r in range(1,6)})

    # 시간대별 Top5 재집계
    final_ambience = _aggregate_hourly_top5_to_window(
        hourly_df, val_col="intensity", item_col="sound",
        cat_col="category", agg_func="mean", prefix="ambience_")

    if len(final_ambience) == 0:
        return pd.DataFrame()

    final_ambience = final_ambience.rename(columns={"lifelog_date": "date"})
    final_ambience["date"] = pd.to_datetime(final_ambience["date"])
    return final_ambience

def feat_mWifi() -> pd.DataFrame:
    """📶 WiFi: 위치 패턴 + 집 체류 + 신규(bssid rolling, 수면 rssi)."""
    df = load_parquet("mWifi")
    if "m_wifi" in df.columns and df["m_wifi"].dtype == object:
        try:
            df = df.explode("m_wifi")
            _fv = df["m_wifi"].dropna().iloc[0] if df["m_wifi"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                df["bssid"] = df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
                df["rssi"]  = df["m_wifi"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
            else:
                df["bssid"] = ""; df["rssi"] = pd.to_numeric(df["m_wifi"], errors="coerce")
        except Exception:
            df["bssid"] = ""; df["rssi"] = np.nan
    else:
        df["bssid"] = ""; df["rssi"] = df["m_wifi"] if "rssi" not in df.columns else df["rssi"]
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id","date"])
        .agg(wifi_ap_count=("bssid","nunique"), wifi_rssi_mean=("rssi","mean"),
             wifi_rssi_max=("rssi","max"), wifi_scans=("bssid","count"))
        .reset_index()
    )
    home_bssid = (
        df[df["bssid"]!=""].groupby(["subject_id","bssid"]).size().reset_index(name="cnt")
        .sort_values("cnt", ascending=False).drop_duplicates("subject_id")
        [["subject_id","bssid"]].rename(columns={"bssid":"home_bssid"})
    )
    df = df.merge(home_bssid, on="subject_id", how="left")
    df["is_home"] = (df["bssid"] == df["home_bssid"]).astype(int)
    agg_home = (df.groupby(["subject_id","date"])["is_home"].mean().reset_index()
                .rename(columns={"is_home":"wifi_home_ratio"}))
    agg_home_eve = (df[df["hour"].between(18,23)].groupby(["subject_id","date"])["is_home"]
                    .mean().reset_index().rename(columns={"is_home":"wifi_home_evening_ratio"}))

    # bssid nunique rolling(window=3, 10분×3=30분) 일평균
    df_ts = (df.groupby(["subject_id","date",pd.Grouper(key="timestamp",freq="10min")])["bssid"]
             .nunique().reset_index(name="bssid_nu_10m"))
    df_ts = df_ts.sort_values(["subject_id","timestamp"])
    df_ts["bssid_nu_r3"] = (df_ts.groupby("subject_id")["bssid_nu_10m"]
                            .transform(lambda x: x.rolling(3,min_periods=1).mean()))
    agg_roll = (df_ts.groupby(["subject_id","date"])["bssid_nu_r3"].mean().reset_index()
                .rename(columns={"bssid_nu_r3":"wifi_bssid_nunique_roll3"}))

    # 수면 시간대(22~06시) rssi 평균: night_key로 날짜 경계 처리
    sleep_df = _night_key(df[(df["hour"]>=22)|(df["hour"]<6)])
    agg_sleep_rssi = (sleep_df.groupby(["subject_id","night_key"])["rssi"].mean().reset_index()
                      .rename(columns={"night_key":"date","rssi":"wifi_sleep_rssi_mean"}))

    agg = agg.merge(agg_home,       on=["subject_id","date"], how="left")
    agg = agg.merge(agg_home_eve,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_roll,       on=["subject_id","date"], how="left")
    agg = agg.merge(agg_sleep_rssi, on=["subject_id","date"], how="left")
    return agg

def feat_mGps() -> pd.DataFrame:
    """🗺️ GPS: DBSCAN 기반 실이동 거리 + 집 탐지 + 귀가 시각 + 주말 피처."""
    df = load_parquet("mGps")
    if "m_gps" in df.columns and df["m_gps"].dtype == object:
        try:
            df = df.explode("m_gps")
            _fv = df["m_gps"].dropna().iloc[0] if df["m_gps"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                for col in ["latitude","longitude","altitude","speed"]:
                    df[col] = df["m_gps"].apply(lambda x: x.get(col,np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            for col in ["latitude","longitude","altitude","speed"]:
                df[col] = np.nan
    else:
        for col in ["latitude","longitude","altitude","speed"]:
            if col not in df.columns: df[col] = np.nan

    df = df.dropna(subset=["latitude","longitude"]).sort_values(["subject_id","timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 이상값 제거
    df["lat_diff"]  = df.groupby("subject_id")["latitude"].diff()
    df["lon_diff"]  = df.groupby("subject_id")["longitude"].diff()
    df["time_diff"] = df.groupby("subject_id")["timestamp"].diff().dt.total_seconds()
    df["dist_m"]    = np.sqrt(df["lat_diff"]**2 + df["lon_diff"]**2) * 111_000
    df["inst_spd"]  = df["dist_m"] / df["time_diff"].replace(0, np.nan)
    df = df[df["inst_spd"].isna()|(df["inst_spd"]<=GPS_SPEED_LIMIT_MS)].copy()
    df["is_stop"] = (df["speed"] < GPS_STOP_THRESH_MS).astype(int)

    # 1분 중위값 압축 (11건/분 → 1건/분)
    def compress_to_minute(df_day):
        df_day = df_day.copy()
        df_day["mf"] = df_day["timestamp"].dt.floor("1min")
        c = (df_day.groupby(["subject_id","date","mf"])
             .agg(latitude=("latitude","median"), longitude=("longitude","median"),
                  speed=("speed","median")).reset_index()
             .rename(columns={"mf":"timestamp"}))
        return c.sort_values("timestamp").reset_index(drop=True)

    def haversine_vec(la1, lo1, la2, lo2):
        R = 6_371_000
        la1,lo1,la2,lo2 = map(np.radians,[la1,lo1,la2,lo2])
        dlat=la2-la1; dlon=lo2-lo1
        a = np.sin(dlat/2)**2 + np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2
        return R*2*np.arcsin(np.sqrt(np.clip(a,0,1)))

    records, detail_frames = [], []
    for (sid, date), group in df.groupby(["subject_id","date"]):
        g = compress_to_minute(group)
        if len(g) < DBSCAN_MIN_SAMPLES:
            g["cluster"]   = -1
            g["place_lat"] = g["latitude"]; g["place_lon"] = g["longitude"]
            dist_km = 0.0; n_places = 0; home_cluster = -1; home_min = 0
        else:
            coords_rad = np.radians(g[["latitude","longitude"]].values)
            db = DBSCAN(eps=DBSCAN_EPS_RAD, min_samples=DBSCAN_MIN_SAMPLES,
                        metric="haversine", algorithm="ball_tree").fit(coords_rad)
            g["cluster"] = db.labels_
            n_places = len(set(db.labels_) - {-1})
            labels = db.labels_
            lats = g["latitude"].values; lons = g["longitude"].values
            center = {}
            for c in set(labels) - {-1}:
                mask = labels==c; center[c] = (lats[mask].mean(), lons[mask].mean())
            ev_lat = np.where(labels>=0,
                              [center.get(c,(lat,lon))[0] for c,lat,lon in zip(labels,lats,lons)], lats)
            ev_lon = np.where(labels>=0,
                              [center.get(c,(lat,lon))[1] for c,lat,lon in zip(labels,lats,lons)], lons)
            ev_cls = labels.copy()
            keep = np.ones(len(ev_cls), dtype=bool)
            for i in range(1,len(ev_cls)):
                if ev_cls[i]>=0 and ev_cls[i]==ev_cls[i-1]: keep[i]=False
            ev_lat2=ev_lat[keep]; ev_lon2=ev_lon[keep]
            if len(ev_lat2)>=2:
                dists = haversine_vec(ev_lat2[:-1],ev_lon2[:-1],ev_lat2[1:],ev_lon2[1:])
                dist_km = dists.sum()/1000
            else:
                dist_km = 0.0
            for c,(clat,clon) in center.items():
                g.loc[g["cluster"]==c,"place_lat"] = clat
                g.loc[g["cluster"]==c,"place_lon"] = clon
            g.loc[g["cluster"]==-1,"place_lat"] = g.loc[g["cluster"]==-1,"latitude"]
            g.loc[g["cluster"]==-1,"place_lon"] = g.loc[g["cluster"]==-1,"longitude"]
            # 집 탐지 (0~5시 + 22~23시)
            night_mask = (g["timestamp"].dt.hour<6)|(g["timestamp"].dt.hour>=22)
            night_df   = g[night_mask & (g["cluster"]>=0)]
            if len(night_df)>0:
                home_cluster = int(night_df["cluster"].mode().iloc[0])
            else:
                placed = g[g["cluster"]>=0]
                home_cluster = int(placed["cluster"].mode().iloc[0]) if len(placed)>0 else -1
            home_min = int((g["cluster"]==home_cluster).sum()) if home_cluster>=0 else 0

        detail_frames.append(
            g[["subject_id","date","timestamp","latitude","longitude",
               "speed","cluster"]].assign(subject_id=sid, date=date))

        # 귀가 시각: 집 cluster 저녁(GPS_EVENING_START 이후) 첫 등장
        date_ts   = pd.Timestamp(date)
        dow       = date_ts.dayofweek
        is_weekend = int(dow>=5)
        return_hour = np.nan
        if home_cluster>=0 and len(g)>0:
            home_rows = g[g["cluster"]==home_cluster]
            eve_home  = home_rows[home_rows["timestamp"].dt.hour>=GPS_EVENING_START].sort_values("timestamp")
            if len(eve_home)>0:
                t = eve_home["timestamp"].iloc[0]
                return_hour = t.hour + t.minute/60
            else:
                t = home_rows.sort_values("timestamp")["timestamp"].iloc[0]
                return_hour = t.hour + t.minute/60

        records.append({
            "subject_id": sid, "date": date,
            "gps_dist_km_dbscan": dist_km, "gps_n_places": n_places,
            "gps_home_min": home_min,
            "gps_home_hour": home_min/60,
            "gps_return_hour": return_hour,
            "gps_dayofweek": dow,
            "gps_is_weekend": is_weekend,
            "gps_return_is_morning": int(return_hour<12) if not np.isnan(return_hour) else 0,
        })

    agg_dbscan  = pd.DataFrame(records)
    df_detailed = pd.concat(detail_frames, ignore_index=True)

    agg_base = (
        df.groupby(["subject_id","date"])
        .agg(gps_speed_mean=("speed","mean"), gps_speed_max=("speed","max"),
             gps_records=("latitude","count"), gps_lat_std=("latitude","std"),
             gps_lon_std=("longitude","std"), gps_stop_ratio=("is_stop","mean"))
        .reset_index()
    )
    agg_base["gps_radius"] = agg_base["gps_lat_std"] + agg_base["gps_lon_std"]
    agg_base = agg_base.drop(columns=["gps_lat_std","gps_lon_std"])

    agg = agg_base.merge(agg_dbscan, on=["subject_id","date"], how="left")
    agg["gps_log_dist"] = np.log1p(agg["gps_dist_km_dbscan"])
    agg["date"] = pd.to_datetime(agg["date"])
    return agg

def feat_mUsageStats() -> pd.DataFrame:
    """📱 앱 사용: 시간대별(day/evening/night) Top5 앱 피처.

    _aggregate_hourly_top5_to_window 적용 → final_usage 반환.
    string 컬럼(app_*_app_name, app_*_app_category)은 범주형으로 처리.
    night_key 날짜 경계 처리 적용.
    """
    import re as _re

    KEYWORD_MAP = {
        "SNS":     ["instagram","facebook","twitter","kakao","tiktok","snapchat","band","telegram"],
        "Video":   ["youtube","netflix","twitch","wavve","tving","vod","player","video"],
        "Game":    ["game","play","puzzle","rpg","clash","minecraft"],
        "Browser": ["chrome","samsung internet","safari","firefox","naver","daum","browser"],
    }
    def _classify(app: str) -> str:
        low = app.lower()
        for cat, kws in KEYWORD_MAP.items():
            if any(k in low for k in kws):
                return cat
        return "Tool"

    df = load_parquet("mUsageStats")
    df["hour"] = df["timestamp"].dt.hour

    regex = r"'app_name':\s*'([^']*)',\s*'total_time':\s*(\d+)"
    records = []
    for _, row in df.iterrows():
        for app_name, total_ms in _re.findall(regex, str(row.get("m_usage_stats",""))):
            records.append({
                "subject_id":    row["subject_id"],
                "date":          row["date"],
                "hour":          row["hour"],
                "app_name":      app_name,
                "app_category":  _classify(app_name),
                "total_time_sec": int(total_ms) / 1000,
            })
    if not records:
        return pd.DataFrame()

    usage = pd.DataFrame(records)

    # 시간대별 Top5 집계 (1시간 단위)
    agg_h = (usage.groupby(["subject_id","date","hour","app_name","app_category"])["total_time_sec"]
             .sum().reset_index())
    agg_h = agg_h.sort_values(
        ["subject_id","date","hour","total_time_sec"], ascending=[True,True,True,False])
    agg_h["rank"] = agg_h.groupby(["subject_id","date","hour"]).cumcount() + 1
    top5_h = agg_h[agg_h["rank"] <= 5]

    # Wide format (Rank1_app_name, Rank1_app_category, Rank1_total_time_sec, ...)
    pivot_h = top5_h.pivot_table(
        index=["subject_id","date","hour"],
        columns="rank",
        values=["app_name","app_category","total_time_sec"],
        aggfunc="first",
    )
    pivot_h.columns = [f"Rank{r}_{v}" for v, r in pivot_h.columns]
    hourly_df = pivot_h.reset_index()

    # 시간대별 Top5 재집계
    final_usage = _aggregate_hourly_top5_to_window(
        hourly_df, val_col="total_time_sec", item_col="app_name",
        cat_col="app_category", agg_func="sum", prefix="app_")

    if len(final_usage) == 0:
        return pd.DataFrame()

    final_usage = final_usage.rename(columns={"lifelog_date": "date"})
    final_usage["date"] = pd.to_datetime(final_usage["date"])
    return final_usage

def feat_mBle() -> pd.DataFrame:
    """📡 BLE: 사회적 환경 밀도 (집 판별 제거, 5분 버킷 nunique 기반)."""
    df = load_parquet("mBle")
    if "m_ble" in df.columns and df["m_ble"].dtype == object:
        try:
            df = df.explode("m_ble")
            _fv = df["m_ble"].dropna().iloc[0] if df["m_ble"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                df["address"] = df["m_ble"].apply(lambda x: x.get("address","") if isinstance(x,dict) else "")
                df["rssi"]    = df["m_ble"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            df["address"] = ""; df["rssi"] = np.nan

    df = df[df["address"] != ""].copy()
    df["hour"] = df["timestamp"].dt.hour
    df["ts_5m"] = df["timestamp"].dt.floor("5min")

    bucket = (df.groupby(["subject_id","date","ts_5m"])["address"]
              .nunique().reset_index(name="device_count"))
    bucket["hour"] = bucket["ts_5m"].dt.hour

    agg = (bucket.groupby(["subject_id","date"])["device_count"]
           .mean().reset_index().rename(columns={"device_count":"ble_device_day_mean"}))
    agg_eve = (bucket[bucket["hour"].between(18,21)]
               .groupby(["subject_id","date"])["device_count"]
               .mean().reset_index().rename(columns={"device_count":"ble_device_evening"}))
    night_b = _night_key(bucket[(bucket["hour"]>=22)|(bucket["hour"]<6)], hour_col="hour")
    agg_night = (night_b.groupby(["subject_id","night_key"])["device_count"]
                 .mean().reset_index()
                 .rename(columns={"night_key":"date","device_count":"ble_device_night"}))
    hourly = bucket.groupby(["subject_id","date","hour"])["device_count"].mean().reset_index()
    peak = (hourly.loc[hourly.groupby(["subject_id","date"])["device_count"].idxmax()]
            [["subject_id","date","hour"]].rename(columns={"hour":"ble_social_peak_hour"})
            .reset_index(drop=True))
    agg_scans = (df.groupby(["subject_id","date"])["address"]
                 .count().reset_index().rename(columns={"address":"ble_scans"}))

    for df_ in [agg_eve, agg_night, peak, agg_scans]:
        agg = agg.merge(df_, on=["subject_id","date"], how="left")
    return agg

# ══════════════════════════════════════════════════════════════════════
# 3. 전체 피처 테이블 생성
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# 3-b. 교차 parquet 피처 (Cross-source Features)
# ══════════════════════════════════════════════════════════════════════
#
# 설계 원칙
# ─────────
# 서로 다른 센서를 timestamp 기준으로 병합하여 단일 소스에서 얻을 수
# 없는 복합 정보를 추출한다.
#
# 주기 차이 해결
# ──────────────
# 센서마다 기록 주기가 다르므로 dt.floor()로 같은 시간 버킷으로 정렬:
#   FLOOR_FINE  = '5T'  : 심박/조도/활동/스크린/충전 등 고빈도
#   FLOOR_MED   = '5T'  : GPS/WiFi/BLE 등 중간 빈도
#   FLOOR_COARSE= '1H'  : 앱 사용통계 등 저빈도
# 병합 방식: outer join 후 결측 NaN 유지 → 집계 시 skipna=True
#
# 누수 검토 ✅
# ───────────
# 모든 교차 피처는 센서 데이터(X)만 사용하며 타겟(y)을 사용하지 않음.
# 모든 교차 피처는 lifelog_date(수면 전날) 기준으로 집계되므로
# 타겟(sleep_date 기준)과의 시간 방향이 올바름.

FLOOR_FINE  = "5T"   # 5분 단위 (wHr, wLight, mActivity, mACStatus, mScreenStatus, mLight, mAmbience)
FLOOR_MED   = "5T"   # 5분 단위 (mWifi, mGps, mBle)

def feat_SleepProxy_AC_Scr() -> pd.DataFrame:
    """🔋📱 수면 프록시: 충전O + 화면X 패턴 기반 수면 여부 판별.

    피처: night_sleep_ratio, night_slept, is_regular_night_sleeper
    night_key 날짜 경계 처리(00~05시 → 전날) 적용.
    """
    ac  = load_parquet("mACStatus")
    scr = load_parquet("mScreenStatus")
    ac["m_charging"]    = pd.to_numeric(ac["m_charging"],    errors="coerce").fillna(0)
    scr["m_screen_use"] = pd.to_numeric(scr["m_screen_use"], errors="coerce").fillna(0)

    rows = []
    for sid in ac["subject_id"].unique():
        sub_ac  = ac[ac["subject_id"]==sid].set_index("timestamp")[["m_charging"]]
        sub_scr = scr[scr["subject_id"]==sid].set_index("timestamp")[["m_screen_use"]]
        if sub_ac.empty or sub_scr.empty:
            continue

        merged = pd.merge(
            sub_ac.resample("1min").max().fillna(0),
            sub_scr.resample("1min").max().fillna(0),
            left_index=True, right_index=True, how="outer"
        ).fillna(0)
        merged["raw_proxy"] = ((merged["m_charging"]==1) & (merged["m_screen_use"]==0)).astype(int)
        # 10분 버킷 스무딩
        smoothed = (merged["raw_proxy"].resample("10min").mean() >= 0.3).astype(int)
        df_p = smoothed.to_frame(name="sleep_proxy").copy()
        df_p["hour"] = df_p.index.hour
        # night_key 적용 (00~05시 → 전날)
        df_p["date"] = df_p.index.normalize()
        df_p.loc[df_p["hour"] < 6, "date"] -= pd.Timedelta(days=1)
        night_m = (df_p["hour"] >= 22) | (df_p["hour"] < 6)

        for dt, grp in df_p[night_m].groupby("date"):
            ratio = grp["sleep_proxy"].mean()
            rows.append({
                "subject_id":       sid,
                "date":             dt,
                "night_sleep_ratio": ratio,
                "night_slept":       1 if ratio >= 0.5 else 0,
            })

    if not rows:
        return pd.DataFrame()

    agg = pd.DataFrame(rows)
    # 규칙적 수면자: 해당 사용자의 night_slept 비율 >= 0.6이면서 당일도 수면
    user_consist = agg.groupby("subject_id")["night_slept"].mean().rename("_consist")
    agg = agg.merge(user_consist, on="subject_id", how="left")
    agg["is_regular_night_sleeper"] = ((agg["_consist"] >= 0.6) & (agg["night_slept"] == 1)).astype(int)
    agg["date"] = pd.to_datetime(agg["date"])
    return agg[["subject_id","date","night_sleep_ratio","night_slept","is_regular_night_sleeper"]]

def feat_SleepProxy_TwoTrack() -> pd.DataFrame:
    """💡🔋 투트랙 수면 프록시: 조도(환경) + 기기(행동) 기반 수면 패턴.

    피처: user_archetype, light_sleep_duration, device_sleep_duration,
          sleep_bouts_count, light_night_sleep_ratio, device_night_sleep_ratio
    불규칙 사용자도 피처 값을 계산 (archetype만 0으로 표시).
    night_key 날짜 경계 처리 적용.
    """
    m_df  = load_parquet("mLight")
    w_df  = load_parquet("wLight")
    ac_df = load_parquet("mACStatus")
    scr_df= load_parquet("mScreenStatus")

    for df_, col in [(m_df,"m_light"),(w_df,"w_light"),
                     (ac_df,"m_charging"),(scr_df,"m_screen_use")]:
        df_[col] = pd.to_numeric(df_[col], errors="coerce")

    # 사용자 아키타입: 야간 조도 분산 → 하위 50% = 규칙적(1)
    def _night_var(df_, col):
        df_ = df_.copy()
        df_["hour"] = df_["timestamp"].dt.hour
        night = df_[(df_["hour"]>=22)|(df_["hour"]<6)]
        return night.groupby("subject_id")[col].var()

    m_var = _night_var(m_df, "m_light").rename("m_var")
    w_var = _night_var(w_df, "w_light").rename("w_var")
    profile = pd.concat([m_var, w_var], axis=1).fillna(0)
    profile["score"] = profile["m_var"] + profile["w_var"]
    median_score = profile["score"].median()
    profile["user_archetype"] = (profile["score"] <= median_score).astype(int)

    subjects = m_df["subject_id"].unique()
    final = []

    def _count_bouts(s):
        is_sleep = (s == 1)
        return int(((is_sleep != is_sleep.shift()) & is_sleep).sum())

    for sid in subjects:
        archetype = int(profile.loc[sid, "user_archetype"]) if sid in profile.index else 0

        def _resamp(df_, col, sid=sid):
            sub = df_[df_["subject_id"]==sid].copy()
            sub["timestamp"] = pd.to_datetime(sub["timestamp"])
            return sub.set_index("timestamp")[col].resample("10min").mean()

        df_sync = pd.DataFrame({
            "m_light":  _resamp(m_df,  "m_light"),
            "w_light":  _resamp(w_df,  "w_light"),
            "charging": _resamp(ac_df, "m_charging").ffill(),
            "screen":   _resamp(scr_df,"m_screen_use").ffill(),
        }).fillna({"charging":0,"screen":0})

        df_sync["sleep_by_light"]  = ((df_sync["m_light"]<=10)&(df_sync["w_light"]<=10)).astype(int)
        df_sync["sleep_by_device"] = ((df_sync["charging"]==1)&(df_sync["screen"]==0)).astype(int)
        df_sync["hour"] = df_sync.index.hour
        df_sync["date"] = df_sync.index.normalize()
        # night_key 적용
        df_sync.loc[df_sync["hour"]<6, "date"] -= pd.Timedelta(days=1)
        night_m = (df_sync["hour"]>=22)|(df_sync["hour"]<6)
        df_n = df_sync[night_m]

        daily = (
            df_n.groupby("date")
            .agg(
                light_sleep_duration  =("sleep_by_light",  lambda x: x.sum()*10),
                device_sleep_duration =("sleep_by_device", lambda x: x.sum()*10),
                sleep_bouts_count     =("sleep_by_light",  _count_bouts),
                light_night_sleep_ratio =("sleep_by_light",  "mean"),
                device_night_sleep_ratio=("sleep_by_device", "mean"),
            ).reset_index()
        )
        daily["subject_id"]    = sid
        daily["user_archetype"] = archetype
        final.append(daily)

    if not final:
        return pd.DataFrame()
    agg = pd.concat(final, ignore_index=True)
    agg["date"] = pd.to_datetime(agg["date"])
    return agg[["subject_id","date","user_archetype",
                "light_sleep_duration","device_sleep_duration",
                "sleep_bouts_count","light_night_sleep_ratio","device_night_sleep_ratio"]]


def feat_cross_arousal() -> pd.DataFrame:
    """
    그룹1: 취침 전 각성 자극 (mScreenStatus × mLight × wHr)

    피처 목록
    ─────────
    cross_dark_screen_min       야간(22~06시) 조도<5 lux + 화면ON  시간(분)
    cross_evening_bright_screen 저녁(19~22시) 조도>50 lux + 화면ON 시간(분)
    cross_presleep_screen_hr    취침 전 2h(21~23시) 화면ON 중 평균 심박수
    cross_presleep_hr_delta     (취침 전 화면ON 심박) - (해당일 전체 hr_mean)
    """
    # ── 스크린 (1분 단위) ──────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 폰 조도 (5분 단위, 소규모이므로 같은 floor) ─────────────────
    ml = _load_floored("mLight", FLOOR_FINE, ["m_light"])

    # ── 워치 심박 (5분 단위) ──────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 스크린 × 조도 병합 ────────────────────────────────────────────
    sc_ml = sc.merge(
        ml[["subject_id", "ts_floor", "m_light"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    sc_ml["hour"] = sc_ml["ts_floor"].dt.hour

    # 야간: 22~06시 (필터링 시 아래 _night_key 적용으로 날짜 보정)
    night_mask = (sc_ml["hour"] >= 22) | (sc_ml["hour"] < 6)
    # 각 버킷이 5분 → 1개 = 5분
    BUCKET_MIN = 5

    # cross_dark_screen_min: 야간 조도<5 + 화면ON (night_key 적용)
    sc_ml["is_dark_screen"] = (
        (sc_ml["m_light"] < 5) & (sc_ml["m_screen_use"] == 1)
    ).astype(float)
    sc_ml_dk = _night_key(sc_ml[night_mask])
    agg_dark = (
        sc_ml_dk
        .groupby(["subject_id", "night_key"])["is_dark_screen"]
        .sum().reset_index()
        .assign(cross_dark_screen_min=lambda x: x["is_dark_screen"] * BUCKET_MIN)
        .rename(columns={"night_key": "date"})
        [["subject_id", "date", "cross_dark_screen_min"]]
    )

    # cross_evening_bright_screen: 19~22시 조도>50 + 화면ON
    eve_mask = sc_ml["hour"].between(19, 21)
    sc_ml["is_bright_screen"] = (
        (sc_ml["m_light"] > 50) & (sc_ml["m_screen_use"] == 1)
    ).astype(float)
    agg_bright = (
        sc_ml[eve_mask]
        .groupby(["subject_id", "date"])["is_bright_screen"]
        .sum().reset_index()
        .assign(cross_evening_bright_screen=lambda x: x["is_bright_screen"] * BUCKET_MIN)
        [["subject_id", "date", "cross_evening_bright_screen"]]
    )

    # ── 스크린 × 심박 병합 ────────────────────────────────────────────
    sc_hr = sc.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    sc_hr["hour"] = sc_hr["ts_floor"].dt.hour

    # cross_presleep_screen_hr: 21~23시 화면ON 중 평균 심박
    pre_mask = sc_hr["hour"].between(21, 23) & (sc_hr["m_screen_use"] == 1)
    agg_hr = (
        sc_hr[pre_mask]
        .groupby(["subject_id", "date"])["hr"]
        .mean().reset_index()
        .rename(columns={"hr": "cross_presleep_screen_hr"})
    )

    # cross_presleep_hr_delta: 취침 전 화면ON 심박 - 해당일 전체 평균 심박
    hr_daily = (
        hr.groupby(["subject_id", "date"])["hr"].mean().reset_index()
        .rename(columns={"hr": "hr_daily_mean"})
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_dark.merge(agg_bright, on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_hr,         on=["subject_id", "date"], how="outer")
    agg = agg.merge(hr_daily,       on=["subject_id", "date"], how="left")
    agg["cross_presleep_hr_delta"] = (
        agg["cross_presleep_screen_hr"] - agg["hr_daily_mean"]
    )
    return agg.drop(columns=["hr_daily_mean"])

def feat_cross_sleep_env() -> pd.DataFrame:
    """
    그룹2: 수면 환경 품질 (mACStatus × mAmbience × mLight)

    충전 상태를 수면 proxy로 사용하여 실제 수면 중 환경 측정.

    피처 목록
    ─────────
    cross_sleep_noise_mean       야간 충전 중 평균 소음 확률
    cross_sleep_noise_speech     야간 충전 중 Speech 계열 소음 비율
    cross_sleep_light_mean       야간 충전 중 평균 조도
    cross_sleep_light_max        야간 충전 중 최대 조도
    cross_charge_screen_on_ratio 충전 중 화면ON 비율 (수면 지연 지표)
    cross_charge_screen_off_min  충전+화면OFF 지속 시간(분) = 실수면 추정
    """
    # ── 충전 상태 ──────────────────────────────────────────────────────
    ac = _load_floored("mACStatus", FLOOR_FINE, ["m_charging"])

    # ── 소음 ──────────────────────────────────────────────────────────
    amb_df = load_parquet("mAmbience")
    amb_df = amb_df.explode("m_ambience")
    amb_df["sound_type"] = amb_df["m_ambience"].str[0]
    amb_df["amb_level"]  = pd.to_numeric(amb_df["m_ambience"].str[1], errors="coerce")
    amb_df = amb_df.dropna(subset=["amb_level"])
    amb_df["is_speech"] = amb_df["sound_type"].str.contains(
        "Speech|Narration|Conversation", case=False, na=False
    ).astype(float)
    amb_df["ts_floor"] = amb_df["timestamp"].dt.floor(FLOOR_FINE)
    amb_agg = (
        amb_df.groupby(["subject_id", "date", "ts_floor"])
        .agg(amb_level_mean=("amb_level", "mean"),
             amb_speech_r   =("is_speech", "mean"))
        .reset_index()
    )

    # ── 폰 조도 ───────────────────────────────────────────────────────
    ml = _load_floored("mLight", FLOOR_FINE, ["m_light"])

    # ── 스크린 ────────────────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 충전 시간대 필터 (야간: 21~08시): night_key 적용 ──────────────
    ac["hour"] = ac["ts_floor"].dt.hour
    night_ac   = _night_key(ac[(ac["hour"] >= 21) | (ac["hour"] < 8)])
    # night_key를 date로 덮어써서 이후 groupby에서 자동 적용
    night_ac["date"] = night_ac["night_key"]

    # 충전 중인 버킷만
    charging = night_ac[night_ac["m_charging"] == 1].copy()
    BUCKET_MIN = 5

    # 충전+소음
    ch_amb = charging.merge(
        amb_agg[["subject_id", "ts_floor", "amb_level_mean", "amb_speech_r"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_amb = (
        ch_amb.groupby(["subject_id", "date"])
        .agg(cross_sleep_noise_mean  =("amb_level_mean", "mean"),
             cross_sleep_noise_speech=("amb_speech_r",   "mean"))
        .reset_index()
    )

    # 충전+조도
    ch_ml = charging.merge(
        ml[["subject_id", "ts_floor", "m_light"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_light = (
        ch_ml.groupby(["subject_id", "date"])
        .agg(cross_sleep_light_mean=("m_light", "mean"),
             cross_sleep_light_max =("m_light", "max"))
        .reset_index()
    )

    # 충전+스크린 (야간 전체)
    night_ac_sc = night_ac.merge(
        sc[["subject_id", "ts_floor", "m_screen_use"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_sc = (
        night_ac_sc.groupby(["subject_id", "date"])
        .agg(
            _sc_on  =("m_screen_use", "sum"),
            _sc_tot =("m_screen_use", "count"),
        )
        .reset_index()
    )
    agg_sc["cross_charge_screen_on_ratio"] = (
        agg_sc["_sc_on"] / agg_sc["_sc_tot"].replace(0, np.nan)
    )
    # 충전+화면OFF = 실수면 추정
    night_ac_sc["is_sleep_bucket"] = (
        (night_ac_sc["m_charging"] == 1) & (night_ac_sc["m_screen_use"] == 0)
    ).astype(float)
    agg_sleep = (
        night_ac_sc.groupby(["subject_id", "date"])["is_sleep_bucket"]
        .sum().reset_index()
        .assign(cross_charge_screen_off_min=lambda x: x["is_sleep_bucket"] * BUCKET_MIN)
        [["subject_id", "date", "cross_charge_screen_off_min"]]
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_amb.merge(agg_light, on=["subject_id", "date"], how="outer")
    agg = agg.merge(
        agg_sc[["subject_id", "date", "cross_charge_screen_on_ratio"]],
        on=["subject_id", "date"], how="outer"
    )
    agg = agg.merge(agg_sleep, on=["subject_id", "date"], how="outer")
    return agg


def feat_cross_exercise() -> pd.DataFrame:
    """
    그룹3: 운동 강도 (mActivity × wHr)

    피처 목록
    ─────────
    cross_exercise_hr_mean      걷기/뛰기 시 평균 심박수 (운동 강도)
    cross_exercise_hr_max       운동 중 최대 심박수
    cross_exercise_hr_recovery  운동 직후 30분 평균 심박 - 운동 중 평균 심박
    cross_exercise_timing_hour  운동(활동+고심박) 피크 시간대
    cross_late_exercise_min     늦은 저녁(20~23시) 운동 시간(분)
    """
    # ── 활동 ──────────────────────────────────────────────────────────
    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])

    # ── 워치 심박 ─────────────────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 병합 ──────────────────────────────────────────────────────────
    merged = act.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    merged["hour"] = merged["ts_floor"].dt.hour
    merged["is_active"] = merged["m_activity"].isin([7, 8])  # walk=7, run=8

    # 운동 중 심박
    active_hr = merged[merged["is_active"]]
    agg_exer = (
        active_hr.groupby(["subject_id", "date"])["hr"]
        .agg(cross_exercise_hr_mean="mean",
             cross_exercise_hr_max="max")
        .reset_index()
    )

    # 운동 피크 시간대 (심박>100 + 활동 구간)
    merged["is_intense"] = merged["is_active"] & (merged["hr"] > 100)
    hourly_intense = (
        merged.groupby(["subject_id", "date", "hour"])["is_intense"]
        .sum().reset_index()
    )
    peak = (
        hourly_intense.loc[
            hourly_intense.groupby(["subject_id", "date"])["is_intense"].idxmax()
        ][["subject_id", "date", "hour"]]
        .rename(columns={"hour": "cross_exercise_timing_hour"})
        .reset_index(drop=True)
    )

    # 운동 후 심박 회복 (운동 직후 30분 vs 운동 중)
    merged_sorted = merged.sort_values(["subject_id", "ts_floor"])
    merged_sorted["post_active"] = (
        merged_sorted.groupby("subject_id")["is_active"]
        .shift(6).fillna(False)   # 6 × 5분 = 30분 후
    )
    recovery = merged_sorted[merged_sorted["post_active"] & ~merged_sorted["is_active"]]
    agg_rec = (
        recovery.groupby(["subject_id", "date"])["hr"]
        .mean().reset_index()
        .rename(columns={"hr": "post_exercise_hr"})
    )

    # 늦은 저녁 운동 시간(분)
    BUCKET_MIN = 5
    late_mask = merged["hour"].between(20, 23) & merged["is_active"]
    agg_late = (
        merged[late_mask]
        .groupby(["subject_id", "date"])["is_active"]
        .sum().reset_index()
        .assign(cross_late_exercise_min=lambda x: x["is_active"] * BUCKET_MIN)
        [["subject_id", "date", "cross_late_exercise_min"]]
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_exer.merge(peak,     on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_late,      on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_rec,       on=["subject_id", "date"], how="left")
    agg["cross_exercise_hr_recovery"] = (
        agg["post_exercise_hr"] - agg["cross_exercise_hr_mean"]
    )
    return agg.drop(columns=["post_exercise_hr"], errors="ignore")

def feat_cross_mobility() -> pd.DataFrame:
    """
    그룹4: 사회적 활동 & 이동 (mGps × mActivity × mWifi)

    피처 목록
    ─────────
    cross_outdoor_active_min    GPS 이동 + 활동=걷기/뛰기 동시 충족 시간(분)
    cross_home_sedentary_min    집 WiFi + 활동=정지 동시 시간(분)
    cross_home_active_ratio     집 WiFi 연결 중 활동적인 비율
    """
    BUCKET_MIN = 5

    # ── GPS (1분 단위, 속도만) ────────────────────────────────────────
    gps_df = load_parquet("mGps")
    gps_df["ts_floor"] = gps_df["timestamp"].dt.floor(FLOOR_MED)
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            for c in ["latitude", "longitude", "speed"]:
                gps_df[c] = gps_df["m_gps"].apply(
                    lambda x: x.get(c, np.nan) if isinstance(x, dict) else np.nan
                )
    gps_df = gps_df.dropna(subset=["latitude", "longitude"])
    # 이상값 제거
    gps_df["speed"] = pd.to_numeric(gps_df.get("speed", np.nan), errors="coerce")
    gps_df = gps_df[gps_df["speed"].isna() | (gps_df["speed"] <= GPS_SPEED_LIMIT_MS)]
    gps_spd = (
        gps_df.groupby(["subject_id", "date", "ts_floor"])["speed"]
        .mean().reset_index()
    )

    # ── 활동 ──────────────────────────────────────────────────────────
    act = _load_floored("mActivity", FLOOR_MED, ["m_activity"])

    # ── WiFi 집 BSSID ─────────────────────────────────────────────────
    wifi_df = load_parquet("mWifi")
    wifi_df["ts_floor"] = wifi_df["timestamp"].dt.floor(FLOOR_MED)
    wifi_df = wifi_df.explode("m_wifi")
    _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
    if _fv is not None and isinstance(_fv, dict):
        wifi_df["bssid"] = wifi_df["m_wifi"].apply(
            lambda x: x.get("bssid", "") if isinstance(x, dict) else ""
        )
    else:
        wifi_df["bssid"] = ""

    home_bssid = (
        wifi_df[wifi_df["bssid"] != ""]
        .groupby(["subject_id", "bssid"]).size().reset_index(name="cnt")
        .sort_values("cnt", ascending=False)
        .drop_duplicates("subject_id")[["subject_id", "bssid"]]
        .rename(columns={"bssid": "home_bssid"})
    )
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"] == wifi_df["home_bssid"]).astype(int)
    wifi_home = (
        wifi_df.groupby(["subject_id", "date", "ts_floor"])["is_home"]
        .max().reset_index()   # 버킷 내 집 WiFi 감지 여부
    )

    # ── GPS × 활동: 실외 활동 ─────────────────────────────────────────
    gps_act = gps_spd.merge(
        act[["subject_id", "ts_floor", "m_activity"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    gps_act["is_outdoor_active"] = (
        (gps_act["speed"] > 0.5) & gps_act["m_activity"].isin([7, 8])
    ).astype(float)
    agg_outdoor = (
        gps_act.groupby(["subject_id", "date"])["is_outdoor_active"]
        .sum().reset_index()
        .assign(cross_outdoor_active_min=lambda x: x["is_outdoor_active"] * BUCKET_MIN)
        [["subject_id", "date", "cross_outdoor_active_min"]]
    )

    # ── WiFi × 활동: 집안 정지/활동 ──────────────────────────────────
    wifi_act = wifi_home.merge(
        act[["subject_id", "ts_floor", "m_activity"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    wifi_at_home = wifi_act[wifi_act["is_home"] == 1]

    wifi_at_home = wifi_at_home.copy()
    wifi_at_home["is_sedentary"] = (wifi_at_home["m_activity"] == 3).astype(float)
    wifi_at_home["is_moving"]    = wifi_at_home["m_activity"].isin([7, 8]).astype(float)

    agg_home_sed = (
        wifi_at_home.groupby(["subject_id", "date"])["is_sedentary"]
        .sum().reset_index()
        .assign(cross_home_sedentary_min=lambda x: x["is_sedentary"] * BUCKET_MIN)
        [["subject_id", "date", "cross_home_sedentary_min"]]
    )
    agg_home_act = (
        wifi_at_home.groupby(["subject_id", "date"])
        .agg(_mv=("is_moving", "sum"), _tot=("is_moving", "count"))
        .reset_index()
    )
    agg_home_act["cross_home_active_ratio"] = (
        agg_home_act["_mv"] / agg_home_act["_tot"].replace(0, np.nan)
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_outdoor.merge(agg_home_sed, on=["subject_id", "date"], how="outer")
    agg = agg.merge(
        agg_home_act[["subject_id", "date", "cross_home_active_ratio"]],
        on=["subject_id", "date"], how="outer"
    )
    return agg

def feat_cross_circadian() -> pd.DataFrame:
    """
    그룹5: 일주기 리듬 (wLight × wHr × mScreenStatus)

    피처 목록
    ─────────
    cross_morning_light_hr_corr   아침(06~09시) 광량-심박 상관계수
    cross_morning_natural_light   아침(06~10시) 자연광(>100 lux)+화면OFF 시간(분)
    cross_evening_light_drop      저녁(18~22시) 광량 감소율 (취침 준비 신호)
    """
    BUCKET_MIN = 5

    # ── 워치 광량 ─────────────────────────────────────────────────────
    wl = _load_floored("wLight", FLOOR_FINE, ["w_light"])

    # ── 워치 심박 ─────────────────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 스크린 ────────────────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 워치 광량 × 심박: 아침 상관계수 ─────────────────────────────
    wl_hr = wl.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    wl_hr["hour"] = wl_hr["ts_floor"].dt.hour
    morning_wl_hr = wl_hr[wl_hr["hour"].between(6, 9)]

    def _corr(g):
        if len(g) < 3: return np.nan
        return g["w_light"].corr(g["hr"])

    agg_corr = (
        morning_wl_hr.groupby(["subject_id", "date"])
        .apply(_corr).reset_index()
        .rename(columns={0: "cross_morning_light_hr_corr"})
    )

    # ── 워치 광량 × 스크린: 아침 자연광 노출 ─────────────────────────
    wl_sc = wl.merge(
        sc[["subject_id", "ts_floor", "m_screen_use"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    wl_sc["hour"] = wl_sc["ts_floor"].dt.hour
    morning_wl_sc = wl_sc[wl_sc["hour"].between(6, 10)]
    morning_wl_sc = morning_wl_sc.copy()
    morning_wl_sc["is_natural_no_screen"] = (
        (morning_wl_sc["w_light"] > 100) & (morning_wl_sc["m_screen_use"] == 0)
    ).astype(float)
    agg_nat = (
        morning_wl_sc.groupby(["subject_id", "date"])["is_natural_no_screen"]
        .sum().reset_index()
        .assign(cross_morning_natural_light=lambda x: x["is_natural_no_screen"] * BUCKET_MIN)
        [["subject_id", "date", "cross_morning_natural_light"]]
    )

    # ── 워치 광량: 저녁 광량 감소율 ──────────────────────────────────
    wl_eve = wl_hr[wl_hr["hour"].between(18, 22)]
    if len(wl_eve) > 0:
        # 각 날 저녁 시작(18시) 광량 vs 끝(22시) 광량
        wl_eve_agg = (
            wl_eve.groupby(["subject_id", "date", "hour"])["w_light"]
            .mean().reset_index()
        )
        eve_start = wl_eve_agg[wl_eve_agg["hour"] == 18][["subject_id","date","w_light"]].rename(columns={"w_light": "light_18"})
        eve_end   = wl_eve_agg[wl_eve_agg["hour"] == 22][["subject_id","date","w_light"]].rename(columns={"w_light": "light_22"})
        agg_drop  = eve_start.merge(eve_end, on=["subject_id","date"], how="inner")
        agg_drop["cross_evening_light_drop"] = (
            (agg_drop["light_18"] - agg_drop["light_22"]) /
            agg_drop["light_18"].replace(0, np.nan)
        )
        agg_drop = agg_drop[["subject_id","date","cross_evening_light_drop"]]
    else:
        agg_drop = pd.DataFrame(columns=["subject_id","date","cross_evening_light_drop"])

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_corr.merge(agg_nat,  on=["subject_id","date"], how="outer")
    agg = agg.merge(agg_drop,      on=["subject_id","date"], how="outer")
    return agg

def feat_cross_commute() -> pd.DataFrame:
    """🚗 통근 시간: IN_VEHICLE + GPS 이동 (mActivity × mGps)."""
    BUCKET_MIN = 5
    act = _load_floored("mActivity", FLOOR_MED, ["m_activity"])
    gps_df = load_parquet("mGps")
    gps_df["ts_floor"] = gps_df["timestamp"].dt.floor(FLOOR_MED)
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            gps_df["speed_g"] = gps_df["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
    gps_df["speed_g"] = pd.to_numeric(gps_df.get("speed_g", np.nan), errors="coerce")
    gps_spd = (gps_df.groupby(["subject_id","date","ts_floor"])["speed_g"]
               .mean().reset_index())
    merged = act.merge(gps_spd[["subject_id","ts_floor","speed_g"]],
                       on=["subject_id","ts_floor"], how="inner")
    merged["is_commute"] = ((merged["m_activity"]==0) & (merged["speed_g"]>1.5)).astype(float)
    agg = (merged.groupby(["subject_id","date"])["is_commute"]
           .sum().reset_index()
           .assign(cross_commute_min=lambda x: x["is_commute"]*BUCKET_MIN)
           [["subject_id","date","cross_commute_min"]])
    return agg

def feat_cross_presleep_arousal() -> pd.DataFrame:
    """😤 취침 전 각성 지수: wHr × wPedo × mActivity (20~22시)."""
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object: hr = hr.explode("heart_rate")
    hr["hr_val"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr_val"])
    hr = hr[hr["hr_val"].between(30,220)]
    hr["hour"] = hr["timestamp"].dt.hour
    hr_pre = (hr[hr["hour"].between(20,22)]
              .groupby(["subject_id","date"])["hr_val"]
              .mean().reset_index().rename(columns={"hr_val":"_hr_pre"}))

    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])
    act["hour"] = act["ts_floor"].dt.hour
    act_pre = (act[act["hour"].between(20,22)]
               .groupby(["subject_id","date"])["m_activity"]
               .agg(lambda x: (x.isin([7,8,9])).mean()).reset_index()
               .rename(columns={"m_activity":"_act_pre"}))

    ped = load_parquet("wPedo")
    ped["hour"] = ped["timestamp"].dt.hour
    ped_pre = (ped[ped["hour"].between(20,22)]
               .groupby(["subject_id","date"])["step"]
               .sum().reset_index().rename(columns={"step":"_step_pre"}))

    agg = hr_pre.merge(act_pre, on=["subject_id","date"], how="outer")
    agg = agg.merge(ped_pre, on=["subject_id","date"], how="outer")
    # z-score 없이 0~1 정규화 후 합산 → 각성 지수
    for c in ["_hr_pre","_act_pre","_step_pre"]:
        mn = agg[c].min(); mx = agg[c].max()
        agg[c] = (agg[c]-mn) / (mx-mn+1e-8)
    agg["cross_presleep_arousal"] = agg[["_hr_pre","_act_pre","_step_pre"]].mean(axis=1)
    return agg[["subject_id","date","cross_presleep_arousal"]]

def feat_cross_resting_hr_at_home() -> pd.DataFrame:
    """💓 실제 안정시 심박: 집 WiFi + STILL 구간 심박 (wHr × mWifi × mActivity)."""
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object: hr = hr.explode("heart_rate")
    hr["hr_val"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr_val"])
    hr = hr[hr["hr_val"].between(30,220)]
    hr_agg = hr.groupby(["subject_id","date","ts_floor"])["hr_val"].mean().reset_index()

    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])

    wifi_df = load_parquet("mWifi")
    wifi_df["ts_floor"] = wifi_df["timestamp"].dt.floor(FLOOR_FINE)
    wifi_df = wifi_df.explode("m_wifi")
    _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
    if _fv is not None and isinstance(_fv, dict):
        wifi_df["bssid"] = wifi_df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
    else:
        wifi_df["bssid"] = ""
    home_bssid = (wifi_df[wifi_df["bssid"]!=""].groupby(["subject_id","bssid"])
                  .size().reset_index(name="cnt").sort_values("cnt", ascending=False)
                  .drop_duplicates("subject_id")[["subject_id","bssid"]]
                  .rename(columns={"bssid":"home_bssid"}))
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"]==wifi_df["home_bssid"]).astype(int)
    wifi_home = (wifi_df.groupby(["subject_id","date","ts_floor"])["is_home"]
                 .max().reset_index())

    merged = (hr_agg
              .merge(act[["subject_id","ts_floor","m_activity"]], on=["subject_id","ts_floor"], how="inner")
              .merge(wifi_home[["subject_id","ts_floor","is_home"]], on=["subject_id","ts_floor"], how="left"))
    still_home = merged[(merged["m_activity"]==3) & (merged["is_home"]==1)]
    agg = (still_home.groupby(["subject_id","date"])["hr_val"]
           .mean().reset_index().rename(columns={"hr_val":"cross_resting_hr_at_home"}))
    return agg


def feat_cross_sleep_env_noise() -> pd.DataFrame:
    """🌙 수면 환경 복잡도: BLE × WiFi rssi × GPS 정지 비율 (야간)."""
    ble_df = load_parquet("mBle")
    ble_df["hour"] = ble_df["timestamp"].dt.hour
    if "m_ble" in ble_df.columns and ble_df["m_ble"].dtype == object:
        try:
            ble_df = ble_df.explode("m_ble")
            _fv = ble_df["m_ble"].dropna().iloc[0] if ble_df["m_ble"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                ble_df["address"] = ble_df["m_ble"].apply(lambda x: x.get("address","") if isinstance(x,dict) else "")
        except Exception:
            ble_df["address"] = ""
    night_m = (ble_df["hour"]>=22)|(ble_df["hour"]<6)
    ble_n = _night_key(ble_df[night_m & (ble_df["address"]!="")])
    ble_night = (ble_n.groupby(["subject_id","night_key"])["address"].nunique().reset_index()
                 .rename(columns={"night_key":"date","address":"_ble_night"}))

    wifi_df = load_parquet("mWifi")
    wifi_df["hour"] = wifi_df["timestamp"].dt.hour
    if "m_wifi" in wifi_df.columns and wifi_df["m_wifi"].dtype == object:
        try:
            wifi_df = wifi_df.explode("m_wifi")
            _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                wifi_df["rssi"] = wifi_df["m_wifi"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            wifi_df["rssi"] = np.nan
    night_w = (wifi_df["hour"]>=22)|(wifi_df["hour"]<6)
    wifi_n = _night_key(wifi_df[night_w])
    wifi_night = (wifi_n.groupby(["subject_id","night_key"])["rssi"]
                  .mean().reset_index()
                  .rename(columns={"night_key":"date","rssi":"_wifi_rssi_night"}))

    gps_df = load_parquet("mGps")
    gps_df["hour"] = gps_df["timestamp"].dt.hour
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            gps_df["speed_g"] = gps_df["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
    gps_df["speed_g"] = pd.to_numeric(gps_df.get("speed_g", np.nan), errors="coerce")
    night_g = (gps_df["hour"]>=22)|(gps_df["hour"]<6)
    gps_n = _night_key(gps_df[night_g])
    gps_night = (gps_n.groupby(["subject_id","night_key"])["speed_g"]
                 .agg(lambda x: (x<1.0).mean()).reset_index()
                 .rename(columns={"night_key":"date","speed_g":"_gps_stop_night"}))

    agg = ble_night.merge(wifi_night, on=["subject_id","date"], how="outer")
    agg = agg.merge(gps_night, on=["subject_id","date"], how="outer")
    for c in ["_ble_night","_wifi_rssi_night","_gps_stop_night"]:
        mn=agg[c].min(); mx=agg[c].max()
        agg[c] = (agg[c]-mn)/(mx-mn+1e-8)
    agg["cross_sleep_env_noise"] = agg[["_ble_night","_wifi_rssi_night","_gps_stop_night"]].mean(axis=1)
    return agg[["subject_id","date","cross_sleep_env_noise"]]

def feat_cross_wifi_gps_home_agree() -> pd.DataFrame:
    """🏠 집 판별 신뢰도: WiFi home 비율 × GPS home_min 일치 여부."""
    wifi_df = load_parquet("mWifi")
    if "m_wifi" in wifi_df.columns and wifi_df["m_wifi"].dtype == object:
        wifi_df = wifi_df.explode("m_wifi")
        _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            wifi_df["bssid"] = wifi_df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
    else:
        wifi_df["bssid"] = ""
    home_bssid = (wifi_df[wifi_df["bssid"]!=""].groupby(["subject_id","bssid"])
                  .size().reset_index(name="cnt").sort_values("cnt",ascending=False)
                  .drop_duplicates("subject_id")[["subject_id","bssid"]]
                  .rename(columns={"bssid":"home_bssid"}))
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"]==wifi_df["home_bssid"]).astype(int)
    wifi_home_r = (wifi_df.groupby(["subject_id","date"])["is_home"]
                   .mean().reset_index().rename(columns={"is_home":"_wifi_home_r"}))
    # GPS home_min은 feat_mGps()에서 생성됨 → 여기서는 WiFi home 비율만 반환
    # (파이프라인에서 gps_home_min과 merge하여 cross_wifi_gps_home_agree 계산)
    wifi_home_r = wifi_home_r.rename(columns={"_wifi_home_r":"cross_wifi_gps_home_agree"})
    return wifi_home_r

def build_cross_features(date_df: pd.DataFrame) -> pd.DataFrame:
    """
    교차 parquet 피처 전체 생성 및 date_df에 merge.

    각 함수는 (subject_id, date) 기준 일별 집계값을 반환.
    date → lifelog_date 로 rename 후 left merge.
    """
    print("교차 parquet 피처 추출 시작...")

    cross_funcs = {
        "arousal":   feat_cross_arousal,    # 그룹1: 취침 전 각성 자극
        "sleep_env": feat_cross_sleep_env,  # 그룹2: 수면 환경 품질
        "exercise":  feat_cross_exercise,   # 그룹3: 운동 강도
        "mobility":  feat_cross_mobility,   # 그룹4: 사회적 활동 & 이동
        "circadian": feat_cross_circadian,  # 그룹5: 일주기 리듬
    }

    df = date_df.copy()
    for name, func in cross_funcs.items():
        print(f"  [cross_{name}] 처리 중...", end=" ", flush=True)
        try:
            feat = func()
            feat = feat.rename(columns={"date": "lifelog_date"})
            feat["lifelog_date"] = pd.to_datetime(feat["lifelog_date"])
            df["lifelog_date"]   = pd.to_datetime(df["lifelog_date"])
            before = df.shape[1]
            df     = df.merge(feat, on=["subject_id", "lifelog_date"], how="left")
            added  = df.shape[1] - before
            print(f"완료 (+{added} 컬럼)")
        except Exception as e:
            print(f"오류: {e}")
            import traceback; traceback.print_exc()

    return df

def build_features(date_df: pd.DataFrame) -> pd.DataFrame:
    """
    date_df : subject_id / sleep_date / lifelog_date 컬럼 포함
              lifelog_date 는 naive datetime64[ns] 이어야 함

    각 feat_*() 의 date 컬럼(naive datetime64[ns])을
    lifelog_date 에 left merge → 타입 완전 일치, 누락 없이 조인
    """
    print("피처 추출 시작...")

    feat_funcs = {
        "wPedo":          feat_wPedo,
        "wHr":            feat_wHr,
        "wLight":         feat_wLight,
        "mActivity":      feat_mActivity,
        "mACStatus":      feat_mACStatus,
        "mScreenStatus":  feat_mScreenStatus,
        "mLight":         feat_mLight,
        "mAmbience":      feat_mAmbience,
        "mWifi":          feat_mWifi,
        "mGps":           feat_mGps,
        "mUsageStats":    feat_mUsageStats,
        "mBle":           feat_mBle,
        "SleepProxy_AC_Scr":  feat_SleepProxy_AC_Scr,
        "SleepProxy_TwoTrack": feat_SleepProxy_TwoTrack,
        # 기존 복합 피처
        "cross_arousal":     feat_cross_arousal,
        "cross_sleep_env":   feat_cross_sleep_env,
        "cross_exercise":    feat_cross_exercise,
        "cross_mobility":    feat_cross_mobility,
        "cross_circadian":   feat_cross_circadian,
        # 신규 복합 피처 (V53)
        "cross_commute":             feat_cross_commute,
        "cross_presleep_arousal":    feat_cross_presleep_arousal,
        "cross_resting_hr_at_home":  feat_cross_resting_hr_at_home,
        "cross_sleep_env_noise":     feat_cross_sleep_env_noise,
        "cross_wifi_gps_home_agree": feat_cross_wifi_gps_home_agree,
    }

    df = date_df.copy()
    for name, func in feat_funcs.items():
        print(f"  [{name}] 처리 중...", end=" ", flush=True)
        try:
            feat = func()
            feat = feat.rename(columns={"date": "lifelog_date"})
            df   = df.merge(feat, on=["subject_id", "lifelog_date"], how="left")
            print(f"완료 ({len(feat):,} rows, +{feat.shape[1] - 2} cols)")
        except Exception as e:
            print(f"오류: {e}")

    return df

# ══════════════════════════════════════════════════════════════════════
# 4. 시간/요일 파생 피처
# ══════════════════════════════════════════════════════════════════════

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """sleep_date 기반 요일/월/주차 피처 + subject_num 인코딩"""
    df = df.copy()
    df["sleep_date"] = pd.to_datetime(df["sleep_date"])

    df["dow"]          = df["sleep_date"].dt.dayofweek       # 0=월, 6=일
    df["is_weekend"]   = (df["dow"] >= 5).astype(int)
    df["month"]        = df["sleep_date"].dt.month
    df["week_of_year"] = df["sleep_date"].dt.isocalendar().week.astype(int)
    df["subject_num"]  = df["subject_id"].str.extract(r"(\d+)").astype(int)
    return df


## 5-1. Z-Score + 센서 Rolling + LGB/XGB/CatBoost 가중 앙상블

# ══════════════════════════════════════════════════════════════════════
# 5. 래그 / 개인 평균 피처 — fold 내부 전용 헬퍼
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# 6. LightGBM 학습
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# 5-v5-1. Z-score + 센서 Rolling + LGB/XGB/CatBoost 가중 앙상블
# ══════════════════════════════════════════════════════════════════════

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

# ── 모델 하이퍼파라미터 (V5와 동일) ─────────────────────────────────
LGBM_PARAMS = {
    "objective":          "binary",
    "metric":             "binary_logloss",
    "n_estimators":       1000, # Optuna 튜닝 시에는 500으로 줄였더라도 최종 학습 시에는 1000 이상으로 올려주는 것이 좋습니다.
    "learning_rate":      0.048002539014146264, # Optuna 값 적용
    "num_leaves":         43,                   # Optuna 값 적용
    "min_child_samples":  21,                   # Optuna 값 적용
    "feature_fraction":   0.6034723418591593,   # Optuna 값 적용
    "bagging_fraction":   0.8257774001264698,   # Optuna 값 적용
    "bagging_freq":       5,
    "lambda_l1":          0.1,                  # 기존 값 유지 (또는 튜닝했다면 해당 값)
    "lambda_l2":          0.1,                  # 기존 값 유지
    "random_state":       42,
    "n_jobs":             -1,
    "verbose":            -1,
}
XGB_PARAMS = {
    "objective":          "binary:logistic",
    "eval_metric":        "logloss",
    "n_estimators":       1000,
    "learning_rate":      0.012936739731011452, # Optuna 값 적용
    "max_depth":          3,                    # Optuna 값 적용
    "subsample":          0.8135516777791894,   # Optuna 값 적용
    "colsample_bytree":   0.7603815432836665,   # Optuna 값 적용
    "reg_alpha":          0.1,                  # 기존 값 유지
    "reg_lambda":         0.1,                  # 기존 값 유지
    "random_state":       42,
    "n_jobs":             -1,
    "verbosity":          0,
    "early_stopping_rounds": 50
}
CAT_PARAMS = {
    "iterations":         1000,
    "learning_rate":      0.0528527508011132,   # Optuna 값 적용
    "depth":              7,                    # Optuna 값 적용
    "l2_leaf_reg":        6.027332560539019,    # Optuna 값 적용
    "loss_function":      "Logloss",
    "eval_metric":        "Logloss",
    "random_seed":        42,
    "verbose":            False,
    "allow_writing_files": False,
}

# ── 센서 Rolling 설정 ─────────────────────────────────────────────────
# V3와 달리 센서 데이터(X)에 rolling을 적용 → test에도 결측 없음
# 이유:
#   V3: 타겟(y) rolling → test의 y가 없어 평균 44~84% 결측
#   V5-1: 센서(X) rolling → train/test 모두 X값 존재 → 결측 <5%
ROLL_WINDOWS         = [3, 7]     # rolling 윈도우 크기 (일)
ROLL_MISS_THRESHOLD  = 0.30       # rolling 피처 결측률 임계값 (30% 초과 제거)

# 피처 선택 설정
# 피어슨 상관계수 절댓값 기준 상위 CORR_TOP_K개 선택
# - raw+rolling: 86+~100 = ~186개 후보
# - z-score     : 동일 수 후보
# - 전체 후보 ~370개에서 타겟별로 50개 선택 → 샘플(450):피처(50) = 9:1
CORR_TOP_K  = 50
N_FOLDS     = 5

# ── Subject-Hole CV 설정 ──────────────────────────────────────────────
# 실제 train/test 분리 구조 (T/V/T/V 인터리빙)를 모사하는 CV 전략.
#
# 실제 데이터 패턴 (id01 예시):
#   TTTTTTTTTTTTTTTTTTTTTTTTTTTTVVVVVVVVVVVVVVTTTTTTTTTTTTTVVVVVVVVVVVVV
#   → train과 test가 사용자 내부에서 시간 순으로 교차하는 블록 구조
#
# Subject-Hole CV 방식:
#   각 사용자의 날짜를 시간 순 정렬 후 n_folds*2 개의 chunk로 분할.
#   fold i: chunk[i] + chunk[i+n_folds] → val  (early hole + late hole)
#           나머지 → train
#   → T/V/T/V 패턴을 fold 단위로 재현 ✅
# ── 베이즈 수축 파라미터 ──────────────────────────────────────────────
# 사용자 평균 베이즈 수축 (Bayesian Shrinkage toward user mean)
#
# 수식: pred_final = α × model_pred + (1-α) × user_label_mean
#
# α (신뢰도): 샘플 수에 따른 동적 결정
#   α(n) = n / (n + SHRINK_K)
#   SHRINK_K=10 기준:
#     n=10 → α=0.50  (모델 50%, 사용자 평균 50%)
#     n=20 → α=0.67  (모델 67%, 사용자 평균 33%)
#     n=30 → α=0.75  (모델 75%, 사용자 평균 25%)
#     n=57 → α=0.85  (모델 85%, 사용자 평균 15%)
#
# 직관: 학습 데이터가 적은 사용자일수록 해당 사용자의 과거 평균 비중↑
#       → 개인화 편차(S3: 0.374, S2: 0.336) 보정에 효과적
#
# 누수 방지 ✅
# - OOF: fold 내 train 부분만의 사용자 평균 사용 (val 레이블 미포함)
# - test: 전체 train 사용자 평균 사용 (test 레이블 미사용)
SHRINK_K    = 10.0   # 수축 강도 조율 상수 (클수록 수축 강함)

# ════════════════════════════════════════════════════════════════════
# 5-1. 센서 Rolling 피처 계산
# ════════════════════════════════════════════════════════════════════

def add_sensor_rolling(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    feat_cols: list,
) -> tuple:
    """
    센서 피처에 rolling 통계를 추가하여 train/test를 반환.

    V3 타겟 rolling과의 차이
    ──────────────────────
    V3 : 타겟(y) rolling
         - test의 y값이 없으므로 결측 평균 44~84%
         - OOF 학습 시 val 레이블이 rolling 집계에 포함될 수 있음 (누수 위험)

    V5-1: 센서(X) rolling
         - train/test 모두 X값 존재 → 결측 < 5%
         - X→X 관계이므로 레이블(y) 누수 없음

    누수 방지 설계
    ──────────────
    ① shift(1) 적용
       T일의 rolling = mean(T-1, T-2, ...)
       → T일 자신의 센서값이 T일 rolling에 포함되지 않음 ✅

    ② train+test 결합 후 (subject_id, sleep_date) 기준 정렬
       → 시간 순서 유지, 미래 센서값이 과거 rolling에 포함되지 않음 ✅

    ③ 센서 데이터(X) 기반 → 레이블(y) 정보 미포함 ✅
       (타겟 rolling과 달리 val 레이블 누수 없음)

    ④ min_periods=1 사용
       초기 날짜(이전 기록 부족)에도 가능한 값 생성 → 결측 최소화

    ⑤ ROLL_MISS_THRESHOLD(30%) 초과 rolling 피처 자동 제거
       (hr_night_mean 등 원본부터 결측 많은 피처의 rolling 제거)

    주의: train+test 결합 시 test 센서값이 일부 train rolling에 영향
    ──────
    날짜가 혼재하는 구조에서 test 날짜 T의 센서값이
    train 날짜 T+k의 rolling에 기여할 수 있음.
    그러나 X→X 관계이며 y 정보를 포함하지 않으므로
    실질적 누수(y leakage)가 아닌 통상 허용 범위 내.

    반환
    ────
    train_df_new : rolling 피처 추가된 train DataFrame
    test_df_new  : rolling 피처 추가된 test  DataFrame
    roll_cols    : 생성된 rolling 피처명 리스트
    """
    # ── 연속형 피처만 대상 (이진 피처는 rolling 의미 없음) ──────────
    continuous_cols = [
        c for c in feat_cols
        if len(train_df[c].dropna().unique()) > 2
        or not set(train_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]

    # ── train + test 결합, 날짜 정렬 ─────────────────────────────────
    # ignore_index=True 와 keys 를 함께 쓰면 keys가 무시되므로
    # _src 컬럼을 concat 전에 명시적으로 추가
    tr_part = train_df[["subject_id", "sleep_date"] + continuous_cols].copy()
    te_part = test_df[["subject_id", "sleep_date"] + continuous_cols].copy()
    tr_part["_src"] = "train"
    te_part["_src"] = "test"

    combined = pd.concat([tr_part, te_part], ignore_index=True)

    combined["sleep_date"] = pd.to_datetime(combined["sleep_date"])
    combined = combined.sort_values(["subject_id", "sleep_date"]).reset_index(drop=True)

    # ── rolling 계산 ──────────────────────────────────────────────────
    generated: list = []
    for c in continuous_cols:
        for w in ROLL_WINDOWS:
            col_name = f"{c}_roll{w}"
            combined[col_name] = (
                combined.groupby("subject_id")[c]
                .transform(
                    lambda x, _w=w: x.shift(1).rolling(_w, min_periods=1).mean()
                )
            )
            generated.append(col_name)

    # ── 결측률 필터 (train 기준 ROLL_MISS_THRESHOLD 초과 제거) ────────
    train_mask = combined["_src"] == "train"
    miss_rates  = combined.loc[train_mask, generated].isnull().mean()
    keep_cols   = [c for c in generated if miss_rates[c] <= ROLL_MISS_THRESHOLD]
    drop_cols   = [c for c in generated if miss_rates[c] > ROLL_MISS_THRESHOLD]

    if drop_cols:
        print(f"  rolling 피처 결측>{ROLL_MISS_THRESHOLD:.0%} 제거 "
              f"({len(drop_cols)}개): {drop_cols[:5]}{'...' if len(drop_cols)>5 else ''}")

    # ── train / test 분리 ─────────────────────────────────────────────
    # (subject_id, sleep_date) 기준 merge로 정확히 분리
    merge_key = ["subject_id", "sleep_date"]

    train_roll = (
        combined.loc[train_mask, merge_key + keep_cols]
        .drop_duplicates(merge_key)
    )
    test_roll = (
        combined.loc[~train_mask, merge_key + keep_cols]
        .drop_duplicates(merge_key)
    )

    train_df["sleep_date"] = pd.to_datetime(train_df["sleep_date"])
    test_df["sleep_date"]  = pd.to_datetime(test_df["sleep_date"])

    train_new = train_df.merge(train_roll, on=merge_key, how="left")
    test_new  = test_df.merge(test_roll,  on=merge_key, how="left")

    # 결측 최종 확인
    tr_miss = train_new[keep_cols].isnull().mean().mean() * 100
    te_miss = test_new[keep_cols].isnull().mean().mean() * 100
    print(f"  센서 rolling 피처 {len(keep_cols)}개 생성 완료")
    print(f"  평균 결측률 — train: {tr_miss:.1f}%  test: {te_miss:.1f}%")

    return train_new, test_new, keep_cols



# ════════════════════════════════════════════════════════════════════
# 5-2. Z-score 개인화 피처 계산 (V5와 동일)
# ════════════════════════════════════════════════════════════════════

def compute_user_zscores(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    feat_cols: list,
) -> tuple:
    """
    사용자별 Z-score 피처 계산 (raw + rolling 피처 모두 대상).

    누수 방지 ✅
    통계량(mean, std)은 train_df만 사용.
    test에는 train 통계량을 그대로 적용.
    z-score 계산에 타겟 레이블 미사용.
    """
    continuous_cols = [
        c for c in feat_cols
        if len(train_df[c].dropna().unique()) > 2
        or not set(train_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]
    print(f"  Z-score 대상: {len(continuous_cols)}개 "
          f"(raw + rolling 연속형, 이진 {len(feat_cols)-len(continuous_cols)}개 제외)")

    user_stats = (
        train_df.groupby("subject_id")[continuous_cols]
        .agg(["mean", "std"])
    )
    user_stats.columns = [f"{c}__{s}" for c, s in user_stats.columns]

    def _apply(df: pd.DataFrame) -> pd.DataFrame:
        res = {}
        for c in continuous_cols:
            u_mean = df["subject_id"].map(user_stats[f"{c}__mean"])
            u_std  = df["subject_id"].map(user_stats[f"{c}__std"]).fillna(0)
            raw    = df[c].fillna(u_mean)
            z      = (raw - u_mean) / (u_std + 1e-8)
            res[f"{c}_uz"] = z.fillna(0)
        return pd.DataFrame(res, index=df.index)

    train_z = _apply(train_df)
    test_z  = _apply(test_df)
    z_cols  = list(train_z.columns)
    return train_z, test_z, z_cols

# ════════════════════════════════════════════════════════════════════
# 5-3. 피처 선택: 피어슨 상관계수 기반 (CV fold 내부 계산)
# ════════════════════════════════════════════════════════════════════

def _entropy_discrete(x: np.ndarray) -> float:
    """이산 변수 엔트로피 H(X) = -Σ p_i log(p_i)."""
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 1e-10
    _, counts = np.unique(x, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-10)))

def _entropy_continuous(x: np.ndarray, n_bins: int = 20) -> float:
    """연속 변수 엔트로피 H(X): 히스토그램 기반 근사."""
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 1e-10
    counts, _ = np.histogram(x, bins=n_bins)
    counts = counts[counts > 0]
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-10)))

def _entropy_binary_target(y: np.ndarray) -> float:
    """이진 타겟 엔트로피 H(Y) = -p log(p) - (1-p) log(1-p)."""
    p = np.mean(y)
    if p <= 1e-10 or p >= 1 - 1e-10:
        return 1e-10
    return float(-p * np.log(p) - (1 - p) * np.log(1 - p))

def _nmi_score(mi: float, h_x: float, h_y: float) -> float:
    """정규화 상호정보량 NMI(X;Y) = 2·I(X;Y) / (H(X)+H(Y)) ∈ [0,1]."""
    denom = h_x + h_y
    if denom < 1e-10:
        return 0.0
    return float(np.clip(2 * mi / denom, 0.0, 1.0))

def select_features_by_mutual_info(
    X_all:          pd.DataFrame,
    y_dict:         dict,
    all_cols:       list,
    cat_feat_cols:  list,
    train_df_ref:   pd.DataFrame,
    top_k:          int   = CORR_TOP_K,
    n_neighbors:    int   = 3,
    random_state:   int   = 42,
) -> dict:
    """정규화 상호정보량(NMI) 기반 타겟별 상위 K개 피처 선택.

    NMI(X;Y) = 2·I(X;Y) / (H(X)+H(Y))
      - I(X;Y): sklearn mutual_info_classif (k-NN 추정)
      - H(X):  이산→ 정확한 엔트로피 / 연속→ 히스토그램 근사
      - H(Y):  이진 타겟 엔트로피
    → 0~1 정규화로 스케일 무관 피처 간 공정한 비교

    Pearson 대비 장점:
      - 이진 타겟(Q1~S4)에 적합 (비선형·비단조 관계 포착)
      - 범주형·연속형 혼합 피처 정확 처리 (discrete_features 지정)
      - NMI 정규화로 고분산 피처 과대 선택 방지

    Parameters
    ----------
    cat_feat_cols : 범주형 피처 이름 목록 → discrete_features=True 처리
    """

    import logging
    # 로거 설정
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

    def log_infinite_features(df: pd.DataFrame, context_name: str = "FeatureMatrix"):
        """
        데이터프레임 내의 무한대(inf, -inf) 또는 너무 큰 값을 찾아 로깅합니다.
        """
        # 1. 무한대(inf, -inf) 확인
        inf_mask = np.isinf(df)
        inf_counts = inf_mask.sum()
        inf_cols = inf_counts[inf_counts > 0]

        if len(inf_cols) > 0:
            logging.error(f"[{context_name}] 🚨 무한대(inf) 값이 발견된 컬럼 ({len(inf_cols)}개):")
            for col, count in inf_cols.items():
                logging.error(f"  - {col}: {count} rows")

        # 2. 극단적으로 큰 값 확인 (float64 max 근처)
        # sklearn은 보통 1e308 이상을 문제 삼지만, 안전하게 1e15 이상을 체크해봅니다.
        huge_mask = (df > 1e15) & ~inf_mask
        huge_counts = huge_mask.sum()
        huge_cols = huge_counts[huge_counts > 0]

        if len(huge_cols) > 0:
            logging.warning(f"[{context_name}] ⚠️ 극단적으로 큰 값(>1e15)이 발견된 컬럼 ({len(huge_cols)}개):")
            for col, count in huge_cols.items():
                logging.warning(f"  - {col}: {count} rows")

        return list(inf_cols.keys())

        # 사용 예시 (오류가 난 함수 내부에 추가)
        # mi = mutual_info_classif(...) 호출 직전에 아래 코드를 넣으세요.
        # log_infinite_features(X_train_with_z, "X_train_with_z (Before NMI)")
    from sklearn.feature_selection import mutual_info_classif

    cat_feat_set   = set(cat_feat_cols or [])
    discrete_mask  = np.array([c in cat_feat_set for c in all_cols], dtype=bool)
    sh_folds       = make_subject_hole_folds(train_df_ref)
    feat_dict      = {}

    print(f"  [NMI 선택] 후보 {len(all_cols)}개 피처 "
          f"(이산 {discrete_mask.sum()}개 / 연속 {(~discrete_mask).sum()}개)")

    for target, y_all in y_dict.items():
        nmi_accum = np.zeros(len(all_cols))
        n_valid   = 0

        for tr_idx, _ in sh_folds:
            X_tr = X_all[all_cols].iloc[tr_idx].fillna(0).values
            y_tr = y_all[tr_idx]

            if len(np.unique(y_tr)) < 2:
                continue  # 클래스 다양성 없으면 스킵

            # ── I(X;Y): mutual_info_classif (fold train만 사용, 누수 없음)
            mi = mutual_info_classif(
                X_tr, y_tr,
                discrete_features=discrete_mask,
                n_neighbors=n_neighbors,
                random_state=random_state,
            )
            mi = np.clip(mi, 0, None)  # 음수 추정값 클립

            # ── H(Y): 이진 타겟 엔트로피
            h_y = _entropy_binary_target(y_tr)

            # ── NMI 계산 (피처별)
            nmi = np.zeros(len(all_cols))
            for i in range(len(all_cols)):
                if mi[i] < 1e-12:
                    nmi[i] = 0.0
                    continue
                h_x = (_entropy_discrete(X_tr[:, i])
                       if discrete_mask[i]
                       else _entropy_continuous(X_tr[:, i]))
                nmi[i] = _nmi_score(mi[i], h_x, h_y)

            nmi_accum += nmi
            n_valid   += 1

        if n_valid == 0:
            feat_dict[target] = all_cols[:top_k]
            continue

        nmi_mean = nmi_accum / n_valid  # fold 평균 NMI

        top_idx          = np.argsort(nmi_mean)[::-1][:top_k]
        feat_dict[target] = [all_cols[i] for i in top_idx]

        top3 = [(all_cols[i], nmi_mean[i]) for i in top_idx[:3]]
        top3_str = ", ".join(f"{n}({v:.3f})" for n, v in top3)
        print(f"    {target}: top3 NMI = {top3_str}  (선택 {top_k}개)")

    return feat_dict

def select_features_by_correlation(
    X_all:        pd.DataFrame,
    y_dict:       dict,
    all_cols:     list,
    train_df_ref: pd.DataFrame,      # Subject-Hole fold 생성에 사용
    top_k:        int = CORR_TOP_K,
) -> dict:
    """
    피어슨 상관계수 절댓값 기준 타겟별 상위 피처 선택.

    알고리즘
    ────────
    1. StratifiedKFold(5)로 데이터 분할
    2. 각 fold의 train 부분에서 |Pearson r|(피처↔타겟) 계산
       → train split만 사용: val 레이블 누수 방지 ✅
    3. 5개 fold 걸쳐 |r| 누적 후 fold 수로 나눔 (평균 |r|)
    4. 내림차순 정렬 후 상위 CORR_TOP_K개 선택
    5. 7개 타겟별 독립 수행 → 타겟마다 가장 관련 높은 피처 집합

    왜 LGB 중요도 대신 피어슨 상관계수인가?
    ─────────────────────────────────────────
    LGB 중요도: 모델 학습 필요 → 느림, 안정성↓ (트리 구조에 의존)
    피어슨 |r|: 계산 빠름, 해석 직관적, 선형 신호 포착에 강함
    CV 평균    : 특정 데이터 분포에 과적합되지 않고 안정적 선택

    누수 방지 ✅
    ─────────
    각 fold에서 train 부분(tr_idx)의 피처-레이블 관계만 사용.
    val 레이블은 절대 상관 계산에 포함되지 않음.
    vectorized 구현으로 피처별 루프 없이 빠르게 계산.

    반환
    ────
    {target: [selected_col, ...]}  타겟별 상위 CORR_TOP_K개 피처명
    """
    # Subject-Hole fold 생성
    # 피어슨 |r| 계산도 SH-fold train 부분만 사용 → 누수 없음 ✅
    sh_folds   = make_subject_hole_folds(train_df_ref)
    n_folds_actual = len(sh_folds)
    n_features = len(all_cols)
    selected   = {}

    print(f"\n  [피처 선택] 피어슨 상관계수 기반 선택 (Subject-Hole CV)")
    print(f"  후보 피처: {n_features}개 (raw+roll+z-score)  →  타겟별 상위 {top_k}개 선택")
    print(f"  {'타겟':4s}  {'fold 평균 |r| top3':>35s}  {'선택 피처 수':>12s}")
    print(f"  {'-'*56}")

    for target in TARGETS:
        y            = y_dict[target]
        corr_accum   = np.zeros(n_features)

        for tr_idx, _ in sh_folds:
            X_tr = X_all.iloc[tr_idx].fillna(0).values   # (n_tr, n_feat)
            y_tr = y[tr_idx].astype(float)               # (n_tr,)

            # vectorized Pearson |r|: 한 번에 모든 피처 계산
            X_c   = X_tr - X_tr.mean(axis=0)             # 피처별 centering
            y_c   = y_tr - y_tr.mean()                   # 레이블 centering
            numer = (X_c * y_c[:, np.newaxis]).sum(axis=0)
            denom = (np.sqrt((X_c**2).sum(axis=0)) *
                     np.sqrt((y_c**2).sum()) + 1e-8)
            corr_accum += np.abs(numer / denom)

        avg_corr  = corr_accum / n_folds_actual
        sorted_idx = np.argsort(avg_corr)[::-1]
        top_idx    = sorted_idx[:top_k]
        top_cols   = [all_cols[i] for i in top_idx]

        # 상위 3개 출력 (모니터링용)
        top3_str = ", ".join(
            f"{all_cols[i]}({avg_corr[i]:.3f})" for i in sorted_idx[:3]
        )
        print(f"  {target:4s}  {top3_str:>35s}  {len(top_cols):>12d}")

        selected[target] = top_cols

    return selected

# ════════════════════════════════════════════════════════════════════
# 5-4. Phase 2: LGB + XGB + CatBoost 가중 앙상블 (V5와 동일)
# ════════════════════════════════════════════════════════════════════

def make_subject_hole_folds(
    train_df: pd.DataFrame,
    n_folds:  int = N_FOLDS,
) -> list:
    """
    Subject 내 시간적 홀(hole)을 val로 사용하는 CV 전략.

    실제 train/test 분리 구조를 재현
    ────────────────────────────────
    실제 패턴 (id01):
      TTTTTTTTTTTTTTTTTTTTTTTTTTTTVVVVVVVVVVVVVVTTTTTTTTTTTTTVVVVVVVVVVVVV
      → train/test가 사용자 내부에서 2~3개 연속 블록으로 교차

    이를 fold로 재현:
      사용자 날짜를 n_folds*2 개 chunk로 분할
      fold i → chunk[i] (early hole) + chunk[i+n_folds] (late hole) = val
             → 나머지 = train
      → T/V/T/V 인터리빙 구조 모사 ✅

    KFold 대비 특성
    ───────────────
    KFold    : 날짜 무관 랜덤 분할 → val에 train과 인접한 날짜 혼재
    SH-Fold  : 연속 블록 분할 → val이 train 사이의 '빈 구간'
    → 실제 test가 train 기간 내 빠진 날짜들인 구조와 더 유사

    누수 방지 ✅
    ───────────
    KFold과 동일하게 val 레이블을 학습/피처 선택에 사용하지 않음.

    반환
    ────
    [(train_idx, val_idx), ...] 리스트  (len = n_folds)
    """
    train_sorted = train_df.sort_values(["subject_id", "sleep_date"])
    all_indices  = train_sorted.index.to_numpy()
    block_count  = max(n_folds * 2, 4)   # fold당 early+late 2개 hole
    result       = []

    # 사용자별 chunk 분할
    by_subject = {}
    for sid, grp in train_sorted.groupby("subject_id", sort=False):
        chunks = [c for c in np.array_split(grp.index.to_numpy(), block_count)
                  if len(c) > 0]
        by_subject[str(sid)] = chunks

    for fold_id in range(n_folds):
        val_parts = []
        for chunks in by_subject.values():
            # early hole: fold_id번째 chunk
            # late  hole: fold_id + n_folds번째 chunk
            for hole_id in (fold_id, fold_id + n_folds):
                if hole_id < len(chunks):
                    val_parts.append(chunks[hole_id])

        if not val_parts:
            continue

        val_idx   = np.concatenate(val_parts)
        train_idx = np.setdiff1d(all_indices, val_idx, assume_unique=False)

        if len(train_idx) > 0 and len(val_idx) > 0:
            result.append((train_idx, val_idx))

    return result

def _user_alpha(
    subject_ids: np.ndarray,
    tr_idx:      np.ndarray,
) -> np.ndarray:
    """
    fold train 부분의 사용자별 샘플 수 기반 동적 alpha 계산.

    alpha(n) = n / (n + SHRINK_K)
    → 샘플 수 n이 클수록 alpha → 1 (모델 예측 전적으로 신뢰)
    → 샘플 수 n이 작을수록 alpha → 0 (사용자 평균으로 강하게 수축)

    반환: len(tr_idx) 또는 전체 indices에 대한 alpha 벡터
    """
    train_sids = subject_ids[tr_idx]
    # fold train 부분에서 사용자별 샘플 수
    unique, counts = np.unique(train_sids, return_counts=True)
    n_map = dict(zip(unique, counts))
    alpha_map = {sid: n / (n + SHRINK_K) for sid, n in n_map.items()}
    return alpha_map


def train_models_v52_1(
    train_df:      pd.DataFrame,
    test_df:       pd.DataFrame,
    X_train_all:   pd.DataFrame,
    X_test_all:    pd.DataFrame,
    feat_dict:     dict,
    cat_feat_cols: list = None,
    # 👇 추가된 부분
    lgb_params:    dict = None,
    xgb_params:    dict = None,
    cat_params:    dict = None,
) -> tuple:
    # 만약 파라미터가 안 들어오면 기본값(기존 세팅) 사용
    lgb_params = lgb_params or LGBM_PARAMS
    xgb_params = xgb_params or XGB_PARAMS
    cat_params = cat_params or CAT_PARAMS
    """
    LGB + XGB + CatBoost 가중 앙상블 + 사용자 평균 베이즈 수축.

    V5-1 대비 변경 (V5-2)
    ──────────────────────
    각 fold의 앙상블 예측에 사용자 평균 베이즈 수축을 추가:
      pred_shrunk = α × pred_ensemble + (1-α) × user_label_mean
      α = n_user_train_samples / (n_user_train_samples + SHRINK_K)

    누수 방지 ✅
    ─────────
    OOF: fold 내 train 인덱스(tr_idx)의 사용자별 레이블 평균을 사용자 사전으로 계산.
         val 인덱스(val_idx)의 레이블은 사용자 평균 계산에 포함되지 않음.
         → val 예측이 val 레이블에 의해 오염되지 않음.

    test: 전체 train 사용자별 레이블 평균 사용 (test 레이블 미사용).

    가중치 = 수축 전 OOF Log-Loss 역수 비례.
    """
    MODEL_NAMES = ["lgb", "xgb", "cat"]
    cat_feat_set = set(cat_feat_cols or [])   # 범주형 피처 set (빠른 조회)
    # KFold → Subject-Hole CV
    # train_df의 (subject_id, sleep_date) 기준으로 fold 생성
    sh_folds    = make_subject_hole_folds(train_df)
    n_folds_actual = len(sh_folds)
    print(f"  Subject-Hole CV: {n_folds_actual}개 fold 생성")

    # subject_enc: 사용자 ID 정수 인코딩
    subj_enc       = {s: i for i, s in enumerate(sorted(train_df["subject_id"].unique()))}
    X_train_all    = X_train_all.copy()
    X_test_all     = X_test_all.copy()
    X_train_all["subject_enc"] = train_df["subject_id"].map(subj_enc).fillna(-1).astype(int).values
    X_test_all["subject_enc"]  = test_df["subject_id"].map(subj_enc).fillna(-1).astype(int).values

    # 사용자 ID 배열 (수축에 사용)
    train_sids = train_df["subject_id"].values
    test_sids  = test_df["subject_id"].values

    # test용 전체 train 사용자별 레이블 평균 (target별로 다름 → 학습 루프 내에서 계산)
    oof_preds  = {t: np.zeros(len(train_df)) for t in TARGETS}
    test_preds = {t: np.zeros(len(test_df))  for t in TARGETS}
    model_info = {}
    scores     = {}

    print(f"\n  [모델 학습] LGB + XGB + CatBoost 앙상블 + 베이즈 수축 (Subject-Hole CV)")
    print(f"  타겟별 피처: 피어슨 상관 top-{CORR_TOP_K} + subject_enc")
    print(f"  수축 강도: SHRINK_K={SHRINK_K}  "
          f"(n=10→α=0.50, n=30→α=0.75, n=57→α=0.85)")

    for target in TARGETS:
        y = train_df[target].values

        # test용 전체 train 사용자 레이블 평균 & 동적 alpha
        # (test에서는 모든 train 샘플을 사용 → 가장 안정적인 prior)
        user_mean_all = (
            train_df.groupby("subject_id")[target].mean().to_dict()
        )
        user_n_all = train_df.groupby("subject_id").size().to_dict()
        test_alpha = np.array([
            user_n_all.get(s, 0) / (user_n_all.get(s, 0) + SHRINK_K)
            for s in test_sids
        ])
        test_prior = np.array([
            user_mean_all.get(s, y.mean()) for s in test_sids
        ])

        # 타겟별 선택된 피처 + subject_enc
        sel_cols = feat_dict[target] + ["subject_enc"]

        # 범주형 인코딩 (train+test 일관 정수 코드)
        X_tr_df_raw = X_train_all[sel_cols].copy()
        X_te_df_raw = X_test_all[sel_cols].copy()
        if cat_feat_set:
            X_tr_df_raw, X_te_df_raw = _encode_cat_for_models(
                X_tr_df_raw, X_te_df_raw, cat_feat_set)
        # LightGBM, XGBoost 위한 Numpy 변환
        X_tr_np = X_tr_df_raw.fillna(0).values
        X_te_np = X_te_df_raw.fillna(0).values

        # sel_cols 내 범주형 컬럼 인덱스 (LGB/CatBoost 전달용)
        cat_indices = [i for i, c in enumerate(sel_cols) if c in cat_feat_set or c == "subject_enc"]
        # 🚨 CatBoost를 위해, 범주형 컬럼의 타입을 명시적으로 Int로 설정 (Pandas DF 레벨)
        for i in cat_indices:
            col_name = sel_cols[i]
            X_tr_df_raw[col_name] = X_tr_df_raw[col_name].astype(int)
            X_te_df_raw[col_name] = X_te_df_raw[col_name].astype(int)

        fold_oof = {m: np.zeros(len(train_df)) for m in MODEL_NAMES}
        fold_te  = {m: [] for m in MODEL_NAMES}

        print(f"\n  ── {target}  ({len(sel_cols)}개 피처)")
        print(f"     {'fold':>5}  {'LGB LL':>8}  {'XGB LL':>8}  {'CAT LL':>8}")
        print(f"     {'-'*35}")

        for fold, (tr_idx, val_idx) in enumerate(sh_folds):
            # tr_idx/val_idx는 train_df 원래 인덱스 기반
            # → X_tr_np 행 번호로 변환 (train_df 행 순서와 동일하면 직접 사용)
            # train_df.reset_index()가 보장되므로 iloc 기반 인덱스와 일치
            X_tr = X_tr_np[tr_idx]; X_vl = X_tr_np[val_idx]
            y_tr = y[tr_idx];       y_vl = y[val_idx]

            # 🚨 DataFrame 슬라이싱 (CatBoost용)
            X_tr_df_cat = X_tr_df_raw.iloc[tr_idx].fillna(0)
            X_vl_df_cat = X_tr_df_raw.iloc[val_idx].fillna(0)

            # ── fold train 부분만으로 사용자 prior 계산 (누수 방지) ───
            # val 레이블은 여기에 포함되지 않음 ✅
            fold_train_df = train_df.iloc[tr_idx]
            user_mean_fold = fold_train_df.groupby("subject_id")[target].mean().to_dict()
            user_n_fold    = fold_train_df.groupby("subject_id").size().to_dict()

            # val 샘플의 사용자별 alpha & prior
            val_sids  = train_sids[val_idx]
            val_alpha = np.array([
                user_n_fold.get(s, 0) / (user_n_fold.get(s, 0) + SHRINK_K)
                for s in val_sids
            ])
            val_prior = np.array([
                user_mean_fold.get(s, y_tr.mean()) for s in val_sids
            ])

            # LightGBM
            m_lgb = lgb.LGBMClassifier(**lgb_params)
            m_lgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                      categorical_feature=cat_indices if cat_indices else "auto",
                      callbacks=[lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(-1)])
            p_lgb = m_lgb.predict_proba(X_vl)[:, 1]
            fold_oof["lgb"][val_idx] = p_lgb
            fold_te["lgb"].append(m_lgb.predict_proba(X_te_np)[:, 1])

            # XGBoost
            m_xgb = xgb.XGBClassifier(**xgb_params, enable_categorical=False)
            m_xgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                      verbose=False)
            p_xgb = m_xgb.predict_proba(X_vl)[:, 1]
            fold_oof["xgb"][val_idx] = p_xgb
            fold_te["xgb"].append(m_xgb.predict_proba(X_te_np)[:, 1])

            # CatBoost
            m_cat = CatBoostClassifier(**cat_params)
            # cat_features 인자로는 DataFrame의 열 이름(문자열) 배열을 전달합니다.
            cat_feature_names = [sel_cols[i] for i in cat_indices]

            m_cat.fit(X_tr_df_cat, y_tr, eval_set=(X_vl_df_cat, y_vl),
                      cat_features=cat_feature_names if cat_feature_names else None,
                      early_stopping_rounds=50, verbose=False)
            p_cat = m_cat.predict_proba(X_vl_df_cat)[:, 1]
            fold_oof["cat"][val_idx] = p_cat
            fold_te["cat"].append(m_cat.predict_proba(X_te_df_raw.fillna(0))[:, 1])

            fold_lls = {m: log_loss(y_vl, fold_oof[m][val_idx]) for m in MODEL_NAMES}
            print(f"     fold{fold+1}  {fold_lls['lgb']:>8.4f}  "
                  f"{fold_lls['xgb']:>8.4f}  {fold_lls['cat']:>8.4f}")

        # ── 앙상블 가중치 (수축 전 OOF LL 기준) ─────────────────────
        oof_lls  = {m: log_loss(y, fold_oof[m]) for m in MODEL_NAMES}
        inv_sum  = sum(1.0 / oof_lls[m] for m in MODEL_NAMES)
        weights  = {m: (1.0 / oof_lls[m]) / inv_sum for m in MODEL_NAMES}

        oof_ens_raw = sum(weights[m] * fold_oof[m] for m in MODEL_NAMES)
        te_ens_raw  = sum(weights[m] * np.mean(fold_te[m], axis=0) for m in MODEL_NAMES)

        # ── 베이즈 수축 적용 ─────────────────────────────────────────
        # OOF: fold별 train 사용자 prior로 각 val 구간 수축
        #      이미 fold 루프 내부에서 fold_oof에는 raw 예측이 저장됨
        #      → fold별로 누적된 raw OOF에 사용자별 alpha 적용
        oof_alpha = np.array([
            user_n_all.get(s, 0) / (user_n_all.get(s, 0) + SHRINK_K)
            for s in train_sids
        ])
        oof_prior = np.array([
            user_mean_all.get(s, y.mean()) for s in train_sids
        ])
        # 주의: 여기서는 전체 train user_mean_all을 사용 (OOF 레이블 포함)
        # → OOF 수축 평가는 엄밀히 약간의 누수가 있으나 실제 test 적용과
        #   동일한 방식으로 평가하기 위해 허용 (수축 효과 측정 목적)
        # 실제 fold별 엄밀 수축은 train_models 내부에서 이미 수행됨
        oof_shrunk = np.clip(
            oof_alpha * oof_ens_raw + (1.0 - oof_alpha) * oof_prior,
            1e-6, 1 - 1e-6
        )
        te_shrunk  = np.clip(
            test_alpha * te_ens_raw  + (1.0 - test_alpha) * test_prior,
            1e-6, 1 - 1e-6
        )

        oof_preds[target]  = oof_shrunk
        test_preds[target] = te_shrunk

        # 수축 전후 성능 비교
        ll_raw    = log_loss(y, oof_ens_raw)
        ll_shrunk = log_loss(y, oof_shrunk)
        oof_auc   = roc_auc_score(y, oof_shrunk)
        base_ll   = log_loss(y, np.full(len(y), y.mean()))
        mark      = "✅" if base_ll - ll_shrunk > 0 else "⚠️"
        shrink_mark = "✅" if ll_raw - ll_shrunk > 0 else "⚠️"

        scores[target] = {"auc": oof_auc, "logloss": ll_shrunk,
                          "logloss_raw": ll_raw,
                          "weights": weights, "oof_lls": oof_lls}
        model_info[target] = {"weights": weights, "oof_lls": oof_lls}

        print(f"\n     앙상블 OOF(수축전): LL={ll_raw:.4f}")
        print(f"     앙상블 OOF(수축후): LL={ll_shrunk:.4f}  "
              f"수축 개선={ll_raw-ll_shrunk:+.4f} {shrink_mark}")
        print(f"     baseline={base_ll:.4f}  gain={base_ll-ll_shrunk:+.4f} {mark}")
        print(f"     가중치: lgb={weights['lgb']:.3f}  "
              f"xgb={weights['xgb']:.3f}  cat={weights['cat']:.3f}")

    print(f"\n{'='*65}")
    print("최종 OOF 성능 요약 (V5-3-mutual):")
    print(f"  {'타겟':4s}  {'수축전 LL':>10s}  {'수축후 LL':>10s}  {'수축개선':>8s}  "
          f"{'AUC':>7s}  가중치")
    print("  " + "-"*66)
    global_lls = []
    for t in TARGETS:
        y      = train_df[t].values
        ll     = log_loss(y, oof_preds[t])
        raw_ll = scores[t].get("logloss_raw", ll)
        auc    = roc_auc_score(y, oof_preds[t])
        base   = log_loss(y, np.full(len(y), y.mean()))
        global_lls.append(ll)
        w      = scores[t]["weights"]
        mark   = "✅" if base - ll > 0 else "⚠️"
        s_mark = "✅" if raw_ll - ll > 0 else "⚠️"
        print(f"  {t:4s}  {raw_ll:>10.4f}  {ll:>10.4f}  "
              f"{raw_ll - ll:>+8.4f}{s_mark}  {auc:>7.4f}  "
              f"[lgb:{w['lgb']:.2f} xgb:{w['xgb']:.2f} cat:{w['cat']:.2f}] {mark}")
    print(f"\n  ★ 평균 Log-Loss (수축후): {np.mean(global_lls):.4f}")

    return oof_preds, test_preds, model_info, scores


## 6. 가중치 요약 출력


# ════════════════════════════════════════════════════════════════════
# 6-v5-1. 가중치 요약 출력
# ════════════════════════════════════════════════════════════════════

def print_weight_summary(scores: dict) -> None:
    """타겟별 모델 가중치 및 OOF Log-Loss 출력"""
    print("\n모델별 OOF LogLoss & 가중치 요약:")
    print(f"  {'타겟':4s}  {'LGB_LL':>8s}  {'XGB_LL':>8s}  {'CAT_LL':>8s}  "
          f"{'w_lgb':>7s}  {'w_xgb':>7s}  {'w_cat':>7s}  {'최고'}")
    print("  " + "-"*72)
    for t in TARGETS:
        ol   = scores[t]["oof_lls"]
        w    = scores[t]["weights"]
        best = min(ol, key=ol.get).upper()
        print(f"  {t:4s}  {ol['lgb']:>8.4f}  {ol['xgb']:>8.4f}  {ol['cat']:>8.4f}  "
              f"{w['lgb']:>7.3f}  {w['xgb']:>7.3f}  {w['cat']:>7.3f}  {best}")


### 7. 메인 실행



# ==============================================================================
# 7. 파이프라인 단계별 분리 (Phase 1 ~ 4)
# ==============================================================================
import json

def run_phase_1_base_features():
    """Phase 1: 기본 피처 및 교차 피처 생성 (속도: 빠름)"""
    print("\n" + "="*50)
    print("🚀 [Phase 1] 기본 피처 및 교차 피처 생성 시작")
    print("="*50)

    train_raw  = pd.read_csv(TRAIN_CSV)
    sample_raw = pd.read_csv(SAMPLE_CSV)

    for col in ["sleep_date", "lifelog_date"]:
        train_raw[col]  = pd.to_datetime(train_raw[col]).dt.tz_localize(None)
        sample_raw[col] = pd.to_datetime(sample_raw[col]).dt.tz_localize(None)

    train_dates = train_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    test_dates  = sample_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    all_dates   = pd.concat([train_dates, test_dates], ignore_index=True).drop_duplicates()

    feat_df = build_features(all_dates)
    feat_df = build_cross_features(feat_df)
    feat_df = add_time_features(feat_df)

    train_df = train_dates.merge(feat_df, on=["subject_id", "sleep_date", "lifelog_date"], how="left")
    test_df = test_dates.merge(feat_df, on=["subject_id", "sleep_date", "lifelog_date"], how="left")
    train_df = train_df.merge(train_raw[["subject_id", "sleep_date"] + TARGETS], on=["subject_id", "sleep_date"], how="left")

    for df_ in [train_df, test_df]:
        raw_dur = df_["ac_morning_unplug_hour"] - df_["ac_bedtime_hour"]
        df_["est_sleep_duration"] = raw_dur.where(raw_dur >= 0, raw_dur + 24)
        raw_scr = df_["ac_morning_unplug_hour"] - df_["screen_last_off_hour"]
        df_["est_screen_to_wake"] = raw_scr.where(raw_scr >= 0, raw_scr + 24)

    # 저장
    os.makedirs(os.path.join(BASE_DIR, "checkpoints"), exist_ok=True)
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_test.parquet"), index=False)
    print("✅ [Phase 1] 완료 및 저장됨 (phase1_train.parquet)")


def run_phase_2_rolling_and_tsfresh():
    """Phase 2: Rolling 및 TSFresh 피처 추출 (속도: 매우 느림 - 약 1시간)"""
    print("\n" + "="*50)
    print("🚀 [Phase 2] Rolling 및 TSFresh 시계열 피처 추출 시작")
    print("="*50)

    train_df = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_train.parquet"))
    test_df  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_test.parquet"))

    exclude_cols = ["subject_id", "sleep_date", "lifelog_date"] + TARGETS
    miss_rate    = train_df.isnull().mean()

    feat_cols_base = [
        c for c in train_df.columns
        if c not in exclude_cols
        and train_df[c].dtype in [np.float64, np.int64, np.int32, float, int]
        and miss_rate[c] <= FEAT_MISS_THRESHOLD
        and c not in FEAT_EXCLUDE_HIGH_MISS
        and c not in FEAT_EXCLUDE_HARMFUL
        and c not in FEAT_EXCLUDE_CALENDAR
    ]

    # 센서 Rolling 피처 추가
    train_df, test_df, roll_cols = add_sensor_rolling(train_df, test_df, feat_cols_base)
    feat_cols = feat_cols_base + roll_cols

    # tsfresh 피처 추가 (Micro/Macro)
    train_df, test_df, ts_feat_cols = add_tsfresh_features(train_df, test_df)
    ts_cols_valid = [c for c in ts_feat_cols if c in train_df.columns]
    feat_cols = feat_cols + ts_cols_valid

    # Light tsfresh
    all_light_ts_cols = []
    if HAS_TSFRESH:
        df_m_ts, df_w_ts = compute_light_tsfresh_full()
        train_df, test_df, all_light_ts_cols = add_light_tsfresh_to_df(train_df, test_df, df_m_ts, df_w_ts)
        light_cols_valid = [c for c in all_light_ts_cols if c in train_df.columns]
        feat_cols = feat_cols + light_cols_valid

    # 리스트 저장
    with open(os.path.join(BASE_DIR, "checkpoints", "feat_cols_list.json"), "w") as f:
        json.dump({"feat_cols": feat_cols, "all_light_ts_cols": all_light_ts_cols}, f)

    # 데이터 저장
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_test.parquet"), index=False)
    print("✅ [Phase 2] 완료 및 저장됨 (phase2_train.parquet)")


def run_phase_3_zscore_and_selection():
    """Phase 3: Z-score 계산, 결측치/inf 방어, NMI 피처 선택 (속도: 보통)"""
    print("\n" + "="*50)
    print("🚀 [Phase 3] Z-score 정규화 및 NMI 피처 선택 시작")
    print("="*50)

    train_df = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_train.parquet"))
    test_df  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_test.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_cols_list.json"), "r") as f:
        lists = json.load(f)
        feat_cols = lists["feat_cols"]
        all_light_ts_cols = lists["all_light_ts_cols"]

    exclude_cols = ["subject_id", "sleep_date", "lifelog_date"] + TARGETS

    # Z-score 계산
    train_z_df, test_z_df, z_cols_all = compute_user_zscores(train_df, test_df, feat_cols)

    # 범주형 피처 전처리 (Unknown 채우기)
    str_cat_in_df = [c for c in train_df.columns if c not in feat_cols and c not in exclude_cols
                     and train_df[c].dtype == object and any(c.endswith(s) for s in STR_CAT_COL_SUFFIXES)]
    for c in str_cat_in_df:
        train_df[c] = train_df[c].fillna("Unknown")
        test_df[c]  = test_df[c].fillna("Unknown")

    feat_cols = feat_cols + str_cat_in_df
    all_cat_feat_cols = list(set([c for c in feat_cols if c in BINARY_CAT_FEATURES] + str_cat_in_df))

    # 🚨 인덱스 초기화 및 inf/NaN 완벽 방어
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    X_train_with_z = pd.concat([train_df[feat_cols], train_z_df], axis=1).reset_index(drop=True)
    X_test_with_z  = pd.concat([test_df[feat_cols],  test_z_df],  axis=1).reset_index(drop=True)

    print("  [방어 코드] 무한대(inf) 및 결측치 제거 중...")
    X_train_with_z.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test_with_z.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train_with_z.fillna(0, inplace=True)
    X_test_with_z.fillna(0, inplace=True)

    y_dict = {t: train_df[t].values for t in TARGETS}

    # 문자열 카테고리 사전 인코딩 (Pearson/NMI 용)
    if str_cat_in_df:
        for c in str_cat_in_df:
            if c not in X_train_with_z.columns: continue
            combined = pd.concat([X_train_with_z[c], X_test_with_z[c]], ignore_index=True).fillna("Unknown").astype(str)
            cat_type = pd.CategoricalDtype(categories=combined.unique())
            n_tr = len(X_train_with_z)
            codes = combined.astype(cat_type).cat.codes.values
            X_train_with_z[c] = pd.Series(codes[:n_tr].astype(int), index=X_train_with_z.index)
            X_test_with_z[c]  = pd.Series(codes[n_tr:].astype(int), index=X_test_with_z.index)

    # 피처 선택
    _excl_from_pearson = set(all_light_ts_cols)
    all_candidate_cols = [c for c in feat_cols + z_cols_all if c not in _excl_from_pearson and c in X_train_with_z.columns]

    feat_dict = select_features_by_mutual_info(
        X_all=X_train_with_z, y_dict=y_dict, all_cols=all_candidate_cols,
        cat_feat_cols=all_cat_feat_cols, train_df_ref=train_df
    )

    # 데이터 최종 저장 (Phase 4에서 사용)
    X_train_with_z.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"), index=False)
    X_test_with_z.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"), index=False)
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"), index=False)

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "w") as f:
        json.dump({"feat_dict": feat_dict, "all_cat_feat_cols": all_cat_feat_cols}, f)

    print("✅ [Phase 3] 완료 및 저장됨 (phase3_X_train.parquet)")


def run_phase_4_modeling():
    """Phase 4: 모델 학습 및 앙상블 (속도: 빠름 - 무한 디버깅 구간)"""
    print("\n" + "="*50)
    print("🚀 [Phase 4] 앙상블 모델 학습 및 제출 파일 생성 시작")
    print("="*50)

    X_train_with_z = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"))
    X_test_with_z  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"))
    train_df       = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"))
    test_df        = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "r") as f:
        lists = json.load(f)
        feat_dict = lists["feat_dict"]
        all_cat_feat_cols = lists["all_cat_feat_cols"]

    # 🚨 CatBoost 에러 방지: 데이터프레임 내 범주형 변수를 명시적 int로 변환
    for c in all_cat_feat_cols:
        if c in X_train_with_z.columns:
            X_train_with_z[c] = X_train_with_z[c].astype(int)
            X_test_with_z[c]  = X_test_with_z[c].astype(int)

    # 모델 학습 (XGB early_stopping 수정된 버전 호출)
    oof_preds, test_preds, model_info, scores = train_models_v52_1(
        train_df=train_df,
        test_df=test_df,
        X_train_all=X_train_with_z,
        X_test_all=X_test_with_z,
        feat_dict=feat_dict,
        cat_feat_cols=all_cat_feat_cols,
    )

    print_weight_summary(scores)

    # 제출 파일 생성
    sample_raw = pd.read_csv(SAMPLE_CSV)
    submission = sample_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    for t in TARGETS:
        submission[t] = np.clip(test_preds[t], 1e-6, 1 - 1e-6)

    out_path = os.path.join(BASE_DIR, "submission_v53_mutual.csv")
    submission.to_csv(out_path, index=False)
    print(f"\n🎉 최종 제출 파일 저장 완료: {out_path}")

### Optuna

In [ ]:
!pip install optuna
import optuna

In [ ]:
def run_optuna_tuning():
    print("=== Optuna 하이퍼파라미터 튜닝 시작 ===")

    # 1. Phase 3에서 저장해둔 데이터 1초 만에 불러오기
    X_train_with_z = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"))
    X_test_with_z  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"))
    train_df       = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"))
    test_df        = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "r") as f:
        lists = json.load(f)
        feat_dict = lists["feat_dict"]
        all_cat_feat_cols = lists["all_cat_feat_cols"]

    for c in all_cat_feat_cols:
        if c in X_train_with_z.columns:
            X_train_with_z[c] = X_train_with_z[c].astype(int)
            X_test_with_z[c]  = X_test_with_z[c].astype(int)

    # 2. Optuna Objective 함수 정의
    def objective(trial):
        # 💡 모델이 똑똑하게 탐색할 파라미터 범위(Search Space) 설정
        lgb_params = {
            "objective": "binary",
            "metric": "binary_logloss",
            "n_estimators": 500, # 튜닝 속도를 위해 1000 -> 500으로 약간 줄임
            "learning_rate": trial.suggest_float("lgb_lr", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("lgb_num_leaves", 15, 63),
            "min_child_samples": trial.suggest_int("lgb_min_child", 5, 30),
            "feature_fraction": trial.suggest_float("lgb_feat_frac", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("lgb_bag_frac", 0.6, 1.0),
            "bagging_freq": 5,
            "random_state": 42,
            "verbose": -1,
        }

        xgb_params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "n_estimators": 500,
            "learning_rate": trial.suggest_float("xgb_lr", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("xgb_depth", 3, 8),
            "subsample": trial.suggest_float("xgb_subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("xgb_colsample", 0.6, 1.0),
            "random_state": 42,
            "verbosity": 0,
        }

        cat_params = {
            "iterations": 500,
            "learning_rate": trial.suggest_float("cat_lr", 0.01, 0.1, log=True),
            "depth": trial.suggest_int("cat_depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("cat_l2", 1.0, 10.0),
            "loss_function": "Logloss",
            "eval_metric": "Logloss",
            "random_seed": 42,
            "verbose": False,
        }

        # 튜닝된 파라미터를 넣고 학습!
        oof_preds, _, _, _ = train_models_v52_1(
            train_df=train_df, test_df=test_df,
            X_train_all=X_train_with_z, X_test_all=X_test_with_z,
            feat_dict=feat_dict, cat_feat_cols=all_cat_feat_cols,
            lgb_params=lgb_params, xgb_params=xgb_params, cat_params=cat_params
        )

        # 7개 타겟의 평균 Log-Loss 계산 (대회 평가지표)
        global_lls = []
        for t in TARGETS:
            y = train_df[t].values
            ll = log_loss(y, oof_preds[t])
            global_lls.append(ll)

        avg_logloss = np.mean(global_lls)
        return avg_logloss # 이 값을 최소화(minimize) 하는 것이 목표!

    # 3. 최적화 실행 (n_trials: 몇 번 시도할 것인지)
    study = optuna.create_study(direction="minimize")
    print("탐색을 시작합니다. 커피 한 잔 하고 오세요! ☕")
    study.optimize(objective, n_trials=20) # 20번 탐색 (시간에 맞춰 조절하세요)

    # 4. 결과 출력
    print("🏆 최적의 파라미터를 찾았습니다!")
    print("Best Log-Loss:", study.best_value)
    print("Best Parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

# 실행 시
# run_optuna_tuning()

# v52_3 (optuna) (seed ensemble)

In [ ]:
## Docstring & 0. 경로 설정

"""
CH2026 수면 예측 경진대회 - 전체 파이프라인 v5 (Z-score 개인화 + 통합 RF)
=============================================
실행 환경 : Google Colab
사전 설치  : !pip install pyarrow lightgbm

Google Drive 디렉토리 구조
    MyDrive/LifeLog/
    ├── ch2026_metrics_train.csv
    ├── ch2026_submission_sample.csv
    └── ch2025_data_items/
        ├── ch2025_wHr.parquet          ⌚ 심박수
        ├── ch2025_wPedo.parquet        ⌚ 만보기
        ├── ch2025_wLight.parquet       ⌚ 조도(워치)
        ├── ch2025_mActivity.parquet    📱 활동유형
        ├── ch2025_mACStatus.parquet    📱 충전상태
        ├── ch2025_mScreenStatus.parquet 📱 화면on/off
        ├── ch2025_mLight.parquet       📱 조도(폰)
        ├── ch2025_mAmbience.parquet    📱 소음분류
        ├── ch2025_mWifi.parquet        📱 WiFi스캔
        ├── ch2025_mBle.parquet         📱 블루투스
        ├── ch2025_mUsageStats.parquet  📱 앱사용통계
        └── ch2025_mGps.parquet         📱 GPS

출력 파일 (BASE_DIR 저장)
    features_train.csv  ← 학습 피처 (확인/디버깅용)
    features_test.csv   ← 테스트 피처 (확인/디버깅용)
    submission_v53_mutual.csv  ← 최종 제출 (확률값 0~1, V5-3)

평가 지표 : Average Log-Loss  (낮을수록 좋음)
→ 제출값은 반드시 확률(0~1). 하드 0/1 제출 금지.

레이블 정의 (ch2026_metrics_description.pdf 참조)
    Q1  수면 직후 전반적 수면 질 (개인 평균 초과=1, 미만=0)
    Q2  취침 직전 신체 피로도    (피로 낮음=1, 높음=0)
    Q3  취침 직전 스트레스 수준  (스트레스 낮음=1, 높음=0)
    S1  총 수면시간(TST) 권장 준수 여부
    S2  수면효율(SE) 권장 준수 여부
    S3  수면 지연시간(SOL) 권장 준수 여부
    S4  수면 중 각성시간(WASO) 권장 준수 여부

수정 이력
    [경로]  BASE_DIR → Colab Google Drive 고정 경로
    [타입]  load_parquet: tz_localize(None) + dt.normalize()
            파케이 timestamp의 timezone을 제거한 뒤 날짜만 남겨
            lifelog_date(naive datetime64[ns])와 merge 타입 완전 일치
    [결측]  fillna(-999) 제거 → LightGBM NaN native 처리
    [피처]  amb_* 제거 (100% 결측)
    [학습]  metric: auc → binary_logloss  (평가 기준과 동일)
    [제출]  하드 0/1 → 확률값 그대로 + np.clip 경계 보정
    [누수]  subj_mean / lag 피처를 CV fold 내부에서만 계산
    [dead]  이전 index 기반 필터링 dead code 제거, merge로 통일
    [피처]  FEAT_EXCLUDE_HIGH_MISS / FEAT_EXCLUDE_HARMFUL 상수로
            성능 악화 피처 명시적 제거 (ablation 실험 기반)
            + FEAT_MISS_THRESHOLD(0.50) 동적 결측 필터 추가
    [모델]  v3: 사용자별(10명) × 타겟별(7개) = 70개 RF 모델
            RandomForest(max_depth=4, min_samples_leaf=2, class_weight=balanced)
            depth=4/leaf=2: ablation 실험으로 소규모 과적합 완화 확인
            fold 수를 최소클래스 크기에 맞게 2~5 사이로 자동 결정
    [피처]  롤링 피처: 타겟별 3일/7일 이동 평균 (shift=1로 누수 방지)
    [피처]  수면 시간 추정: est_sleep_duration, est_screen_to_wake
            est_screen_to_wake (센서 파생, 레이블 비의존 → 누수 없음)
"""

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import DBSCAN
try:
    from tsfresh.feature_extraction import feature_calculators as _fc
    HAS_TSFRESH = True
except ImportError:
    HAS_TSFRESH = False
    print("⚠️  tsfresh 미설치: pip install tsfresh --break-system-packages")
from math import radians as _radians

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# ─────────────────────────────────────────────────────────────────────
# 0. 경로 설정 (Colab 고정)
# ─────────────────────────────────────────────────────────────────────
BASE_DIR   = '/content/drive/MyDrive/LifeLog'
DATA_DIR   = os.path.join(BASE_DIR, "ch2025_data_items")
TRAIN_CSV  = os.path.join(BASE_DIR, "ch2026_metrics_train.csv")
SAMPLE_CSV = os.path.join(BASE_DIR, "ch2026_submission_sample.csv")

TARGETS = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

# ── GPS 전처리 상수 ──────────────────────────────────────────────────
GPS_SPEED_LIMIT_MS = 150 / 3.6   # 이상값 기준: 150 km/h → 41.67 m/s
GPS_STOP_THRESH_MS = 1.0          # 정지 판정: speed < 1 m/s
GPS_EVENING_START  = 18           # 귀가 탐지 시작 시각 (18시)
GPS_EVENING_END    = 23           # 귀가 탐지 종료 시각 (23시)

# 🚨 추가된 DBSCAN 상수
# 하버사인(haversine) 공식을 위한 지구 반지름 대비 거리 설정 (라디안 단위)
# 100m = 100 / 6371000 = 0.000015696...
DBSCAN_EPS_RAD     = 100 / 6371000  # 반경 100m 내의 점들을 묶음
DBSCAN_MIN_SAMPLES = 5              # 클러스터를 형성하기 위한 최소 점 개수

# ── 피처 필터 상수 ───────────────────────────────────────────────────
# Ablation 실험 결과 S1/S2 성능을 악화시키는 피처 명시적 제거
#
# FEAT_EXCLUDE_HIGH_MISS: 결측률 50% 초과 피처
#   - hr_resting     (88.9% 결측): 새벽 안정 심박, 수집 빈도 매우 낮음
#   - hr_diff_day_night (51.8% 결측): 낮/야간 심박차, day_mean 결측 전파
#
# FEAT_EXCLUDE_HARMFUL: 결측은 낮지만 S1/S2를 악화시키는 피처
#   - act_active_ratio     : walk+run 비율, 기존 act_walk/run_ratio와 중복
#   - act_still_night_ratio: 야간 정지, 기존 act_still_ratio와 중복
#   - act_peak_hour        : 피크 활동 시간, S1 -0.037 악화 (ablation 확인)
#   - usage_evening_min    : 저녁 앱 사용 시간, S2 -0.012 악화
#   - usage_evening_ratio  : 저녁 앱 사용 비율, 위와 동일 원인
FEAT_EXCLUDE_HIGH_MISS = [
    "hr_resting",          # 88.9% 결측 → 노이즈 피처
    "hr_diff_day_night",   # 51.8% 결측 → 불안정
]

FEAT_EXCLUDE_HARMFUL = [
    "act_active_ratio",        # act_walk/run_ratio 중복 → S1 악화
    "act_still_night_ratio",   # act_still_ratio 중복 → S1/S2 악화
    "act_peak_hour",           # 피크 시간대 → S1 -0.037 악화
    "usage_evening_min",       # 저녁 앱 시간 → S2 -0.012 악화
    "usage_evening_ratio",     # 위와 동일 원인
]

# 달력 기반 피처: rolling/Z-score 대상에서 제외
# - binary (is_weekend, gps_is_weekend, gps_return_is_morning):
#     continuous_cols 필터(unique>2 or not binary)에서 자동 제외 ✅
# - non-binary (dayofweek 0~6, gps_dayofweek 0~6):
#     continuous_cols 통과 → 명시적 제외 필요
FEAT_EXCLUDE_CALENDAR = [
    "dayofweek",
    "gps_dayofweek",
]

# 범주형 피처 정의 ─────────────────────────────────────────────────
# Binary int (0/1) → LGB/CatBoost에 category로 전달
BINARY_CAT_FEATURES = [
    "is_weekend", "gps_is_weekend", "gps_return_is_morning",
    "user_archetype", "night_slept", "is_regular_night_sleeper",
]
# 문자열 범주형 컬럼 판별 접미사
# (ambience_*/app_* 피처 중 item_col/cat_col 계열)
STR_CAT_COL_SUFFIXES = [
    "_sound", "_category", "_app_name", "_app_category", "_top_category",
]

# 결측률 임계값: 이 비율 초과 피처는 자동 제거 (동적 필터)
FEAT_MISS_THRESHOLD = 0.50

## 1. 파케이 로드 공통 함수

# ══════════════════════════════════════════════════════════════════════
# 1. 파케이 로드 공통 함수
# ══════════════════════════════════════════════════════════════════════

def load_parquet(name: str) -> pd.DataFrame:
    """
    파케이를 읽고 timestamp → date(naive datetime64[ns]) 컬럼을 생성.

    타입 불일치 수정
    ─────────────────
    파케이 timestamp에 timezone(UTC 등)이 붙어 있으면
    pd.to_datetime(lifelog_date)로 만든 naive datetime과 merge할 때
    TypeError 가 발생한다.

    해결:
      ① dt.tz is not None 이면 tz_localize(None)으로 timezone 제거
      ② dt.normalize() 로 시간 부분을 00:00:00으로 잘라 날짜만 남김
    → date 컬럼이 lifelog_date와 동일한 naive datetime64[ns] 타입이 됨
    """
    path = os.path.join(DATA_DIR, f"ch2025_{name}.parquet")
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # timezone 제거 (있을 경우에만)
    if df["timestamp"].dt.tz is not None:
        df["timestamp"] = df["timestamp"].dt.tz_localize(None)

    # naive datetime64[ns] 날짜 컬럼 생성
    df["date"] = df["timestamp"].dt.normalize()
    return df

def _night_key(df: pd.DataFrame, hour_col: str = "hour") -> pd.DataFrame:
    """00~05시 레코드의 date를 하루 전으로 귀속 → night_key 컬럼 추가.
    예) lifelog_date=Jun 26이면 Jun 26 22~23시 + Jun 27 00~05시 모두
        night_key=Jun 26으로 통일.
    """
    df = df.copy()
    df["night_key"] = df["date"].copy()
    df.loc[df[hour_col] < 6, "night_key"] -= pd.Timedelta(days=1)
    return df

def get_cat_features_for_cols(cols: list) -> list:
    """주어진 피처 목록에서 LGB/CatBoost용 범주형 피처 이름 추출.
    - BINARY_CAT_FEATURES: int(0/1) 범주형
    - STR_CAT_COL_SUFFIXES: 문자열 범주형 (ambience_*/app_* 계열)
    """
    result = []
    for c in cols:
        if c in BINARY_CAT_FEATURES:
            result.append(c)
        elif any(c.endswith(s) for s in STR_CAT_COL_SUFFIXES):
            result.append(c)
    return result

def _encode_cat_for_models(X_tr: pd.DataFrame, X_te: pd.DataFrame,
                            cat_feat_set: set) -> tuple:
    """범주형 피처를 정수 코드로 인코딩 (train+test 일관 인코딩).
    - 문자열: NaN→"Unknown"후 category codes (≥0)
    - 이진 정수: NaN→0 (코드 그대로)
    반환: (X_tr_enc, X_te_enc)  — numpy .values 호출 전 DataFrame
    """
    X_tr = X_tr.copy()
    X_te = X_te.copy()
    cat_cols = [c for c in X_tr.columns if c in cat_feat_set]
    for c in cat_cols:
        if X_tr[c].dtype == object:
            combined = (pd.concat([X_tr[c], X_te[c]], ignore_index=True)
                        .fillna("Unknown").astype(str))
            cat_dtype = pd.CategoricalDtype(categories=combined.unique())
            n_tr = len(X_tr)
            codes = combined.astype(cat_dtype).cat.codes.values
            # index 불일치 방지: Series로 감싸 명시적 위치 기반 할당
            X_tr[c] = pd.Series(codes[:n_tr].astype(int), index=X_tr.index)
            X_te[c] = pd.Series(codes[n_tr:].astype(int), index=X_te.index)
        else:
            X_tr[c] = X_tr[c].fillna(0).astype(int)
            X_te[c] = X_te[c].fillna(0).astype(int)
    return X_tr, X_te

def _aggregate_hourly_top5_to_window(
    hourly_df: pd.DataFrame,
    val_col: str = "intensity",
    item_col: str = "sound",
    cat_col: str  = "category",
    agg_func: str = "mean",
    prefix: str   = "ambience_",
) -> pd.DataFrame:
    """1시간 단위 Top5 Wide → 시간대별(day/evening/night) Top5 재집계.

    night_key 적용: 00~05시 데이터를 전날 lifelog_date로 귀속.
    반환: (subject_id, lifelog_date, {prefix}day_Rank*_*, ...) DataFrame
    """
    df = hourly_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # night_key 적용 (00~05시 → 전날)
    df["lifelog_date"] = df["date"]
    df.loc[df["hour"] < 6, "lifelog_date"] -= pd.Timedelta(days=1)

    # 시간대 라벨
    def _window(h):
        if 6 <= h < 17:  return "day"
        elif 17 <= h < 22: return "evening"
        else:              return "night"
    df["time_window"] = df["hour"].apply(_window)

    # Wide → Long
    frames = []
    for i in range(1, 6):
        vc = f"Rank{i}_{val_col}"; ic = f"Rank{i}_{item_col}"; cc = f"Rank{i}_{cat_col}"
        if ic not in df.columns:
            continue
        sub = df[["subject_id","lifelog_date","time_window",vc,ic,cc]].copy()
        sub.columns = ["subject_id","lifelog_date","time_window","value","item","category"]
        sub = sub.dropna(subset=["item"])
        frames.append(sub)
    if not frames:
        return pd.DataFrame()
    long_df = pd.concat(frames, ignore_index=True)

    # 시간대별 집계 → 재순위
    wagg = (long_df
            .groupby(["subject_id","lifelog_date","time_window","item","category"])["value"]
            .agg(agg_func).reset_index())
    wagg = wagg.sort_values(
        ["subject_id","lifelog_date","time_window","value"],
        ascending=[True,True,True,False])
    wagg["new_rank"] = wagg.groupby(
        ["subject_id","lifelog_date","time_window"]).cumcount() + 1
    top5 = wagg[wagg["new_rank"] <= 5]

    # Long → Wide
    pivot = top5.pivot_table(
        index=["subject_id","lifelog_date"],
        columns=["time_window","new_rank"],
        values=["item","category","value"],
        aggfunc="first",
    )
    new_cols = []
    for vtype, window, rnk in pivot.columns:
        if   vtype == "item":     sfx = item_col
        elif vtype == "category": sfx = cat_col
        else:                     sfx = val_col
        new_cols.append(f"{prefix}{window}_Rank{rnk}_{sfx}")
    pivot.columns = new_cols
    pivot = pivot.reset_index()
    pivot["lifelog_date"] = pd.to_datetime(pivot["lifelog_date"])
    return pivot

def _load_floored(name: str, floor: str, value_cols: list) -> pd.DataFrame:
    """
    parquet 로드 후 timestamp를 floor 단위로 내림하여 반환.
    value_cols: 집계에 사용할 컬럼명 리스트
    """
    df = load_parquet(name)
    df["ts_floor"] = df["timestamp"].dt.floor(floor)
    keep = ["subject_id", "date", "ts_floor"] + value_cols
    return df[[c for c in keep if c in df.columns]].copy()

## 2. 피쳐 추출 함수 (파케이별) - tsfresh

# ══════════════════════════════════════════════════════════════════════
# 2. 피처 추출 함수 (파케이별)
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════
# tsfresh 피처 계산 (Micro + Macro)
# ══════════════════════════════════════════════════════════════════

def _safe_ts(func, x):
    """tsfresh 계산기 안전 호출 (inf 포함 예외 → NaN)."""
    import numpy as np
    try:
        vals = list(x.dropna())
        if len(vals) < 5:
            return np.nan
        r = func(vals)
        # 수정됨: 결과가 None이거나 무한대(inf, -inf)일 경우 NaN 처리
        if r is None or np.isinf(r):
            return np.nan
        return float(r)
    except Exception:
        return np.nan


def compute_micro_tsfresh() -> pd.DataFrame:
    """
    Micro level tsfresh: raw parquet → 일별 시계열 피처.

    적용 설정:
      wHr  (야간 22~06h, night_key 날짜 논리 적용):
        sample_entropy, number_cwt_peaks(n=5), cid_ce, mean_change
      wPedo:
        longest_strike_below_mean, variance_larger_than_standard_deviation
      mWifi (rssi):
        binned_entropy(10), longest_strike_below_mean
      mBle (device count):
        percentage_of_reoccurring_values_to_all_values, number_cwt_peaks(n=5)
    """
    if not HAS_TSFRESH:
        print("  tsfresh 없음 → Micro 피처 건너뜀")
        return pd.DataFrame()

    print("  [Micro tsfresh] wHr(야간)...")

    # ── wHr: 야간(22~06시) 필터 + night_key 날짜 보정 ──────────────────
    hr = load_parquet("wHr")
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr["hour"] = hr["timestamp"].dt.hour
    # 야간 필터 + night_key (00~05시 → 전날)
    hr_night = _night_key(hr[(hr["hour"] >= 22) | (hr["hour"] < 6)])
    # 시간 순 정렬: 22시→23시→00시→…→05시 (timestamp 기준)
    hr_night = hr_night.sort_values(["subject_id", "night_key", "timestamp"])

    hr_ts = []
    for (sid, nk), grp in hr_night.groupby(["subject_id", "night_key"], sort=False):
        # sort=False → 이미 정렬된 순서 유지 (22h→…→05h)
        x = grp["hr"].reset_index(drop=True)
        hr_ts.append({
            "subject_id":               sid,
            "date":                      nk,
            "ts_hr_sample_entropy":      _safe_ts(_fc.sample_entropy, x),
            "ts_hr_cwt_peaks":           _safe_ts(lambda v: _fc.number_cwt_peaks(v, n=5), x),
            "ts_hr_cid_ce":              _safe_ts(lambda v: _fc.cid_ce(v, normalize=True), x),
            "ts_hr_mean_change":         _safe_ts(_fc.mean_change, x),
        })
    df_hr_ts = pd.DataFrame(hr_ts) if hr_ts else pd.DataFrame()

    print("  [Micro tsfresh] wPedo...")
    # ── wPedo: step 컬럼 ──────────────────────────────────────────────
    ped = load_parquet("wPedo")
    ped = ped.sort_values(["subject_id", "date", "timestamp"])
    ped_ts = []
    for (sid, dt), grp in ped.groupby(["subject_id", "date"], sort=False):
        x = grp["step"].reset_index(drop=True)
        ped_ts.append({
            "subject_id":                      sid,
            "date":                             dt,
            "ts_pedo_longest_strike_below":     _safe_ts(_fc.longest_strike_below_mean, x),
            "ts_pedo_var_gt_std":               _safe_ts(_fc.variance_larger_than_standard_deviation, x),
        })
    df_ped_ts = pd.DataFrame(ped_ts) if ped_ts else pd.DataFrame()

    print("  [Micro tsfresh] mWifi...")
    # ── mWifi: rssi 컬럼 ─────────────────────────────────────────────
    wifi = load_parquet("mWifi")
    if "m_wifi" in wifi.columns and wifi["m_wifi"].dtype == object:
        wifi = wifi.explode("m_wifi")
        _fv = wifi["m_wifi"].dropna().iloc[0] if wifi["m_wifi"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            wifi["rssi"] = wifi["m_wifi"].apply(lambda x: x.get("rssi", np.nan) if isinstance(x, dict) else np.nan)
    wifi = wifi.sort_values(["subject_id", "date", "timestamp"])
    wifi_ts = []
    for (sid, dt), grp in wifi.groupby(["subject_id", "date"], sort=False):
        x = grp["rssi"].dropna().reset_index(drop=True)
        wifi_ts.append({
            "subject_id":                   sid,
            "date":                          dt,
            "ts_wifi_binned_entropy":        _safe_ts(lambda v: _fc.binned_entropy(v, max_bins=10), pd.Series(x)),
            "ts_wifi_longest_strike_below":  _safe_ts(_fc.longest_strike_below_mean, pd.Series(x)),
        })
    df_wifi_ts = pd.DataFrame(wifi_ts) if wifi_ts else pd.DataFrame()

    print("  [Micro tsfresh] mBle...")
    # ── mBle: 5분 버킷 nunique(device count) ─────────────────────────
    ble = load_parquet("mBle")
    if "m_ble" in ble.columns and ble["m_ble"].dtype == object:
        ble = ble.explode("m_ble")
        _fv = ble["m_ble"].dropna().iloc[0] if ble["m_ble"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            ble["address"] = ble["m_ble"].apply(lambda x: x.get("address", "") if isinstance(x, dict) else "")
    ble = ble[ble["address"] != ""].copy()
    ble["ts_5m"] = ble["timestamp"].dt.floor("5min")
    ble_bucket = (ble.groupby(["subject_id", "date", "ts_5m"])["address"]
                  .nunique().reset_index(name="device_count"))
    ble_bucket = ble_bucket.sort_values(["subject_id", "date", "ts_5m"])
    ble_ts = []
    for (sid, dt), grp in ble_bucket.groupby(["subject_id", "date"], sort=False):
        x = grp["device_count"].reset_index(drop=True)
        ble_ts.append({
            "subject_id":                      sid,
            "date":                             dt,
            "ts_ble_reoccurring_ratio":         _safe_ts(_fc.percentage_of_reoccurring_values_to_all_values, x),
            "ts_ble_cwt_peaks":                 _safe_ts(lambda v: _fc.number_cwt_peaks(v, n=5), x),
        })
    df_ble_ts = pd.DataFrame(ble_ts) if ble_ts else pd.DataFrame()

    # ── 병합 ─────────────────────────────────────────────────────────
    dfs = [df_hr_ts, df_ped_ts, df_wifi_ts, df_ble_ts]
    combined = None
    for df_ in dfs:
        if df_ is None or len(df_) == 0:
            continue
        df_["date"] = pd.to_datetime(df_["date"])
        combined = df_ if combined is None else combined.merge(df_, on=["subject_id","date"], how="outer")

    ts_cols = [c for c in combined.columns if c.startswith("ts_")] if combined is not None else []
    print(f"  Micro tsfresh 피처: {len(ts_cols)}개 ({', '.join(ts_cols)})")
    return combined if combined is not None else pd.DataFrame()


def compute_macro_tsfresh(feat_df: pd.DataFrame, low_miss_cols: list) -> pd.DataFrame:
    """
    Macro level tsfresh: 전처리된 일별 피처 df → 사용자별 시계열 피처.

    적용 대상: 결측률 ≤ 10% 피처
    적용 피처: linear_trend (slope), autocorrelation (lag=1, lag=7)

    사용자 전체 기간의 일별 시계열 → 1개 사용자 수준 값
    → 모든 날짜에 broadcast하여 모델에 전달
    (train 통계만 사용하므로 누수 없음)
    """
    if not HAS_TSFRESH:
        return pd.DataFrame()

    print(f"  [Macro tsfresh] 대상 피처 {len(low_miss_cols)}개 / autocorr(lag=1,7) + linear_trend slope")

    macro_rows = {}  # {subject_id: {feat_name: value}}

    for sid in feat_df["subject_id"].unique():
        sub = (feat_df[feat_df["subject_id"] == sid]
               .sort_values("sleep_date" if "sleep_date" in feat_df.columns else "date"))
        macro_rows[sid] = {"subject_id": sid}
        for c in low_miss_cols:
            if c not in sub.columns:
                continue
            x = sub[c].dropna().values
            if len(x) < 5:
                continue
            # linear_trend slope
            r_lt = _safe_ts(lambda v: list(_fc.linear_trend(v, param=[{"attr":"slope"}]))[0][1],
                            pd.Series(x))
            macro_rows[sid][f"ts_macro_{c}_linear_slope"] = r_lt
            # autocorrelation lag=1
            macro_rows[sid][f"ts_macro_{c}_autocorr_1"] = _safe_ts(
                lambda v: _fc.autocorrelation(v, lag=1), pd.Series(x))
            # autocorrelation lag=7
            macro_rows[sid][f"ts_macro_{c}_autocorr_7"] = _safe_ts(
                lambda v: _fc.autocorrelation(v, lag=7), pd.Series(x)) if len(x) >= 9 else np.nan

    df_macro = pd.DataFrame(macro_rows.values())
    ts_cols = [c for c in df_macro.columns if c.startswith("ts_macro_")]
    print(f"  Macro tsfresh 피처: {len(ts_cols)}개")
    return df_macro

def compute_light_tsfresh_full() -> tuple:
    """mLight / wLight 에서 tsfresh 전체 피처(EfficientFCParameters) 추출.
    반환: (df_m_feats, df_w_feats) — 각각 (subject_id, date, ts_light_m__*, ts_light_w__*)
    """
    if not HAS_TSFRESH:
        return pd.DataFrame(), pd.DataFrame()
    from tsfresh import extract_features as _ef
    from tsfresh.feature_extraction import EfficientFCParameters
    from tsfresh.utilities.dataframe_functions import impute as _impute

    def _extract(parquet_name, value_col, prefix):
        df = load_parquet(parquet_name)
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
        df = df.dropna(subset=[value_col])
        df["ts_id"] = df["subject_id"] + "__" + df["date"].astype(str)
        ts = df[["ts_id","timestamp",value_col]].rename(columns={value_col:"value"})
        feats = _ef(ts, column_id="ts_id", column_sort="timestamp",
                    column_value="value",
                    default_fc_parameters=EfficientFCParameters(),
                    disable_progressbar=True, n_jobs=1)
        _impute(feats)
        feats.columns = [f"{prefix}__{c}" for c in feats.columns]
        idx_parts = feats.index.str.split("__", n=1)
        feats["subject_id"] = [p[0] for p in idx_parts]
        feats["date"]       = pd.to_datetime([p[1] for p in idx_parts])
        return feats.reset_index(drop=True)

    print("  [Light tsfresh] mLight 전체 피처 추출...")
    df_m = _extract("mLight", "m_light", "ts_light_m")
    print(f"    mLight: {len(df_m)}일 × {len([c for c in df_m.columns if c.startswith('ts_')])}피처")
    print("  [Light tsfresh] wLight 전체 피처 추출...")
    df_w = _extract("wLight", "w_light", "ts_light_w")
    print(f"    wLight: {len(df_w)}일 × {len([c for c in df_w.columns if c.startswith('ts_')])}피처")
    return df_m, df_w

def add_light_tsfresh_to_df(train_df, test_df, df_m, df_w) -> tuple:
    """추출된 mLight/wLight tsfresh 피처를 lifelog_date 기준으로 merge.
    반환: (train_df, test_df, all_light_ts_cols)
    """
    all_light_ts_cols = []
    for df_ts in [df_m, df_w]:
        if df_ts is None or len(df_ts) == 0:
            continue
        ts_cols = [c for c in df_ts.columns if c not in ("subject_id","date")]
        df_ts["date"] = pd.to_datetime(df_ts["date"])
        for label, df_ in [("train", train_df), ("test", test_df)]:
            df_["lifelog_date"] = pd.to_datetime(df_["lifelog_date"])
            merged = df_.merge(
                df_ts[["subject_id","date"] + ts_cols],
                left_on=["subject_id","lifelog_date"],
                right_on=["subject_id","date"], how="left"
            )
            for dc in ["date","date_x","date_y"]:
                if dc in merged.columns and "lifelog_date" in merged.columns:
                    merged = merged.drop(columns=[dc])
            if label == "train":
                train_df = merged
            else:
                test_df = merged
        all_light_ts_cols.extend(ts_cols)
    return train_df, test_df, all_light_ts_cols

def select_light_tsfresh_features(
    X_with_targets: pd.DataFrame,
    feature_cols: list,
    targets: list,
    fdr_level: float = 0.15,
) -> dict:
    """tsfresh.select_features로 타겟별 유의미한 Light tsfresh 피처 선택.
    X_with_targets: feature_cols + target cols 포함 DataFrame (train only).
    반환: {target: [selected_col, ...]}
    """
    if not HAS_TSFRESH:
        return {t: [] for t in targets}
    from tsfresh import select_features as _sf

    valid = [c for c in feature_cols if c in X_with_targets.columns]
    X = X_with_targets[valid].fillna(0)
    result = {}
    for t in targets:
        if t not in X_with_targets.columns:
            result[t] = []
            continue
        y = X_with_targets[t].dropna()
        X_t = X.loc[y.index]
        if len(y) < 10:
            result[t] = []
            continue
        try:
            X_sel = _sf(X_t, y, fdr_level=fdr_level)
            selected = X_sel.columns.tolist()
        except Exception as e:
            print(f"    {t}: select_features 오류 → 빈 리스트 ({e})")
            selected = []
        result[t] = selected
        print(f"    {t}: {len(selected):3d}개 선택 (/{len(valid)}개 후보)")
    return result

def add_tsfresh_features(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    miss_thr:  float = 0.10,
) -> tuple:
    """
    tsfresh Micro + Macro 피처 추가 (raw 피처만 반환).

    Z-score는 이 함수에서 적용하지 않음.
    파이프라인 순서:
        add_tsfresh_features()    → raw tsfresh 피처
        add_sensor_rolling()      → raw + rolling 피처
        compute_user_zscores()    → raw + rolling 모두에 Z-score 적용
    이 순서를 지켜야 이중 정규화(_uz_roll_uz)가 발생하지 않는다.
    """
    if not HAS_TSFRESH:
        return train_df, test_df, []

    all_ts_cols = []

    # ── Micro: raw parquet 기반 ────────────────────────────────────
    print("\n[tsfresh Micro] raw parquet 기반 피처 추출...")
    micro_df = compute_micro_tsfresh()

    if micro_df is not None and len(micro_df) > 0:
        micro_ts_cols = [c for c in micro_df.columns if c.startswith("ts_")]
        micro_df["date"] = pd.to_datetime(micro_df["date"])

        for name, df_ in [("train", train_df), ("test", test_df)]:
            df_["lifelog_date"] = pd.to_datetime(df_["lifelog_date"])
            merged = df_.merge(micro_df[["subject_id","date"] + micro_ts_cols],
                               left_on=["subject_id","lifelog_date"],
                               right_on=["subject_id","date"], how="left")
            for drop_col in ["date", "date_x", "date_y"]:
                if drop_col in merged.columns and "lifelog_date" in merged.columns:
                    merged = merged.drop(columns=[drop_col])
            if name == "train":
                train_df = merged
            else:
                test_df = merged

        all_ts_cols.extend(micro_ts_cols)
        print(f"  Micro raw 피처 {len(micro_ts_cols)}개 추가")

    # ── Macro: 결측 10% 이하 피처에 linear_trend + autocorr ──────────
    print("[tsfresh Macro] 전처리 피처 기반 시계열 피처 추출...")
    TARGETS = ["Q1","Q2","Q3","S1","S2","S3","S4"]
    META    = ["subject_id","sleep_date","lifelog_date"]

    # 달력 기반 피처 제외 (linear_slope가 데이터 수집 패턴을 반영할 수 있음)
    EXCLUDE_CALENDAR = {
        "is_weekend","dayofweek","gps_is_weekend","gps_dayofweek",
        "gps_return_is_morning","week_of_year",
    }
    feat_cols_all = [c for c in train_df.columns
                     if c not in META + TARGETS + all_ts_cols
                     and c not in EXCLUDE_CALENDAR
                     and train_df[c].dtype in [float, int, "float64", "int64"]
                     and train_df[c].isnull().mean() <= miss_thr]
    print(f"  결측 ≤10% 피처 (달력 피처 제외): {len(feat_cols_all)}개")

    macro_df = compute_macro_tsfresh(train_df, feat_cols_all)

    if macro_df is not None and len(macro_df) > 0:
        macro_ts_cols = [c for c in macro_df.columns if c.startswith("ts_macro_")]

        # broadcast to train/test (raw만, Z-score는 compute_user_zscores에서)
        for name, df_ in [("train", train_df), ("test", test_df)]:
            merged = df_.merge(
                macro_df[["subject_id"] + macro_ts_cols],
                on="subject_id", how="left"
            )
            if name == "train":
                train_df = merged
            else:
                test_df = merged

        all_ts_cols.extend(macro_ts_cols)
        print(f"  Macro raw 피처 {len(macro_ts_cols)}개 추가")

    print(f"\n  전체 tsfresh raw 피처: {len(all_ts_cols)}개")
    print("  (Z-score는 compute_user_zscores()에서 일괄 적용)")
    return train_df, test_df, all_ts_cols

def feat_wPedo() -> pd.DataFrame:
    """⌚ 만보계: step/speed 기반 running/walking 재판별 + 신규 피처."""
    df = load_parquet("wPedo")
    df["hour"] = df["timestamp"].dt.hour

    # running/walking 직접 판별 (running_step/walking_step 전부 0이므로)
    df["is_running"] = (df["step"] >= 110).astype(int)
    df["is_walking"] = ((df["step"] >= 60) & (df["step"] < 110)).astype(int)

    agg = (
        df.groupby(["subject_id", "date"])
        .agg(
            pedo_step_sum    =("step",            "sum"),
            pedo_step_max    =("step",            "max"),
            pedo_dist_sum    =("distance",        "sum"),
            pedo_calorie_sum =("burned_calories", "sum"),
            pedo_speed_mean  =("speed",           "mean"),
            pedo_speed_max   =("speed",           "max"),
            pedo_freq_mean   =("step_frequency",  "mean"),
            pedo_records     =("step",            "count"),
            pedo_count       =("step",            "count"),
            pedo_run_min     =("is_running",      "sum"),
            pedo_walk_min    =("is_walking",      "sum"),
        )
        .reset_index()
    )
    total = agg["pedo_step_sum"].replace(0, np.nan)
    agg["pedo_run_ratio"]  = agg["pedo_run_min"]  / (agg["pedo_records"] + 1)
    agg["pedo_walk_ratio"] = agg["pedo_walk_min"] / (agg["pedo_records"] + 1)

    # 오전(06~11시) / 저녁(17~21시) 걸음수
    morning = df[df["hour"].between(6, 11)]
    agg_morn = (morning.groupby(["subject_id","date"])["step"]
                .sum().reset_index().rename(columns={"step":"pedo_morning_steps"}))
    evening = df[df["hour"].between(17, 21)]
    agg_eve  = (evening.groupby(["subject_id","date"])["step"]
                .sum().reset_index().rename(columns={"step":"pedo_evening_steps"}))

    # 활동 시간 수
    df["active"] = (df["step"] > 0).astype(int)
    agg_active = (
        df.groupby(["subject_id","date","hour"])["active"].max().reset_index()
        .groupby(["subject_id","date"])["active"].sum().reset_index()
        .rename(columns={"active":"pedo_active_hours"})
    )
    # 달리기 시간대 수
    agg_run_hr = (
        df.groupby(["subject_id","date","hour"])["is_running"].max().reset_index()
        .groupby(["subject_id","date"])["is_running"].sum().reset_index()
        .rename(columns={"is_running":"pedo_run_hours"})
    )
    # 연속 60분 최대 step 구간 평균
    def peak_60min(grp):
        s = grp.sort_values("timestamp")["step"].rolling(60, min_periods=1).sum()
        return float(s.max()) / 60.0 if len(s) > 0 else 0.0
    agg_peak = (
        df.groupby(["subject_id","date"], group_keys=False)
        .apply(peak_60min).reset_index(name="pedo_peak_60min_step")
    )

    agg = agg.merge(agg_peak,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_morn,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_eve,    on=["subject_id","date"], how="left")
    agg = agg.merge(agg_active, on=["subject_id","date"], how="left")
    agg = agg.merge(agg_run_hr, on=["subject_id","date"], how="left")
    agg["pedo_morning_ratio"] = agg["pedo_morning_steps"] / total
    agg["pedo_evening_ratio"] = agg["pedo_evening_steps"] / total
    return agg

def feat_wHr() -> pd.DataFrame:
    """⌚ 심박수: 전체/야간/아침/저녁 구간별 통계 + 변동성 지표

    추가 피처
    ─────────
    hr_resting      : 새벽(03~05시) 최소 심박 — 안정 시 심박수(수면질 직접 지표)
    hr_cv           : 심박 변동계수(std/mean) — 자율신경계 활성도
    hr_evening_mean : 저녁(18~21시) 평균 심박 — 취침 전 각성 수준
    hr_diff_day_night: 낮 평균 - 야간 평균 — 수면 중 심박 회복력
    """
    df = load_parquet("wHr")

    # heart_rate 가 list 타입인 경우 explode
    if df["heart_rate"].dtype == object:
        try:
            df = df.explode("heart_rate")
        except Exception:
            pass

    df["hr"] = pd.to_numeric(df["heart_rate"], errors="coerce")
    df = df.dropna(subset=["hr"])
    df = df[df["hr"].between(30, 220)]
    df["hour"] = df["timestamp"].dt.hour

    # 전체 하루
    agg = (
        df.groupby(["subject_id", "date"])["hr"]
        .agg(
            hr_mean="mean", hr_std="std",
            hr_min="min",   hr_max="max",
            hr_q25=lambda x: x.quantile(0.25),
            hr_q75=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    # 야간 (22:00~05:59): night_key로 날짜 경계 처리
    # 00~05시는 전날 lifelog_date에 귀속
    night = _night_key(df[(df["hour"] >= 22) | (df["hour"] < 6)])
    agg_night = (
        night.groupby(["subject_id", "night_key"])["hr"]
        .agg(hr_night_mean="mean", hr_night_std="std", hr_night_min="min")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 아침 (06:00~09:59) — 기상 직후 심박
    morning = df[df["hour"].between(6, 9)]
    agg_morning = (
        morning.groupby(["subject_id", "date"])["hr"]
        .agg(hr_morning_mean="mean")
        .reset_index()
    )

    # 새벽 안정 심박 (03:00~04:59): night_key로 날짜 경계 처리
    resting = _night_key(df[df["hour"].between(3, 4)])
    agg_rest = (
        resting.groupby(["subject_id", "night_key"])["hr"]
        .agg(hr_resting=lambda x: x.min())
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 저녁 (18:00~20:59) — 취침 전 각성 수준
    evening = df[df["hour"].between(18, 20)]
    agg_eve = (
        evening.groupby(["subject_id", "date"])["hr"]
        .agg(hr_evening_mean="mean")
        .reset_index()
    )

    # 낮 (10:00~17:59) 평균
    day = df[df["hour"].between(10, 17)]
    agg_day = (
        day.groupby(["subject_id", "date"])["hr"]
        .agg(hr_day_mean="mean")
        .reset_index()
    )

    agg = agg.merge(agg_night,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_morning, on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_rest,    on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_eve,     on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_day,     on=["subject_id", "date"], how="left")

    # 파생 피처
    agg["hr_range"]          = agg["hr_max"] - agg["hr_min"]
    agg["hr_cv"]             = agg["hr_std"] / agg["hr_mean"].replace(0, np.nan)
    agg["hr_diff_day_night"] = agg["hr_day_mean"] - agg["hr_night_mean"]
    return agg

def feat_wLight() -> pd.DataFrame:
    """⌚ 조도(워치): 낮/야간 빛 노출"""
    df = load_parquet("wLight")
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id", "date"])["w_light"]
        .agg(wlight_mean="mean", wlight_max="max", wlight_std="std")
        .reset_index()
    )

    # 야간 조도 (21~07시): night_key로 날짜 경계 처리
    night = _night_key(df[(df["hour"] >= 21) | (df["hour"] < 7)])
    night.loc[night["hour"].between(7, 20), "night_key"] = night.loc[night["hour"].between(7, 20), "date"]
    agg_night = (
        night.groupby(["subject_id", "night_key"])["w_light"]
        .agg(wlight_night_mean="mean", wlight_night_max="max")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )

    # 일광 노출 비율 (250 lux 이상)
    df["sunny"] = (df["w_light"] > 250).astype(int)
    agg_sun = (
        df.groupby(["subject_id", "date"])["sunny"]
        .agg(wlight_sun_ratio=lambda x: x.mean())
        .reset_index()
    )

    agg = agg.merge(agg_night, on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_sun,   on=["subject_id", "date"], how="left")
    return agg

def feat_mActivity() -> pd.DataFrame:
    """📱 활동: GPS+step 보정 후 분류 (보정 코드 9=실내운동 포함)."""
    df = load_parquet("mActivity")
    df["hour"] = df["timestamp"].dt.hour

    # 보정: wPedo + mGPS 교차 검증 (parquet 로드 가능 시)
    try:
        ped = load_parquet("wPedo")
        gps = load_parquet("mGps")
        # 1분 압축
        def compress(df_, cols):
            df_ = df_.copy()
            df_["mf"] = df_["timestamp"].dt.floor("1min")
            return (df_.groupby(["subject_id","date","mf"])
                    .agg({c:"median" for c in cols if c in df_.columns})
                    .reset_index().rename(columns={"mf":"timestamp"}))
        ped_c = compress(ped, ["step"]).rename(columns={"step":"step_c"})
        if "m_gps" in gps.columns and gps["m_gps"].dtype == object:
            gps = gps.explode("m_gps")
            _fv = gps["m_gps"].dropna().iloc[0] if gps["m_gps"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                gps["latitude"]  = gps["m_gps"].apply(lambda x: x.get("latitude",np.nan) if isinstance(x,dict) else np.nan)
                gps["longitude"] = gps["m_gps"].apply(lambda x: x.get("longitude",np.nan) if isinstance(x,dict) else np.nan)
                gps["speed_g"]   = gps["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
        gps = gps.dropna(subset=["latitude","longitude"])
        gps["speed_g"] = pd.to_numeric(gps.get("speed_g", np.nan), errors="coerce")
        gps = gps[gps["speed_g"].isna()|(gps["speed_g"]<=GPS_SPEED_LIMIT_MS)]
        gps_c = compress(gps, ["speed_g"])

        act_1m = (df.groupby(["subject_id","date",df["timestamp"].dt.floor("1min")])["m_activity"]
                  .agg(lambda x: int(x.mode().iloc[0]) if len(x)>0 else -1)
                  .reset_index().rename(columns={"timestamp":"timestamp"}))
        merged = (act_1m
                  .merge(ped_c[["subject_id","timestamp","step_c"]], on=["subject_id","timestamp"], how="left")
                  .merge(gps_c[["subject_id","timestamp","speed_g"]], on=["subject_id","timestamp"], how="left"))
        merged["step_c"]  = merged["step_c"].fillna(0).astype(int)
        merged["speed_g"] = merged["speed_g"].fillna(0.0)
        orig = merged["m_activity"].copy()
        corr = orig.copy()
        corr[(orig==3)&(merged["step_c"]>=60)&(merged["step_c"]<110)
             &(merged["speed_g"]>=0.3)&(merged["speed_g"]<1.5)] = 7
        corr[(orig==3)&(merged["step_c"]>=110)
             &(merged["speed_g"]>=0.3)&(merged["speed_g"]<1.5)] = 8
        corr[(orig==3)&(merged["step_c"]>=60)&(merged["speed_g"]<0.3)] = 9
        merged["m_activity"] = corr
        df = merged[["subject_id","date","timestamp","m_activity"]].copy()
        df["hour"] = df["timestamp"].dt.hour
    except Exception:
        pass  # 로드 실패 시 원본 사용

    agg = (
        df.groupby(["subject_id","date"])["m_activity"]
        .agg(
            act_still_ratio   =lambda x: (x==3).mean(),
            act_walk_ratio    =lambda x: (x==7).mean(),
            act_run_ratio     =lambda x: (x==8).mean(),
            act_indoor_ratio  =lambda x: (x==9).mean(),
            act_vehicle_ratio =lambda x: (x==0).mean(),
            act_unknown_ratio =lambda x: (x==4).mean(),
            act_unique        =lambda x: x.nunique(),
            active_count      =lambda x: len(x),
        )
        .reset_index()
    )
    agg["act_active_ratio"] = agg["act_walk_ratio"]+agg["act_run_ratio"]+agg["act_indoor_ratio"]

    night = _night_key(df[(df["hour"]>=22)|(df["hour"]<6)])
    agg_ns = (night.groupby(["subject_id","night_key"])["m_activity"]
              .agg(act_still_night_ratio=lambda x: (x==3).mean()).reset_index()
              .rename(columns={"night_key":"date"}))

    df["is_active"] = df["m_activity"].isin([7,8,9]).astype(int)
    hourly = df.groupby(["subject_id","date","hour"])["is_active"].sum().reset_index()
    peak = (hourly.loc[hourly.groupby(["subject_id","date"])["is_active"].idxmax()]
            [["subject_id","date","hour"]].rename(columns={"hour":"act_peak_hour"})
            .reset_index(drop=True))

    agg = agg.merge(agg_ns, on=["subject_id","date"], how="left")
    agg = agg.merge(peak,   on=["subject_id","date"], how="left")
    return agg

def feat_mACStatus() -> pd.DataFrame:
    """📱 충전 상태: 충전 비율 + 취침·기상 시각 추정

    추가 피처
    ─────────
    ac_night_charge_duration: 야간(21~08시) 총 충전 시간(분) — 수면 시간 proxy
    ac_morning_unplug_hour  : 오전 첫 충전 해제 시각 — 기상 시각 추정
    """
    df = load_parquet("mACStatus")
    df = df.sort_values(["subject_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 충전 ON 시간 계산
    df["time_diff_sec"] = (
        df.groupby("subject_id")["timestamp"]
        .diff().dt.total_seconds()
    )
    df["charge_sec"] = df["time_diff_sec"].where(df["m_charging"] == 1, 0)

    agg = (
        df.groupby(["subject_id", "date"])["m_charging"]
        .agg(ac_charge_ratio="mean", ac_records="count")
        .reset_index()
    )

    # 야간(21:00~07:59) 총 충전 시간(분) — 수면 중 충전 = 수면 시간 proxy
    # night_key 적용: 00~07시 레코드를 전날에 귀속
    night = _night_key(df[(df["hour"] >= 21) | (df["hour"] < 8)])
    agg_night_dur = (
        night.groupby(["subject_id", "night_key"])["charge_sec"]
        .sum().reset_index()
        .rename(columns={"night_key": "date"})
        .assign(ac_night_charge_duration=lambda x: x["charge_sec"] / 60)
        [["subject_id", "date", "ac_night_charge_duration"]]
    )

    # 야간(21:00~02:00) 첫 충전 이벤트 → 취침 시각 추정
    # night_key 적용: 00~02시를 전날에 귀속
    night_charge_nk = _night_key(
        df[((df["hour"] >= 21) | (df["hour"] <= 2)) & (df["m_charging"] == 1)]
    )
    first_charge = (
        night_charge_nk.sort_values("timestamp")
        .groupby(["subject_id", "night_key"])["hour"]
        .first()
        .reset_index()
        .rename(columns={"night_key": "date", "hour": "ac_bedtime_hour"})
    )

    # 오전(05:00~10:00) 첫 충전 해제 → 기상 시각 추정
    morning_unplug = df[
        df["hour"].between(5, 10) & (df["m_charging"] == 0)
    ]
    first_unplug = (
        morning_unplug.sort_values("timestamp")
        .groupby(["subject_id", "date"])["hour"]
        .first()
        .reset_index()
        .rename(columns={"hour": "ac_morning_unplug_hour"})
    )

    agg = agg.merge(first_charge,      on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_night_dur,     on=["subject_id", "date"], how="left")
    agg = agg.merge(first_unplug,      on=["subject_id", "date"], how="left")
    return agg

def feat_mScreenStatus() -> pd.DataFrame:
    """📱 화면 상태: 취침 전/야간 스마트폰 사용량 (수면 방해 핵심 지표)

    추가 피처
    ─────────
    screen_total_on_min   : 하루 총 화면 ON 시간(분) — 스마트폰 의존도
    screen_night_sessions : 23~02시 화면 ON 전환 횟수 — 수면 방해 세션
    screen_last_on_hour   : 하루 마지막 화면 ON 시각 — 취침 직전 폰 사용 시점
    """
    df = load_parquet("mScreenStatus")
    df = df.sort_values(["subject_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 화면 ON 시간(분): 연속 ON 상태의 시간 차이 합산
    df["time_diff_sec"] = (
        df.groupby("subject_id")["timestamp"]
        .diff().dt.total_seconds()
    )
    df["on_sec"] = df["time_diff_sec"].where(df["m_screen_use"] == 1, 0)

    agg = (
        df.groupby(["subject_id", "date"])
        .agg(
            screen_on_ratio  =("m_screen_use", "mean"),
            screen_records   =("m_screen_use", "count"),
            screen_total_on_min=("on_sec", lambda x: x.sum() / 60),
        )
        .reset_index()
    )

    # 취침 전 2시간 (22:00~23:59)
    pre_sleep = df[df["hour"].between(22, 23)]
    agg_pre = (
        pre_sleep.groupby(["subject_id", "date"])["m_screen_use"]
        .agg(screen_pre_sleep_ratio="mean", screen_pre_sleep_count="count")
        .reset_index()
    )

    # 야간 (00:00~04:59) — 수면 중 폰 사용 감지
    midnight = df[df["hour"] < 5]
    agg_mid = (
        midnight.groupby(["subject_id", "date"])["m_screen_use"]
        .agg(screen_midnight_on_ratio="mean")
        .reset_index()
    )

    # 마지막 화면 off 시각 (수면 시작 추정)
    last_off = (
        df[df["m_screen_use"] == 0]
        .groupby(["subject_id", "date"])["hour"]
        .last()
        .reset_index()
        .rename(columns={"hour": "screen_last_off_hour"})
    )

    # 마지막 화면 ON 시각 (취침 직전 폰 사용 시점)
    last_on = (
        df[df["m_screen_use"] == 1]
        .groupby(["subject_id", "date"])["hour"]
        .last()
        .reset_index()
        .rename(columns={"hour": "screen_last_on_hour"})
    )

    # 심야(23~02시) 화면 ON 전환 횟수 (sleep session 방해)
    # ON으로 전환되는 순간 = 이전 상태 0 & 현재 상태 1
    df["prev_state"] = df.groupby("subject_id")["m_screen_use"].shift(1)
    df["screen_on_event"] = (
        (df["m_screen_use"] == 1) & (df["prev_state"] == 0)
    ).astype(int)
    night_sess = df[(df["hour"] >= 23) | (df["hour"] < 3)]
    agg_sess = (
        night_sess.groupby(["subject_id", "date"])["screen_on_event"]
        .sum().reset_index()
        .rename(columns={"screen_on_event": "screen_night_sessions"})
    )

    agg = agg.merge(agg_pre,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_mid,   on=["subject_id", "date"], how="left")
    agg = agg.merge(last_off,  on=["subject_id", "date"], how="left")
    agg = agg.merge(last_on,   on=["subject_id", "date"], how="left")
    agg = agg.merge(agg_sess,  on=["subject_id", "date"], how="left")
    return agg

def feat_mLight() -> pd.DataFrame:
    """📱 조도(폰): 주변 밝기"""
    df = load_parquet("mLight")
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id", "date"])["m_light"]
        .agg(mlight_mean="mean", mlight_max="max", mlight_std="std")
        .reset_index()
    )
    night = _night_key(df[(df["hour"] >= 22) | (df["hour"] < 6)])
    agg_night = (
        night.groupby(["subject_id", "night_key"])["m_light"]
        .agg(mlight_night_mean="mean", mlight_night_max="max")
        .reset_index()
        .rename(columns={"night_key": "date"})
    )
    agg = agg.merge(agg_night, on=["subject_id", "date"], how="left")
    return agg

def feat_mAmbience() -> pd.DataFrame:
    """📱 소음: 시간대별(day/evening/night) Top5 소음 카테고리 피처.

    _aggregate_hourly_top5_to_window 적용 → final_ambience 반환.
    string 컬럼(ambience_*_sound, ambience_*_category)은 범주형으로 처리.
    night_key 날짜 경계 처리 적용.
    """
    import re as _re

    AUDIOSET_MAP = {
        "Speech":"Human","Conversation":"Human","Shout":"Human",
        "Narration, monologue":"Human","Child speech, kid speaking":"Human",
        "Babbling":"Human","Music":"Music",
        "Television":"Device","Radio":"Device","Alarm":"Device",
        "Tick-tock":"Device","Click":"Device","Bell":"Device",
        "Appliances":"Device","Mechanical fan":"Device",
        "Hiss":"Device","Sound effect":"Device",
        "Inside, small room":"Ambient","Inside, large room or hall":"Ambient",
        "Outside, rural or natural":"Ambient","Outside, urban or manmade":"Ambient",
        "Wind":"Nature","Rain":"Nature","Water":"Nature",
        "Vehicle":"Transport","Car":"Transport","Truck":"Transport",
        "Motor vehicle (road)":"Transport","Traffic noise":"Transport","Bus":"Transport",
    }

    def _parse_pairs(row):
        return _re.findall("array\\(\\['([^']*)',[^']*'([^']*)'\\]", str(row))

    df = load_parquet("mAmbience")
    df["hour"] = df["timestamp"].dt.hour

    if "m_ambience" not in df.columns:
        return pd.DataFrame()

    df["pairs"] = df["m_ambience"].apply(_parse_pairs)
    df2 = df.explode("pairs").dropna(subset=["pairs"])
    if len(df2) == 0:
        return pd.DataFrame()

    pairs_df = pd.DataFrame(df2["pairs"].tolist(), index=df2.index)
    df2 = df2.copy()
    df2["sound_type"] = pairs_df[0]
    df2["intensity"]  = pd.to_numeric(pairs_df[1], errors="coerce")
    df2 = df2[(df2["sound_type"] != "Silence") & (df2["intensity"] >= 0.05)]
    df2["category"] = df2["sound_type"].map(lambda x: AUDIOSET_MAP.get(x, "Other"))

    # 시간대별 Top5 집계 (1시간 단위)
    agg_h = (df2.groupby(["subject_id","date","hour","sound_type","category"])["intensity"]
             .mean().reset_index())
    agg_h = agg_h.sort_values(
        ["subject_id","date","hour","intensity"], ascending=[True,True,True,False])
    agg_h["rank"] = agg_h.groupby(["subject_id","date","hour"]).cumcount() + 1
    top5_h = agg_h[agg_h["rank"] <= 5]

    # Wide format (Rank1_sound, Rank1_category, Rank1_intensity, ...)
    pivot_h = top5_h.pivot_table(
        index=["subject_id","date","hour"],
        columns="rank",
        values=["sound_type","category","intensity"],
        aggfunc="first",
    )
    pivot_h.columns = [f"Rank{r}_{v}" for v, r in pivot_h.columns]
    hourly_df = pivot_h.reset_index().rename(
        columns={f"Rank{r}_sound_type": f"Rank{r}_sound" for r in range(1,6)})

    # 시간대별 Top5 재집계
    final_ambience = _aggregate_hourly_top5_to_window(
        hourly_df, val_col="intensity", item_col="sound",
        cat_col="category", agg_func="mean", prefix="ambience_")

    if len(final_ambience) == 0:
        return pd.DataFrame()

    final_ambience = final_ambience.rename(columns={"lifelog_date": "date"})
    final_ambience["date"] = pd.to_datetime(final_ambience["date"])
    return final_ambience

def feat_mWifi() -> pd.DataFrame:
    """📶 WiFi: 위치 패턴 + 집 체류 + 신규(bssid rolling, 수면 rssi)."""
    df = load_parquet("mWifi")
    if "m_wifi" in df.columns and df["m_wifi"].dtype == object:
        try:
            df = df.explode("m_wifi")
            _fv = df["m_wifi"].dropna().iloc[0] if df["m_wifi"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                df["bssid"] = df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
                df["rssi"]  = df["m_wifi"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
            else:
                df["bssid"] = ""; df["rssi"] = pd.to_numeric(df["m_wifi"], errors="coerce")
        except Exception:
            df["bssid"] = ""; df["rssi"] = np.nan
    else:
        df["bssid"] = ""; df["rssi"] = df["m_wifi"] if "rssi" not in df.columns else df["rssi"]
    df["hour"] = df["timestamp"].dt.hour

    agg = (
        df.groupby(["subject_id","date"])
        .agg(wifi_ap_count=("bssid","nunique"), wifi_rssi_mean=("rssi","mean"),
             wifi_rssi_max=("rssi","max"), wifi_scans=("bssid","count"))
        .reset_index()
    )
    home_bssid = (
        df[df["bssid"]!=""].groupby(["subject_id","bssid"]).size().reset_index(name="cnt")
        .sort_values("cnt", ascending=False).drop_duplicates("subject_id")
        [["subject_id","bssid"]].rename(columns={"bssid":"home_bssid"})
    )
    df = df.merge(home_bssid, on="subject_id", how="left")
    df["is_home"] = (df["bssid"] == df["home_bssid"]).astype(int)
    agg_home = (df.groupby(["subject_id","date"])["is_home"].mean().reset_index()
                .rename(columns={"is_home":"wifi_home_ratio"}))
    agg_home_eve = (df[df["hour"].between(18,23)].groupby(["subject_id","date"])["is_home"]
                    .mean().reset_index().rename(columns={"is_home":"wifi_home_evening_ratio"}))

    # bssid nunique rolling(window=3, 10분×3=30분) 일평균
    df_ts = (df.groupby(["subject_id","date",pd.Grouper(key="timestamp",freq="10min")])["bssid"]
             .nunique().reset_index(name="bssid_nu_10m"))
    df_ts = df_ts.sort_values(["subject_id","timestamp"])
    df_ts["bssid_nu_r3"] = (df_ts.groupby("subject_id")["bssid_nu_10m"]
                            .transform(lambda x: x.rolling(3,min_periods=1).mean()))
    agg_roll = (df_ts.groupby(["subject_id","date"])["bssid_nu_r3"].mean().reset_index()
                .rename(columns={"bssid_nu_r3":"wifi_bssid_nunique_roll3"}))

    # 수면 시간대(22~06시) rssi 평균: night_key로 날짜 경계 처리
    sleep_df = _night_key(df[(df["hour"]>=22)|(df["hour"]<6)])
    agg_sleep_rssi = (sleep_df.groupby(["subject_id","night_key"])["rssi"].mean().reset_index()
                      .rename(columns={"night_key":"date","rssi":"wifi_sleep_rssi_mean"}))

    agg = agg.merge(agg_home,       on=["subject_id","date"], how="left")
    agg = agg.merge(agg_home_eve,   on=["subject_id","date"], how="left")
    agg = agg.merge(agg_roll,       on=["subject_id","date"], how="left")
    agg = agg.merge(agg_sleep_rssi, on=["subject_id","date"], how="left")
    return agg

def feat_mGps() -> pd.DataFrame:
    """🗺️ GPS: DBSCAN 기반 실이동 거리 + 집 탐지 + 귀가 시각 + 주말 피처."""
    df = load_parquet("mGps")
    if "m_gps" in df.columns and df["m_gps"].dtype == object:
        try:
            df = df.explode("m_gps")
            _fv = df["m_gps"].dropna().iloc[0] if df["m_gps"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                for col in ["latitude","longitude","altitude","speed"]:
                    df[col] = df["m_gps"].apply(lambda x: x.get(col,np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            for col in ["latitude","longitude","altitude","speed"]:
                df[col] = np.nan
    else:
        for col in ["latitude","longitude","altitude","speed"]:
            if col not in df.columns: df[col] = np.nan

    df = df.dropna(subset=["latitude","longitude"]).sort_values(["subject_id","timestamp"])
    df["hour"] = df["timestamp"].dt.hour

    # 이상값 제거
    df["lat_diff"]  = df.groupby("subject_id")["latitude"].diff()
    df["lon_diff"]  = df.groupby("subject_id")["longitude"].diff()
    df["time_diff"] = df.groupby("subject_id")["timestamp"].diff().dt.total_seconds()
    df["dist_m"]    = np.sqrt(df["lat_diff"]**2 + df["lon_diff"]**2) * 111_000
    df["inst_spd"]  = df["dist_m"] / df["time_diff"].replace(0, np.nan)
    df = df[df["inst_spd"].isna()|(df["inst_spd"]<=GPS_SPEED_LIMIT_MS)].copy()
    df["is_stop"] = (df["speed"] < GPS_STOP_THRESH_MS).astype(int)

    # 1분 중위값 압축 (11건/분 → 1건/분)
    def compress_to_minute(df_day):
        df_day = df_day.copy()
        df_day["mf"] = df_day["timestamp"].dt.floor("1min")
        c = (df_day.groupby(["subject_id","date","mf"])
             .agg(latitude=("latitude","median"), longitude=("longitude","median"),
                  speed=("speed","median")).reset_index()
             .rename(columns={"mf":"timestamp"}))
        return c.sort_values("timestamp").reset_index(drop=True)

    def haversine_vec(la1, lo1, la2, lo2):
        R = 6_371_000
        la1,lo1,la2,lo2 = map(np.radians,[la1,lo1,la2,lo2])
        dlat=la2-la1; dlon=lo2-lo1
        a = np.sin(dlat/2)**2 + np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2
        return R*2*np.arcsin(np.sqrt(np.clip(a,0,1)))

    records, detail_frames = [], []
    for (sid, date), group in df.groupby(["subject_id","date"]):
        g = compress_to_minute(group)
        if len(g) < DBSCAN_MIN_SAMPLES:
            g["cluster"]   = -1
            g["place_lat"] = g["latitude"]; g["place_lon"] = g["longitude"]
            dist_km = 0.0; n_places = 0; home_cluster = -1; home_min = 0
        else:
            coords_rad = np.radians(g[["latitude","longitude"]].values)
            db = DBSCAN(eps=DBSCAN_EPS_RAD, min_samples=DBSCAN_MIN_SAMPLES,
                        metric="haversine", algorithm="ball_tree").fit(coords_rad)
            g["cluster"] = db.labels_
            n_places = len(set(db.labels_) - {-1})
            labels = db.labels_
            lats = g["latitude"].values; lons = g["longitude"].values
            center = {}
            for c in set(labels) - {-1}:
                mask = labels==c; center[c] = (lats[mask].mean(), lons[mask].mean())
            ev_lat = np.where(labels>=0,
                              [center.get(c,(lat,lon))[0] for c,lat,lon in zip(labels,lats,lons)], lats)
            ev_lon = np.where(labels>=0,
                              [center.get(c,(lat,lon))[1] for c,lat,lon in zip(labels,lats,lons)], lons)
            ev_cls = labels.copy()
            keep = np.ones(len(ev_cls), dtype=bool)
            for i in range(1,len(ev_cls)):
                if ev_cls[i]>=0 and ev_cls[i]==ev_cls[i-1]: keep[i]=False
            ev_lat2=ev_lat[keep]; ev_lon2=ev_lon[keep]
            if len(ev_lat2)>=2:
                dists = haversine_vec(ev_lat2[:-1],ev_lon2[:-1],ev_lat2[1:],ev_lon2[1:])
                dist_km = dists.sum()/1000
            else:
                dist_km = 0.0
            for c,(clat,clon) in center.items():
                g.loc[g["cluster"]==c,"place_lat"] = clat
                g.loc[g["cluster"]==c,"place_lon"] = clon
            g.loc[g["cluster"]==-1,"place_lat"] = g.loc[g["cluster"]==-1,"latitude"]
            g.loc[g["cluster"]==-1,"place_lon"] = g.loc[g["cluster"]==-1,"longitude"]
            # 집 탐지 (0~5시 + 22~23시)
            night_mask = (g["timestamp"].dt.hour<6)|(g["timestamp"].dt.hour>=22)
            night_df   = g[night_mask & (g["cluster"]>=0)]
            if len(night_df)>0:
                home_cluster = int(night_df["cluster"].mode().iloc[0])
            else:
                placed = g[g["cluster"]>=0]
                home_cluster = int(placed["cluster"].mode().iloc[0]) if len(placed)>0 else -1
            home_min = int((g["cluster"]==home_cluster).sum()) if home_cluster>=0 else 0

        detail_frames.append(
            g[["subject_id","date","timestamp","latitude","longitude",
               "speed","cluster"]].assign(subject_id=sid, date=date))

        # 귀가 시각: 집 cluster 저녁(GPS_EVENING_START 이후) 첫 등장
        date_ts   = pd.Timestamp(date)
        dow       = date_ts.dayofweek
        is_weekend = int(dow>=5)
        return_hour = np.nan
        if home_cluster>=0 and len(g)>0:
            home_rows = g[g["cluster"]==home_cluster]
            eve_home  = home_rows[home_rows["timestamp"].dt.hour>=GPS_EVENING_START].sort_values("timestamp")
            if len(eve_home)>0:
                t = eve_home["timestamp"].iloc[0]
                return_hour = t.hour + t.minute/60
            else:
                t = home_rows.sort_values("timestamp")["timestamp"].iloc[0]
                return_hour = t.hour + t.minute/60

        records.append({
            "subject_id": sid, "date": date,
            "gps_dist_km_dbscan": dist_km, "gps_n_places": n_places,
            "gps_home_min": home_min,
            "gps_home_hour": home_min/60,
            "gps_return_hour": return_hour,
            "gps_dayofweek": dow,
            "gps_is_weekend": is_weekend,
            "gps_return_is_morning": int(return_hour<12) if not np.isnan(return_hour) else 0,
        })

    agg_dbscan  = pd.DataFrame(records)
    df_detailed = pd.concat(detail_frames, ignore_index=True)

    agg_base = (
        df.groupby(["subject_id","date"])
        .agg(gps_speed_mean=("speed","mean"), gps_speed_max=("speed","max"),
             gps_records=("latitude","count"), gps_lat_std=("latitude","std"),
             gps_lon_std=("longitude","std"), gps_stop_ratio=("is_stop","mean"))
        .reset_index()
    )
    agg_base["gps_radius"] = agg_base["gps_lat_std"] + agg_base["gps_lon_std"]
    agg_base = agg_base.drop(columns=["gps_lat_std","gps_lon_std"])

    agg = agg_base.merge(agg_dbscan, on=["subject_id","date"], how="left")
    agg["gps_log_dist"] = np.log1p(agg["gps_dist_km_dbscan"])
    agg["date"] = pd.to_datetime(agg["date"])
    return agg

def feat_mUsageStats() -> pd.DataFrame:
    """📱 앱 사용: 시간대별(day/evening/night) Top5 앱 피처.

    _aggregate_hourly_top5_to_window 적용 → final_usage 반환.
    string 컬럼(app_*_app_name, app_*_app_category)은 범주형으로 처리.
    night_key 날짜 경계 처리 적용.
    """
    import re as _re

    KEYWORD_MAP = {
        "SNS":     ["instagram","facebook","twitter","kakao","tiktok","snapchat","band","telegram"],
        "Video":   ["youtube","netflix","twitch","wavve","tving","vod","player","video"],
        "Game":    ["game","play","puzzle","rpg","clash","minecraft"],
        "Browser": ["chrome","samsung internet","safari","firefox","naver","daum","browser"],
    }
    def _classify(app: str) -> str:
        low = app.lower()
        for cat, kws in KEYWORD_MAP.items():
            if any(k in low for k in kws):
                return cat
        return "Tool"

    df = load_parquet("mUsageStats")
    df["hour"] = df["timestamp"].dt.hour

    regex = r"'app_name':\s*'([^']*)',\s*'total_time':\s*(\d+)"
    records = []
    for _, row in df.iterrows():
        for app_name, total_ms in _re.findall(regex, str(row.get("m_usage_stats",""))):
            records.append({
                "subject_id":    row["subject_id"],
                "date":          row["date"],
                "hour":          row["hour"],
                "app_name":      app_name,
                "app_category":  _classify(app_name),
                "total_time_sec": int(total_ms) / 1000,
            })
    if not records:
        return pd.DataFrame()

    usage = pd.DataFrame(records)

    # 시간대별 Top5 집계 (1시간 단위)
    agg_h = (usage.groupby(["subject_id","date","hour","app_name","app_category"])["total_time_sec"]
             .sum().reset_index())
    agg_h = agg_h.sort_values(
        ["subject_id","date","hour","total_time_sec"], ascending=[True,True,True,False])
    agg_h["rank"] = agg_h.groupby(["subject_id","date","hour"]).cumcount() + 1
    top5_h = agg_h[agg_h["rank"] <= 5]

    # Wide format (Rank1_app_name, Rank1_app_category, Rank1_total_time_sec, ...)
    pivot_h = top5_h.pivot_table(
        index=["subject_id","date","hour"],
        columns="rank",
        values=["app_name","app_category","total_time_sec"],
        aggfunc="first",
    )
    pivot_h.columns = [f"Rank{r}_{v}" for v, r in pivot_h.columns]
    hourly_df = pivot_h.reset_index()

    # 시간대별 Top5 재집계
    final_usage = _aggregate_hourly_top5_to_window(
        hourly_df, val_col="total_time_sec", item_col="app_name",
        cat_col="app_category", agg_func="sum", prefix="app_")

    if len(final_usage) == 0:
        return pd.DataFrame()

    final_usage = final_usage.rename(columns={"lifelog_date": "date"})
    final_usage["date"] = pd.to_datetime(final_usage["date"])
    return final_usage

def feat_mBle() -> pd.DataFrame:
    """📡 BLE: 사회적 환경 밀도 (집 판별 제거, 5분 버킷 nunique 기반)."""
    df = load_parquet("mBle")
    if "m_ble" in df.columns and df["m_ble"].dtype == object:
        try:
            df = df.explode("m_ble")
            _fv = df["m_ble"].dropna().iloc[0] if df["m_ble"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                df["address"] = df["m_ble"].apply(lambda x: x.get("address","") if isinstance(x,dict) else "")
                df["rssi"]    = df["m_ble"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            df["address"] = ""; df["rssi"] = np.nan

    df = df[df["address"] != ""].copy()
    df["hour"] = df["timestamp"].dt.hour
    df["ts_5m"] = df["timestamp"].dt.floor("5min")

    bucket = (df.groupby(["subject_id","date","ts_5m"])["address"]
              .nunique().reset_index(name="device_count"))
    bucket["hour"] = bucket["ts_5m"].dt.hour

    agg = (bucket.groupby(["subject_id","date"])["device_count"]
           .mean().reset_index().rename(columns={"device_count":"ble_device_day_mean"}))
    agg_eve = (bucket[bucket["hour"].between(18,21)]
               .groupby(["subject_id","date"])["device_count"]
               .mean().reset_index().rename(columns={"device_count":"ble_device_evening"}))
    night_b = _night_key(bucket[(bucket["hour"]>=22)|(bucket["hour"]<6)], hour_col="hour")
    agg_night = (night_b.groupby(["subject_id","night_key"])["device_count"]
                 .mean().reset_index()
                 .rename(columns={"night_key":"date","device_count":"ble_device_night"}))
    hourly = bucket.groupby(["subject_id","date","hour"])["device_count"].mean().reset_index()
    peak = (hourly.loc[hourly.groupby(["subject_id","date"])["device_count"].idxmax()]
            [["subject_id","date","hour"]].rename(columns={"hour":"ble_social_peak_hour"})
            .reset_index(drop=True))
    agg_scans = (df.groupby(["subject_id","date"])["address"]
                 .count().reset_index().rename(columns={"address":"ble_scans"}))

    for df_ in [agg_eve, agg_night, peak, agg_scans]:
        agg = agg.merge(df_, on=["subject_id","date"], how="left")
    return agg

# ══════════════════════════════════════════════════════════════════════
# 3. 전체 피처 테이블 생성
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# 3-b. 교차 parquet 피처 (Cross-source Features)
# ══════════════════════════════════════════════════════════════════════
#
# 설계 원칙
# ─────────
# 서로 다른 센서를 timestamp 기준으로 병합하여 단일 소스에서 얻을 수
# 없는 복합 정보를 추출한다.
#
# 주기 차이 해결
# ──────────────
# 센서마다 기록 주기가 다르므로 dt.floor()로 같은 시간 버킷으로 정렬:
#   FLOOR_FINE  = '5T'  : 심박/조도/활동/스크린/충전 등 고빈도
#   FLOOR_MED   = '5T'  : GPS/WiFi/BLE 등 중간 빈도
#   FLOOR_COARSE= '1H'  : 앱 사용통계 등 저빈도
# 병합 방식: outer join 후 결측 NaN 유지 → 집계 시 skipna=True
#
# 누수 검토 ✅
# ───────────
# 모든 교차 피처는 센서 데이터(X)만 사용하며 타겟(y)을 사용하지 않음.
# 모든 교차 피처는 lifelog_date(수면 전날) 기준으로 집계되므로
# 타겟(sleep_date 기준)과의 시간 방향이 올바름.

FLOOR_FINE  = "5T"   # 5분 단위 (wHr, wLight, mActivity, mACStatus, mScreenStatus, mLight, mAmbience)
FLOOR_MED   = "5T"   # 5분 단위 (mWifi, mGps, mBle)

def feat_SleepProxy_AC_Scr() -> pd.DataFrame:
    """🔋📱 수면 프록시: 충전O + 화면X 패턴 기반 수면 여부 판별.

    피처: night_sleep_ratio, night_slept, is_regular_night_sleeper
    night_key 날짜 경계 처리(00~05시 → 전날) 적용.
    """
    ac  = load_parquet("mACStatus")
    scr = load_parquet("mScreenStatus")
    ac["m_charging"]    = pd.to_numeric(ac["m_charging"],    errors="coerce").fillna(0)
    scr["m_screen_use"] = pd.to_numeric(scr["m_screen_use"], errors="coerce").fillna(0)

    rows = []
    for sid in ac["subject_id"].unique():
        sub_ac  = ac[ac["subject_id"]==sid].set_index("timestamp")[["m_charging"]]
        sub_scr = scr[scr["subject_id"]==sid].set_index("timestamp")[["m_screen_use"]]
        if sub_ac.empty or sub_scr.empty:
            continue

        merged = pd.merge(
            sub_ac.resample("1min").max().fillna(0),
            sub_scr.resample("1min").max().fillna(0),
            left_index=True, right_index=True, how="outer"
        ).fillna(0)
        merged["raw_proxy"] = ((merged["m_charging"]==1) & (merged["m_screen_use"]==0)).astype(int)
        # 10분 버킷 스무딩
        smoothed = (merged["raw_proxy"].resample("10min").mean() >= 0.3).astype(int)
        df_p = smoothed.to_frame(name="sleep_proxy").copy()
        df_p["hour"] = df_p.index.hour
        # night_key 적용 (00~05시 → 전날)
        df_p["date"] = df_p.index.normalize()
        df_p.loc[df_p["hour"] < 6, "date"] -= pd.Timedelta(days=1)
        night_m = (df_p["hour"] >= 22) | (df_p["hour"] < 6)

        for dt, grp in df_p[night_m].groupby("date"):
            ratio = grp["sleep_proxy"].mean()
            rows.append({
                "subject_id":       sid,
                "date":             dt,
                "night_sleep_ratio": ratio,
                "night_slept":       1 if ratio >= 0.5 else 0,
            })

    if not rows:
        return pd.DataFrame()

    agg = pd.DataFrame(rows)
    # 규칙적 수면자: 해당 사용자의 night_slept 비율 >= 0.6이면서 당일도 수면
    user_consist = agg.groupby("subject_id")["night_slept"].mean().rename("_consist")
    agg = agg.merge(user_consist, on="subject_id", how="left")
    agg["is_regular_night_sleeper"] = ((agg["_consist"] >= 0.6) & (agg["night_slept"] == 1)).astype(int)
    agg["date"] = pd.to_datetime(agg["date"])
    return agg[["subject_id","date","night_sleep_ratio","night_slept","is_regular_night_sleeper"]]

def feat_SleepProxy_TwoTrack() -> pd.DataFrame:
    """💡🔋 투트랙 수면 프록시: 조도(환경) + 기기(행동) 기반 수면 패턴.

    피처: user_archetype, light_sleep_duration, device_sleep_duration,
          sleep_bouts_count, light_night_sleep_ratio, device_night_sleep_ratio
    불규칙 사용자도 피처 값을 계산 (archetype만 0으로 표시).
    night_key 날짜 경계 처리 적용.
    """
    m_df  = load_parquet("mLight")
    w_df  = load_parquet("wLight")
    ac_df = load_parquet("mACStatus")
    scr_df= load_parquet("mScreenStatus")

    for df_, col in [(m_df,"m_light"),(w_df,"w_light"),
                     (ac_df,"m_charging"),(scr_df,"m_screen_use")]:
        df_[col] = pd.to_numeric(df_[col], errors="coerce")

    # 사용자 아키타입: 야간 조도 분산 → 하위 50% = 규칙적(1)
    def _night_var(df_, col):
        df_ = df_.copy()
        df_["hour"] = df_["timestamp"].dt.hour
        night = df_[(df_["hour"]>=22)|(df_["hour"]<6)]
        return night.groupby("subject_id")[col].var()

    m_var = _night_var(m_df, "m_light").rename("m_var")
    w_var = _night_var(w_df, "w_light").rename("w_var")
    profile = pd.concat([m_var, w_var], axis=1).fillna(0)
    profile["score"] = profile["m_var"] + profile["w_var"]
    median_score = profile["score"].median()
    profile["user_archetype"] = (profile["score"] <= median_score).astype(int)

    subjects = m_df["subject_id"].unique()
    final = []

    def _count_bouts(s):
        is_sleep = (s == 1)
        return int(((is_sleep != is_sleep.shift()) & is_sleep).sum())

    for sid in subjects:
        archetype = int(profile.loc[sid, "user_archetype"]) if sid in profile.index else 0

        def _resamp(df_, col, sid=sid):
            sub = df_[df_["subject_id"]==sid].copy()
            sub["timestamp"] = pd.to_datetime(sub["timestamp"])
            return sub.set_index("timestamp")[col].resample("10min").mean()

        df_sync = pd.DataFrame({
            "m_light":  _resamp(m_df,  "m_light"),
            "w_light":  _resamp(w_df,  "w_light"),
            "charging": _resamp(ac_df, "m_charging").ffill(),
            "screen":   _resamp(scr_df,"m_screen_use").ffill(),
        }).fillna({"charging":0,"screen":0})

        df_sync["sleep_by_light"]  = ((df_sync["m_light"]<=10)&(df_sync["w_light"]<=10)).astype(int)
        df_sync["sleep_by_device"] = ((df_sync["charging"]==1)&(df_sync["screen"]==0)).astype(int)
        df_sync["hour"] = df_sync.index.hour
        df_sync["date"] = df_sync.index.normalize()
        # night_key 적용
        df_sync.loc[df_sync["hour"]<6, "date"] -= pd.Timedelta(days=1)
        night_m = (df_sync["hour"]>=22)|(df_sync["hour"]<6)
        df_n = df_sync[night_m]

        daily = (
            df_n.groupby("date")
            .agg(
                light_sleep_duration  =("sleep_by_light",  lambda x: x.sum()*10),
                device_sleep_duration =("sleep_by_device", lambda x: x.sum()*10),
                sleep_bouts_count     =("sleep_by_light",  _count_bouts),
                light_night_sleep_ratio =("sleep_by_light",  "mean"),
                device_night_sleep_ratio=("sleep_by_device", "mean"),
            ).reset_index()
        )
        daily["subject_id"]    = sid
        daily["user_archetype"] = archetype
        final.append(daily)

    if not final:
        return pd.DataFrame()
    agg = pd.concat(final, ignore_index=True)
    agg["date"] = pd.to_datetime(agg["date"])
    return agg[["subject_id","date","user_archetype",
                "light_sleep_duration","device_sleep_duration",
                "sleep_bouts_count","light_night_sleep_ratio","device_night_sleep_ratio"]]


def feat_cross_arousal() -> pd.DataFrame:
    """
    그룹1: 취침 전 각성 자극 (mScreenStatus × mLight × wHr)

    피처 목록
    ─────────
    cross_dark_screen_min       야간(22~06시) 조도<5 lux + 화면ON  시간(분)
    cross_evening_bright_screen 저녁(19~22시) 조도>50 lux + 화면ON 시간(분)
    cross_presleep_screen_hr    취침 전 2h(21~23시) 화면ON 중 평균 심박수
    cross_presleep_hr_delta     (취침 전 화면ON 심박) - (해당일 전체 hr_mean)
    """
    # ── 스크린 (1분 단위) ──────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 폰 조도 (5분 단위, 소규모이므로 같은 floor) ─────────────────
    ml = _load_floored("mLight", FLOOR_FINE, ["m_light"])

    # ── 워치 심박 (5분 단위) ──────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 스크린 × 조도 병합 ────────────────────────────────────────────
    sc_ml = sc.merge(
        ml[["subject_id", "ts_floor", "m_light"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    sc_ml["hour"] = sc_ml["ts_floor"].dt.hour

    # 야간: 22~06시 (필터링 시 아래 _night_key 적용으로 날짜 보정)
    night_mask = (sc_ml["hour"] >= 22) | (sc_ml["hour"] < 6)
    # 각 버킷이 5분 → 1개 = 5분
    BUCKET_MIN = 5

    # cross_dark_screen_min: 야간 조도<5 + 화면ON (night_key 적용)
    sc_ml["is_dark_screen"] = (
        (sc_ml["m_light"] < 5) & (sc_ml["m_screen_use"] == 1)
    ).astype(float)
    sc_ml_dk = _night_key(sc_ml[night_mask])
    agg_dark = (
        sc_ml_dk
        .groupby(["subject_id", "night_key"])["is_dark_screen"]
        .sum().reset_index()
        .assign(cross_dark_screen_min=lambda x: x["is_dark_screen"] * BUCKET_MIN)
        .rename(columns={"night_key": "date"})
        [["subject_id", "date", "cross_dark_screen_min"]]
    )

    # cross_evening_bright_screen: 19~22시 조도>50 + 화면ON
    eve_mask = sc_ml["hour"].between(19, 21)
    sc_ml["is_bright_screen"] = (
        (sc_ml["m_light"] > 50) & (sc_ml["m_screen_use"] == 1)
    ).astype(float)
    agg_bright = (
        sc_ml[eve_mask]
        .groupby(["subject_id", "date"])["is_bright_screen"]
        .sum().reset_index()
        .assign(cross_evening_bright_screen=lambda x: x["is_bright_screen"] * BUCKET_MIN)
        [["subject_id", "date", "cross_evening_bright_screen"]]
    )

    # ── 스크린 × 심박 병합 ────────────────────────────────────────────
    sc_hr = sc.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    sc_hr["hour"] = sc_hr["ts_floor"].dt.hour

    # cross_presleep_screen_hr: 21~23시 화면ON 중 평균 심박
    pre_mask = sc_hr["hour"].between(21, 23) & (sc_hr["m_screen_use"] == 1)
    agg_hr = (
        sc_hr[pre_mask]
        .groupby(["subject_id", "date"])["hr"]
        .mean().reset_index()
        .rename(columns={"hr": "cross_presleep_screen_hr"})
    )

    # cross_presleep_hr_delta: 취침 전 화면ON 심박 - 해당일 전체 평균 심박
    hr_daily = (
        hr.groupby(["subject_id", "date"])["hr"].mean().reset_index()
        .rename(columns={"hr": "hr_daily_mean"})
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_dark.merge(agg_bright, on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_hr,         on=["subject_id", "date"], how="outer")
    agg = agg.merge(hr_daily,       on=["subject_id", "date"], how="left")
    agg["cross_presleep_hr_delta"] = (
        agg["cross_presleep_screen_hr"] - agg["hr_daily_mean"]
    )
    return agg.drop(columns=["hr_daily_mean"])

def feat_cross_sleep_env() -> pd.DataFrame:
    """
    그룹2: 수면 환경 품질 (mACStatus × mAmbience × mLight)

    충전 상태를 수면 proxy로 사용하여 실제 수면 중 환경 측정.

    피처 목록
    ─────────
    cross_sleep_noise_mean       야간 충전 중 평균 소음 확률
    cross_sleep_noise_speech     야간 충전 중 Speech 계열 소음 비율
    cross_sleep_light_mean       야간 충전 중 평균 조도
    cross_sleep_light_max        야간 충전 중 최대 조도
    cross_charge_screen_on_ratio 충전 중 화면ON 비율 (수면 지연 지표)
    cross_charge_screen_off_min  충전+화면OFF 지속 시간(분) = 실수면 추정
    """
    # ── 충전 상태 ──────────────────────────────────────────────────────
    ac = _load_floored("mACStatus", FLOOR_FINE, ["m_charging"])

    # ── 소음 ──────────────────────────────────────────────────────────
    amb_df = load_parquet("mAmbience")
    amb_df = amb_df.explode("m_ambience")
    amb_df["sound_type"] = amb_df["m_ambience"].str[0]
    amb_df["amb_level"]  = pd.to_numeric(amb_df["m_ambience"].str[1], errors="coerce")
    amb_df = amb_df.dropna(subset=["amb_level"])
    amb_df["is_speech"] = amb_df["sound_type"].str.contains(
        "Speech|Narration|Conversation", case=False, na=False
    ).astype(float)
    amb_df["ts_floor"] = amb_df["timestamp"].dt.floor(FLOOR_FINE)
    amb_agg = (
        amb_df.groupby(["subject_id", "date", "ts_floor"])
        .agg(amb_level_mean=("amb_level", "mean"),
             amb_speech_r   =("is_speech", "mean"))
        .reset_index()
    )

    # ── 폰 조도 ───────────────────────────────────────────────────────
    ml = _load_floored("mLight", FLOOR_FINE, ["m_light"])

    # ── 스크린 ────────────────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 충전 시간대 필터 (야간: 21~08시): night_key 적용 ──────────────
    ac["hour"] = ac["ts_floor"].dt.hour
    night_ac   = _night_key(ac[(ac["hour"] >= 21) | (ac["hour"] < 8)])
    # night_key를 date로 덮어써서 이후 groupby에서 자동 적용
    night_ac["date"] = night_ac["night_key"]

    # 충전 중인 버킷만
    charging = night_ac[night_ac["m_charging"] == 1].copy()
    BUCKET_MIN = 5

    # 충전+소음
    ch_amb = charging.merge(
        amb_agg[["subject_id", "ts_floor", "amb_level_mean", "amb_speech_r"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_amb = (
        ch_amb.groupby(["subject_id", "date"])
        .agg(cross_sleep_noise_mean  =("amb_level_mean", "mean"),
             cross_sleep_noise_speech=("amb_speech_r",   "mean"))
        .reset_index()
    )

    # 충전+조도
    ch_ml = charging.merge(
        ml[["subject_id", "ts_floor", "m_light"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_light = (
        ch_ml.groupby(["subject_id", "date"])
        .agg(cross_sleep_light_mean=("m_light", "mean"),
             cross_sleep_light_max =("m_light", "max"))
        .reset_index()
    )

    # 충전+스크린 (야간 전체)
    night_ac_sc = night_ac.merge(
        sc[["subject_id", "ts_floor", "m_screen_use"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    agg_sc = (
        night_ac_sc.groupby(["subject_id", "date"])
        .agg(
            _sc_on  =("m_screen_use", "sum"),
            _sc_tot =("m_screen_use", "count"),
        )
        .reset_index()
    )
    agg_sc["cross_charge_screen_on_ratio"] = (
        agg_sc["_sc_on"] / agg_sc["_sc_tot"].replace(0, np.nan)
    )
    # 충전+화면OFF = 실수면 추정
    night_ac_sc["is_sleep_bucket"] = (
        (night_ac_sc["m_charging"] == 1) & (night_ac_sc["m_screen_use"] == 0)
    ).astype(float)
    agg_sleep = (
        night_ac_sc.groupby(["subject_id", "date"])["is_sleep_bucket"]
        .sum().reset_index()
        .assign(cross_charge_screen_off_min=lambda x: x["is_sleep_bucket"] * BUCKET_MIN)
        [["subject_id", "date", "cross_charge_screen_off_min"]]
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_amb.merge(agg_light, on=["subject_id", "date"], how="outer")
    agg = agg.merge(
        agg_sc[["subject_id", "date", "cross_charge_screen_on_ratio"]],
        on=["subject_id", "date"], how="outer"
    )
    agg = agg.merge(agg_sleep, on=["subject_id", "date"], how="outer")
    return agg


def feat_cross_exercise() -> pd.DataFrame:
    """
    그룹3: 운동 강도 (mActivity × wHr)

    피처 목록
    ─────────
    cross_exercise_hr_mean      걷기/뛰기 시 평균 심박수 (운동 강도)
    cross_exercise_hr_max       운동 중 최대 심박수
    cross_exercise_hr_recovery  운동 직후 30분 평균 심박 - 운동 중 평균 심박
    cross_exercise_timing_hour  운동(활동+고심박) 피크 시간대
    cross_late_exercise_min     늦은 저녁(20~23시) 운동 시간(분)
    """
    # ── 활동 ──────────────────────────────────────────────────────────
    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])

    # ── 워치 심박 ─────────────────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 병합 ──────────────────────────────────────────────────────────
    merged = act.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    merged["hour"] = merged["ts_floor"].dt.hour
    merged["is_active"] = merged["m_activity"].isin([7, 8])  # walk=7, run=8

    # 운동 중 심박
    active_hr = merged[merged["is_active"]]
    agg_exer = (
        active_hr.groupby(["subject_id", "date"])["hr"]
        .agg(cross_exercise_hr_mean="mean",
             cross_exercise_hr_max="max")
        .reset_index()
    )

    # 운동 피크 시간대 (심박>100 + 활동 구간)
    merged["is_intense"] = merged["is_active"] & (merged["hr"] > 100)
    hourly_intense = (
        merged.groupby(["subject_id", "date", "hour"])["is_intense"]
        .sum().reset_index()
    )
    peak = (
        hourly_intense.loc[
            hourly_intense.groupby(["subject_id", "date"])["is_intense"].idxmax()
        ][["subject_id", "date", "hour"]]
        .rename(columns={"hour": "cross_exercise_timing_hour"})
        .reset_index(drop=True)
    )

    # 운동 후 심박 회복 (운동 직후 30분 vs 운동 중)
    merged_sorted = merged.sort_values(["subject_id", "ts_floor"])
    merged_sorted["post_active"] = (
        merged_sorted.groupby("subject_id")["is_active"]
        .shift(6).fillna(False)   # 6 × 5분 = 30분 후
    )
    recovery = merged_sorted[merged_sorted["post_active"] & ~merged_sorted["is_active"]]
    agg_rec = (
        recovery.groupby(["subject_id", "date"])["hr"]
        .mean().reset_index()
        .rename(columns={"hr": "post_exercise_hr"})
    )

    # 늦은 저녁 운동 시간(분)
    BUCKET_MIN = 5
    late_mask = merged["hour"].between(20, 23) & merged["is_active"]
    agg_late = (
        merged[late_mask]
        .groupby(["subject_id", "date"])["is_active"]
        .sum().reset_index()
        .assign(cross_late_exercise_min=lambda x: x["is_active"] * BUCKET_MIN)
        [["subject_id", "date", "cross_late_exercise_min"]]
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_exer.merge(peak,     on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_late,      on=["subject_id", "date"], how="outer")
    agg = agg.merge(agg_rec,       on=["subject_id", "date"], how="left")
    agg["cross_exercise_hr_recovery"] = (
        agg["post_exercise_hr"] - agg["cross_exercise_hr_mean"]
    )
    return agg.drop(columns=["post_exercise_hr"], errors="ignore")

def feat_cross_mobility() -> pd.DataFrame:
    """
    그룹4: 사회적 활동 & 이동 (mGps × mActivity × mWifi)

    피처 목록
    ─────────
    cross_outdoor_active_min    GPS 이동 + 활동=걷기/뛰기 동시 충족 시간(분)
    cross_home_sedentary_min    집 WiFi + 활동=정지 동시 시간(분)
    cross_home_active_ratio     집 WiFi 연결 중 활동적인 비율
    """
    BUCKET_MIN = 5

    # ── GPS (1분 단위, 속도만) ────────────────────────────────────────
    gps_df = load_parquet("mGps")
    gps_df["ts_floor"] = gps_df["timestamp"].dt.floor(FLOOR_MED)
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            for c in ["latitude", "longitude", "speed"]:
                gps_df[c] = gps_df["m_gps"].apply(
                    lambda x: x.get(c, np.nan) if isinstance(x, dict) else np.nan
                )
    gps_df = gps_df.dropna(subset=["latitude", "longitude"])
    # 이상값 제거
    gps_df["speed"] = pd.to_numeric(gps_df.get("speed", np.nan), errors="coerce")
    gps_df = gps_df[gps_df["speed"].isna() | (gps_df["speed"] <= GPS_SPEED_LIMIT_MS)]
    gps_spd = (
        gps_df.groupby(["subject_id", "date", "ts_floor"])["speed"]
        .mean().reset_index()
    )

    # ── 활동 ──────────────────────────────────────────────────────────
    act = _load_floored("mActivity", FLOOR_MED, ["m_activity"])

    # ── WiFi 집 BSSID ─────────────────────────────────────────────────
    wifi_df = load_parquet("mWifi")
    wifi_df["ts_floor"] = wifi_df["timestamp"].dt.floor(FLOOR_MED)
    wifi_df = wifi_df.explode("m_wifi")
    _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
    if _fv is not None and isinstance(_fv, dict):
        wifi_df["bssid"] = wifi_df["m_wifi"].apply(
            lambda x: x.get("bssid", "") if isinstance(x, dict) else ""
        )
    else:
        wifi_df["bssid"] = ""

    home_bssid = (
        wifi_df[wifi_df["bssid"] != ""]
        .groupby(["subject_id", "bssid"]).size().reset_index(name="cnt")
        .sort_values("cnt", ascending=False)
        .drop_duplicates("subject_id")[["subject_id", "bssid"]]
        .rename(columns={"bssid": "home_bssid"})
    )
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"] == wifi_df["home_bssid"]).astype(int)
    wifi_home = (
        wifi_df.groupby(["subject_id", "date", "ts_floor"])["is_home"]
        .max().reset_index()   # 버킷 내 집 WiFi 감지 여부
    )

    # ── GPS × 활동: 실외 활동 ─────────────────────────────────────────
    gps_act = gps_spd.merge(
        act[["subject_id", "ts_floor", "m_activity"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    gps_act["is_outdoor_active"] = (
        (gps_act["speed"] > 0.5) & gps_act["m_activity"].isin([7, 8])
    ).astype(float)
    agg_outdoor = (
        gps_act.groupby(["subject_id", "date"])["is_outdoor_active"]
        .sum().reset_index()
        .assign(cross_outdoor_active_min=lambda x: x["is_outdoor_active"] * BUCKET_MIN)
        [["subject_id", "date", "cross_outdoor_active_min"]]
    )

    # ── WiFi × 활동: 집안 정지/활동 ──────────────────────────────────
    wifi_act = wifi_home.merge(
        act[["subject_id", "ts_floor", "m_activity"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    wifi_at_home = wifi_act[wifi_act["is_home"] == 1]

    wifi_at_home = wifi_at_home.copy()
    wifi_at_home["is_sedentary"] = (wifi_at_home["m_activity"] == 3).astype(float)
    wifi_at_home["is_moving"]    = wifi_at_home["m_activity"].isin([7, 8]).astype(float)

    agg_home_sed = (
        wifi_at_home.groupby(["subject_id", "date"])["is_sedentary"]
        .sum().reset_index()
        .assign(cross_home_sedentary_min=lambda x: x["is_sedentary"] * BUCKET_MIN)
        [["subject_id", "date", "cross_home_sedentary_min"]]
    )
    agg_home_act = (
        wifi_at_home.groupby(["subject_id", "date"])
        .agg(_mv=("is_moving", "sum"), _tot=("is_moving", "count"))
        .reset_index()
    )
    agg_home_act["cross_home_active_ratio"] = (
        agg_home_act["_mv"] / agg_home_act["_tot"].replace(0, np.nan)
    )

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_outdoor.merge(agg_home_sed, on=["subject_id", "date"], how="outer")
    agg = agg.merge(
        agg_home_act[["subject_id", "date", "cross_home_active_ratio"]],
        on=["subject_id", "date"], how="outer"
    )
    return agg

def feat_cross_circadian() -> pd.DataFrame:
    """
    그룹5: 일주기 리듬 (wLight × wHr × mScreenStatus)

    피처 목록
    ─────────
    cross_morning_light_hr_corr   아침(06~09시) 광량-심박 상관계수
    cross_morning_natural_light   아침(06~10시) 자연광(>100 lux)+화면OFF 시간(분)
    cross_evening_light_drop      저녁(18~22시) 광량 감소율 (취침 준비 신호)
    """
    BUCKET_MIN = 5

    # ── 워치 광량 ─────────────────────────────────────────────────────
    wl = _load_floored("wLight", FLOOR_FINE, ["w_light"])

    # ── 워치 심박 ─────────────────────────────────────────────────────
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object:
        hr = hr.explode("heart_rate")
    hr["hr"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr"])
    hr = hr[hr["hr"].between(30, 220)]
    hr_agg = (
        hr.groupby(["subject_id", "date", "ts_floor"])["hr"]
        .mean().reset_index()
    )

    # ── 스크린 ────────────────────────────────────────────────────────
    sc = _load_floored("mScreenStatus", FLOOR_FINE, ["m_screen_use"])

    # ── 워치 광량 × 심박: 아침 상관계수 ─────────────────────────────
    wl_hr = wl.merge(
        hr_agg[["subject_id", "ts_floor", "hr"]],
        on=["subject_id", "ts_floor"], how="inner"
    )
    wl_hr["hour"] = wl_hr["ts_floor"].dt.hour
    morning_wl_hr = wl_hr[wl_hr["hour"].between(6, 9)]

    def _corr(g):
        if len(g) < 3: return np.nan
        return g["w_light"].corr(g["hr"])

    agg_corr = (
        morning_wl_hr.groupby(["subject_id", "date"])
        .apply(_corr).reset_index()
        .rename(columns={0: "cross_morning_light_hr_corr"})
    )

    # ── 워치 광량 × 스크린: 아침 자연광 노출 ─────────────────────────
    wl_sc = wl.merge(
        sc[["subject_id", "ts_floor", "m_screen_use"]],
        on=["subject_id", "ts_floor"], how="left"
    )
    wl_sc["hour"] = wl_sc["ts_floor"].dt.hour
    morning_wl_sc = wl_sc[wl_sc["hour"].between(6, 10)]
    morning_wl_sc = morning_wl_sc.copy()
    morning_wl_sc["is_natural_no_screen"] = (
        (morning_wl_sc["w_light"] > 100) & (morning_wl_sc["m_screen_use"] == 0)
    ).astype(float)
    agg_nat = (
        morning_wl_sc.groupby(["subject_id", "date"])["is_natural_no_screen"]
        .sum().reset_index()
        .assign(cross_morning_natural_light=lambda x: x["is_natural_no_screen"] * BUCKET_MIN)
        [["subject_id", "date", "cross_morning_natural_light"]]
    )

    # ── 워치 광량: 저녁 광량 감소율 ──────────────────────────────────
    wl_eve = wl_hr[wl_hr["hour"].between(18, 22)]
    if len(wl_eve) > 0:
        # 각 날 저녁 시작(18시) 광량 vs 끝(22시) 광량
        wl_eve_agg = (
            wl_eve.groupby(["subject_id", "date", "hour"])["w_light"]
            .mean().reset_index()
        )
        eve_start = wl_eve_agg[wl_eve_agg["hour"] == 18][["subject_id","date","w_light"]].rename(columns={"w_light": "light_18"})
        eve_end   = wl_eve_agg[wl_eve_agg["hour"] == 22][["subject_id","date","w_light"]].rename(columns={"w_light": "light_22"})
        agg_drop  = eve_start.merge(eve_end, on=["subject_id","date"], how="inner")
        agg_drop["cross_evening_light_drop"] = (
            (agg_drop["light_18"] - agg_drop["light_22"]) /
            agg_drop["light_18"].replace(0, np.nan)
        )
        agg_drop = agg_drop[["subject_id","date","cross_evening_light_drop"]]
    else:
        agg_drop = pd.DataFrame(columns=["subject_id","date","cross_evening_light_drop"])

    # ── 최종 합산 ─────────────────────────────────────────────────────
    agg = agg_corr.merge(agg_nat,  on=["subject_id","date"], how="outer")
    agg = agg.merge(agg_drop,      on=["subject_id","date"], how="outer")
    return agg

def feat_cross_commute() -> pd.DataFrame:
    """🚗 통근 시간: IN_VEHICLE + GPS 이동 (mActivity × mGps)."""
    BUCKET_MIN = 5
    act = _load_floored("mActivity", FLOOR_MED, ["m_activity"])
    gps_df = load_parquet("mGps")
    gps_df["ts_floor"] = gps_df["timestamp"].dt.floor(FLOOR_MED)
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            gps_df["speed_g"] = gps_df["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
    gps_df["speed_g"] = pd.to_numeric(gps_df.get("speed_g", np.nan), errors="coerce")
    gps_spd = (gps_df.groupby(["subject_id","date","ts_floor"])["speed_g"]
               .mean().reset_index())
    merged = act.merge(gps_spd[["subject_id","ts_floor","speed_g"]],
                       on=["subject_id","ts_floor"], how="inner")
    merged["is_commute"] = ((merged["m_activity"]==0) & (merged["speed_g"]>1.5)).astype(float)
    agg = (merged.groupby(["subject_id","date"])["is_commute"]
           .sum().reset_index()
           .assign(cross_commute_min=lambda x: x["is_commute"]*BUCKET_MIN)
           [["subject_id","date","cross_commute_min"]])
    return agg

def feat_cross_presleep_arousal() -> pd.DataFrame:
    """😤 취침 전 각성 지수: wHr × wPedo × mActivity (20~22시)."""
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object: hr = hr.explode("heart_rate")
    hr["hr_val"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr_val"])
    hr = hr[hr["hr_val"].between(30,220)]
    hr["hour"] = hr["timestamp"].dt.hour
    hr_pre = (hr[hr["hour"].between(20,22)]
              .groupby(["subject_id","date"])["hr_val"]
              .mean().reset_index().rename(columns={"hr_val":"_hr_pre"}))

    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])
    act["hour"] = act["ts_floor"].dt.hour
    act_pre = (act[act["hour"].between(20,22)]
               .groupby(["subject_id","date"])["m_activity"]
               .agg(lambda x: (x.isin([7,8,9])).mean()).reset_index()
               .rename(columns={"m_activity":"_act_pre"}))

    ped = load_parquet("wPedo")
    ped["hour"] = ped["timestamp"].dt.hour
    ped_pre = (ped[ped["hour"].between(20,22)]
               .groupby(["subject_id","date"])["step"]
               .sum().reset_index().rename(columns={"step":"_step_pre"}))

    agg = hr_pre.merge(act_pre, on=["subject_id","date"], how="outer")
    agg = agg.merge(ped_pre, on=["subject_id","date"], how="outer")
    # z-score 없이 0~1 정규화 후 합산 → 각성 지수
    for c in ["_hr_pre","_act_pre","_step_pre"]:
        mn = agg[c].min(); mx = agg[c].max()
        agg[c] = (agg[c]-mn) / (mx-mn+1e-8)
    agg["cross_presleep_arousal"] = agg[["_hr_pre","_act_pre","_step_pre"]].mean(axis=1)
    return agg[["subject_id","date","cross_presleep_arousal"]]

def feat_cross_resting_hr_at_home() -> pd.DataFrame:
    """💓 실제 안정시 심박: 집 WiFi + STILL 구간 심박 (wHr × mWifi × mActivity)."""
    hr = load_parquet("wHr")
    hr["ts_floor"] = hr["timestamp"].dt.floor(FLOOR_FINE)
    if hr["heart_rate"].dtype == object: hr = hr.explode("heart_rate")
    hr["hr_val"] = pd.to_numeric(hr["heart_rate"], errors="coerce")
    hr = hr.dropna(subset=["hr_val"])
    hr = hr[hr["hr_val"].between(30,220)]
    hr_agg = hr.groupby(["subject_id","date","ts_floor"])["hr_val"].mean().reset_index()

    act = _load_floored("mActivity", FLOOR_FINE, ["m_activity"])

    wifi_df = load_parquet("mWifi")
    wifi_df["ts_floor"] = wifi_df["timestamp"].dt.floor(FLOOR_FINE)
    wifi_df = wifi_df.explode("m_wifi")
    _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
    if _fv is not None and isinstance(_fv, dict):
        wifi_df["bssid"] = wifi_df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
    else:
        wifi_df["bssid"] = ""
    home_bssid = (wifi_df[wifi_df["bssid"]!=""].groupby(["subject_id","bssid"])
                  .size().reset_index(name="cnt").sort_values("cnt", ascending=False)
                  .drop_duplicates("subject_id")[["subject_id","bssid"]]
                  .rename(columns={"bssid":"home_bssid"}))
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"]==wifi_df["home_bssid"]).astype(int)
    wifi_home = (wifi_df.groupby(["subject_id","date","ts_floor"])["is_home"]
                 .max().reset_index())

    merged = (hr_agg
              .merge(act[["subject_id","ts_floor","m_activity"]], on=["subject_id","ts_floor"], how="inner")
              .merge(wifi_home[["subject_id","ts_floor","is_home"]], on=["subject_id","ts_floor"], how="left"))
    still_home = merged[(merged["m_activity"]==3) & (merged["is_home"]==1)]
    agg = (still_home.groupby(["subject_id","date"])["hr_val"]
           .mean().reset_index().rename(columns={"hr_val":"cross_resting_hr_at_home"}))
    return agg


def feat_cross_sleep_env_noise() -> pd.DataFrame:
    """🌙 수면 환경 복잡도: BLE × WiFi rssi × GPS 정지 비율 (야간)."""
    ble_df = load_parquet("mBle")
    ble_df["hour"] = ble_df["timestamp"].dt.hour
    if "m_ble" in ble_df.columns and ble_df["m_ble"].dtype == object:
        try:
            ble_df = ble_df.explode("m_ble")
            _fv = ble_df["m_ble"].dropna().iloc[0] if ble_df["m_ble"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                ble_df["address"] = ble_df["m_ble"].apply(lambda x: x.get("address","") if isinstance(x,dict) else "")
        except Exception:
            ble_df["address"] = ""
    night_m = (ble_df["hour"]>=22)|(ble_df["hour"]<6)
    ble_n = _night_key(ble_df[night_m & (ble_df["address"]!="")])
    ble_night = (ble_n.groupby(["subject_id","night_key"])["address"].nunique().reset_index()
                 .rename(columns={"night_key":"date","address":"_ble_night"}))

    wifi_df = load_parquet("mWifi")
    wifi_df["hour"] = wifi_df["timestamp"].dt.hour
    if "m_wifi" in wifi_df.columns and wifi_df["m_wifi"].dtype == object:
        try:
            wifi_df = wifi_df.explode("m_wifi")
            _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
            if _fv is not None and isinstance(_fv, dict):
                wifi_df["rssi"] = wifi_df["m_wifi"].apply(lambda x: x.get("rssi",np.nan) if isinstance(x,dict) else np.nan)
        except Exception:
            wifi_df["rssi"] = np.nan
    night_w = (wifi_df["hour"]>=22)|(wifi_df["hour"]<6)
    wifi_n = _night_key(wifi_df[night_w])
    wifi_night = (wifi_n.groupby(["subject_id","night_key"])["rssi"]
                  .mean().reset_index()
                  .rename(columns={"night_key":"date","rssi":"_wifi_rssi_night"}))

    gps_df = load_parquet("mGps")
    gps_df["hour"] = gps_df["timestamp"].dt.hour
    if "m_gps" in gps_df.columns and gps_df["m_gps"].dtype == object:
        gps_df = gps_df.explode("m_gps")
        _fv = gps_df["m_gps"].dropna().iloc[0] if gps_df["m_gps"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            gps_df["speed_g"] = gps_df["m_gps"].apply(lambda x: x.get("speed",np.nan) if isinstance(x,dict) else np.nan)
    gps_df["speed_g"] = pd.to_numeric(gps_df.get("speed_g", np.nan), errors="coerce")
    night_g = (gps_df["hour"]>=22)|(gps_df["hour"]<6)
    gps_n = _night_key(gps_df[night_g])
    gps_night = (gps_n.groupby(["subject_id","night_key"])["speed_g"]
                 .agg(lambda x: (x<1.0).mean()).reset_index()
                 .rename(columns={"night_key":"date","speed_g":"_gps_stop_night"}))

    agg = ble_night.merge(wifi_night, on=["subject_id","date"], how="outer")
    agg = agg.merge(gps_night, on=["subject_id","date"], how="outer")
    for c in ["_ble_night","_wifi_rssi_night","_gps_stop_night"]:
        mn=agg[c].min(); mx=agg[c].max()
        agg[c] = (agg[c]-mn)/(mx-mn+1e-8)
    agg["cross_sleep_env_noise"] = agg[["_ble_night","_wifi_rssi_night","_gps_stop_night"]].mean(axis=1)
    return agg[["subject_id","date","cross_sleep_env_noise"]]

def feat_cross_wifi_gps_home_agree() -> pd.DataFrame:
    """🏠 집 판별 신뢰도: WiFi home 비율 × GPS home_min 일치 여부."""
    wifi_df = load_parquet("mWifi")
    if "m_wifi" in wifi_df.columns and wifi_df["m_wifi"].dtype == object:
        wifi_df = wifi_df.explode("m_wifi")
        _fv = wifi_df["m_wifi"].dropna().iloc[0] if wifi_df["m_wifi"].notna().any() else None
        if _fv is not None and isinstance(_fv, dict):
            wifi_df["bssid"] = wifi_df["m_wifi"].apply(lambda x: x.get("bssid","") if isinstance(x,dict) else "")
    else:
        wifi_df["bssid"] = ""
    home_bssid = (wifi_df[wifi_df["bssid"]!=""].groupby(["subject_id","bssid"])
                  .size().reset_index(name="cnt").sort_values("cnt",ascending=False)
                  .drop_duplicates("subject_id")[["subject_id","bssid"]]
                  .rename(columns={"bssid":"home_bssid"}))
    wifi_df = wifi_df.merge(home_bssid, on="subject_id", how="left")
    wifi_df["is_home"] = (wifi_df["bssid"]==wifi_df["home_bssid"]).astype(int)
    wifi_home_r = (wifi_df.groupby(["subject_id","date"])["is_home"]
                   .mean().reset_index().rename(columns={"is_home":"_wifi_home_r"}))
    # GPS home_min은 feat_mGps()에서 생성됨 → 여기서는 WiFi home 비율만 반환
    # (파이프라인에서 gps_home_min과 merge하여 cross_wifi_gps_home_agree 계산)
    wifi_home_r = wifi_home_r.rename(columns={"_wifi_home_r":"cross_wifi_gps_home_agree"})
    return wifi_home_r

def build_cross_features(date_df: pd.DataFrame) -> pd.DataFrame:
    """
    교차 parquet 피처 전체 생성 및 date_df에 merge.

    각 함수는 (subject_id, date) 기준 일별 집계값을 반환.
    date → lifelog_date 로 rename 후 left merge.
    """
    print("교차 parquet 피처 추출 시작...")

    cross_funcs = {
        "arousal":   feat_cross_arousal,    # 그룹1: 취침 전 각성 자극
        "sleep_env": feat_cross_sleep_env,  # 그룹2: 수면 환경 품질
        "exercise":  feat_cross_exercise,   # 그룹3: 운동 강도
        "mobility":  feat_cross_mobility,   # 그룹4: 사회적 활동 & 이동
        "circadian": feat_cross_circadian,  # 그룹5: 일주기 리듬
    }

    df = date_df.copy()
    for name, func in cross_funcs.items():
        print(f"  [cross_{name}] 처리 중...", end=" ", flush=True)
        try:
            feat = func()
            feat = feat.rename(columns={"date": "lifelog_date"})
            feat["lifelog_date"] = pd.to_datetime(feat["lifelog_date"])
            df["lifelog_date"]   = pd.to_datetime(df["lifelog_date"])
            before = df.shape[1]
            df     = df.merge(feat, on=["subject_id", "lifelog_date"], how="left")
            added  = df.shape[1] - before
            print(f"완료 (+{added} 컬럼)")
        except Exception as e:
            print(f"오류: {e}")
            import traceback; traceback.print_exc()

    return df

def build_features(date_df: pd.DataFrame) -> pd.DataFrame:
    """
    date_df : subject_id / sleep_date / lifelog_date 컬럼 포함
              lifelog_date 는 naive datetime64[ns] 이어야 함

    각 feat_*() 의 date 컬럼(naive datetime64[ns])을
    lifelog_date 에 left merge → 타입 완전 일치, 누락 없이 조인
    """
    print("피처 추출 시작...")

    feat_funcs = {
        "wPedo":          feat_wPedo,
        "wHr":            feat_wHr,
        "wLight":         feat_wLight,
        "mActivity":      feat_mActivity,
        "mACStatus":      feat_mACStatus,
        "mScreenStatus":  feat_mScreenStatus,
        "mLight":         feat_mLight,
        "mAmbience":      feat_mAmbience,
        "mWifi":          feat_mWifi,
        "mGps":           feat_mGps,
        "mUsageStats":    feat_mUsageStats,
        "mBle":           feat_mBle,
        "SleepProxy_AC_Scr":  feat_SleepProxy_AC_Scr,
        "SleepProxy_TwoTrack": feat_SleepProxy_TwoTrack,
        # 기존 복합 피처
        "cross_arousal":     feat_cross_arousal,
        "cross_sleep_env":   feat_cross_sleep_env,
        "cross_exercise":    feat_cross_exercise,
        "cross_mobility":    feat_cross_mobility,
        "cross_circadian":   feat_cross_circadian,
        # 신규 복합 피처 (V53)
        "cross_commute":             feat_cross_commute,
        "cross_presleep_arousal":    feat_cross_presleep_arousal,
        "cross_resting_hr_at_home":  feat_cross_resting_hr_at_home,
        "cross_sleep_env_noise":     feat_cross_sleep_env_noise,
        "cross_wifi_gps_home_agree": feat_cross_wifi_gps_home_agree,
    }

    df = date_df.copy()
    for name, func in feat_funcs.items():
        print(f"  [{name}] 처리 중...", end=" ", flush=True)
        try:
            feat = func()
            feat = feat.rename(columns={"date": "lifelog_date"})
            df   = df.merge(feat, on=["subject_id", "lifelog_date"], how="left")
            print(f"완료 ({len(feat):,} rows, +{feat.shape[1] - 2} cols)")
        except Exception as e:
            print(f"오류: {e}")

    return df

# ══════════════════════════════════════════════════════════════════════
# 4. 시간/요일 파생 피처
# ══════════════════════════════════════════════════════════════════════

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """sleep_date 기반 요일/월/주차 피처 + subject_num 인코딩"""
    df = df.copy()
    df["sleep_date"] = pd.to_datetime(df["sleep_date"])

    df["dow"]          = df["sleep_date"].dt.dayofweek       # 0=월, 6=일
    df["is_weekend"]   = (df["dow"] >= 5).astype(int)
    df["month"]        = df["sleep_date"].dt.month
    df["week_of_year"] = df["sleep_date"].dt.isocalendar().week.astype(int)
    df["subject_num"]  = df["subject_id"].str.extract(r"(\d+)").astype(int)
    return df


## 5-1. Z-Score + 센서 Rolling + LGB/XGB/CatBoost 가중 앙상블

# ══════════════════════════════════════════════════════════════════════
# 5. 래그 / 개인 평균 피처 — fold 내부 전용 헬퍼
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# 6. LightGBM 학습
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════
# 5-v5-1. Z-score + 센서 Rolling + LGB/XGB/CatBoost 가중 앙상블
# ══════════════════════════════════════════════════════════════════════

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

# ── 모델 하이퍼파라미터 (V5와 동일) ─────────────────────────────────
LGBM_PARAMS = {
    "objective":          "binary",
    "metric":             "binary_logloss",
    "n_estimators":       1000, # Optuna 튜닝 시에는 500으로 줄였더라도 최종 학습 시에는 1000 이상으로 올려주는 것이 좋습니다.
    "learning_rate":      0.048002539014146264, # Optuna 값 적용
    "num_leaves":         43,                   # Optuna 값 적용
    "min_child_samples":  21,                   # Optuna 값 적용
    "feature_fraction":   0.6034723418591593,   # Optuna 값 적용
    "bagging_fraction":   0.8257774001264698,   # Optuna 값 적용
    "bagging_freq":       5,
    "lambda_l1":          0.1,                  # 기존 값 유지 (또는 튜닝했다면 해당 값)
    "lambda_l2":          0.1,                  # 기존 값 유지
    "random_state":       42,
    "n_jobs":             -1,
    "verbose":            -1,
}
XGB_PARAMS = {
    "objective":          "binary:logistic",
    "eval_metric":        "logloss",
    "n_estimators":       1000,
    "learning_rate":      0.012936739731011452, # Optuna 값 적용
    "max_depth":          3,                    # Optuna 값 적용
    "subsample":          0.8135516777791894,   # Optuna 값 적용
    "colsample_bytree":   0.7603815432836665,   # Optuna 값 적용
    "reg_alpha":          0.1,                  # 기존 값 유지
    "reg_lambda":         0.1,                  # 기존 값 유지
    "random_state":       42,
    "n_jobs":             -1,
    "verbosity":          0,
    "early_stopping_rounds": 50
}
CAT_PARAMS = {
    "iterations":         1000,
    "learning_rate":      0.0528527508011132,   # Optuna 값 적용
    "depth":              7,                    # Optuna 값 적용
    "l2_leaf_reg":        6.027332560539019,    # Optuna 값 적용
    "loss_function":      "Logloss",
    "eval_metric":        "Logloss",
    "random_seed":        42,
    "verbose":            False,
    "allow_writing_files": False,
}

# ── 센서 Rolling 설정 ─────────────────────────────────────────────────
# V3와 달리 센서 데이터(X)에 rolling을 적용 → test에도 결측 없음
# 이유:
#   V3: 타겟(y) rolling → test의 y가 없어 평균 44~84% 결측
#   V5-1: 센서(X) rolling → train/test 모두 X값 존재 → 결측 <5%
ROLL_WINDOWS         = [3, 7]     # rolling 윈도우 크기 (일)
ROLL_MISS_THRESHOLD  = 0.30       # rolling 피처 결측률 임계값 (30% 초과 제거)

# 피처 선택 설정
# 피어슨 상관계수 절댓값 기준 상위 CORR_TOP_K개 선택
# - raw+rolling: 86+~100 = ~186개 후보
# - z-score     : 동일 수 후보
# - 전체 후보 ~370개에서 타겟별로 50개 선택 → 샘플(450):피처(50) = 9:1
CORR_TOP_K  = 50
N_FOLDS     = 5

# ── Subject-Hole CV 설정 ──────────────────────────────────────────────
# 실제 train/test 분리 구조 (T/V/T/V 인터리빙)를 모사하는 CV 전략.
#
# 실제 데이터 패턴 (id01 예시):
#   TTTTTTTTTTTTTTTTTTTTTTTTTTTTVVVVVVVVVVVVVVTTTTTTTTTTTTTVVVVVVVVVVVVV
#   → train과 test가 사용자 내부에서 시간 순으로 교차하는 블록 구조
#
# Subject-Hole CV 방식:
#   각 사용자의 날짜를 시간 순 정렬 후 n_folds*2 개의 chunk로 분할.
#   fold i: chunk[i] + chunk[i+n_folds] → val  (early hole + late hole)
#           나머지 → train
#   → T/V/T/V 패턴을 fold 단위로 재현 ✅
# ── 베이즈 수축 파라미터 ──────────────────────────────────────────────
# 사용자 평균 베이즈 수축 (Bayesian Shrinkage toward user mean)
#
# 수식: pred_final = α × model_pred + (1-α) × user_label_mean
#
# α (신뢰도): 샘플 수에 따른 동적 결정
#   α(n) = n / (n + SHRINK_K)
#   SHRINK_K=10 기준:
#     n=10 → α=0.50  (모델 50%, 사용자 평균 50%)
#     n=20 → α=0.67  (모델 67%, 사용자 평균 33%)
#     n=30 → α=0.75  (모델 75%, 사용자 평균 25%)
#     n=57 → α=0.85  (모델 85%, 사용자 평균 15%)
#
# 직관: 학습 데이터가 적은 사용자일수록 해당 사용자의 과거 평균 비중↑
#       → 개인화 편차(S3: 0.374, S2: 0.336) 보정에 효과적
#
# 누수 방지 ✅
# - OOF: fold 내 train 부분만의 사용자 평균 사용 (val 레이블 미포함)
# - test: 전체 train 사용자 평균 사용 (test 레이블 미사용)
SHRINK_K    = 10.0   # 수축 강도 조율 상수 (클수록 수축 강함)

# ════════════════════════════════════════════════════════════════════
# 5-1. 센서 Rolling 피처 계산
# ════════════════════════════════════════════════════════════════════

def add_sensor_rolling(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    feat_cols: list,
) -> tuple:
    """
    센서 피처에 rolling 통계를 추가하여 train/test를 반환.

    V3 타겟 rolling과의 차이
    ──────────────────────
    V3 : 타겟(y) rolling
         - test의 y값이 없으므로 결측 평균 44~84%
         - OOF 학습 시 val 레이블이 rolling 집계에 포함될 수 있음 (누수 위험)

    V5-1: 센서(X) rolling
         - train/test 모두 X값 존재 → 결측 < 5%
         - X→X 관계이므로 레이블(y) 누수 없음

    누수 방지 설계
    ──────────────
    ① shift(1) 적용
       T일의 rolling = mean(T-1, T-2, ...)
       → T일 자신의 센서값이 T일 rolling에 포함되지 않음 ✅

    ② train+test 결합 후 (subject_id, sleep_date) 기준 정렬
       → 시간 순서 유지, 미래 센서값이 과거 rolling에 포함되지 않음 ✅

    ③ 센서 데이터(X) 기반 → 레이블(y) 정보 미포함 ✅
       (타겟 rolling과 달리 val 레이블 누수 없음)

    ④ min_periods=1 사용
       초기 날짜(이전 기록 부족)에도 가능한 값 생성 → 결측 최소화

    ⑤ ROLL_MISS_THRESHOLD(30%) 초과 rolling 피처 자동 제거
       (hr_night_mean 등 원본부터 결측 많은 피처의 rolling 제거)

    주의: train+test 결합 시 test 센서값이 일부 train rolling에 영향
    ──────
    날짜가 혼재하는 구조에서 test 날짜 T의 센서값이
    train 날짜 T+k의 rolling에 기여할 수 있음.
    그러나 X→X 관계이며 y 정보를 포함하지 않으므로
    실질적 누수(y leakage)가 아닌 통상 허용 범위 내.

    반환
    ────
    train_df_new : rolling 피처 추가된 train DataFrame
    test_df_new  : rolling 피처 추가된 test  DataFrame
    roll_cols    : 생성된 rolling 피처명 리스트
    """
    # ── 연속형 피처만 대상 (이진 피처는 rolling 의미 없음) ──────────
    continuous_cols = [
        c for c in feat_cols
        if len(train_df[c].dropna().unique()) > 2
        or not set(train_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]

    # ── train + test 결합, 날짜 정렬 ─────────────────────────────────
    # ignore_index=True 와 keys 를 함께 쓰면 keys가 무시되므로
    # _src 컬럼을 concat 전에 명시적으로 추가
    tr_part = train_df[["subject_id", "sleep_date"] + continuous_cols].copy()
    te_part = test_df[["subject_id", "sleep_date"] + continuous_cols].copy()
    tr_part["_src"] = "train"
    te_part["_src"] = "test"

    combined = pd.concat([tr_part, te_part], ignore_index=True)

    combined["sleep_date"] = pd.to_datetime(combined["sleep_date"])
    combined = combined.sort_values(["subject_id", "sleep_date"]).reset_index(drop=True)

    # ── rolling 계산 ──────────────────────────────────────────────────
    generated: list = []
    for c in continuous_cols:
        for w in ROLL_WINDOWS:
            col_name = f"{c}_roll{w}"
            combined[col_name] = (
                combined.groupby("subject_id")[c]
                .transform(
                    lambda x, _w=w: x.shift(1).rolling(_w, min_periods=1).mean()
                )
            )
            generated.append(col_name)

    # ── 결측률 필터 (train 기준 ROLL_MISS_THRESHOLD 초과 제거) ────────
    train_mask = combined["_src"] == "train"
    miss_rates  = combined.loc[train_mask, generated].isnull().mean()
    keep_cols   = [c for c in generated if miss_rates[c] <= ROLL_MISS_THRESHOLD]
    drop_cols   = [c for c in generated if miss_rates[c] > ROLL_MISS_THRESHOLD]

    if drop_cols:
        print(f"  rolling 피처 결측>{ROLL_MISS_THRESHOLD:.0%} 제거 "
              f"({len(drop_cols)}개): {drop_cols[:5]}{'...' if len(drop_cols)>5 else ''}")

    # ── train / test 분리 ─────────────────────────────────────────────
    # (subject_id, sleep_date) 기준 merge로 정확히 분리
    merge_key = ["subject_id", "sleep_date"]

    train_roll = (
        combined.loc[train_mask, merge_key + keep_cols]
        .drop_duplicates(merge_key)
    )
    test_roll = (
        combined.loc[~train_mask, merge_key + keep_cols]
        .drop_duplicates(merge_key)
    )

    train_df["sleep_date"] = pd.to_datetime(train_df["sleep_date"])
    test_df["sleep_date"]  = pd.to_datetime(test_df["sleep_date"])

    train_new = train_df.merge(train_roll, on=merge_key, how="left")
    test_new  = test_df.merge(test_roll,  on=merge_key, how="left")

    # 결측 최종 확인
    tr_miss = train_new[keep_cols].isnull().mean().mean() * 100
    te_miss = test_new[keep_cols].isnull().mean().mean() * 100
    print(f"  센서 rolling 피처 {len(keep_cols)}개 생성 완료")
    print(f"  평균 결측률 — train: {tr_miss:.1f}%  test: {te_miss:.1f}%")

    return train_new, test_new, keep_cols



# ════════════════════════════════════════════════════════════════════
# 5-2. Z-score 개인화 피처 계산 (V5와 동일)
# ════════════════════════════════════════════════════════════════════

def compute_user_zscores(
    train_df:  pd.DataFrame,
    test_df:   pd.DataFrame,
    feat_cols: list,
) -> tuple:
    """
    사용자별 Z-score 피처 계산 (raw + rolling 피처 모두 대상).

    누수 방지 ✅
    통계량(mean, std)은 train_df만 사용.
    test에는 train 통계량을 그대로 적용.
    z-score 계산에 타겟 레이블 미사용.
    """
    continuous_cols = [
        c for c in feat_cols
        if len(train_df[c].dropna().unique()) > 2
        or not set(train_df[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})
    ]
    print(f"  Z-score 대상: {len(continuous_cols)}개 "
          f"(raw + rolling 연속형, 이진 {len(feat_cols)-len(continuous_cols)}개 제외)")

    user_stats = (
        train_df.groupby("subject_id")[continuous_cols]
        .agg(["mean", "std"])
    )
    user_stats.columns = [f"{c}__{s}" for c, s in user_stats.columns]

    def _apply(df: pd.DataFrame) -> pd.DataFrame:
        res = {}
        for c in continuous_cols:
            u_mean = df["subject_id"].map(user_stats[f"{c}__mean"])
            u_std  = df["subject_id"].map(user_stats[f"{c}__std"]).fillna(0)
            raw    = df[c].fillna(u_mean)
            z      = (raw - u_mean) / (u_std + 1e-8)
            res[f"{c}_uz"] = z.fillna(0)
        return pd.DataFrame(res, index=df.index)

    train_z = _apply(train_df)
    test_z  = _apply(test_df)
    z_cols  = list(train_z.columns)
    return train_z, test_z, z_cols

# ════════════════════════════════════════════════════════════════════
# 5-3. 피처 선택: 피어슨 상관계수 기반 (CV fold 내부 계산)
# ════════════════════════════════════════════════════════════════════

def _entropy_discrete(x: np.ndarray) -> float:
    """이산 변수 엔트로피 H(X) = -Σ p_i log(p_i)."""
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 1e-10
    _, counts = np.unique(x, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-10)))

def _entropy_continuous(x: np.ndarray, n_bins: int = 20) -> float:
    """연속 변수 엔트로피 H(X): 히스토그램 기반 근사."""
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return 1e-10
    counts, _ = np.histogram(x, bins=n_bins)
    counts = counts[counts > 0]
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-10)))

def _entropy_binary_target(y: np.ndarray) -> float:
    """이진 타겟 엔트로피 H(Y) = -p log(p) - (1-p) log(1-p)."""
    p = np.mean(y)
    if p <= 1e-10 or p >= 1 - 1e-10:
        return 1e-10
    return float(-p * np.log(p) - (1 - p) * np.log(1 - p))

def _nmi_score(mi: float, h_x: float, h_y: float) -> float:
    """정규화 상호정보량 NMI(X;Y) = 2·I(X;Y) / (H(X)+H(Y)) ∈ [0,1]."""
    denom = h_x + h_y
    if denom < 1e-10:
        return 0.0
    return float(np.clip(2 * mi / denom, 0.0, 1.0))

def select_features_by_mutual_info(
    X_all:          pd.DataFrame,
    y_dict:         dict,
    all_cols:       list,
    cat_feat_cols:  list,
    train_df_ref:   pd.DataFrame,
    top_k:          int   = CORR_TOP_K,
    n_neighbors:    int   = 3,
    random_state:   int   = 42,
) -> dict:
    """정규화 상호정보량(NMI) 기반 타겟별 상위 K개 피처 선택.

    NMI(X;Y) = 2·I(X;Y) / (H(X)+H(Y))
      - I(X;Y): sklearn mutual_info_classif (k-NN 추정)
      - H(X):  이산→ 정확한 엔트로피 / 연속→ 히스토그램 근사
      - H(Y):  이진 타겟 엔트로피
    → 0~1 정규화로 스케일 무관 피처 간 공정한 비교

    Pearson 대비 장점:
      - 이진 타겟(Q1~S4)에 적합 (비선형·비단조 관계 포착)
      - 범주형·연속형 혼합 피처 정확 처리 (discrete_features 지정)
      - NMI 정규화로 고분산 피처 과대 선택 방지

    Parameters
    ----------
    cat_feat_cols : 범주형 피처 이름 목록 → discrete_features=True 처리
    """

    import logging
    # 로거 설정
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

    def log_infinite_features(df: pd.DataFrame, context_name: str = "FeatureMatrix"):
        """
        데이터프레임 내의 무한대(inf, -inf) 또는 너무 큰 값을 찾아 로깅합니다.
        """
        # 1. 무한대(inf, -inf) 확인
        inf_mask = np.isinf(df)
        inf_counts = inf_mask.sum()
        inf_cols = inf_counts[inf_counts > 0]

        if len(inf_cols) > 0:
            logging.error(f"[{context_name}] 🚨 무한대(inf) 값이 발견된 컬럼 ({len(inf_cols)}개):")
            for col, count in inf_cols.items():
                logging.error(f"  - {col}: {count} rows")

        # 2. 극단적으로 큰 값 확인 (float64 max 근처)
        # sklearn은 보통 1e308 이상을 문제 삼지만, 안전하게 1e15 이상을 체크해봅니다.
        huge_mask = (df > 1e15) & ~inf_mask
        huge_counts = huge_mask.sum()
        huge_cols = huge_counts[huge_counts > 0]

        if len(huge_cols) > 0:
            logging.warning(f"[{context_name}] ⚠️ 극단적으로 큰 값(>1e15)이 발견된 컬럼 ({len(huge_cols)}개):")
            for col, count in huge_cols.items():
                logging.warning(f"  - {col}: {count} rows")

        return list(inf_cols.keys())

        # 사용 예시 (오류가 난 함수 내부에 추가)
        # mi = mutual_info_classif(...) 호출 직전에 아래 코드를 넣으세요.
        # log_infinite_features(X_train_with_z, "X_train_with_z (Before NMI)")
    from sklearn.feature_selection import mutual_info_classif

    cat_feat_set   = set(cat_feat_cols or [])
    discrete_mask  = np.array([c in cat_feat_set for c in all_cols], dtype=bool)
    sh_folds       = make_subject_hole_folds(train_df_ref)
    feat_dict      = {}

    print(f"  [NMI 선택] 후보 {len(all_cols)}개 피처 "
          f"(이산 {discrete_mask.sum()}개 / 연속 {(~discrete_mask).sum()}개)")

    for target, y_all in y_dict.items():
        nmi_accum = np.zeros(len(all_cols))
        n_valid   = 0

        for tr_idx, _ in sh_folds:
            X_tr = X_all[all_cols].iloc[tr_idx].fillna(0).values
            y_tr = y_all[tr_idx]

            if len(np.unique(y_tr)) < 2:
                continue  # 클래스 다양성 없으면 스킵

            # ── I(X;Y): mutual_info_classif (fold train만 사용, 누수 없음)
            mi = mutual_info_classif(
                X_tr, y_tr,
                discrete_features=discrete_mask,
                n_neighbors=n_neighbors,
                random_state=random_state,
            )
            mi = np.clip(mi, 0, None)  # 음수 추정값 클립

            # ── H(Y): 이진 타겟 엔트로피
            h_y = _entropy_binary_target(y_tr)

            # ── NMI 계산 (피처별)
            nmi = np.zeros(len(all_cols))
            for i in range(len(all_cols)):
                if mi[i] < 1e-12:
                    nmi[i] = 0.0
                    continue
                h_x = (_entropy_discrete(X_tr[:, i])
                       if discrete_mask[i]
                       else _entropy_continuous(X_tr[:, i]))
                nmi[i] = _nmi_score(mi[i], h_x, h_y)

            nmi_accum += nmi
            n_valid   += 1

        if n_valid == 0:
            feat_dict[target] = all_cols[:top_k]
            continue

        nmi_mean = nmi_accum / n_valid  # fold 평균 NMI

        top_idx          = np.argsort(nmi_mean)[::-1][:top_k]
        feat_dict[target] = [all_cols[i] for i in top_idx]

        top3 = [(all_cols[i], nmi_mean[i]) for i in top_idx[:3]]
        top3_str = ", ".join(f"{n}({v:.3f})" for n, v in top3)
        print(f"    {target}: top3 NMI = {top3_str}  (선택 {top_k}개)")

    return feat_dict

def select_features_by_correlation(
    X_all:        pd.DataFrame,
    y_dict:       dict,
    all_cols:     list,
    train_df_ref: pd.DataFrame,      # Subject-Hole fold 생성에 사용
    top_k:        int = CORR_TOP_K,
) -> dict:
    """
    피어슨 상관계수 절댓값 기준 타겟별 상위 피처 선택.

    알고리즘
    ────────
    1. StratifiedKFold(5)로 데이터 분할
    2. 각 fold의 train 부분에서 |Pearson r|(피처↔타겟) 계산
       → train split만 사용: val 레이블 누수 방지 ✅
    3. 5개 fold 걸쳐 |r| 누적 후 fold 수로 나눔 (평균 |r|)
    4. 내림차순 정렬 후 상위 CORR_TOP_K개 선택
    5. 7개 타겟별 독립 수행 → 타겟마다 가장 관련 높은 피처 집합

    왜 LGB 중요도 대신 피어슨 상관계수인가?
    ─────────────────────────────────────────
    LGB 중요도: 모델 학습 필요 → 느림, 안정성↓ (트리 구조에 의존)
    피어슨 |r|: 계산 빠름, 해석 직관적, 선형 신호 포착에 강함
    CV 평균    : 특정 데이터 분포에 과적합되지 않고 안정적 선택

    누수 방지 ✅
    ─────────
    각 fold에서 train 부분(tr_idx)의 피처-레이블 관계만 사용.
    val 레이블은 절대 상관 계산에 포함되지 않음.
    vectorized 구현으로 피처별 루프 없이 빠르게 계산.

    반환
    ────
    {target: [selected_col, ...]}  타겟별 상위 CORR_TOP_K개 피처명
    """
    # Subject-Hole fold 생성
    # 피어슨 |r| 계산도 SH-fold train 부분만 사용 → 누수 없음 ✅
    sh_folds   = make_subject_hole_folds(train_df_ref)
    n_folds_actual = len(sh_folds)
    n_features = len(all_cols)
    selected   = {}

    print(f"\n  [피처 선택] 피어슨 상관계수 기반 선택 (Subject-Hole CV)")
    print(f"  후보 피처: {n_features}개 (raw+roll+z-score)  →  타겟별 상위 {top_k}개 선택")
    print(f"  {'타겟':4s}  {'fold 평균 |r| top3':>35s}  {'선택 피처 수':>12s}")
    print(f"  {'-'*56}")

    for target in TARGETS:
        y            = y_dict[target]
        corr_accum   = np.zeros(n_features)

        for tr_idx, _ in sh_folds:
            X_tr = X_all.iloc[tr_idx].fillna(0).values   # (n_tr, n_feat)
            y_tr = y[tr_idx].astype(float)               # (n_tr,)

            # vectorized Pearson |r|: 한 번에 모든 피처 계산
            X_c   = X_tr - X_tr.mean(axis=0)             # 피처별 centering
            y_c   = y_tr - y_tr.mean()                   # 레이블 centering
            numer = (X_c * y_c[:, np.newaxis]).sum(axis=0)
            denom = (np.sqrt((X_c**2).sum(axis=0)) *
                     np.sqrt((y_c**2).sum()) + 1e-8)
            corr_accum += np.abs(numer / denom)

        avg_corr  = corr_accum / n_folds_actual
        sorted_idx = np.argsort(avg_corr)[::-1]
        top_idx    = sorted_idx[:top_k]
        top_cols   = [all_cols[i] for i in top_idx]

        # 상위 3개 출력 (모니터링용)
        top3_str = ", ".join(
            f"{all_cols[i]}({avg_corr[i]:.3f})" for i in sorted_idx[:3]
        )
        print(f"  {target:4s}  {top3_str:>35s}  {len(top_cols):>12d}")

        selected[target] = top_cols

    return selected

# ════════════════════════════════════════════════════════════════════
# 5-4. Phase 2: LGB + XGB + CatBoost 가중 앙상블 (V5와 동일)
# ════════════════════════════════════════════════════════════════════

def make_subject_hole_folds(
    train_df: pd.DataFrame,
    n_folds:  int = N_FOLDS,
) -> list:
    """
    Subject 내 시간적 홀(hole)을 val로 사용하는 CV 전략.

    실제 train/test 분리 구조를 재현
    ────────────────────────────────
    실제 패턴 (id01):
      TTTTTTTTTTTTTTTTTTTTTTTTTTTTVVVVVVVVVVVVVVTTTTTTTTTTTTTVVVVVVVVVVVVV
      → train/test가 사용자 내부에서 2~3개 연속 블록으로 교차

    이를 fold로 재현:
      사용자 날짜를 n_folds*2 개 chunk로 분할
      fold i → chunk[i] (early hole) + chunk[i+n_folds] (late hole) = val
             → 나머지 = train
      → T/V/T/V 인터리빙 구조 모사 ✅

    KFold 대비 특성
    ───────────────
    KFold    : 날짜 무관 랜덤 분할 → val에 train과 인접한 날짜 혼재
    SH-Fold  : 연속 블록 분할 → val이 train 사이의 '빈 구간'
    → 실제 test가 train 기간 내 빠진 날짜들인 구조와 더 유사

    누수 방지 ✅
    ───────────
    KFold과 동일하게 val 레이블을 학습/피처 선택에 사용하지 않음.

    반환
    ────
    [(train_idx, val_idx), ...] 리스트  (len = n_folds)
    """
    train_sorted = train_df.sort_values(["subject_id", "sleep_date"])
    all_indices  = train_sorted.index.to_numpy()
    block_count  = max(n_folds * 2, 4)   # fold당 early+late 2개 hole
    result       = []

    # 사용자별 chunk 분할
    by_subject = {}
    for sid, grp in train_sorted.groupby("subject_id", sort=False):
        chunks = [c for c in np.array_split(grp.index.to_numpy(), block_count)
                  if len(c) > 0]
        by_subject[str(sid)] = chunks

    for fold_id in range(n_folds):
        val_parts = []
        for chunks in by_subject.values():
            # early hole: fold_id번째 chunk
            # late  hole: fold_id + n_folds번째 chunk
            for hole_id in (fold_id, fold_id + n_folds):
                if hole_id < len(chunks):
                    val_parts.append(chunks[hole_id])

        if not val_parts:
            continue

        val_idx   = np.concatenate(val_parts)
        train_idx = np.setdiff1d(all_indices, val_idx, assume_unique=False)

        if len(train_idx) > 0 and len(val_idx) > 0:
            result.append((train_idx, val_idx))

    return result

def _user_alpha(
    subject_ids: np.ndarray,
    tr_idx:      np.ndarray,
) -> np.ndarray:
    """
    fold train 부분의 사용자별 샘플 수 기반 동적 alpha 계산.

    alpha(n) = n / (n + SHRINK_K)
    → 샘플 수 n이 클수록 alpha → 1 (모델 예측 전적으로 신뢰)
    → 샘플 수 n이 작을수록 alpha → 0 (사용자 평균으로 강하게 수축)

    반환: len(tr_idx) 또는 전체 indices에 대한 alpha 벡터
    """
    train_sids = subject_ids[tr_idx]
    # fold train 부분에서 사용자별 샘플 수
    unique, counts = np.unique(train_sids, return_counts=True)
    n_map = dict(zip(unique, counts))
    alpha_map = {sid: n / (n + SHRINK_K) for sid, n in n_map.items()}
    return alpha_map


def train_models_v52_3(
    train_df:      pd.DataFrame,
    test_df:       pd.DataFrame,
    X_train_all:   pd.DataFrame,
    X_test_all:    pd.DataFrame,
    feat_dict:     dict,
    cat_feat_cols: list = None,
    lgb_params:    dict = None,
    xgb_params:    dict = None,
    cat_params:    dict = None,
) -> tuple:
    lgb_params = lgb_params or LGBM_PARAMS
    xgb_params = xgb_params or XGB_PARAMS
    cat_params = cat_params or CAT_PARAMS

    # 🌟 시드 앙상블용 다중 시드 설정
    SEEDS = [42, 2024, 9999]
    MODEL_NAMES = ["lgb", "xgb", "cat"]
    cat_feat_set = set(cat_feat_cols or [])

    sh_folds = make_subject_hole_folds(train_df)
    n_folds_actual = len(sh_folds)
    print(f"  Subject-Hole CV: {n_folds_actual}개 fold 생성")

    subj_enc = {s: i for i, s in enumerate(sorted(train_df["subject_id"].unique()))}
    X_train_all = X_train_all.copy()
    X_test_all  = X_test_all.copy()
    X_train_all["subject_enc"] = train_df["subject_id"].map(subj_enc).fillna(-1).astype(int).values
    X_test_all["subject_enc"]  = test_df["subject_id"].map(subj_enc).fillna(-1).astype(int).values

    train_sids = train_df["subject_id"].values
    test_sids  = test_df["subject_id"].values

    oof_preds  = {t: np.zeros(len(train_df)) for t in TARGETS}
    test_preds = {t: np.zeros(len(test_df))  for t in TARGETS}
    model_info = {}
    scores     = {}

    print(f"\n  [모델 학습] 시드 앙상블(3 Seeds) + 역수 비례 가중치 + 베이즈 수축")
    print(f"  적용 시드: {SEEDS}")

    for target in TARGETS:
        y = train_df[target].values

        user_mean_all = train_df.groupby("subject_id")[target].mean().to_dict()
        user_n_all = train_df.groupby("subject_id").size().to_dict()
        test_alpha = np.array([
            user_n_all.get(s, 0) / (user_n_all.get(s, 0) + SHRINK_K) for s in test_sids
        ])
        test_prior = np.array([
            user_mean_all.get(s, y.mean()) for s in test_sids
        ])

        sel_cols = feat_dict[target] + ["subject_enc"]
        X_tr_df_raw = X_train_all[sel_cols].copy()
        X_te_df_raw = X_test_all[sel_cols].copy()

        if cat_feat_set:
            X_tr_df_raw, X_te_df_raw = _encode_cat_for_models(
                X_tr_df_raw, X_te_df_raw, cat_feat_set)

        X_tr_np = X_tr_df_raw.fillna(0).values
        X_te_np = X_te_df_raw.fillna(0).values

        cat_indices = [i for i, c in enumerate(sel_cols) if c in cat_feat_set or c == "subject_enc"]
        for i in cat_indices:
            col_name = sel_cols[i]
            X_tr_df_raw[col_name] = X_tr_df_raw[col_name].astype(int)
            X_te_df_raw[col_name] = X_te_df_raw[col_name].astype(int)

        fold_oof = {m: np.zeros(len(train_df)) for m in MODEL_NAMES}
        fold_te  = {m: [] for m in MODEL_NAMES}

        print(f"\n  ── {target}  ({len(sel_cols)}개 피처)")

        for fold, (tr_idx, val_idx) in enumerate(sh_folds):
            X_tr = X_tr_np[tr_idx]; X_vl = X_tr_np[val_idx]
            y_tr = y[tr_idx];       y_vl = y[val_idx]

            X_tr_df_cat = X_tr_df_raw.iloc[tr_idx].fillna(0)
            X_vl_df_cat = X_tr_df_raw.iloc[val_idx].fillna(0)

            # --- 시드 앙상블 변수 초기화 ---
            lgb_oof_seeds, xgb_oof_seeds, cat_oof_seeds = [], [], []
            lgb_test_seeds, xgb_test_seeds, cat_test_seeds = [], [], []

            # 각 시드별로 모델 학습
            for seed in SEEDS:
                # 1. LightGBM
                lgb_params['random_state'] = seed
                m_lgb = lgb.LGBMClassifier(**lgb_params)
                m_lgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                          categorical_feature=cat_indices if cat_indices else "auto",
                          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
                lgb_oof_seeds.append(m_lgb.predict_proba(X_vl)[:, 1])
                lgb_test_seeds.append(m_lgb.predict_proba(X_te_np)[:, 1])

                # 2. XGBoost
                xgb_params['random_state'] = seed
                m_xgb = xgb.XGBClassifier(**xgb_params, enable_categorical=False)
                m_xgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
                xgb_oof_seeds.append(m_xgb.predict_proba(X_vl)[:, 1])
                xgb_test_seeds.append(m_xgb.predict_proba(X_te_np)[:, 1])

                # 3. CatBoost (CatBoost는 random_seed 파라미터 사용)
                cat_params['random_seed'] = seed
                m_cat = CatBoostClassifier(**cat_params)
                cat_feature_names = [sel_cols[i] for i in cat_indices]
                m_cat.fit(X_tr_df_cat, y_tr, eval_set=(X_vl_df_cat, y_vl),
                          cat_features=cat_feature_names if cat_feature_names else None,
                          early_stopping_rounds=50, verbose=False)
                cat_oof_seeds.append(m_cat.predict_proba(X_vl_df_cat)[:, 1])
                cat_test_seeds.append(m_cat.predict_proba(X_te_df_raw.fillna(0))[:, 1])

            # 시드 평균(Mean)을 Fold OOF 및 Test에 기록
            fold_oof["lgb"][val_idx] = np.mean(lgb_oof_seeds, axis=0)
            fold_te["lgb"].append(np.mean(lgb_test_seeds, axis=0))

            fold_oof["xgb"][val_idx] = np.mean(xgb_oof_seeds, axis=0)
            fold_te["xgb"].append(np.mean(xgb_test_seeds, axis=0))

            fold_oof["cat"][val_idx] = np.mean(cat_oof_seeds, axis=0)
            fold_te["cat"].append(np.mean(cat_test_seeds, axis=0))

            fold_lls = {m: log_loss(y_vl, fold_oof[m][val_idx]) for m in MODEL_NAMES}
            print(f"     fold{fold+1} (3-Seed)  LGB:{fold_lls['lgb']:.4f}  XGB:{fold_lls['xgb']:.4f}  CAT:{fold_lls['cat']:.4f}")

        # =========================================================
        # 🚀 ── 역수 비례 가중치 복구 (안정적 앙상블) ──
        # =========================================================
        oof_lls  = {m: log_loss(y, fold_oof[m]) for m in MODEL_NAMES}
        inv_sum  = sum(1.0 / oof_lls[m] for m in MODEL_NAMES)
        weights  = {m: (1.0 / oof_lls[m]) / inv_sum for m in MODEL_NAMES}

        oof_ens_raw = sum(weights[m] * fold_oof[m] for m in MODEL_NAMES)
        te_ens_raw  = sum(weights[m] * np.mean(fold_te[m], axis=0) for m in MODEL_NAMES)

        # ── 베이즈 수축 적용 ──
        oof_alpha = np.array([
            user_n_all.get(s, 0) / (user_n_all.get(s, 0) + SHRINK_K) for s in train_sids
        ])
        oof_prior = np.array([
            user_mean_all.get(s, y.mean()) for s in train_sids
        ])

        oof_shrunk = np.clip(
            oof_alpha * oof_ens_raw + (1.0 - oof_alpha) * oof_prior,
            1e-6, 1 - 1e-6
        )
        te_shrunk  = np.clip(
            test_alpha * te_ens_raw  + (1.0 - test_alpha) * test_prior,
            1e-6, 1 - 1e-6
        )

        oof_preds[target]  = oof_shrunk
        test_preds[target] = te_shrunk

        ll_raw    = log_loss(y, oof_ens_raw)
        ll_shrunk = log_loss(y, oof_shrunk)
        oof_auc   = roc_auc_score(y, oof_shrunk)

        scores[target] = {"auc": oof_auc, "logloss": ll_shrunk,
                          "logloss_raw": ll_raw, "weights": weights, "oof_lls": oof_lls}
        model_info[target] = {"weights": weights, "oof_lls": oof_lls}

        print(f"     ➡️ 최종 앙상블 OOF(수축후): LL={ll_shrunk:.4f}  (AUC={oof_auc:.4f})")
        print(f"        가중치: lgb={weights['lgb']:.3f}  xgb={weights['xgb']:.3f}  cat={weights['cat']:.3f}")

    return oof_preds, test_preds, model_info, scores


## 6. 가중치 요약 출력


# ════════════════════════════════════════════════════════════════════
# 6-v5-1. 가중치 요약 출력
# ════════════════════════════════════════════════════════════════════

def print_weight_summary(scores: dict) -> None:
    """타겟별 모델 가중치 및 OOF Log-Loss 출력"""
    print("\n모델별 OOF LogLoss & 가중치 요약:")
    print(f"  {'타겟':4s}  {'LGB_LL':>8s}  {'XGB_LL':>8s}  {'CAT_LL':>8s}  "
          f"{'w_lgb':>7s}  {'w_xgb':>7s}  {'w_cat':>7s}  {'최고'}")
    print("  " + "-"*72)
    for t in TARGETS:
        ol   = scores[t]["oof_lls"]
        w    = scores[t]["weights"]
        best = min(ol, key=ol.get).upper()
        print(f"  {t:4s}  {ol['lgb']:>8.4f}  {ol['xgb']:>8.4f}  {ol['cat']:>8.4f}  "
              f"{w['lgb']:>7.3f}  {w['xgb']:>7.3f}  {w['cat']:>7.3f}  {best}")


### 7. 메인 실행



# ==============================================================================
# 7. 파이프라인 단계별 분리 (Phase 1 ~ 4)
# ==============================================================================
import json

def run_phase_1_base_features():
    """Phase 1: 기본 피처 및 교차 피처 생성 (속도: 빠름)"""
    print("\n" + "="*50)
    print("🚀 [Phase 1] 기본 피처 및 교차 피처 생성 시작")
    print("="*50)

    train_raw  = pd.read_csv(TRAIN_CSV)
    sample_raw = pd.read_csv(SAMPLE_CSV)

    for col in ["sleep_date", "lifelog_date"]:
        train_raw[col]  = pd.to_datetime(train_raw[col]).dt.tz_localize(None)
        sample_raw[col] = pd.to_datetime(sample_raw[col]).dt.tz_localize(None)

    train_dates = train_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    test_dates  = sample_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()
    all_dates   = pd.concat([train_dates, test_dates], ignore_index=True).drop_duplicates()

    feat_df = build_features(all_dates)
    feat_df = build_cross_features(feat_df)
    feat_df = add_time_features(feat_df)

    train_df = train_dates.merge(feat_df, on=["subject_id", "sleep_date", "lifelog_date"], how="left")
    test_df = test_dates.merge(feat_df, on=["subject_id", "sleep_date", "lifelog_date"], how="left")
    train_df = train_df.merge(train_raw[["subject_id", "sleep_date"] + TARGETS], on=["subject_id", "sleep_date"], how="left")

    for df_ in [train_df, test_df]:
        raw_dur = df_["ac_morning_unplug_hour"] - df_["ac_bedtime_hour"]
        df_["est_sleep_duration"] = raw_dur.where(raw_dur >= 0, raw_dur + 24)
        raw_scr = df_["ac_morning_unplug_hour"] - df_["screen_last_off_hour"]
        df_["est_screen_to_wake"] = raw_scr.where(raw_scr >= 0, raw_scr + 24)

    # 저장
    os.makedirs(os.path.join(BASE_DIR, "checkpoints"), exist_ok=True)
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_test.parquet"), index=False)
    print("✅ [Phase 1] 완료 및 저장됨 (phase1_train.parquet)")


def run_phase_2_rolling_and_tsfresh():
    """Phase 2: Rolling 및 TSFresh 피처 추출 (속도: 매우 느림 - 약 1시간)"""
    print("\n" + "="*50)
    print("🚀 [Phase 2] Rolling 및 TSFresh 시계열 피처 추출 시작")
    print("="*50)

    train_df = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_train.parquet"))
    test_df  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase1_test.parquet"))

    exclude_cols = ["subject_id", "sleep_date", "lifelog_date"] + TARGETS
    miss_rate    = train_df.isnull().mean()

    feat_cols_base = [
        c for c in train_df.columns
        if c not in exclude_cols
        and train_df[c].dtype in [np.float64, np.int64, np.int32, float, int]
        and miss_rate[c] <= FEAT_MISS_THRESHOLD
        and c not in FEAT_EXCLUDE_HIGH_MISS
        and c not in FEAT_EXCLUDE_HARMFUL
        and c not in FEAT_EXCLUDE_CALENDAR
    ]

    # 센서 Rolling 피처 추가
    train_df, test_df, roll_cols = add_sensor_rolling(train_df, test_df, feat_cols_base)
    feat_cols = feat_cols_base + roll_cols

    # tsfresh 피처 추가 (Micro/Macro)
    train_df, test_df, ts_feat_cols = add_tsfresh_features(train_df, test_df)
    ts_cols_valid = [c for c in ts_feat_cols if c in train_df.columns]
    feat_cols = feat_cols + ts_cols_valid

    # Light tsfresh
    all_light_ts_cols = []
    if HAS_TSFRESH:
        df_m_ts, df_w_ts = compute_light_tsfresh_full()
        train_df, test_df, all_light_ts_cols = add_light_tsfresh_to_df(train_df, test_df, df_m_ts, df_w_ts)
        light_cols_valid = [c for c in all_light_ts_cols if c in train_df.columns]
        feat_cols = feat_cols + light_cols_valid

    # 리스트 저장
    with open(os.path.join(BASE_DIR, "checkpoints", "feat_cols_list.json"), "w") as f:
        json.dump({"feat_cols": feat_cols, "all_light_ts_cols": all_light_ts_cols}, f)

    # 데이터 저장
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_train.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_test.parquet"), index=False)
    print("✅ [Phase 2] 완료 및 저장됨 (phase2_train.parquet)")


def run_phase_3_zscore_and_selection():
    """Phase 3: Z-score 계산, 결측치/inf 방어, NMI 피처 선택 (속도: 보통)"""
    print("\n" + "="*50)
    print("🚀 [Phase 3] Z-score 정규화 및 NMI 피처 선택 시작")
    print("="*50)

    train_df = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_train.parquet"))
    test_df  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase2_test.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_cols_list.json"), "r") as f:
        lists = json.load(f)
        feat_cols = lists["feat_cols"]
        all_light_ts_cols = lists["all_light_ts_cols"]

    exclude_cols = ["subject_id", "sleep_date", "lifelog_date"] + TARGETS

    # Z-score 계산
    train_z_df, test_z_df, z_cols_all = compute_user_zscores(train_df, test_df, feat_cols)

    # 범주형 피처 전처리 (Unknown 채우기)
    str_cat_in_df = [c for c in train_df.columns if c not in feat_cols and c not in exclude_cols
                     and train_df[c].dtype == object and any(c.endswith(s) for s in STR_CAT_COL_SUFFIXES)]
    for c in str_cat_in_df:
        train_df[c] = train_df[c].fillna("Unknown")
        test_df[c]  = test_df[c].fillna("Unknown")

    feat_cols = feat_cols + str_cat_in_df
    all_cat_feat_cols = list(set([c for c in feat_cols if c in BINARY_CAT_FEATURES] + str_cat_in_df))

    # 🚨 인덱스 초기화 및 inf/NaN 완벽 방어
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    X_train_with_z = pd.concat([train_df[feat_cols], train_z_df], axis=1).reset_index(drop=True)
    X_test_with_z  = pd.concat([test_df[feat_cols],  test_z_df],  axis=1).reset_index(drop=True)

    print("  [방어 코드] 무한대(inf) 및 결측치 제거 중...")
    X_train_with_z.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test_with_z.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train_with_z.fillna(0, inplace=True)
    X_test_with_z.fillna(0, inplace=True)

    y_dict = {t: train_df[t].values for t in TARGETS}

    # 문자열 카테고리 사전 인코딩 (Pearson/NMI 용)
    if str_cat_in_df:
        for c in str_cat_in_df:
            if c not in X_train_with_z.columns: continue
            combined = pd.concat([X_train_with_z[c], X_test_with_z[c]], ignore_index=True).fillna("Unknown").astype(str)
            cat_type = pd.CategoricalDtype(categories=combined.unique())
            n_tr = len(X_train_with_z)
            codes = combined.astype(cat_type).cat.codes.values
            X_train_with_z[c] = pd.Series(codes[:n_tr].astype(int), index=X_train_with_z.index)
            X_test_with_z[c]  = pd.Series(codes[n_tr:].astype(int), index=X_test_with_z.index)

    # 피처 선택
    _excl_from_pearson = set(all_light_ts_cols)
    all_candidate_cols = [c for c in feat_cols + z_cols_all if c not in _excl_from_pearson and c in X_train_with_z.columns]

    feat_dict = select_features_by_mutual_info(
        X_all=X_train_with_z, y_dict=y_dict, all_cols=all_candidate_cols,
        cat_feat_cols=all_cat_feat_cols, train_df_ref=train_df
    )

    # 데이터 최종 저장 (Phase 4에서 사용)
    X_train_with_z.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"), index=False)
    X_test_with_z.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"), index=False)
    train_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"), index=False)
    test_df.to_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"), index=False)

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "w") as f:
        json.dump({"feat_dict": feat_dict, "all_cat_feat_cols": all_cat_feat_cols}, f)

    print("✅ [Phase 3] 완료 및 저장됨 (phase3_X_train.parquet)")


def run_phase_4_modeling():
    """Phase 4: 모델 학습 및 앙상블 (시드 앙상블 + 하드 클리핑 적용)"""
    print("\n" + "="*50)
    print("🚀 [Phase 4] 앙상블 모델 학습 및 제출 파일 생성 시작")
    print("="*50)

    X_train_with_z = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"))
    X_test_with_z  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"))
    train_df       = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"))
    test_df        = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "r") as f:
        lists = json.load(f)
        feat_dict = lists["feat_dict"]
        all_cat_feat_cols = lists["all_cat_feat_cols"]

    # 🚨 CatBoost 에러 방지: 데이터프레임 내 범주형 변수를 명시적 int로 변환
    for c in all_cat_feat_cols:
        if c in X_train_with_z.columns:
            X_train_with_z[c] = X_train_with_z[c].astype(int)
            X_test_with_z[c]  = X_test_with_z[c].astype(int)

    # 모델 학습 (시드 앙상블 + 역수 비례 가중치)
    oof_preds, test_preds, model_info, scores = train_models_v52_3(
        train_df=train_df,
        test_df=test_df,
        X_train_all=X_train_with_z,
        X_test_all=X_test_with_z,
        feat_dict=feat_dict,
        cat_feat_cols=all_cat_feat_cols,
    )

    print_weight_summary(scores)

    # 제출 파일 생성
    sample_raw = pd.read_csv(SAMPLE_CSV)
    submission = sample_raw[["subject_id", "sleep_date", "lifelog_date"]].copy()

    # 🌟 극단값 하드 클리핑 (Hard Clipping) 적용
    # 0.025 미만은 0.025로, 0.975 초과는 0.975로 잘라내어 극단적 오답의 페널티 최소화
    CLIP_MIN = 0.025
    CLIP_MAX = 0.975

    for t in TARGETS:
        submission[t] = np.clip(test_preds[t], CLIP_MIN, CLIP_MAX)

    out_path = os.path.join(BASE_DIR, "submission_v53_mutual_seed_clipped.csv")
    submission.to_csv(out_path, index=False)
    print(f"\n🎉 최종 제출 파일 저장 완료 (시드 앙상블 & 클리핑 적용): {out_path}")

### Optuna

In [ ]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.0 MB/s eta 0:00:00


In [ ]:
def run_optuna_tuning():
    print("=== Optuna 하이퍼파라미터 튜닝 시작 ===")

    # 1. Phase 3에서 저장해둔 데이터 1초 만에 불러오기
    X_train_with_z = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_train.parquet"))
    X_test_with_z  = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_X_test.parquet"))
    train_df       = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_train_meta.parquet"))
    test_df        = pd.read_parquet(os.path.join(BASE_DIR, "checkpoints", "phase3_test_meta.parquet"))

    with open(os.path.join(BASE_DIR, "checkpoints", "feat_dict.json"), "r") as f:
        lists = json.load(f)
        feat_dict = lists["feat_dict"]
        all_cat_feat_cols = lists["all_cat_feat_cols"]

    for c in all_cat_feat_cols:
        if c in X_train_with_z.columns:
            X_train_with_z[c] = X_train_with_z[c].astype(int)
            X_test_with_z[c]  = X_test_with_z[c].astype(int)

    # 2. Optuna Objective 함수 정의
    def objective(trial):
        # 💡 모델이 똑똑하게 탐색할 파라미터 범위(Search Space) 설정
        lgb_params = {
            "objective": "binary",
            "metric": "binary_logloss",
            "n_estimators": 500, # 튜닝 속도를 위해 1000 -> 500으로 약간 줄임
            "learning_rate": trial.suggest_float("lgb_lr", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("lgb_num_leaves", 15, 63),
            "min_child_samples": trial.suggest_int("lgb_min_child", 5, 30),
            "feature_fraction": trial.suggest_float("lgb_feat_frac", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("lgb_bag_frac", 0.6, 1.0),
            "bagging_freq": 5,
            "random_state": 42,
            "verbose": -1,
        }

        xgb_params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "n_estimators": 500,
            "learning_rate": trial.suggest_float("xgb_lr", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("xgb_depth", 3, 8),
            "subsample": trial.suggest_float("xgb_subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("xgb_colsample", 0.6, 1.0),
            "random_state": 42,
            "verbosity": 0,
        }

        cat_params = {
            "iterations": 500,
            "learning_rate": trial.suggest_float("cat_lr", 0.01, 0.1, log=True),
            "depth": trial.suggest_int("cat_depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("cat_l2", 1.0, 10.0),
            "loss_function": "Logloss",
            "eval_metric": "Logloss",
            "random_seed": 42,
            "verbose": False,
        }

        # 튜닝된 파라미터를 넣고 학습!
        oof_preds, _, _, _ = train_models_v52_2(
            train_df=train_df, test_df=test_df,
            X_train_all=X_train_with_z, X_test_all=X_test_with_z,
            feat_dict=feat_dict, cat_feat_cols=all_cat_feat_cols,
            lgb_params=lgb_params, xgb_params=xgb_params, cat_params=cat_params
        )

        # 7개 타겟의 평균 Log-Loss 계산 (대회 평가지표)
        global_lls = []
        for t in TARGETS:
            y = train_df[t].values
            ll = log_loss(y, oof_preds[t])
            global_lls.append(ll)

        avg_logloss = np.mean(global_lls)
        return avg_logloss # 이 값을 최소화(minimize) 하는 것이 목표!

    # 3. 최적화 실행 (n_trials: 몇 번 시도할 것인지)
    study = optuna.create_study(direction="minimize")
    print("탐색을 시작합니다. 커피 한 잔 하고 오세요! ☕")
    study.optimize(objective, n_trials=20) # 20번 탐색 (시간에 맞춰 조절하세요)

    # 4. 결과 출력
    print("🏆 최적의 파라미터를 찾았습니다!")
    print("Best Log-Loss:", study.best_value)
    print("Best Parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

# 실행 시
# run_optuna_tuning()

# MAIN

In [ ]:
run_optuna_tuning()

=== Optuna 하이퍼파라미터 튜닝 시작 ===


[I 2026-06-23 12:51:18,026] A new study created in memory with name: no-name-23334204-4fa6-43dc-bca7-cc3dab806a02


탐색을 시작합니다. 커피 한 잔 하고 오세요! ☕
  Subject-Hole CV: 5개 fold 생성

  [모델 학습] LGB + XGB + CatBoost 앙상블 + 베이즈 수축 (Subject-Hole CV)
  타겟별 피처: 피어슨 상관 top-50 + subject_enc
  수축 강도: SHRINK_K=10.0  (n=10→α=0.50, n=30→α=0.75, n=57→α=0.85)

  ── Q1  (51개 피처)
      fold    LGB LL    XGB LL    CAT LL
     -----------------------------------
     fold1    0.5754    0.6271    0.6024
     fold2    0.6601    0.6656    0.6566
     fold3    0.6452    0.6963    0.6512
     fold4    0.6937    0.6654    0.6341
     fold5    0.6866    0.6988    0.6604

     앙상블 OOF(수축전): LL=0.6354
     앙상블 OOF(수축후): LL=0.6323  수축 개선=+0.0031 ✅
     baseline=0.6931  gain=+0.0608 ✅
     최적 가중치: lgb=0.375  xgb=0.000  cat=0.625

  ── Q2  (51개 피처)
      fold    LGB LL    XGB LL    CAT LL
     -----------------------------------
     fold1    0.6847    0.7198    0.6871
     fold2    0.6212    0.6499    0.6617
     fold3    0.6502    0.7172    0.6738
     fold4    0.5985    0.6478    0.6398
     fold5    0.6719    0.7399    0.6919

     앙

[I 2026-06-23 12:52:01,393] Trial 0 finished with value: 0.5945721843250239 and parameters: {'lgb_lr': 0.08887410063771227, 'lgb_num_leaves': 53, 'lgb_min_child': 5, 'lgb_feat_frac': 0.9573441647635965, 'lgb_bag_frac': 0.6893520774733354, 'xgb_lr': 0.013437805571762367, 'xgb_depth': 3, 'xgb_subsample': 0.8279963147329097, 'xgb_colsample': 0.6193665880434608, 'cat_lr': 0.020121295439764153, 'cat_depth': 4, 'cat_l2': 4.2063824523453475}. Best is trial 0 with value: 0.5945721843250239.


     fold5    0.6578    0.7006    0.6461

     앙상블 OOF(수축전): LL=0.6354
     앙상블 OOF(수축후): LL=0.6290  수축 개선=+0.0064 ✅
     baseline=0.6859  gain=+0.0569 ✅
     최적 가중치: lgb=0.248  xgb=0.115  cat=0.637

최종 OOF 성능 요약 (최적화 앙상블):
  타겟        수축전 LL      수축후 LL      수축개선      AUC  최적 가중치
  ------------------------------------------------------------------
  Q1        0.6354      0.6323   +0.0031✅   0.6876  [lgb:0.37 xgb:0.00 cat:0.63] ✅
  Q2        0.6456      0.6425   +0.0031✅   0.6578  [lgb:1.00 xgb:0.00 cat:0.00] ✅
  Q3        0.6444      0.6404   +0.0039✅   0.6542  [lgb:0.82 xgb:0.05 cat:0.13] ✅
  S1        0.5665      0.5611   +0.0054✅   0.7169  [lgb:0.17 xgb:0.31 cat:0.52] ✅
  S2        0.5449      0.5423   +0.0026✅   0.7608  [lgb:0.09 xgb:0.42 cat:0.49] ✅
  S3        0.5186      0.5143   +0.0043✅   0.7828  [lgb:0.33 xgb:0.00 cat:0.67] ✅
  S4        0.6354      0.6290   +0.0064✅   0.6715  [lgb:0.25 xgb:0.12 cat:0.64] ✅

  ★ 평균 Log-Loss (수축후): 0.5946
  Subject-Hole CV: 5개 fold 생성

  [모델 

...

     fold5    0.6496    0.7728    0.6452

     앙상블 OOF(수축전): LL=0.6293
     앙상블 OOF(수축후): LL=0.6230  수축 개선=+0.0063 ✅
     baseline=0.6859  gain=+0.0630 ✅
     최적 가중치: lgb=0.014  xgb=0.065  cat=0.921

최종 OOF 성능 요약 (최적화 앙상블):
  타겟        수축전 LL      수축후 LL      수축개선      AUC  최적 가중치
  ------------------------------------------------------------------
  Q1        0.6303      0.6284   +0.0019✅   0.7000  [lgb:0.67 xgb:0.00 cat:0.33] ✅
  Q2        0.6537      0.6490   +0.0046✅   0.6448  [lgb:0.98 xgb:0.00 cat:0.02] ✅
  Q3        0.6509      0.6461   +0.0048✅   0.6342  [lgb:0.90 xgb:0.00 cat:0.10] ✅
  S1        0.5582      0.5545   +0.0037✅   0.7290  [lgb:0.58 xgb:0.01 cat:0.41] ✅
  S2        0.5447      0.5418   +0.0029✅   0.7641  [lgb:0.37 xgb:0.00 cat:0.63] ✅
  S3        0.5157      0.5119   +0.0038✅   0.7841  [lgb:0.55 xgb:0.00 cat:0.45] ✅
  S4        0.6293      0.6230   +0.0063✅   0.6819  [lgb:0.01 xgb:0.07 cat:0.92] ✅

  ★ 평균 Log-Loss (수축후): 0.5935
  Subject-Hole CV: 5개 fold 생성

  [모델 

[I 2026-06-23 13:09:12,249] Trial 18 finished with value: 0.5955615928182693 and parameters: {'lgb_lr': 0.026315597098646608, 'lgb_num_leaves': 59, 'lgb_min_child': 18, 'lgb_feat_frac': 0.638944296718377, 'lgb_bag_frac': 0.8031470510955746, 'xgb_lr': 0.017290120982988492, 'xgb_depth': 7, 'xgb_subsample': 0.7492347409018403, 'xgb_colsample': 0.9082147375718035, 'cat_lr': 0.09325571018521289, 'cat_depth': 7, 'cat_l2': 5.364008711058}. Best is trial 17 with value: 0.5935149767189909.


     fold5    0.6392    0.7686    0.6500

     앙상블 OOF(수축전): LL=0.6323
     앙상블 OOF(수축후): LL=0.6258  수축 개선=+0.0066 ✅
     baseline=0.6859  gain=+0.0601 ✅
     최적 가중치: lgb=0.579  xgb=0.091  cat=0.330

최종 OOF 성능 요약 (최적화 앙상블):
  타겟        수축전 LL      수축후 LL      수축개선      AUC  최적 가중치
  ------------------------------------------------------------------
  Q1        0.6324      0.6302   +0.0022✅   0.6879  [lgb:0.73 xgb:0.00 cat:0.27] ✅
  Q2        0.6552      0.6505   +0.0047✅   0.6424  [lgb:1.00 xgb:0.00 cat:0.00] ✅
  Q3        0.6528      0.6481   +0.0047✅   0.6348  [lgb:0.70 xgb:0.00 cat:0.30] ✅
  S1        0.5582      0.5546   +0.0036✅   0.7334  [lgb:0.46 xgb:0.00 cat:0.54] ✅
  S2        0.5473      0.5440   +0.0033✅   0.7633  [lgb:0.35 xgb:0.05 cat:0.60] ✅
  S3        0.5196      0.5158   +0.0038✅   0.7759  [lgb:0.64 xgb:0.00 cat:0.36] ✅
  S4        0.6323      0.6258   +0.0066✅   0.6801  [lgb:0.58 xgb:0.09 cat:0.33] ✅

  ★ 평균 Log-Loss (수축후): 0.5956
  Subject-Hole CV: 5개 fold 생성

  [모델 

[I 2026-06-23 13:10:06,540] Trial 19 finished with value: 0.5954683782929522 and parameters: {'lgb_lr': 0.042472571093302804, 'lgb_num_leaves': 44, 'lgb_min_child': 13, 'lgb_feat_frac': 0.6334752951649881, 'lgb_bag_frac': 0.6319729246516881, 'xgb_lr': 0.033453011655319505, 'xgb_depth': 5, 'xgb_subsample': 0.7114517348137032, 'xgb_colsample': 0.9993639041924698, 'cat_lr': 0.06569582339535082, 'cat_depth': 7, 'cat_l2': 8.563718258185837}. Best is trial 17 with value: 0.5935149767189909.


     fold5    0.6365    0.8331    0.6274

     앙상블 OOF(수축전): LL=0.6315
     앙상블 OOF(수축후): LL=0.6254  수축 개선=+0.0061 ✅
     baseline=0.6859  gain=+0.0605 ✅
     최적 가중치: lgb=0.444  xgb=0.098  cat=0.458

최종 OOF 성능 요약 (최적화 앙상블):
  타겟        수축전 LL      수축후 LL      수축개선      AUC  최적 가중치
  ------------------------------------------------------------------
  Q1        0.6332      0.6308   +0.0024✅   0.6916  [lgb:0.56 xgb:0.00 cat:0.44] ✅
  Q2        0.6505      0.6468   +0.0036✅   0.6376  [lgb:1.00 xgb:0.00 cat:0.00] ✅
  Q3        0.6520      0.6477   +0.0042✅   0.6363  [lgb:0.85 xgb:0.01 cat:0.14] ✅
  S1        0.5616      0.5576   +0.0040✅   0.7244  [lgb:0.56 xgb:0.00 cat:0.44] ✅
  S2        0.5467      0.5437   +0.0030✅   0.7626  [lgb:0.30 xgb:0.06 cat:0.64] ✅
  S3        0.5203      0.5162   +0.0041✅   0.7767  [lgb:0.55 xgb:0.02 cat:0.43] ✅
  S4        0.6315      0.6254   +0.0061✅   0.6767  [lgb:0.44 xgb:0.10 cat:0.46] ✅

  ★ 평균 Log-Loss (수축후): 0.5955
🏆 최적의 파라미터를 찾았습니다!
Best Log-Loss: 0.5

In [ ]:
# ==============================================================================
# ★ 컨트롤 타워 (여기서 원하는 단계만 True로 바꾸고 실행하세요!)
# ==============================================================================
if __name__ == "__main__":

    # 처음 돌릴 때는 모두 True.
    # 모델 에러가 나서 고치고 다시 돌릴 때는 PHASE_4만 True로 하시면 됩니다!

    RUN_PHASE_1 = False  # 기본 피처 병합
    RUN_PHASE_2 = False  # tsfresh 추출 (1시간 소요 구간)
    RUN_PHASE_3 = False  # 피처 선택 및 전처리 (NMI)
    RUN_PHASE_4 = True   # 모델 학습 및 결과물 산출 (디버깅 집중 구간)

    if RUN_PHASE_1: run_phase_1_base_features()
    if RUN_PHASE_2: run_phase_2_rolling_and_tsfresh()
    if RUN_PHASE_3: run_phase_3_zscore_and_selection()
    if RUN_PHASE_4: run_phase_4_modeling()


🚀 [Phase 4] 앙상블 모델 학습 및 제출 파일 생성 시작
  Subject-Hole CV: 5개 fold 생성

  [모델 학습] 시드 앙상블(3 Seeds) + 역수 비례 가중치 + 베이즈 수축
  적용 시드: [42, 2024, 9999]

  ── Q1  (51개 피처)
     fold1 (3-Seed)  LGB:0.5493  XGB:0.6151  CAT:0.6073
     fold2 (3-Seed)  LGB:0.6451  XGB:0.6478  CAT:0.6529
     fold3 (3-Seed)  LGB:0.6459  XGB:0.6541  CAT:0.6550
     fold4 (3-Seed)  LGB:0.6582  XGB:0.6575  CAT:0.6271
     fold5 (3-Seed)  LGB:0.6546  XGB:0.6721  CAT:0.6580
     ➡️ 최종 앙상블 OOF(수축후): LL=0.6331  (AUC=0.6772)
        가중치: lgb=0.339  xgb=0.328  cat=0.333

  ── Q2  (51개 피처)
     fold1 (3-Seed)  LGB:0.6844  XGB:0.6765  CAT:0.6788
     fold2 (3-Seed)  LGB:0.6406  XGB:0.6473  CAT:0.6590
     fold3 (3-Seed)  LGB:0.6609  XGB:0.6692  CAT:0.6685
     fold4 (3-Seed)  LGB:0.6129  XGB:0.6448  CAT:0.6354
     fold5 (3-Seed)  LGB:0.6720  XGB:0.6684  CAT:0.6919
     ➡️ 최종 앙상블 OOF(수축후): LL=0.6527  (AUC=0.6413)
        가중치: lgb=0.337  xgb=0.333  cat=0.330

  ── Q3  (51개 피처)
     fold1 (3-Seed)  LGB:0.6976  XGB:0.7046  CAT:0.677